# Colab Session: opportunity-10m
Generated from colab-cli history log.

**Session Created**: 2026-10-08 10:27:15
- Endpoint: `gpu-t4-s-kkb-usw4b0-23ceoq23r05gr`
- Hardware: `T4`

In [ ]:
# CELL: ENV
# Colab ships torch/pandas/numpy; install research extras.
%pip install -q --upgrade-strategy only-if-needed pyarrow onnx onnxruntime requests scikit-learn

import json
import math
import random
import time
from datetime import datetime, timedelta, timezone
from pathlib import Path
from typing import Any, Dict, List, Optional, Sequence, Tuple

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset

print(f"PyTorch: {torch.__version__}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("WARNING: No GPU detected — training will run on CPU and be much slower.")
    print("Runtime -> Change runtime type -> select a GPU (T4), then re-run this cell.")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 86.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.0/23.6 MB ? eta -:--:--

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╸ 23.6/23.6 MB 185.3 MB/s eta 0:00:01

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.6/23.6 MB 68.8 MB/s eta 0:00:00


PyTorch: 2.11.0+cu130
GPU: Tesla T4


In [ ]:
# CELL: CONFIG
CONFIG: Dict[str, Any] = {
    "symbol": "BTCUSD",
    "base_url": "https://api.india.delta.exchange",
    "history_days": 900,
    "resolution_5m": "5m",
    "seed": 42,
    "tp_atr_mult": 1.5,
    "sl_atr_mult": 1.0,
    "horizon_bars": 3,
    "atr_period": 14,
    "window_len": 60,
    "round_trip_cost_pct": 0.0016,
    "minority_rate": 0.15,
    "ambiguous_win_max": 0.50,
    "train_fraction": 0.70,
    "val_fraction": 0.15,
    "purge_bars": 3,
    "d_model": 64,
    "nhead": 4,
    "num_layers": 2,
    "dropout": 0.10,
    "ff_mult": 4,
    "lr": 1e-3,
    "weight_decay": 1e-4,
    "batch_size": 64,
    "epochs": 40,
    "patience": 6,
    "cls_loss_weight": 1.0,
    "reg_loss_weight": 0.35,
    "export_dir": "export/opportunity_10m",
    "ready_to_promote": False,
}

FEATURE_COLS: Tuple[str, ...] = (
    "ret_1",
    "atr_pct",
    "rsi",
    "ema_dist_atr",
    "candle_body",
    "volume_z",
)
LABEL_NAMES: Tuple[str, ...] = ("LONG", "SHORT", "NO_TRADE")
LABEL_LONG, LABEL_SHORT, LABEL_NO_TRADE = 0, 1, 2
print(json.dumps(CONFIG, indent=2))


{
  "symbol": "BTCUSD",
  "base_url": "https://api.india.delta.exchange",
  "history_days": 900,
  "resolution_5m": "5m",
  "seed": 42,
  "tp_atr_mult": 1.5,
  "sl_atr_mult": 1.0,
  "horizon_bars": 3,
  "atr_period": 14,
  "window_len": 60,
  "round_trip_cost_pct": 0.0016,
  "minority_rate": 0.15,
  "ambiguous_win_max": 0.5,
  "train_fraction": 0.7,
  "val_fraction": 0.15,
  "purge_bars": 3,
  "d_model": 64,
  "nhead": 4,
  "num_layers": 2,
  "dropout": 0.1,
  "ff_mult": 4,
  "lr": 0.001,
  "weight_decay": 0.0001,
  "batch_size": 64,
  "epochs": 40,
  "patience": 6,
  "cls_loss_weight": 1.0,
  "reg_loss_weight": 0.35,
  "export_dir": "export/opportunity_10m",
  "ready_to_promote": false
}


In [ ]:
# CELL: SEED
def set_seed(seed: int) -> None:
    random.seed(int(seed))
    np.random.seed(int(seed))
    torch.manual_seed(int(seed))
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(int(seed))


set_seed(int(CONFIG["seed"]))
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device", DEVICE)


device cuda


In [ ]:
# CELL: FETCH
DELTA_CANDLE_PAGE_BARS = 2000
REQUEST_DELAY_SECONDS = 0.2
EMPTY_PAGE_RETRIES = 3
MIN_OHLCV_COMPLETENESS = 0.95
_RESOLUTION_MINUTES = {"5m": 5, "10m": 10}


def _bar_seconds(resolution: str) -> int:
    if resolution not in _RESOLUTION_MINUTES:
        raise ValueError(f"Unsupported resolution {resolution!r}")
    return int(_RESOLUTION_MINUTES[resolution]) * 60


def _parse_candle_rows(payload: Any) -> List[Dict[str, Any]]:
    if not isinstance(payload, dict):
        return []
    result = payload.get("result", [])
    if isinstance(result, dict):
        result = result.get("candles", []) or []
    if not isinstance(result, list):
        return []
    return [row for row in result if isinstance(row, dict) and "time" in row]


def _candle_epoch(row: Dict[str, Any]) -> Optional[int]:
    try:
        return int(row["time"])
    except (KeyError, TypeError, ValueError):
        return None


def _request_candle_page(
    url: str,
    *,
    symbol: str,
    resolution: str,
    start_ts: int,
    end_ts: int,
) -> List[Dict[str, Any]]:
    import requests

    last_error: Optional[Exception] = None
    for attempt in range(EMPTY_PAGE_RETRIES):
        try:
            response = requests.get(
                url,
                params={
                    "symbol": symbol,
                    "resolution": resolution,
                    "start": int(start_ts),
                    "end": int(end_ts),
                },
                timeout=30,
            )
            response.raise_for_status()
            payload = response.json()
        except Exception as exc:
            last_error = exc
            time.sleep(REQUEST_DELAY_SECONDS * (attempt + 1))
            continue
        if payload.get("success") is False:
            last_error = RuntimeError(f"Delta candles error for {symbol}: {payload}")
            time.sleep(REQUEST_DELAY_SECONDS * (attempt + 1))
            continue
        rows = _parse_candle_rows(payload)
        if rows:
            return rows
        last_error = None
        time.sleep(REQUEST_DELAY_SECONDS * (attempt + 1))
    if last_error is not None:
        raise last_error
    return []


def _rows_to_frame(all_rows: List[Dict[str, Any]]) -> pd.DataFrame:
    df = pd.DataFrame(all_rows)
    expected_cols = ["time", "open", "high", "low", "close", "volume"]
    df = df[[c for c in expected_cols if c in df.columns]]
    df["time"] = pd.to_datetime(df["time"], unit="s", utc=True)
    df = df.drop_duplicates(subset="time").sort_values("time").reset_index(drop=True)
    for col in ["open", "high", "low", "close", "volume"]:
        if col in df.columns:
            df[col] = df[col].astype(float)
    return df


def fetch_candles(
    symbol: str,
    resolution: str,
    start_ts: int,
    end_ts: int,
    base_url: str,
    *,
    page_bars: int = DELTA_CANDLE_PAGE_BARS,
) -> pd.DataFrame:
    url = f"{base_url.rstrip('/')}/v2/history/candles"
    bar_seconds = _bar_seconds(resolution)
    page_bars = max(1, int(page_bars))
    start_ts, end_ts = int(start_ts), int(end_ts)
    if end_ts <= start_ts:
        raise ValueError("end_ts must be greater than start_ts")
    all_rows: List[Dict[str, Any]] = []
    cursor_end = end_ts
    seen_oldest: Optional[int] = None
    max_pages = max(2, (end_ts - start_ts) // bar_seconds + 5)
    for _ in range(max_pages):
        if cursor_end <= start_ts:
            break
        page_start = max(start_ts, cursor_end - page_bars * bar_seconds)
        if page_start >= cursor_end:
            break
        rows = _request_candle_page(
            url,
            symbol=symbol,
            resolution=resolution,
            start_ts=page_start,
            end_ts=cursor_end,
        )
        if not rows:
            break
        epochs = [e for e in (_candle_epoch(r) for r in rows) if e is not None]
        if not epochs:
            break
        oldest = min(epochs)
        if seen_oldest is not None and oldest >= seen_oldest:
            break
        seen_oldest = oldest
        all_rows.extend(rows)
        cursor_end = oldest - 1
        time.sleep(REQUEST_DELAY_SECONDS)
    if not all_rows:
        raise RuntimeError(f"No candle data returned for symbol={symbol}")
    return _rows_to_frame(all_rows)


def validate_ohlcv_completeness(
    df: pd.DataFrame,
    resolution: str,
    *,
    min_completeness: float = MIN_OHLCV_COMPLETENESS,
    symbol: str = "",
) -> Dict[str, Any]:
    if df is None or df.empty or "time" not in df.columns:
        raise ValueError("Cannot validate completeness on an empty OHLCV frame")
    bar_seconds = _bar_seconds(resolution)
    time_col = df["time"]
    epochs = (pd.to_datetime(time_col, utc=True).astype("int64") // 10**9).tolist()
    if len(epochs) < 2:
        raise ValueError("Need at least 2 OHLCV bars to validate completeness")
    gaps = 0
    for prev, cur in zip(epochs, epochs[1:]):
        if int(cur) - int(prev) > bar_seconds * 1.5:
            gaps += 1
    completeness = 1.0 - (gaps / max(1, len(epochs) - 1))
    report = {
        "rows": len(epochs),
        "gaps": int(gaps),
        "completeness": round(float(completeness), 4),
    }
    if completeness < min_completeness:
        label = f"{symbol} {resolution}".strip()
        raise ValueError(
            f"OHLCV completeness {completeness:.1%} below {min_completeness:.0%} "
            f"for {label or 'candles'}"
        )
    return report


end_dt = datetime.now(timezone.utc)
start_dt = end_dt - timedelta(days=int(CONFIG["history_days"]))
print(
    f"Fetching {CONFIG['symbol']} 5m OHLCV for {CONFIG['history_days']}d "
    f"from {CONFIG['base_url']}..."
)
df5m = fetch_candles(
    str(CONFIG["symbol"]),
    str(CONFIG["resolution_5m"]),
    int(start_dt.timestamp()),
    int(end_dt.timestamp()),
    str(CONFIG["base_url"]),
)
ohlcv_report = validate_ohlcv_completeness(
    df5m, "5m", symbol=str(CONFIG["symbol"])
)
print(
    f"OHLCV {CONFIG['symbol']} 5m: {ohlcv_report['rows']} bars, "
    f"completeness {ohlcv_report['completeness']:.1%}, gaps={ohlcv_report['gaps']}"
)


Fetching BTCUSD 5m OHLCV for 900d from https://api.india.delta.exchange...


OHLCV BTCUSD 5m: 259200 bars, completeness 100.0%, gaps=0


In [ ]:
# CELL: BARS_10M
def build_10m_ohlcv_from_5m(df5m: pd.DataFrame) -> pd.DataFrame:
    """Exchange-aligned 10m bars from two closed 5m candles. Drop a partial last bucket."""
    src = df5m.copy()
    src["time"] = pd.to_datetime(src["time"], utc=True)
    required = ("open", "high", "low", "close", "volume")
    missing = [c for c in required if c not in src.columns]
    if missing:
        raise ValueError(f"5m OHLCV missing columns: {missing}")
    if src.empty:
        return src.iloc[0:0].copy()
    grouped = src.set_index("time").sort_index()
    agg = grouped.resample("10min", label="left", closed="left").agg(
        {
            "open": "first",
            "high": "max",
            "low": "min",
            "close": "last",
            "volume": "sum",
        }
    )
    counts = grouped["close"].resample("10min", label="left", closed="left").count()
    complete = counts >= 2
    agg = agg.loc[complete].dropna(subset=["open", "high", "low", "close"])
    out = agg.reset_index()
    return out.sort_values("time").reset_index(drop=True)


bars_10m = build_10m_ohlcv_from_5m(df5m)
print(f"10m bars={len(bars_10m)} span={bars_10m['time'].iloc[0]} → {bars_10m['time'].iloc[-1]}")


10m bars=129600 span=2024-04-21 10:30:00+00:00 → 2026-10-08 10:20:00+00:00


In [ ]:
# CELL: FEATURES
def _wilder_rsi(close: pd.Series, period: int = 14) -> pd.Series:
    delta = close.diff()
    gain = delta.clip(lower=0.0)
    loss = -delta.clip(upper=0.0)
    avg_gain = gain.ewm(alpha=1.0 / period, min_periods=period, adjust=False).mean()
    avg_loss = loss.ewm(alpha=1.0 / period, min_periods=period, adjust=False).mean()
    rs = avg_gain / avg_loss.replace(0.0, np.nan)
    return 100.0 - (100.0 / (1.0 + rs))


def add_opportunity_features(
    df: pd.DataFrame,
    *,
    atr_period: int = 14,
    rsi_period: int = 14,
    ema_span: int = 20,
    vol_window: int = 20,
) -> pd.DataFrame:
    """Causal 10m features. ATR(14) is also stored for the label path."""
    out = df.copy()
    close = out["close"].astype(np.float64)
    high = out["high"].astype(np.float64)
    low = out["low"].astype(np.float64)
    open_ = out["open"].astype(np.float64)
    volume = out["volume"].astype(np.float64)
    prev_close = close.shift(1)
    tr = pd.concat(
        [
            high - low,
            (high - prev_close).abs(),
            (low - prev_close).abs(),
        ],
        axis=1,
    ).max(axis=1)
    atr = tr.rolling(int(atr_period), min_periods=int(atr_period)).mean()
    out["atr"] = atr
    out["ret_1"] = close.pct_change()
    out["atr_pct"] = atr / close.replace(0.0, np.nan)
    out["rsi"] = _wilder_rsi(close, period=int(rsi_period))
    ema = close.ewm(span=int(ema_span), adjust=False, min_periods=int(ema_span)).mean()
    out["ema_dist_atr"] = (close - ema) / atr.replace(0.0, np.nan)
    out["candle_body"] = (close - open_) / atr.replace(0.0, np.nan)
    vol_mean = volume.rolling(int(vol_window), min_periods=int(vol_window)).mean()
    vol_std = volume.rolling(int(vol_window), min_periods=int(vol_window)).std()
    out["volume_z"] = (volume - vol_mean) / vol_std.replace(0.0, np.nan)
    return out


featured = add_opportunity_features(
    bars_10m, atr_period=int(CONFIG["atr_period"])
)
print("feature null rates")
print(featured[list(FEATURE_COLS)].isna().mean().round(4))


feature null rates
ret_1           0.0000
atr_pct         0.0001
rsi             0.0001
ema_dist_atr    0.0001
candle_body     0.0001
volume_z        0.0001
dtype: float64


In [ ]:
# CELL: LABELS
# Opportunity labels on a 10-minute series. Path is bars t+1 .. t+horizon only.

import numpy as np
import pandas as pd

_EPS = 1e-9
HORIZON_BARS_DEFAULT = 3
ATR_PERIOD_DEFAULT = 14
TP_ATR_MULT_DEFAULT = 1.5
SL_ATR_MULT_DEFAULT = 1.0

TP_FIRST = 0
SL_FIRST = 1
NEITHER = 2
AMBIGUOUS = 3

LABEL_LONG = 0
LABEL_SHORT = 1
LABEL_NO_TRADE = 2
LABEL_UNLABELED = -1
LABEL_NAMES = ("LONG", "SHORT", "NO_TRADE")


def tp_sl_from_path(
    fwd_high: np.ndarray,
    fwd_low: np.ndarray,
    entry: float,
    atr: float,
    tp_mult: float,
    sl_mult: float,
    side: str,
) -> int:
    """Walk future highs/lows. Same-bar TP and SL is AMBIGUOUS, not a win."""
    high = np.asarray(fwd_high, dtype=np.float64).ravel()
    low = np.asarray(fwd_low, dtype=np.float64).ravel()
    if high.size == 0 or high.size != low.size:
        return int(NEITHER)
    denom = max(float(atr), _EPS)
    entry_f = float(entry)
    if str(side).strip().upper() == "SHORT":
        tp_level = entry_f - float(tp_mult) * denom
        sl_level = entry_f + float(sl_mult) * denom
        tp_hit = low <= tp_level
        sl_hit = high >= sl_level
    else:
        tp_level = entry_f + float(tp_mult) * denom
        sl_level = entry_f - float(sl_mult) * denom
        tp_hit = high >= tp_level
        sl_hit = low <= sl_level
    for j in range(high.size):
        hit_tp = bool(tp_hit[j])
        hit_sl = bool(sl_hit[j])
        if hit_tp and hit_sl:
            return int(AMBIGUOUS)
        if hit_tp:
            return int(TP_FIRST)
        if hit_sl:
            return int(SL_FIRST)
    return int(NEITHER)


def _side_win(code: int) -> int:
    return 1 if int(code) == int(TP_FIRST) else 0


def _causal_atr(df: pd.DataFrame, period: int = ATR_PERIOD_DEFAULT) -> np.ndarray:
    if "atr" in df.columns:
        atr = df["atr"].to_numpy(dtype=np.float64)
        if np.isfinite(atr).any():
            return atr
    close = df["close"].to_numpy(dtype=np.float64)
    high = df["high"].to_numpy(dtype=np.float64)
    low = df["low"].to_numpy(dtype=np.float64)
    prev = np.concatenate([[close[0]], close[:-1]])
    tr = np.maximum(high - low, np.maximum(np.abs(high - prev), np.abs(low - prev)))
    return pd.Series(tr).rolling(int(period), min_periods=int(period)).mean().to_numpy(
        dtype=np.float64
    )


def compute_opportunity_labels(
    df: pd.DataFrame,
    *,
    tp_mult: float = TP_ATR_MULT_DEFAULT,
    sl_mult: float = SL_ATR_MULT_DEFAULT,
    horizon_bars: int = HORIZON_BARS_DEFAULT,
    atr_period: int = ATR_PERIOD_DEFAULT,
) -> pd.DataFrame:
    """Label each closed 10m bar. Last ``horizon_bars`` rows stay unlabeled."""
    out = df.copy().reset_index(drop=True)
    n = len(out)
    k = int(horizon_bars)
    close = out["close"].to_numpy(dtype=np.float64)
    high = out["high"].to_numpy(dtype=np.float64)
    low = out["low"].to_numpy(dtype=np.float64)
    atr = _causal_atr(out, period=int(atr_period))
    n_valid = max(n - k, 0)

    trade_label = np.full(n, LABEL_UNLABELED, dtype=np.int64)
    long_result = np.full(n, np.nan, dtype=np.float64)
    short_result = np.full(n, np.nan, dtype=np.float64)
    long_code = np.full(n, np.nan, dtype=np.float64)
    short_code = np.full(n, np.nan, dtype=np.float64)
    upside = np.full(n, np.nan, dtype=np.float64)
    downside = np.full(n, np.nan, dtype=np.float64)
    timeout_ret = np.full(n, np.nan, dtype=np.float64)
    long_pnl_pct = np.full(n, np.nan, dtype=np.float64)
    short_pnl_pct = np.full(n, np.nan, dtype=np.float64)

    for i in range(n_valid):
        atr_i = float(atr[i]) if i < len(atr) else float("nan")
        if not np.isfinite(atr_i) or atr_i <= _EPS:
            continue
        entry = float(close[i])
        if not np.isfinite(entry) or abs(entry) <= _EPS:
            continue
        fwd_high = high[i + 1 : i + 1 + k]
        fwd_low = low[i + 1 : i + 1 + k]
        fwd_close = close[i + 1 : i + 1 + k]
        if fwd_high.size < k:
            continue
        l_code = tp_sl_from_path(fwd_high, fwd_low, entry, atr_i, tp_mult, sl_mult, "LONG")
        s_code = tp_sl_from_path(fwd_high, fwd_low, entry, atr_i, tp_mult, sl_mult, "SHORT")
        l_win = _side_win(l_code)
        s_win = _side_win(s_code)
        if l_win == 1 and s_win == 0:
            label = LABEL_LONG
        elif s_win == 1 and l_win == 0:
            label = LABEL_SHORT
        else:
            label = LABEL_NO_TRADE
        trade_label[i] = label
        long_result[i] = float(l_win)
        short_result[i] = float(s_win)
        long_code[i] = float(l_code)
        short_code[i] = float(s_code)
        denom = max(atr_i, _EPS)
        upside[i] = max((float(np.max(fwd_high)) - entry) / denom, 0.0)
        downside[i] = max((entry - float(np.min(fwd_low))) / denom, 0.0)
        end_close = float(fwd_close[-1])
        timeout_ret[i] = (end_close - entry) / entry
        tp_pct = float(tp_mult) * denom / entry
        sl_pct = float(sl_mult) * denom / entry
        if l_code == TP_FIRST:
            long_pnl_pct[i] = tp_pct
        elif l_code == SL_FIRST:
            long_pnl_pct[i] = -sl_pct
        elif l_code == NEITHER:
            long_pnl_pct[i] = timeout_ret[i]
        if s_code == TP_FIRST:
            short_pnl_pct[i] = tp_pct
        elif s_code == SL_FIRST:
            short_pnl_pct[i] = -sl_pct
        elif s_code == NEITHER:
            short_pnl_pct[i] = -timeout_ret[i]

    out["atr"] = atr
    out["trade_label"] = trade_label
    out["long_result"] = long_result
    out["short_result"] = short_result
    out["long_code"] = long_code
    out["short_code"] = short_code
    out["upside"] = upside
    out["downside"] = downside
    out["timeout_ret"] = timeout_ret
    out["long_pnl_pct"] = long_pnl_pct
    out["short_pnl_pct"] = short_pnl_pct
    names = np.array(["UNLABELED"] * n, dtype=object)
    names[trade_label == LABEL_LONG] = "LONG"
    names[trade_label == LABEL_SHORT] = "SHORT"
    names[trade_label == LABEL_NO_TRADE] = "NO_TRADE"
    out["trade_name"] = names
    return out


In [ ]:
# CELL: APPLY_LABELS
labeled = compute_opportunity_labels(
    featured,
    tp_mult=float(CONFIG["tp_atr_mult"]),
    sl_mult=float(CONFIG["sl_atr_mult"]),
    horizon_bars=int(CONFIG["horizon_bars"]),
    atr_period=int(CONFIG["atr_period"]),
)
valid = labeled["trade_label"].to_numpy() >= 0
print(f"labeled rows={int(valid.sum())} / {len(labeled)}")
print(labeled.loc[valid, "trade_name"].value_counts(normalize=True).round(4))


labeled rows=129584 / 129600
trade_name
NO_TRADE    0.6529
SHORT       0.1757
LONG        0.1715
Name: proportion, dtype: float64


In [ ]:
# CELL: GATE
def train_period_mask(n: int, train_fraction: float) -> np.ndarray:
    cut = int(n * float(train_fraction))
    mask = np.zeros(n, dtype=bool)
    mask[:cut] = True
    return mask


def summarize_label_gate(
    df: pd.DataFrame,
    *,
    train_fraction: float,
    round_trip_cost_pct: float,
    minority_rate: float,
    ambiguous_win_max: float,
) -> Dict[str, Any]:
    n = len(df)
    in_train = train_period_mask(n, train_fraction)
    labels = df["trade_label"].to_numpy(dtype=np.int64)
    usable = in_train & (labels >= 0)
    n_use = int(usable.sum())
    if n_use == 0:
        raise RuntimeError("Train-period gate has no labeled rows")
    mix = {}
    mix_ok = True
    for cid, name in enumerate(LABEL_NAMES):
        rate = float(np.mean(labels[usable] == cid))
        mix[name] = rate
        if rate < float(minority_rate):
            mix_ok = False
    long_code = df["long_code"].to_numpy(dtype=np.float64)
    short_code = df["short_code"].to_numpy(dtype=np.float64)
    long_pnl = df["long_pnl_pct"].to_numpy(dtype=np.float64) - float(round_trip_cost_pct)
    short_pnl = df["short_pnl_pct"].to_numpy(dtype=np.float64) - float(round_trip_cost_pct)
    long_exp_mask = usable & np.isfinite(long_pnl)
    short_exp_mask = usable & np.isfinite(short_pnl)
    long_expectancy = float(np.mean(long_pnl[long_exp_mask])) if long_exp_mask.any() else float("nan")
    short_expectancy = (
        float(np.mean(short_pnl[short_exp_mask])) if short_exp_mask.any() else float("nan")
    )

    def _ambig_share(codes: np.ndarray) -> float:
        finite = usable & np.isfinite(codes)
        tp = finite & (codes == TP_FIRST)
        amb = finite & (codes == AMBIGUOUS)
        denom = int(tp.sum() + amb.sum())
        if denom == 0:
            return 0.0
        return float(amb.sum()) / float(denom)

    long_ambig = _ambig_share(long_code)
    short_ambig = _ambig_share(short_code)
    reasons: List[str] = []
    if not mix_ok:
        reasons.append("class_mix_below_floor")
    if not (np.isfinite(long_expectancy) and long_expectancy > 0.0):
        reasons.append("long_expectancy_not_positive")
    if not (np.isfinite(short_expectancy) and short_expectancy > 0.0):
        reasons.append("short_expectancy_not_positive")
    if long_ambig >= float(ambiguous_win_max):
        reasons.append("long_ambiguous_majority")
    if short_ambig >= float(ambiguous_win_max):
        reasons.append("short_ambiguous_majority")
    passed = len(reasons) == 0
    return {
        "passed": passed,
        "n_train_labeled": n_use,
        "class_mix": mix,
        "minority_rate": float(minority_rate),
        "long_expectancy_pct": long_expectancy,
        "short_expectancy_pct": short_expectancy,
        "round_trip_cost_pct": float(round_trip_cost_pct),
        "long_ambiguous_share": long_ambig,
        "short_ambiguous_share": short_ambig,
        "tp_atr_mult": float(CONFIG["tp_atr_mult"]),
        "sl_atr_mult": float(CONFIG["sl_atr_mult"]),
        "reasons": reasons,
    }


EXPORT_DIR = Path(str(CONFIG["export_dir"]))
EXPORT_DIR.mkdir(parents=True, exist_ok=True)
label_report = summarize_label_gate(
    labeled,
    train_fraction=float(CONFIG["train_fraction"]),
    round_trip_cost_pct=float(CONFIG["round_trip_cost_pct"]),
    minority_rate=float(CONFIG["minority_rate"]),
    ambiguous_win_max=float(CONFIG["ambiguous_win_max"]),
)
report_path = EXPORT_DIR / "label_report.json"
report_path.write_text(json.dumps(label_report, indent=2), encoding="utf-8")
print(json.dumps(label_report, indent=2))
print("wrote", report_path)
if not label_report["passed"]:
    raise RuntimeError(
        "Train-period label gate failed: "
        f"{label_report['reasons']}. Change CONFIG tp/sl ATR multiples and re-run "
        "from the top. Do not fit the network."
    )
print("Gate passed. Training may start.")


{
  "passed": false,
  "n_train_labeled": 90707,
  "class_mix": {
    "LONG": 0.17130982173371403,
    "SHORT": 0.17500303173955703,
    "NO_TRADE": 0.6536871465267289
  },
  "minority_rate": 0.15,
  "long_expectancy_pct": -0.001595113182739019,
  "short_expectancy_pct": -0.001595480227763668,
  "round_trip_cost_pct": 0.0016,
  "long_ambiguous_share": 0.017389654736309598,
  "short_ambiguous_share": 0.018123337663140966,
  "tp_atr_mult": 1.5,
  "sl_atr_mult": 1.0,
  "reasons": [
    "long_expectancy_not_positive",
    "short_expectancy_not_positive"
  ]
}
wrote export/opportunity_10m/label_report.json


RuntimeError: Train-period label gate failed: ['long_expectancy_not_positive', 'short_expectancy_not_positive']. Change CONFIG tp/sl ATR multiples and re-run from the top. Do not fit the network.

In [ ]:
# CELL: SPLIT
def purged_index_splits(
    n: int,
    *,
    train_fraction: float,
    val_fraction: float,
    purge_bars: int,
) -> Dict[str, np.ndarray]:
    """Time-ordered split. Purge ``purge_bars`` on each side of every cut."""
    n = int(n)
    purge = int(purge_bars)
    train_end = int(n * float(train_fraction))
    val_len = int(n * float(val_fraction))
    train_idx = np.arange(0, max(train_end - purge, 0), dtype=np.int64)
    val_start = min(train_end + purge, n)
    val_end = min(val_start + val_len, n)
    val_idx = np.arange(val_start, max(val_end - purge, val_start), dtype=np.int64)
    test_start = min(val_end + purge, n)
    test_idx = np.arange(test_start, n, dtype=np.int64)
    return {"train": train_idx, "val": val_idx, "test": test_idx}


def development_walk_forward(
    n_dev: int,
    *,
    folds: int = 4,
    purge_bars: int,
) -> List[Tuple[np.ndarray, np.ndarray]]:
    """Expanding-window folds inside the development prefix only."""
    folds = max(2, int(folds))
    purge = int(purge_bars)
    edges = [int(n_dev * (i + 1) / folds) for i in range(folds)]
    out: List[Tuple[np.ndarray, np.ndarray]] = []
    for i in range(1, folds):
        train_end = edges[i - 1]
        val_end = edges[i]
        tr = np.arange(0, max(train_end - purge, 0), dtype=np.int64)
        va = np.arange(min(train_end + purge, n_dev), val_end, dtype=np.int64)
        if len(tr) == 0 or len(va) == 0:
            continue
        out.append((tr, va))
    return out


splits = purged_index_splits(
    len(labeled),
    train_fraction=float(CONFIG["train_fraction"]),
    val_fraction=float(CONFIG["val_fraction"]),
    purge_bars=int(CONFIG["purge_bars"]),
)
for key, idx in splits.items():
    print(key, "n", len(idx), "range", int(idx[0]) if len(idx) else None, int(idx[-1]) if len(idx) else None)
dev_end = int(splits["test"][0]) if len(splits["test"]) else len(labeled)
wf_folds = development_walk_forward(
    dev_end, folds=4, purge_bars=int(CONFIG["purge_bars"])
)
print("walk-forward folds", len(wf_folds), "dev_end", dev_end)


train n 90717 range 0 90716
val n 19437 range 90723 110159
test n 19434 range 110166 129599
walk-forward folds 3 dev_end 110166


In [ ]:
# CELL: DATASET
class SequenceDataset(Dataset):
    def __init__(
        self,
        windows: np.ndarray,
        y_cls: np.ndarray,
        y_reg: np.ndarray,
    ) -> None:
        self.windows = np.asarray(windows, dtype=np.float32)
        self.y_cls = np.asarray(y_cls, dtype=np.int64)
        self.y_reg = np.asarray(y_reg, dtype=np.float32)

    def __len__(self) -> int:
        return int(self.windows.shape[0])

    def __getitem__(self, idx: int):
        return (
            torch.from_numpy(self.windows[idx]),
            torch.tensor(self.y_cls[idx], dtype=torch.long),
            torch.from_numpy(self.y_reg[idx]),
        )


def build_windows(
    frame: pd.DataFrame,
    *,
    window_len: int,
    feature_cols: Sequence[str],
) -> Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    feats = frame.loc[:, list(feature_cols)].to_numpy(dtype=np.float64)
    y_cls = frame["trade_label"].to_numpy(dtype=np.int64)
    y_reg = frame.loc[:, ["upside", "downside"]].to_numpy(dtype=np.float64)
    w = int(window_len)
    n = len(frame)
    xs: List[np.ndarray] = []
    cs: List[int] = []
    rs: List[np.ndarray] = []
    ends: List[int] = []
    for end in range(w - 1, n):
        if y_cls[end] < 0:
            continue
        if not np.isfinite(y_reg[end]).all():
            continue
        window = feats[end - w + 1 : end + 1]
        if not np.isfinite(window).all():
            continue
        xs.append(window)
        cs.append(int(y_cls[end]))
        rs.append(y_reg[end])
        ends.append(int(end))
    if not xs:
        empty_x = np.zeros((0, w, len(feature_cols)), dtype=np.float32)
        return empty_x, np.zeros((0,), dtype=np.int64), np.zeros((0, 2), dtype=np.float32), np.zeros((0,), dtype=np.int64)
    return (
        np.stack(xs, axis=0).astype(np.float32),
        np.asarray(cs, dtype=np.int64),
        np.stack(rs, axis=0).astype(np.float32),
        np.asarray(ends, dtype=np.int64),
    )


all_x, all_y, all_r, all_ends = build_windows(
    labeled, window_len=int(CONFIG["window_len"]), feature_cols=FEATURE_COLS
)
print("windows", all_x.shape, "class mix", {n: float(np.mean(all_y == i)) for i, n in enumerate(LABEL_NAMES)})


def _split_windows(split_idx: np.ndarray) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    allowed = set(int(i) for i in split_idx.tolist())
    keep = np.array([int(e) in allowed for e in all_ends], dtype=bool)
    return all_x[keep], all_y[keep], all_r[keep]


x_train, y_train, r_train = _split_windows(splits["train"])
x_val, y_val, r_val = _split_windows(splits["val"])
x_test, y_test, r_test = _split_windows(splits["test"])
print("train", x_train.shape, "val", x_val.shape, "test", x_test.shape)

feat_mean = x_train.mean(axis=(0, 1), keepdims=True)
feat_std = x_train.std(axis=(0, 1), keepdims=True)
feat_std = np.where(feat_std < 1e-6, 1.0, feat_std)
reg_mean = r_train.mean(axis=0, keepdims=True)
reg_std = r_train.std(axis=0, keepdims=True)
reg_std = np.where(reg_std < 1e-6, 1.0, reg_std)


def _scale_x(x: np.ndarray) -> np.ndarray:
    return ((x - feat_mean) / feat_std).astype(np.float32)


def _scale_r(r: np.ndarray) -> np.ndarray:
    return ((r - reg_mean) / reg_std).astype(np.float32)


train_ds = SequenceDataset(_scale_x(x_train), y_train, _scale_r(r_train))
val_ds = SequenceDataset(_scale_x(x_val), y_val, _scale_r(r_val))
test_ds = SequenceDataset(_scale_x(x_test), y_test, _scale_r(r_test))
train_loader = DataLoader(train_ds, batch_size=int(CONFIG["batch_size"]), shuffle=True, drop_last=False)
val_loader = DataLoader(val_ds, batch_size=int(CONFIG["batch_size"]), shuffle=False)
test_loader = DataLoader(test_ds, batch_size=int(CONFIG["batch_size"]), shuffle=False)


windows (129519, 60, 6) class mix {'LONG': 0.17149607393509833, 'SHORT': 0.17563446289733553, 'NO_TRADE': 0.6528694631675661}


train (90639, 60, 6) val (19437, 60, 6) test (19431, 60, 6)


In [ ]:
# CELL: MODEL
class OpportunityEncoder(nn.Module):
    """Single 10m encoder: 3-class opportunity plus upside/downside."""

    def __init__(
        self,
        n_features: int,
        *,
        d_model: int = 64,
        nhead: int = 4,
        num_layers: int = 2,
        dropout: float = 0.1,
        ff_mult: int = 4,
        n_classes: int = 3,
    ) -> None:
        super().__init__()
        self.input_proj = nn.Linear(int(n_features), int(d_model))
        layer = nn.TransformerEncoderLayer(
            d_model=int(d_model),
            nhead=int(nhead),
            dim_feedforward=int(d_model) * int(ff_mult),
            dropout=float(dropout),
            batch_first=True,
            activation="gelu",
        )
        self.encoder = nn.TransformerEncoder(layer, num_layers=int(num_layers))
        self.norm = nn.LayerNorm(int(d_model))
        self.cls_head = nn.Linear(int(d_model), int(n_classes))
        self.upside_head = nn.Linear(int(d_model), 1)
        self.downside_head = nn.Linear(int(d_model), 1)

    def forward(self, x: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        h = self.norm(self.encoder(self.input_proj(x)))
        pooled = h[:, -1, :]
        logits = self.cls_head(pooled)
        upside = self.upside_head(pooled).squeeze(-1)
        downside = self.downside_head(pooled).squeeze(-1)
        return logits, upside, downside


model = OpportunityEncoder(
    len(FEATURE_COLS),
    d_model=int(CONFIG["d_model"]),
    nhead=int(CONFIG["nhead"]),
    num_layers=int(CONFIG["num_layers"]),
    dropout=float(CONFIG["dropout"]),
    ff_mult=int(CONFIG["ff_mult"]),
).to(DEVICE)
print(model)
print("params", sum(p.numel() for p in model.parameters()))


OpportunityEncoder(
  (input_proj): Linear(in_features=6, out_features=64, bias=True)
  (encoder): TransformerEncoder(
    (layers): ModuleList(
      (0-1): 2 x TransformerEncoderLayer(
        (self_attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=64, out_features=64, bias=True)
        )
        (linear1): Linear(in_features=64, out_features=256, bias=True)
        (dropout): Dropout(p=0.1, inplace=False)
        (linear2): Linear(in_features=256, out_features=64, bias=True)
        (norm1): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
        (norm2): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
        (dropout1): Dropout(p=0.1, inplace=False)
        (dropout2): Dropout(p=0.1, inplace=False)
      )
    )
  )
  (norm): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
  (cls_head): Linear(in_features=64, out_features=3, bias=True)
  (upside_head): Linear(in_features=64, out_features=1, bias=True)
  (downside_head): Li

In [ ]:
# CELL: TRAIN
def batch_loss(
    logits: torch.Tensor,
    pred_up: torch.Tensor,
    pred_dn: torch.Tensor,
    y_cls: torch.Tensor,
    y_reg: torch.Tensor,
) -> torch.Tensor:
    valid = y_cls >= 0
    if not bool(valid.any()):
        return logits.sum() * 0.0
    ce = nn.functional.cross_entropy(logits[valid], y_cls[valid])
    sm = nn.SmoothL1Loss()
    reg = sm(pred_up[valid], y_reg[valid, 0]) + sm(pred_dn[valid], y_reg[valid, 1])
    return float(CONFIG["cls_loss_weight"]) * ce + float(CONFIG["reg_loss_weight"]) * reg


def run_epoch(loader: DataLoader, *, train: bool) -> float:
    if train:
        model.train()
    else:
        model.eval()
    total = 0.0
    n = 0
    ctx = torch.enable_grad() if train else torch.no_grad()
    with ctx:
        for xb, yb, rb in loader:
            xb = xb.to(DEVICE)
            yb = yb.to(DEVICE)
            rb = rb.to(DEVICE)
            logits, up, dn = model(xb)
            loss = batch_loss(logits, up, dn, yb, rb)
            if train:
                optimizer.zero_grad(set_to_none=True)
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
            total += float(loss.item()) * int(xb.size(0))
            n += int(xb.size(0))
    return total / max(n, 1)


optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=float(CONFIG["lr"]),
    weight_decay=float(CONFIG["weight_decay"]),
)
best_state = None
best_val = float("inf")
stale = 0
history: List[Dict[str, float]] = []
for epoch in range(1, int(CONFIG["epochs"]) + 1):
    tr = run_epoch(train_loader, train=True)
    va = run_epoch(val_loader, train=False)
    history.append({"epoch": epoch, "train_loss": tr, "val_loss": va})
    print(f"epoch {epoch:02d}  train={tr:.4f}  val={va:.4f}")
    if va < best_val - 1e-4:
        best_val = va
        stale = 0
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
    else:
        stale += 1
        if stale >= int(CONFIG["patience"]):
            print("early stopping")
            break
if best_state is not None:
    model.load_state_dict(best_state)
print("best val loss", best_val)


epoch 01  train=0.9798  val=0.9778


epoch 02  train=0.9733  val=0.9815


epoch 03  train=0.9713  val=0.9744


epoch 04  train=0.9702  val=0.9719


epoch 05  train=0.9694  val=0.9779


epoch 06  train=0.9686  val=0.9735


epoch 07  train=0.9681  val=0.9730


epoch 08  train=0.9670  val=0.9721


epoch 09  train=0.9661  val=0.9761


epoch 10  train=0.9654  val=0.9731
early stopping
best val loss 0.9719380616908665


In [ ]:
# CELL: EVAL
def predict_loader(loader: DataLoader) -> Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    model.eval()
    logits_l: List[np.ndarray] = []
    up_l: List[np.ndarray] = []
    dn_l: List[np.ndarray] = []
    y_l: List[np.ndarray] = []
    r_l: List[np.ndarray] = []
    with torch.no_grad():
        for xb, yb, rb in loader:
            xb = xb.to(DEVICE)
            logits, up, dn = model(xb)
            logits_l.append(logits.cpu().numpy())
            up_l.append(up.cpu().numpy())
            dn_l.append(dn.cpu().numpy())
            y_l.append(yb.numpy())
            r_l.append(rb.numpy())
    return (
        np.concatenate(logits_l, axis=0),
        np.concatenate(up_l, axis=0),
        np.concatenate(dn_l, axis=0),
        np.concatenate(y_l, axis=0),
        np.concatenate(r_l, axis=0),
    )


def _unscale_reg(pred: np.ndarray, col: int) -> np.ndarray:
    return pred * float(reg_std[0, col]) + float(reg_mean[0, col])


def _unscale_true(arr: np.ndarray) -> np.ndarray:
    return arr * reg_std + reg_mean


def confusion(y_true: np.ndarray, y_pred: np.ndarray, n_classes: int = 3) -> np.ndarray:
    mat = np.zeros((n_classes, n_classes), dtype=np.int64)
    for t, p in zip(y_true.tolist(), y_pred.tolist()):
        if 0 <= int(t) < n_classes and 0 <= int(p) < n_classes:
            mat[int(t), int(p)] += 1
    return mat


def class_precision(mat: np.ndarray, cls: int) -> float:
    col = mat[:, cls]
    denom = float(col.sum())
    if denom <= 0:
        return float("nan")
    return float(mat[cls, cls]) / denom


def balanced_accuracy(mat: np.ndarray) -> float:
    recalls = []
    for i in range(mat.shape[0]):
        denom = float(mat[i].sum())
        recalls.append(float(mat[i, i]) / denom if denom else float("nan"))
    finite = [r for r in recalls if np.isfinite(r)]
    return float(np.mean(finite)) if finite else float("nan")


def corr(a: np.ndarray, b: np.ndarray) -> float:
    if a.size < 3:
        return float("nan")
    if float(np.std(a)) < 1e-12 or float(np.std(b)) < 1e-12:
        return float("nan")
    return float(np.corrcoef(a, b)[0, 1])


def signal_expectancy(
    y_true: np.ndarray,
    y_pred: np.ndarray,
    frame: pd.DataFrame,
    ends: np.ndarray,
    split_idx: np.ndarray,
    pred_cls: np.ndarray,
) -> Dict[str, float]:
    allowed = set(int(i) for i in split_idx.tolist())
    keep = np.array([int(e) in allowed for e in all_ends], dtype=bool)
    row_idx = all_ends[keep]
    cost = float(CONFIG["round_trip_cost_pct"])
    out: Dict[str, float] = {}
    for cls, name, pnl_col in (
        (LABEL_LONG, "LONG", "long_pnl_pct"),
        (LABEL_SHORT, "SHORT", "short_pnl_pct"),
    ):
        take = pred_cls == cls
        if not bool(take.any()):
            out[f"{name}_n"] = 0.0
            out[f"{name}_expectancy"] = float("nan")
            continue
        pnl = frame.loc[row_idx[take], pnl_col].to_numpy(dtype=np.float64) - cost
        finite = np.isfinite(pnl)
        out[f"{name}_n"] = float(finite.sum())
        out[f"{name}_expectancy"] = float(np.mean(pnl[finite])) if finite.any() else float("nan")
    return out


logits_test, up_s, dn_s, y_true, r_true_s = predict_loader(test_loader)
pred_cls = logits_test.argmax(axis=1)
up_hat = _unscale_reg(up_s, 0)
dn_hat = _unscale_reg(dn_s, 1)
r_true = _unscale_true(r_true_s)
cm = confusion(y_true, pred_cls)
print("test confusion rows=true cols=pred LONG/SHORT/NO_TRADE")
print(cm)
long_p = class_precision(cm, LABEL_LONG)
short_p = class_precision(cm, LABEL_SHORT)
bacc = balanced_accuracy(cm)
up_c = corr(up_hat, r_true[:, 0])
dn_c = corr(dn_hat, r_true[:, 1])
exp_report = signal_expectancy(
    y_true, pred_cls, labeled, all_ends, splits["test"], pred_cls
)
eval_report = {
    "n_test": int(len(y_true)),
    "confusion": cm.tolist(),
    "precision_LONG": long_p,
    "precision_SHORT": short_p,
    "balanced_accuracy": bacc,
    "upside_corr": up_c,
    "downside_corr": dn_c,
    "signal_expectancy": exp_report,
    "note": (
        "Balanced accuracy is reported and is not the acceptance number; "
        "NO_TRADE is the large class. Use LONG/SHORT precision and expectancy."
    ),
}
print(json.dumps(eval_report, indent=2, default=float))


test confusion rows=true cols=pred LONG/SHORT/NO_TRADE
[[    2    14  3425]
 [    3     9  3434]
 [    2    10 12532]]
{
  "n_test": 19431,
  "confusion": [
    [
      2,
      14,
      3425
    ],
    [
      3,
      9,
      3434
    ],
    [
      2,
      10,
      12532
    ]
  ],
  "precision_LONG": 0.2857142857142857,
  "precision_SHORT": 0.2727272727272727,
  "balanced_accuracy": 0.33407877249076123,
  "upside_corr": 0.1719896429976165,
  "downside_corr": 0.1518864600543995,
  "signal_expectancy": {
    "LONG_n": 6.0,
    "LONG_expectancy": -0.000725148732289013,
    "SHORT_n": 33.0,
    "SHORT_expectancy": -0.0017777758745918773
  },
  "note": "Balanced accuracy is reported and is not the acceptance number; NO_TRADE is the large class. Use LONG/SHORT precision and expectancy."
}


In [ ]:
# CELL: WALK_FORWARD
wf_rows: List[Dict[str, Any]] = []
for fold_i, (tr_idx, va_idx) in enumerate(wf_folds, start=1):
    x_tr, y_tr, r_tr = _split_windows(tr_idx)
    x_va, y_va, r_va = _split_windows(va_idx)
    if len(x_tr) < 32 or len(x_va) < 8:
        continue
    mean = x_tr.mean(axis=(0, 1), keepdims=True)
    std = np.where(x_tr.std(axis=(0, 1), keepdims=True) < 1e-6, 1.0, x_tr.std(axis=(0, 1), keepdims=True))
    rmean = r_tr.mean(axis=0, keepdims=True)
    rstd = np.where(r_tr.std(axis=0, keepdims=True) < 1e-6, 1.0, r_tr.std(axis=0, keepdims=True))
    fold_model = OpportunityEncoder(
        len(FEATURE_COLS),
        d_model=int(CONFIG["d_model"]),
        nhead=int(CONFIG["nhead"]),
        num_layers=int(CONFIG["num_layers"]),
        dropout=float(CONFIG["dropout"]),
        ff_mult=int(CONFIG["ff_mult"]),
    ).to(DEVICE)
    opt = torch.optim.AdamW(
        fold_model.parameters(),
        lr=float(CONFIG["lr"]),
        weight_decay=float(CONFIG["weight_decay"]),
    )
    tr_loader = DataLoader(
        SequenceDataset(((x_tr - mean) / std).astype(np.float32), y_tr, ((r_tr - rmean) / rstd).astype(np.float32)),
        batch_size=int(CONFIG["batch_size"]),
        shuffle=True,
    )
    va_loader = DataLoader(
        SequenceDataset(((x_va - mean) / std).astype(np.float32), y_va, ((r_va - rmean) / rstd).astype(np.float32)),
        batch_size=int(CONFIG["batch_size"]),
        shuffle=False,
    )
    best = float("inf")
    stale = 0
    best_w = None
    for epoch in range(1, min(12, int(CONFIG["epochs"])) + 1):
        fold_model.train()
        for xb, yb, rb in tr_loader:
            xb, yb, rb = xb.to(DEVICE), yb.to(DEVICE), rb.to(DEVICE)
            logits, up, dn = fold_model(xb)
            loss = batch_loss(logits, up, dn, yb, rb)
            opt.zero_grad(set_to_none=True)
            loss.backward()
            opt.step()
        fold_model.eval()
        total = 0.0
        n = 0
        with torch.no_grad():
            for xb, yb, rb in va_loader:
                xb, yb, rb = xb.to(DEVICE), yb.to(DEVICE), rb.to(DEVICE)
                logits, up, dn = fold_model(xb)
                loss = batch_loss(logits, up, dn, yb, rb)
                total += float(loss.item()) * int(xb.size(0))
                n += int(xb.size(0))
        va = total / max(n, 1)
        if va < best:
            best = va
            stale = 0
            best_w = {k: v.detach().cpu().clone() for k, v in fold_model.state_dict().items()}
        else:
            stale += 1
            if stale >= 3:
                break
    wf_rows.append({"fold": fold_i, "n_train": int(len(x_tr)), "n_val": int(len(x_va)), "best_val": best})
    del fold_model, best_w
print(json.dumps({"walk_forward": wf_rows}, indent=2))


{
  "walk_forward": [
    {
      "fold": 1,
      "n_train": 27460,
      "n_val": 27539,
      "best_val": 0.9539579672640093
    },
    {
      "fold": 2,
      "n_train": 55002,
      "n_val": 27538,
      "best_val": 0.990724825858637
    },
    {
      "fold": 3,
      "n_train": 82543,
      "n_val": 27539,
      "best_val": 0.9735735853866596
    }
  ]
}


In [ ]:
# CELL: EXPORT
EXPORT_DIR.mkdir(parents=True, exist_ok=True)
model_cpu = OpportunityEncoder(
    len(FEATURE_COLS),
    d_model=int(CONFIG["d_model"]),
    nhead=int(CONFIG["nhead"]),
    num_layers=int(CONFIG["num_layers"]),
    dropout=float(CONFIG["dropout"]),
    ff_mult=int(CONFIG["ff_mult"]),
)
model_cpu.load_state_dict(model.state_dict())
model_cpu.eval()
dummy = torch.zeros(1, int(CONFIG["window_len"]), len(FEATURE_COLS), dtype=torch.float32)
onnx_path = EXPORT_DIR / "btcusd_opportunity_10m.onnx"
torch.onnx.export(
    model_cpu,
    dummy,
    str(onnx_path),
    input_names=["features"],
    output_names=["trade_logits", "upside", "downside"],
    dynamic_axes={
        "features": {0: "batch"},
        "trade_logits": {0: "batch"},
        "upside": {0: "batch"},
        "downside": {0: "batch"},
    },
    opset_version=17,
)
weights_path = EXPORT_DIR / "opportunity_10m.pt"
torch.save(model.state_dict(), weights_path)
metadata = {
    "model_family": "jacksparrow_opportunity_10m",
    "timeframe": "10m",
    "horizon_bars": int(CONFIG["horizon_bars"]),
    "horizon_minutes": 30,
    "tp_atr_mult": float(CONFIG["tp_atr_mult"]),
    "sl_atr_mult": float(CONFIG["sl_atr_mult"]),
    "round_trip_cost_pct": float(CONFIG["round_trip_cost_pct"]),
    "window_len": int(CONFIG["window_len"]),
    "feature_cols": list(FEATURE_COLS),
    "label_names": list(LABEL_NAMES),
    "feat_mean": feat_mean.reshape(-1).tolist(),
    "feat_std": feat_std.reshape(-1).tolist(),
    "reg_mean": reg_mean.reshape(-1).tolist(),
    "reg_std": reg_std.reshape(-1).tolist(),
    "ready_to_promote": False,
    "promote_reason": "research_opportunity_10m_not_live",
    "eval": eval_report,
    "label_gate": label_report,
    "walk_forward": wf_rows,
    "history": history,
}
(EXPORT_DIR / "metadata.json").write_text(json.dumps(metadata, indent=2, default=float), encoding="utf-8")
(EXPORT_DIR / "eval_report.json").write_text(json.dumps(eval_report, indent=2, default=float), encoding="utf-8")
print("wrote", onnx_path)
print("wrote", weights_path)
print("ready_to_promote=false")
try:
    from google.colab import files

    zip_path = EXPORT_DIR.with_suffix(".zip")
    import shutil

    shutil.make_archive(str(EXPORT_DIR), "zip", root_dir=str(EXPORT_DIR))
    files.download(str(zip_path) if zip_path.exists() else str(EXPORT_DIR) + ".zip")
except ImportError:
    print("Not Colab; files remain under", EXPORT_DIR)


ModuleNotFoundError: No module named 'onnxscript'

*File Operation*: `ls` on `content`

*File Operation*: `ls` on `/content/export/opportunity_10m`

*File Operation*: `ls` on `content`

*File Operation*: `ls` on `content`

*File Operation*: `ls` on `/content/export`

*File Operation*: `ls` on `/content/export/opportunity_10m`

**Session Created**: 2026-10-08 11:14:33
- Endpoint: `gpu-t4-s-kkb-usw4b1-1xclpydwpbllk`
- Hardware: `T4`

In [ ]:
# CELL: ENV
# Colab ships torch/pandas/numpy; install research extras.
%pip install -q --upgrade-strategy only-if-needed pyarrow onnx onnxruntime requests scikit-learn

import json
import math
import random
import time
from datetime import datetime, timedelta, timezone
from pathlib import Path
from typing import Any, Dict, List, Optional, Sequence, Tuple

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset

print(f"PyTorch: {torch.__version__}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("WARNING: No GPU detected — training will run on CPU and be much slower.")
    print("Runtime -> Change runtime type -> select a GPU (T4), then re-run this cell.")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 77.6 MB/s eta 0:00:00
   ━━━━━━━━━━━╺━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.7/23.6 MB 201.0 MB/s eta 0:00:01

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.6/23.6 MB 84.0 MB/s eta 0:00:00


PyTorch: 2.11.0+cu130
GPU: Tesla T4


In [ ]:
# CELL: CONFIG
CONFIG: Dict[str, Any] = {
    "symbol": "BTCUSD",
    "base_url": "https://api.india.delta.exchange",
    "history_days": 900,
    "resolution_5m": "5m",
    "seed": 42,
    "tp_atr_mult": 1.5,
    "sl_atr_mult": 1.0,
    "horizon_bars": 3,
    "atr_period": 14,
    "window_len": 60,
    "round_trip_cost_pct": 0.0016,
    "minority_rate": 0.15,
    "ambiguous_win_max": 0.50,
    "train_fraction": 0.70,
    "val_fraction": 0.15,
    "purge_bars": 3,
    "d_model": 64,
    "nhead": 4,
    "num_layers": 2,
    "dropout": 0.10,
    "ff_mult": 4,
    "lr": 1e-3,
    "weight_decay": 1e-4,
    "batch_size": 64,
    "epochs": 40,
    "patience": 6,
    "cls_loss_weight": 1.0,
    "reg_loss_weight": 0.35,
    "export_dir": "export/opportunity_10m",
    "ready_to_promote": False,
}

FEATURE_COLS: Tuple[str, ...] = (
    "ret_1",
    "atr_pct",
    "rsi",
    "ema_dist_atr",
    "candle_body",
    "volume_z",
)
LABEL_NAMES: Tuple[str, ...] = ("LONG", "SHORT", "NO_TRADE")
LABEL_LONG, LABEL_SHORT, LABEL_NO_TRADE = 0, 1, 2
print(json.dumps(CONFIG, indent=2))


{
  "symbol": "BTCUSD",
  "base_url": "https://api.india.delta.exchange",
  "history_days": 900,
  "resolution_5m": "5m",
  "seed": 42,
  "tp_atr_mult": 1.5,
  "sl_atr_mult": 1.0,
  "horizon_bars": 3,
  "atr_period": 14,
  "window_len": 60,
  "round_trip_cost_pct": 0.0016,
  "minority_rate": 0.15,
  "ambiguous_win_max": 0.5,
  "train_fraction": 0.7,
  "val_fraction": 0.15,
  "purge_bars": 3,
  "d_model": 64,
  "nhead": 4,
  "num_layers": 2,
  "dropout": 0.1,
  "ff_mult": 4,
  "lr": 0.001,
  "weight_decay": 0.0001,
  "batch_size": 64,
  "epochs": 40,
  "patience": 6,
  "cls_loss_weight": 1.0,
  "reg_loss_weight": 0.35,
  "export_dir": "export/opportunity_10m",
  "ready_to_promote": false
}


In [ ]:
# CELL: SEED
def set_seed(seed: int) -> None:
    random.seed(int(seed))
    np.random.seed(int(seed))
    torch.manual_seed(int(seed))
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(int(seed))


set_seed(int(CONFIG["seed"]))
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device", DEVICE)


device cuda


In [ ]:
# CELL: FETCH
DELTA_CANDLE_PAGE_BARS = 2000
REQUEST_DELAY_SECONDS = 0.2
EMPTY_PAGE_RETRIES = 3
MIN_OHLCV_COMPLETENESS = 0.95
_RESOLUTION_MINUTES = {"5m": 5, "10m": 10}


def _bar_seconds(resolution: str) -> int:
    if resolution not in _RESOLUTION_MINUTES:
        raise ValueError(f"Unsupported resolution {resolution!r}")
    return int(_RESOLUTION_MINUTES[resolution]) * 60


def _parse_candle_rows(payload: Any) -> List[Dict[str, Any]]:
    if not isinstance(payload, dict):
        return []
    result = payload.get("result", [])
    if isinstance(result, dict):
        result = result.get("candles", []) or []
    if not isinstance(result, list):
        return []
    return [row for row in result if isinstance(row, dict) and "time" in row]


def _candle_epoch(row: Dict[str, Any]) -> Optional[int]:
    try:
        return int(row["time"])
    except (KeyError, TypeError, ValueError):
        return None


def _request_candle_page(
    url: str,
    *,
    symbol: str,
    resolution: str,
    start_ts: int,
    end_ts: int,
) -> List[Dict[str, Any]]:
    import requests

    last_error: Optional[Exception] = None
    for attempt in range(EMPTY_PAGE_RETRIES):
        try:
            response = requests.get(
                url,
                params={
                    "symbol": symbol,
                    "resolution": resolution,
                    "start": int(start_ts),
                    "end": int(end_ts),
                },
                timeout=30,
            )
            response.raise_for_status()
            payload = response.json()
        except Exception as exc:
            last_error = exc
            time.sleep(REQUEST_DELAY_SECONDS * (attempt + 1))
            continue
        if payload.get("success") is False:
            last_error = RuntimeError(f"Delta candles error for {symbol}: {payload}")
            time.sleep(REQUEST_DELAY_SECONDS * (attempt + 1))
            continue
        rows = _parse_candle_rows(payload)
        if rows:
            return rows
        last_error = None
        time.sleep(REQUEST_DELAY_SECONDS * (attempt + 1))
    if last_error is not None:
        raise last_error
    return []


def _rows_to_frame(all_rows: List[Dict[str, Any]]) -> pd.DataFrame:
    df = pd.DataFrame(all_rows)
    expected_cols = ["time", "open", "high", "low", "close", "volume"]
    df = df[[c for c in expected_cols if c in df.columns]]
    df["time"] = pd.to_datetime(df["time"], unit="s", utc=True)
    df = df.drop_duplicates(subset="time").sort_values("time").reset_index(drop=True)
    for col in ["open", "high", "low", "close", "volume"]:
        if col in df.columns:
            df[col] = df[col].astype(float)
    return df


def fetch_candles(
    symbol: str,
    resolution: str,
    start_ts: int,
    end_ts: int,
    base_url: str,
    *,
    page_bars: int = DELTA_CANDLE_PAGE_BARS,
) -> pd.DataFrame:
    url = f"{base_url.rstrip('/')}/v2/history/candles"
    bar_seconds = _bar_seconds(resolution)
    page_bars = max(1, int(page_bars))
    start_ts, end_ts = int(start_ts), int(end_ts)
    if end_ts <= start_ts:
        raise ValueError("end_ts must be greater than start_ts")
    all_rows: List[Dict[str, Any]] = []
    cursor_end = end_ts
    seen_oldest: Optional[int] = None
    max_pages = max(2, (end_ts - start_ts) // bar_seconds + 5)
    for _ in range(max_pages):
        if cursor_end <= start_ts:
            break
        page_start = max(start_ts, cursor_end - page_bars * bar_seconds)
        if page_start >= cursor_end:
            break
        rows = _request_candle_page(
            url,
            symbol=symbol,
            resolution=resolution,
            start_ts=page_start,
            end_ts=cursor_end,
        )
        if not rows:
            break
        epochs = [e for e in (_candle_epoch(r) for r in rows) if e is not None]
        if not epochs:
            break
        oldest = min(epochs)
        if seen_oldest is not None and oldest >= seen_oldest:
            break
        seen_oldest = oldest
        all_rows.extend(rows)
        cursor_end = oldest - 1
        time.sleep(REQUEST_DELAY_SECONDS)
    if not all_rows:
        raise RuntimeError(f"No candle data returned for symbol={symbol}")
    return _rows_to_frame(all_rows)


def validate_ohlcv_completeness(
    df: pd.DataFrame,
    resolution: str,
    *,
    min_completeness: float = MIN_OHLCV_COMPLETENESS,
    symbol: str = "",
) -> Dict[str, Any]:
    if df is None or df.empty or "time" not in df.columns:
        raise ValueError("Cannot validate completeness on an empty OHLCV frame")
    bar_seconds = _bar_seconds(resolution)
    time_col = df["time"]
    epochs = (pd.to_datetime(time_col, utc=True).astype("int64") // 10**9).tolist()
    if len(epochs) < 2:
        raise ValueError("Need at least 2 OHLCV bars to validate completeness")
    gaps = 0
    for prev, cur in zip(epochs, epochs[1:]):
        if int(cur) - int(prev) > bar_seconds * 1.5:
            gaps += 1
    completeness = 1.0 - (gaps / max(1, len(epochs) - 1))
    report = {
        "rows": len(epochs),
        "gaps": int(gaps),
        "completeness": round(float(completeness), 4),
    }
    if completeness < min_completeness:
        label = f"{symbol} {resolution}".strip()
        raise ValueError(
            f"OHLCV completeness {completeness:.1%} below {min_completeness:.0%} "
            f"for {label or 'candles'}"
        )
    return report


end_dt = datetime.now(timezone.utc)
start_dt = end_dt - timedelta(days=int(CONFIG["history_days"]))
print(
    f"Fetching {CONFIG['symbol']} 5m OHLCV for {CONFIG['history_days']}d "
    f"from {CONFIG['base_url']}..."
)
df5m = fetch_candles(
    str(CONFIG["symbol"]),
    str(CONFIG["resolution_5m"]),
    int(start_dt.timestamp()),
    int(end_dt.timestamp()),
    str(CONFIG["base_url"]),
)
ohlcv_report = validate_ohlcv_completeness(
    df5m, "5m", symbol=str(CONFIG["symbol"])
)
print(
    f"OHLCV {CONFIG['symbol']} 5m: {ohlcv_report['rows']} bars, "
    f"completeness {ohlcv_report['completeness']:.1%}, gaps={ohlcv_report['gaps']}"
)


Fetching BTCUSD 5m OHLCV for 900d from https://api.india.delta.exchange...


OHLCV BTCUSD 5m: 259200 bars, completeness 100.0%, gaps=0


In [ ]:
# CELL: BARS_10M
def build_10m_ohlcv_from_5m(df5m: pd.DataFrame) -> pd.DataFrame:
    """Exchange-aligned 10m bars from two closed 5m candles. Drop a partial last bucket."""
    src = df5m.copy()
    src["time"] = pd.to_datetime(src["time"], utc=True)
    required = ("open", "high", "low", "close", "volume")
    missing = [c for c in required if c not in src.columns]
    if missing:
        raise ValueError(f"5m OHLCV missing columns: {missing}")
    if src.empty:
        return src.iloc[0:0].copy()
    grouped = src.set_index("time").sort_index()
    agg = grouped.resample("10min", label="left", closed="left").agg(
        {
            "open": "first",
            "high": "max",
            "low": "min",
            "close": "last",
            "volume": "sum",
        }
    )
    counts = grouped["close"].resample("10min", label="left", closed="left").count()
    complete = counts >= 2
    agg = agg.loc[complete].dropna(subset=["open", "high", "low", "close"])
    out = agg.reset_index()
    return out.sort_values("time").reset_index(drop=True)


bars_10m = build_10m_ohlcv_from_5m(df5m)
print(f"10m bars={len(bars_10m)} span={bars_10m['time'].iloc[0]} → {bars_10m['time'].iloc[-1]}")


10m bars=129599 span=2024-04-21 11:20:00+00:00 → 2026-10-08 11:00:00+00:00


In [ ]:
# CELL: FEATURES
def _wilder_rsi(close: pd.Series, period: int = 14) -> pd.Series:
    delta = close.diff()
    gain = delta.clip(lower=0.0)
    loss = -delta.clip(upper=0.0)
    avg_gain = gain.ewm(alpha=1.0 / period, min_periods=period, adjust=False).mean()
    avg_loss = loss.ewm(alpha=1.0 / period, min_periods=period, adjust=False).mean()
    rs = avg_gain / avg_loss.replace(0.0, np.nan)
    return 100.0 - (100.0 / (1.0 + rs))


def add_opportunity_features(
    df: pd.DataFrame,
    *,
    atr_period: int = 14,
    rsi_period: int = 14,
    ema_span: int = 20,
    vol_window: int = 20,
) -> pd.DataFrame:
    """Causal 10m features. ATR(14) is also stored for the label path."""
    out = df.copy()
    close = out["close"].astype(np.float64)
    high = out["high"].astype(np.float64)
    low = out["low"].astype(np.float64)
    open_ = out["open"].astype(np.float64)
    volume = out["volume"].astype(np.float64)
    prev_close = close.shift(1)
    tr = pd.concat(
        [
            high - low,
            (high - prev_close).abs(),
            (low - prev_close).abs(),
        ],
        axis=1,
    ).max(axis=1)
    atr = tr.rolling(int(atr_period), min_periods=int(atr_period)).mean()
    out["atr"] = atr
    out["ret_1"] = close.pct_change()
    out["atr_pct"] = atr / close.replace(0.0, np.nan)
    out["rsi"] = _wilder_rsi(close, period=int(rsi_period))
    ema = close.ewm(span=int(ema_span), adjust=False, min_periods=int(ema_span)).mean()
    out["ema_dist_atr"] = (close - ema) / atr.replace(0.0, np.nan)
    out["candle_body"] = (close - open_) / atr.replace(0.0, np.nan)
    vol_mean = volume.rolling(int(vol_window), min_periods=int(vol_window)).mean()
    vol_std = volume.rolling(int(vol_window), min_periods=int(vol_window)).std()
    out["volume_z"] = (volume - vol_mean) / vol_std.replace(0.0, np.nan)
    return out


featured = add_opportunity_features(
    bars_10m, atr_period=int(CONFIG["atr_period"])
)
print("feature null rates")
print(featured[list(FEATURE_COLS)].isna().mean().round(4))


feature null rates
ret_1           0.0000
atr_pct         0.0001
rsi             0.0001
ema_dist_atr    0.0001
candle_body     0.0001
volume_z        0.0001
dtype: float64


In [ ]:
# CELL: LABELS
# Opportunity labels on a 10-minute series. Path is bars t+1 .. t+horizon only.

import numpy as np
import pandas as pd

_EPS = 1e-9
HORIZON_BARS_DEFAULT = 3
ATR_PERIOD_DEFAULT = 14
TP_ATR_MULT_DEFAULT = 1.5
SL_ATR_MULT_DEFAULT = 1.0

TP_FIRST = 0
SL_FIRST = 1
NEITHER = 2
AMBIGUOUS = 3

LABEL_LONG = 0
LABEL_SHORT = 1
LABEL_NO_TRADE = 2
LABEL_UNLABELED = -1
LABEL_NAMES = ("LONG", "SHORT", "NO_TRADE")


def tp_sl_from_path(
    fwd_high: np.ndarray,
    fwd_low: np.ndarray,
    entry: float,
    atr: float,
    tp_mult: float,
    sl_mult: float,
    side: str,
) -> int:
    """Walk future highs/lows. Same-bar TP and SL is AMBIGUOUS, not a win."""
    high = np.asarray(fwd_high, dtype=np.float64).ravel()
    low = np.asarray(fwd_low, dtype=np.float64).ravel()
    if high.size == 0 or high.size != low.size:
        return int(NEITHER)
    denom = max(float(atr), _EPS)
    entry_f = float(entry)
    if str(side).strip().upper() == "SHORT":
        tp_level = entry_f - float(tp_mult) * denom
        sl_level = entry_f + float(sl_mult) * denom
        tp_hit = low <= tp_level
        sl_hit = high >= sl_level
    else:
        tp_level = entry_f + float(tp_mult) * denom
        sl_level = entry_f - float(sl_mult) * denom
        tp_hit = high >= tp_level
        sl_hit = low <= sl_level
    for j in range(high.size):
        hit_tp = bool(tp_hit[j])
        hit_sl = bool(sl_hit[j])
        if hit_tp and hit_sl:
            return int(AMBIGUOUS)
        if hit_tp:
            return int(TP_FIRST)
        if hit_sl:
            return int(SL_FIRST)
    return int(NEITHER)


def _side_win(code: int) -> int:
    return 1 if int(code) == int(TP_FIRST) else 0


def _causal_atr(df: pd.DataFrame, period: int = ATR_PERIOD_DEFAULT) -> np.ndarray:
    if "atr" in df.columns:
        atr = df["atr"].to_numpy(dtype=np.float64)
        if np.isfinite(atr).any():
            return atr
    close = df["close"].to_numpy(dtype=np.float64)
    high = df["high"].to_numpy(dtype=np.float64)
    low = df["low"].to_numpy(dtype=np.float64)
    prev = np.concatenate([[close[0]], close[:-1]])
    tr = np.maximum(high - low, np.maximum(np.abs(high - prev), np.abs(low - prev)))
    return pd.Series(tr).rolling(int(period), min_periods=int(period)).mean().to_numpy(
        dtype=np.float64
    )


def compute_opportunity_labels(
    df: pd.DataFrame,
    *,
    tp_mult: float = TP_ATR_MULT_DEFAULT,
    sl_mult: float = SL_ATR_MULT_DEFAULT,
    horizon_bars: int = HORIZON_BARS_DEFAULT,
    atr_period: int = ATR_PERIOD_DEFAULT,
) -> pd.DataFrame:
    """Label each closed 10m bar. Last ``horizon_bars`` rows stay unlabeled."""
    out = df.copy().reset_index(drop=True)
    n = len(out)
    k = int(horizon_bars)
    close = out["close"].to_numpy(dtype=np.float64)
    high = out["high"].to_numpy(dtype=np.float64)
    low = out["low"].to_numpy(dtype=np.float64)
    atr = _causal_atr(out, period=int(atr_period))
    n_valid = max(n - k, 0)

    trade_label = np.full(n, LABEL_UNLABELED, dtype=np.int64)
    long_result = np.full(n, np.nan, dtype=np.float64)
    short_result = np.full(n, np.nan, dtype=np.float64)
    long_code = np.full(n, np.nan, dtype=np.float64)
    short_code = np.full(n, np.nan, dtype=np.float64)
    upside = np.full(n, np.nan, dtype=np.float64)
    downside = np.full(n, np.nan, dtype=np.float64)
    timeout_ret = np.full(n, np.nan, dtype=np.float64)
    long_pnl_pct = np.full(n, np.nan, dtype=np.float64)
    short_pnl_pct = np.full(n, np.nan, dtype=np.float64)

    for i in range(n_valid):
        atr_i = float(atr[i]) if i < len(atr) else float("nan")
        if not np.isfinite(atr_i) or atr_i <= _EPS:
            continue
        entry = float(close[i])
        if not np.isfinite(entry) or abs(entry) <= _EPS:
            continue
        fwd_high = high[i + 1 : i + 1 + k]
        fwd_low = low[i + 1 : i + 1 + k]
        fwd_close = close[i + 1 : i + 1 + k]
        if fwd_high.size < k:
            continue
        l_code = tp_sl_from_path(fwd_high, fwd_low, entry, atr_i, tp_mult, sl_mult, "LONG")
        s_code = tp_sl_from_path(fwd_high, fwd_low, entry, atr_i, tp_mult, sl_mult, "SHORT")
        l_win = _side_win(l_code)
        s_win = _side_win(s_code)
        if l_win == 1 and s_win == 0:
            label = LABEL_LONG
        elif s_win == 1 and l_win == 0:
            label = LABEL_SHORT
        else:
            label = LABEL_NO_TRADE
        trade_label[i] = label
        long_result[i] = float(l_win)
        short_result[i] = float(s_win)
        long_code[i] = float(l_code)
        short_code[i] = float(s_code)
        denom = max(atr_i, _EPS)
        upside[i] = max((float(np.max(fwd_high)) - entry) / denom, 0.0)
        downside[i] = max((entry - float(np.min(fwd_low))) / denom, 0.0)
        end_close = float(fwd_close[-1])
        timeout_ret[i] = (end_close - entry) / entry
        tp_pct = float(tp_mult) * denom / entry
        sl_pct = float(sl_mult) * denom / entry
        if l_code == TP_FIRST:
            long_pnl_pct[i] = tp_pct
        elif l_code == SL_FIRST:
            long_pnl_pct[i] = -sl_pct
        elif l_code == NEITHER:
            long_pnl_pct[i] = timeout_ret[i]
        if s_code == TP_FIRST:
            short_pnl_pct[i] = tp_pct
        elif s_code == SL_FIRST:
            short_pnl_pct[i] = -sl_pct
        elif s_code == NEITHER:
            short_pnl_pct[i] = -timeout_ret[i]

    out["atr"] = atr
    out["trade_label"] = trade_label
    out["long_result"] = long_result
    out["short_result"] = short_result
    out["long_code"] = long_code
    out["short_code"] = short_code
    out["upside"] = upside
    out["downside"] = downside
    out["timeout_ret"] = timeout_ret
    out["long_pnl_pct"] = long_pnl_pct
    out["short_pnl_pct"] = short_pnl_pct
    names = np.array(["UNLABELED"] * n, dtype=object)
    names[trade_label == LABEL_LONG] = "LONG"
    names[trade_label == LABEL_SHORT] = "SHORT"
    names[trade_label == LABEL_NO_TRADE] = "NO_TRADE"
    out["trade_name"] = names
    return out


In [ ]:
# CELL: APPLY_LABELS
labeled = compute_opportunity_labels(
    featured,
    tp_mult=float(CONFIG["tp_atr_mult"]),
    sl_mult=float(CONFIG["sl_atr_mult"]),
    horizon_bars=int(CONFIG["horizon_bars"]),
    atr_period=int(CONFIG["atr_period"]),
)
valid = labeled["trade_label"].to_numpy() >= 0
print(f"labeled rows={int(valid.sum())} / {len(labeled)}")
print(labeled.loc[valid, "trade_name"].value_counts(normalize=True).round(4))


labeled rows=129583 / 129599
trade_name
NO_TRADE    0.6529
SHORT       0.1757
LONG        0.1715
Name: proportion, dtype: float64


In [ ]:
# CELL: GATE
def train_period_mask(n: int, train_fraction: float) -> np.ndarray:
    cut = int(n * float(train_fraction))
    mask = np.zeros(n, dtype=bool)
    mask[:cut] = True
    return mask


def summarize_label_gate(
    df: pd.DataFrame,
    *,
    train_fraction: float,
    round_trip_cost_pct: float,
    minority_rate: float,
    ambiguous_win_max: float,
) -> Dict[str, Any]:
    n = len(df)
    in_train = train_period_mask(n, train_fraction)
    labels = df["trade_label"].to_numpy(dtype=np.int64)
    usable = in_train & (labels >= 0)
    n_use = int(usable.sum())
    if n_use == 0:
        raise RuntimeError("Train-period gate has no labeled rows")
    mix = {}
    mix_ok = True
    for cid, name in enumerate(LABEL_NAMES):
        rate = float(np.mean(labels[usable] == cid))
        mix[name] = rate
        if rate < float(minority_rate):
            mix_ok = False
    long_code = df["long_code"].to_numpy(dtype=np.float64)
    short_code = df["short_code"].to_numpy(dtype=np.float64)
    long_pnl = df["long_pnl_pct"].to_numpy(dtype=np.float64) - float(round_trip_cost_pct)
    short_pnl = df["short_pnl_pct"].to_numpy(dtype=np.float64) - float(round_trip_cost_pct)
    long_exp_mask = usable & np.isfinite(long_pnl)
    short_exp_mask = usable & np.isfinite(short_pnl)
    long_expectancy = float(np.mean(long_pnl[long_exp_mask])) if long_exp_mask.any() else float("nan")
    short_expectancy = (
        float(np.mean(short_pnl[short_exp_mask])) if short_exp_mask.any() else float("nan")
    )

    def _ambig_share(codes: np.ndarray) -> float:
        finite = usable & np.isfinite(codes)
        tp = finite & (codes == TP_FIRST)
        amb = finite & (codes == AMBIGUOUS)
        denom = int(tp.sum() + amb.sum())
        if denom == 0:
            return 0.0
        return float(amb.sum()) / float(denom)

    long_ambig = _ambig_share(long_code)
    short_ambig = _ambig_share(short_code)
    reasons: List[str] = []
    if not mix_ok:
        reasons.append("class_mix_below_floor")
    if not (np.isfinite(long_expectancy) and long_expectancy > 0.0):
        reasons.append("long_expectancy_not_positive")
    if not (np.isfinite(short_expectancy) and short_expectancy > 0.0):
        reasons.append("short_expectancy_not_positive")
    if long_ambig >= float(ambiguous_win_max):
        reasons.append("long_ambiguous_majority")
    if short_ambig >= float(ambiguous_win_max):
        reasons.append("short_ambiguous_majority")
    passed = len(reasons) == 0
    return {
        "passed": passed,
        "n_train_labeled": n_use,
        "class_mix": mix,
        "minority_rate": float(minority_rate),
        "long_expectancy_pct": long_expectancy,
        "short_expectancy_pct": short_expectancy,
        "round_trip_cost_pct": float(round_trip_cost_pct),
        "long_ambiguous_share": long_ambig,
        "short_ambiguous_share": short_ambig,
        "tp_atr_mult": float(CONFIG["tp_atr_mult"]),
        "sl_atr_mult": float(CONFIG["sl_atr_mult"]),
        "reasons": reasons,
    }


EXPORT_DIR = Path(str(CONFIG["export_dir"]))
EXPORT_DIR.mkdir(parents=True, exist_ok=True)
label_report = summarize_label_gate(
    labeled,
    train_fraction=float(CONFIG["train_fraction"]),
    round_trip_cost_pct=float(CONFIG["round_trip_cost_pct"]),
    minority_rate=float(CONFIG["minority_rate"]),
    ambiguous_win_max=float(CONFIG["ambiguous_win_max"]),
)
report_path = EXPORT_DIR / "label_report.json"
report_path.write_text(json.dumps(label_report, indent=2), encoding="utf-8")
print(json.dumps(label_report, indent=2))
print("wrote", report_path)
if not label_report["passed"]:
    raise RuntimeError(
        "Train-period label gate failed: "
        f"{label_report['reasons']}. Change CONFIG tp/sl ATR multiples and re-run "
        "from the top. Do not fit the network."
    )
print("Gate passed. Training may start.")


{
  "passed": false,
  "n_train_labeled": 90706,
  "class_mix": {
    "LONG": 0.17131171036094636,
    "SHORT": 0.1749829118250171,
    "NO_TRADE": 0.6537053778140366
  },
  "minority_rate": 0.15,
  "long_expectancy_pct": -0.0015950013071000019,
  "short_expectancy_pct": -0.0015955378001310072,
  "round_trip_cost_pct": 0.0016,
  "long_ambiguous_share": 0.017389654736309598,
  "short_ambiguous_share": 0.018125579956696565,
  "tp_atr_mult": 1.5,
  "sl_atr_mult": 1.0,
  "reasons": [
    "long_expectancy_not_positive",
    "short_expectancy_not_positive"
  ]
}
wrote export/opportunity_10m/label_report.json


RuntimeError: Train-period label gate failed: ['long_expectancy_not_positive', 'short_expectancy_not_positive']. Change CONFIG tp/sl ATR multiples and re-run from the top. Do not fit the network.

In [ ]:
# CELL: SPLIT
def purged_index_splits(
    n: int,
    *,
    train_fraction: float,
    val_fraction: float,
    purge_bars: int,
) -> Dict[str, np.ndarray]:
    """Time-ordered split. Purge ``purge_bars`` on each side of every cut."""
    n = int(n)
    purge = int(purge_bars)
    train_end = int(n * float(train_fraction))
    val_len = int(n * float(val_fraction))
    train_idx = np.arange(0, max(train_end - purge, 0), dtype=np.int64)
    val_start = min(train_end + purge, n)
    val_end = min(val_start + val_len, n)
    val_idx = np.arange(val_start, max(val_end - purge, val_start), dtype=np.int64)
    test_start = min(val_end + purge, n)
    test_idx = np.arange(test_start, n, dtype=np.int64)
    return {"train": train_idx, "val": val_idx, "test": test_idx}


def development_walk_forward(
    n_dev: int,
    *,
    folds: int = 4,
    purge_bars: int,
) -> List[Tuple[np.ndarray, np.ndarray]]:
    """Expanding-window folds inside the development prefix only."""
    folds = max(2, int(folds))
    purge = int(purge_bars)
    edges = [int(n_dev * (i + 1) / folds) for i in range(folds)]
    out: List[Tuple[np.ndarray, np.ndarray]] = []
    for i in range(1, folds):
        train_end = edges[i - 1]
        val_end = edges[i]
        tr = np.arange(0, max(train_end - purge, 0), dtype=np.int64)
        va = np.arange(min(train_end + purge, n_dev), val_end, dtype=np.int64)
        if len(tr) == 0 or len(va) == 0:
            continue
        out.append((tr, va))
    return out


splits = purged_index_splits(
    len(labeled),
    train_fraction=float(CONFIG["train_fraction"]),
    val_fraction=float(CONFIG["val_fraction"]),
    purge_bars=int(CONFIG["purge_bars"]),
)
for key, idx in splits.items():
    print(key, "n", len(idx), "range", int(idx[0]) if len(idx) else None, int(idx[-1]) if len(idx) else None)
dev_end = int(splits["test"][0]) if len(splits["test"]) else len(labeled)
wf_folds = development_walk_forward(
    dev_end, folds=4, purge_bars=int(CONFIG["purge_bars"])
)
print("walk-forward folds", len(wf_folds), "dev_end", dev_end)


train n 90716 range 0 90715
val n 19436 range 90722 110157
test n 19435 range 110164 129598
walk-forward folds 3 dev_end 110164


In [ ]:
# CELL: DATASET
class SequenceDataset(Dataset):
    def __init__(
        self,
        windows: np.ndarray,
        y_cls: np.ndarray,
        y_reg: np.ndarray,
    ) -> None:
        self.windows = np.asarray(windows, dtype=np.float32)
        self.y_cls = np.asarray(y_cls, dtype=np.int64)
        self.y_reg = np.asarray(y_reg, dtype=np.float32)

    def __len__(self) -> int:
        return int(self.windows.shape[0])

    def __getitem__(self, idx: int):
        return (
            torch.from_numpy(self.windows[idx]),
            torch.tensor(self.y_cls[idx], dtype=torch.long),
            torch.from_numpy(self.y_reg[idx]),
        )


def build_windows(
    frame: pd.DataFrame,
    *,
    window_len: int,
    feature_cols: Sequence[str],
) -> Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    feats = frame.loc[:, list(feature_cols)].to_numpy(dtype=np.float64)
    y_cls = frame["trade_label"].to_numpy(dtype=np.int64)
    y_reg = frame.loc[:, ["upside", "downside"]].to_numpy(dtype=np.float64)
    w = int(window_len)
    n = len(frame)
    xs: List[np.ndarray] = []
    cs: List[int] = []
    rs: List[np.ndarray] = []
    ends: List[int] = []
    for end in range(w - 1, n):
        if y_cls[end] < 0:
            continue
        if not np.isfinite(y_reg[end]).all():
            continue
        window = feats[end - w + 1 : end + 1]
        if not np.isfinite(window).all():
            continue
        xs.append(window)
        cs.append(int(y_cls[end]))
        rs.append(y_reg[end])
        ends.append(int(end))
    if not xs:
        empty_x = np.zeros((0, w, len(feature_cols)), dtype=np.float32)
        return empty_x, np.zeros((0,), dtype=np.int64), np.zeros((0, 2), dtype=np.float32), np.zeros((0,), dtype=np.int64)
    return (
        np.stack(xs, axis=0).astype(np.float32),
        np.asarray(cs, dtype=np.int64),
        np.stack(rs, axis=0).astype(np.float32),
        np.asarray(ends, dtype=np.int64),
    )


all_x, all_y, all_r, all_ends = build_windows(
    labeled, window_len=int(CONFIG["window_len"]), feature_cols=FEATURE_COLS
)
print("windows", all_x.shape, "class mix", {n: float(np.mean(all_y == i)) for i, n in enumerate(LABEL_NAMES)})


def _split_windows(split_idx: np.ndarray) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    allowed = set(int(i) for i in split_idx.tolist())
    keep = np.array([int(e) in allowed for e in all_ends], dtype=bool)
    return all_x[keep], all_y[keep], all_r[keep]


x_train, y_train, r_train = _split_windows(splits["train"])
x_val, y_val, r_val = _split_windows(splits["val"])
x_test, y_test, r_test = _split_windows(splits["test"])
print("train", x_train.shape, "val", x_val.shape, "test", x_test.shape)

feat_mean = x_train.mean(axis=(0, 1), keepdims=True)
feat_std = x_train.std(axis=(0, 1), keepdims=True)
feat_std = np.where(feat_std < 1e-6, 1.0, feat_std)
reg_mean = r_train.mean(axis=0, keepdims=True)
reg_std = r_train.std(axis=0, keepdims=True)
reg_std = np.where(reg_std < 1e-6, 1.0, reg_std)


def _scale_x(x: np.ndarray) -> np.ndarray:
    return ((x - feat_mean) / feat_std).astype(np.float32)


def _scale_r(r: np.ndarray) -> np.ndarray:
    return ((r - reg_mean) / reg_std).astype(np.float32)


train_ds = SequenceDataset(_scale_x(x_train), y_train, _scale_r(r_train))
val_ds = SequenceDataset(_scale_x(x_val), y_val, _scale_r(r_val))
test_ds = SequenceDataset(_scale_x(x_test), y_test, _scale_r(r_test))
train_loader = DataLoader(train_ds, batch_size=int(CONFIG["batch_size"]), shuffle=True, drop_last=False)
val_loader = DataLoader(val_ds, batch_size=int(CONFIG["batch_size"]), shuffle=False)
test_loader = DataLoader(test_ds, batch_size=int(CONFIG["batch_size"]), shuffle=False)


windows (129518, 60, 6) class mix {'LONG': 0.17148967711051746, 'SHORT': 0.17563581895952687, 'NO_TRADE': 0.6528745039299557}
train (90638, 60, 6) val (19436, 60, 6) test (19432, 60, 6)


In [ ]:
# CELL: MODEL
class OpportunityEncoder(nn.Module):
    """Single 10m encoder: 3-class opportunity plus upside/downside."""

    def __init__(
        self,
        n_features: int,
        *,
        d_model: int = 64,
        nhead: int = 4,
        num_layers: int = 2,
        dropout: float = 0.1,
        ff_mult: int = 4,
        n_classes: int = 3,
    ) -> None:
        super().__init__()
        self.input_proj = nn.Linear(int(n_features), int(d_model))
        layer = nn.TransformerEncoderLayer(
            d_model=int(d_model),
            nhead=int(nhead),
            dim_feedforward=int(d_model) * int(ff_mult),
            dropout=float(dropout),
            batch_first=True,
            activation="gelu",
        )
        self.encoder = nn.TransformerEncoder(layer, num_layers=int(num_layers))
        self.norm = nn.LayerNorm(int(d_model))
        self.cls_head = nn.Linear(int(d_model), int(n_classes))
        self.upside_head = nn.Linear(int(d_model), 1)
        self.downside_head = nn.Linear(int(d_model), 1)

    def forward(self, x: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        h = self.norm(self.encoder(self.input_proj(x)))
        pooled = h[:, -1, :]
        logits = self.cls_head(pooled)
        upside = self.upside_head(pooled).squeeze(-1)
        downside = self.downside_head(pooled).squeeze(-1)
        return logits, upside, downside


model = OpportunityEncoder(
    len(FEATURE_COLS),
    d_model=int(CONFIG["d_model"]),
    nhead=int(CONFIG["nhead"]),
    num_layers=int(CONFIG["num_layers"]),
    dropout=float(CONFIG["dropout"]),
    ff_mult=int(CONFIG["ff_mult"]),
).to(DEVICE)
print(model)
print("params", sum(p.numel() for p in model.parameters()))


OpportunityEncoder(
  (input_proj): Linear(in_features=6, out_features=64, bias=True)
  (encoder): TransformerEncoder(
    (layers): ModuleList(
      (0-1): 2 x TransformerEncoderLayer(
        (self_attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=64, out_features=64, bias=True)
        )
        (linear1): Linear(in_features=64, out_features=256, bias=True)
        (dropout): Dropout(p=0.1, inplace=False)
        (linear2): Linear(in_features=256, out_features=64, bias=True)
        (norm1): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
        (norm2): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
        (dropout1): Dropout(p=0.1, inplace=False)
        (dropout2): Dropout(p=0.1, inplace=False)
      )
    )
  )
  (norm): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
  (cls_head): Linear(in_features=64, out_features=3, bias=True)
  (upside_head): Linear(in_features=64, out_features=1, bias=True)
  (downside_head): Li

In [ ]:
# CELL: TRAIN
def batch_loss(
    logits: torch.Tensor,
    pred_up: torch.Tensor,
    pred_dn: torch.Tensor,
    y_cls: torch.Tensor,
    y_reg: torch.Tensor,
) -> torch.Tensor:
    valid = y_cls >= 0
    if not bool(valid.any()):
        return logits.sum() * 0.0
    ce = nn.functional.cross_entropy(logits[valid], y_cls[valid])
    sm = nn.SmoothL1Loss()
    reg = sm(pred_up[valid], y_reg[valid, 0]) + sm(pred_dn[valid], y_reg[valid, 1])
    return float(CONFIG["cls_loss_weight"]) * ce + float(CONFIG["reg_loss_weight"]) * reg


def run_epoch(loader: DataLoader, *, train: bool) -> float:
    if train:
        model.train()
    else:
        model.eval()
    total = 0.0
    n = 0
    ctx = torch.enable_grad() if train else torch.no_grad()
    with ctx:
        for xb, yb, rb in loader:
            xb = xb.to(DEVICE)
            yb = yb.to(DEVICE)
            rb = rb.to(DEVICE)
            logits, up, dn = model(xb)
            loss = batch_loss(logits, up, dn, yb, rb)
            if train:
                optimizer.zero_grad(set_to_none=True)
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
            total += float(loss.item()) * int(xb.size(0))
            n += int(xb.size(0))
    return total / max(n, 1)


optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=float(CONFIG["lr"]),
    weight_decay=float(CONFIG["weight_decay"]),
)
best_state = None
best_val = float("inf")
stale = 0
history: List[Dict[str, float]] = []
for epoch in range(1, int(CONFIG["epochs"]) + 1):
    tr = run_epoch(train_loader, train=True)
    va = run_epoch(val_loader, train=False)
    history.append({"epoch": epoch, "train_loss": tr, "val_loss": va})
    print(f"epoch {epoch:02d}  train={tr:.4f}  val={va:.4f}")
    if va < best_val - 1e-4:
        best_val = va
        stale = 0
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
    else:
        stale += 1
        if stale >= int(CONFIG["patience"]):
            print("early stopping")
            break
if best_state is not None:
    model.load_state_dict(best_state)
print("best val loss", best_val)


epoch 01  train=0.9800  val=0.9751


epoch 02  train=0.9734  val=0.9756


epoch 03  train=0.9714  val=0.9760


epoch 04  train=0.9703  val=0.9759


epoch 05  train=0.9690  val=0.9721


...

epoch 07  train=0.9675  val=0.9776


epoch 08  train=0.9668  val=0.9731


epoch 09  train=0.9660  val=0.9795


epoch 10  train=0.9660  val=0.9738


epoch 11  train=0.9650  val=0.9743
early stopping
best val loss 0.972128979559665


In [ ]:
# CELL: EVAL
def predict_loader(loader: DataLoader) -> Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    model.eval()
    logits_l: List[np.ndarray] = []
    up_l: List[np.ndarray] = []
    dn_l: List[np.ndarray] = []
    y_l: List[np.ndarray] = []
    r_l: List[np.ndarray] = []
    with torch.no_grad():
        for xb, yb, rb in loader:
            xb = xb.to(DEVICE)
            logits, up, dn = model(xb)
            logits_l.append(logits.cpu().numpy())
            up_l.append(up.cpu().numpy())
            dn_l.append(dn.cpu().numpy())
            y_l.append(yb.numpy())
            r_l.append(rb.numpy())
    return (
        np.concatenate(logits_l, axis=0),
        np.concatenate(up_l, axis=0),
        np.concatenate(dn_l, axis=0),
        np.concatenate(y_l, axis=0),
        np.concatenate(r_l, axis=0),
    )


def _unscale_reg(pred: np.ndarray, col: int) -> np.ndarray:
    return pred * float(reg_std[0, col]) + float(reg_mean[0, col])


def _unscale_true(arr: np.ndarray) -> np.ndarray:
    return arr * reg_std + reg_mean


def confusion(y_true: np.ndarray, y_pred: np.ndarray, n_classes: int = 3) -> np.ndarray:
    mat = np.zeros((n_classes, n_classes), dtype=np.int64)
    for t, p in zip(y_true.tolist(), y_pred.tolist()):
        if 0 <= int(t) < n_classes and 0 <= int(p) < n_classes:
            mat[int(t), int(p)] += 1
    return mat


def class_precision(mat: np.ndarray, cls: int) -> float:
    col = mat[:, cls]
    denom = float(col.sum())
    if denom <= 0:
        return float("nan")
    return float(mat[cls, cls]) / denom


def balanced_accuracy(mat: np.ndarray) -> float:
    recalls = []
    for i in range(mat.shape[0]):
        denom = float(mat[i].sum())
        recalls.append(float(mat[i, i]) / denom if denom else float("nan"))
    finite = [r for r in recalls if np.isfinite(r)]
    return float(np.mean(finite)) if finite else float("nan")


def corr(a: np.ndarray, b: np.ndarray) -> float:
    if a.size < 3:
        return float("nan")
    if float(np.std(a)) < 1e-12 or float(np.std(b)) < 1e-12:
        return float("nan")
    return float(np.corrcoef(a, b)[0, 1])


def signal_expectancy(
    y_true: np.ndarray,
    y_pred: np.ndarray,
    frame: pd.DataFrame,
    ends: np.ndarray,
    split_idx: np.ndarray,
    pred_cls: np.ndarray,
) -> Dict[str, float]:
    allowed = set(int(i) for i in split_idx.tolist())
    keep = np.array([int(e) in allowed for e in all_ends], dtype=bool)
    row_idx = all_ends[keep]
    cost = float(CONFIG["round_trip_cost_pct"])
    out: Dict[str, float] = {}
    for cls, name, pnl_col in (
        (LABEL_LONG, "LONG", "long_pnl_pct"),
        (LABEL_SHORT, "SHORT", "short_pnl_pct"),
    ):
        take = pred_cls == cls
        if not bool(take.any()):
            out[f"{name}_n"] = 0.0
            out[f"{name}_expectancy"] = float("nan")
            continue
        pnl = frame.loc[row_idx[take], pnl_col].to_numpy(dtype=np.float64) - cost
        finite = np.isfinite(pnl)
        out[f"{name}_n"] = float(finite.sum())
        out[f"{name}_expectancy"] = float(np.mean(pnl[finite])) if finite.any() else float("nan")
    return out


logits_test, up_s, dn_s, y_true, r_true_s = predict_loader(test_loader)
pred_cls = logits_test.argmax(axis=1)
up_hat = _unscale_reg(up_s, 0)
dn_hat = _unscale_reg(dn_s, 1)
r_true = _unscale_true(r_true_s)
cm = confusion(y_true, pred_cls)
print("test confusion rows=true cols=pred LONG/SHORT/NO_TRADE")
print(cm)
long_p = class_precision(cm, LABEL_LONG)
short_p = class_precision(cm, LABEL_SHORT)
bacc = balanced_accuracy(cm)
up_c = corr(up_hat, r_true[:, 0])
dn_c = corr(dn_hat, r_true[:, 1])
exp_report = signal_expectancy(
    y_true, pred_cls, labeled, all_ends, splits["test"], pred_cls
)
eval_report = {
    "n_test": int(len(y_true)),
    "confusion": cm.tolist(),
    "precision_LONG": long_p,
    "precision_SHORT": short_p,
    "balanced_accuracy": bacc,
    "upside_corr": up_c,
    "downside_corr": dn_c,
    "signal_expectancy": exp_report,
    "note": (
        "Balanced accuracy is reported and is not the acceptance number; "
        "NO_TRADE is the large class. Use LONG/SHORT precision and expectancy."
    ),
}
print(json.dumps(eval_report, indent=2, default=float))


test confusion rows=true cols=pred LONG/SHORT/NO_TRADE
[[   13     0  3428]
 [    7     0  3438]
 [    6     0 12540]]
{
  "n_test": 19432,
  "confusion": [
    [
      13,
      0,
      3428
    ],
    [
      7,
      0,
      3438
    ],
    [
      6,
      0,
      12540
    ]
  ],
  "precision_LONG": 0.5,
  "precision_SHORT": NaN,
  "balanced_accuracy": 0.3344332438144629,
  "upside_corr": 0.12344369590851521,
  "downside_corr": 0.14560067758720174,
  "signal_expectancy": {
    "LONG_n": 25.0,
    "LONG_expectancy": -0.00087199749588296,
    "SHORT_n": 0.0,
    "SHORT_expectancy": NaN
  },
  "note": "Balanced accuracy is reported and is not the acceptance number; NO_TRADE is the large class. Use LONG/SHORT precision and expectancy."
}


In [ ]:
# CELL: WALK_FORWARD
wf_rows: List[Dict[str, Any]] = []
for fold_i, (tr_idx, va_idx) in enumerate(wf_folds, start=1):
    x_tr, y_tr, r_tr = _split_windows(tr_idx)
    x_va, y_va, r_va = _split_windows(va_idx)
    if len(x_tr) < 32 or len(x_va) < 8:
        continue
    mean = x_tr.mean(axis=(0, 1), keepdims=True)
    std = np.where(x_tr.std(axis=(0, 1), keepdims=True) < 1e-6, 1.0, x_tr.std(axis=(0, 1), keepdims=True))
    rmean = r_tr.mean(axis=0, keepdims=True)
    rstd = np.where(r_tr.std(axis=0, keepdims=True) < 1e-6, 1.0, r_tr.std(axis=0, keepdims=True))
    fold_model = OpportunityEncoder(
        len(FEATURE_COLS),
        d_model=int(CONFIG["d_model"]),
        nhead=int(CONFIG["nhead"]),
        num_layers=int(CONFIG["num_layers"]),
        dropout=float(CONFIG["dropout"]),
        ff_mult=int(CONFIG["ff_mult"]),
    ).to(DEVICE)
    opt = torch.optim.AdamW(
        fold_model.parameters(),
        lr=float(CONFIG["lr"]),
        weight_decay=float(CONFIG["weight_decay"]),
    )
    tr_loader = DataLoader(
        SequenceDataset(((x_tr - mean) / std).astype(np.float32), y_tr, ((r_tr - rmean) / rstd).astype(np.float32)),
        batch_size=int(CONFIG["batch_size"]),
        shuffle=True,
    )
    va_loader = DataLoader(
        SequenceDataset(((x_va - mean) / std).astype(np.float32), y_va, ((r_va - rmean) / rstd).astype(np.float32)),
        batch_size=int(CONFIG["batch_size"]),
        shuffle=False,
    )
    best = float("inf")
    stale = 0
    best_w = None
    for epoch in range(1, min(12, int(CONFIG["epochs"])) + 1):
        fold_model.train()
        for xb, yb, rb in tr_loader:
            xb, yb, rb = xb.to(DEVICE), yb.to(DEVICE), rb.to(DEVICE)
            logits, up, dn = fold_model(xb)
            loss = batch_loss(logits, up, dn, yb, rb)
            opt.zero_grad(set_to_none=True)
            loss.backward()
            opt.step()
        fold_model.eval()
        total = 0.0
        n = 0
        with torch.no_grad():
            for xb, yb, rb in va_loader:
                xb, yb, rb = xb.to(DEVICE), yb.to(DEVICE), rb.to(DEVICE)
                logits, up, dn = fold_model(xb)
                loss = batch_loss(logits, up, dn, yb, rb)
                total += float(loss.item()) * int(xb.size(0))
                n += int(xb.size(0))
        va = total / max(n, 1)
        if va < best:
            best = va
            stale = 0
            best_w = {k: v.detach().cpu().clone() for k, v in fold_model.state_dict().items()}
        else:
            stale += 1
            if stale >= 3:
                break
    wf_rows.append({"fold": fold_i, "n_train": int(len(x_tr)), "n_val": int(len(x_va)), "best_val": best})
    del fold_model, best_w
print(json.dumps({"walk_forward": wf_rows}, indent=2))


{
  "walk_forward": [
    {
      "fold": 1,
      "n_train": 27460,
      "n_val": 27538,
      "best_val": 0.95504779982856
    },
    {
      "fold": 2,
      "n_train": 55001,
      "n_val": 27538,
      "best_val": 0.9899022125177925
    },
    {
      "fold": 3,
      "n_train": 82542,
      "n_val": 27538,
      "best_val": 0.9726020430538491
    }
  ]
}


In [ ]:
# CELL: EXPORT
EXPORT_DIR.mkdir(parents=True, exist_ok=True)
model_cpu = OpportunityEncoder(
    len(FEATURE_COLS),
    d_model=int(CONFIG["d_model"]),
    nhead=int(CONFIG["nhead"]),
    num_layers=int(CONFIG["num_layers"]),
    dropout=float(CONFIG["dropout"]),
    ff_mult=int(CONFIG["ff_mult"]),
)
model_cpu.load_state_dict(model.state_dict())
model_cpu.eval()
dummy = torch.zeros(1, int(CONFIG["window_len"]), len(FEATURE_COLS), dtype=torch.float32)
onnx_path = EXPORT_DIR / "btcusd_opportunity_10m.onnx"
torch.onnx.export(
    model_cpu,
    dummy,
    str(onnx_path),
    input_names=["features"],
    output_names=["trade_logits", "upside", "downside"],
    dynamic_axes={
        "features": {0: "batch"},
        "trade_logits": {0: "batch"},
        "upside": {0: "batch"},
        "downside": {0: "batch"},
    },
    opset_version=17,
)
weights_path = EXPORT_DIR / "opportunity_10m.pt"
torch.save(model.state_dict(), weights_path)
metadata = {
    "model_family": "jacksparrow_opportunity_10m",
    "timeframe": "10m",
    "horizon_bars": int(CONFIG["horizon_bars"]),
    "horizon_minutes": 30,
    "tp_atr_mult": float(CONFIG["tp_atr_mult"]),
    "sl_atr_mult": float(CONFIG["sl_atr_mult"]),
    "round_trip_cost_pct": float(CONFIG["round_trip_cost_pct"]),
    "window_len": int(CONFIG["window_len"]),
    "feature_cols": list(FEATURE_COLS),
    "label_names": list(LABEL_NAMES),
    "feat_mean": feat_mean.reshape(-1).tolist(),
    "feat_std": feat_std.reshape(-1).tolist(),
    "reg_mean": reg_mean.reshape(-1).tolist(),
    "reg_std": reg_std.reshape(-1).tolist(),
    "ready_to_promote": False,
    "promote_reason": "research_opportunity_10m_not_live",
    "eval": eval_report,
    "label_gate": label_report,
    "walk_forward": wf_rows,
    "history": history,
}
(EXPORT_DIR / "metadata.json").write_text(json.dumps(metadata, indent=2, default=float), encoding="utf-8")
(EXPORT_DIR / "eval_report.json").write_text(json.dumps(eval_report, indent=2, default=float), encoding="utf-8")
print("wrote", onnx_path)
print("wrote", weights_path)
print("ready_to_promote=false")
try:
    from google.colab import files

    zip_path = EXPORT_DIR.with_suffix(".zip")
    import shutil

    shutil.make_archive(str(EXPORT_DIR), "zip", root_dir=str(EXPORT_DIR))
    files.download(str(zip_path) if zip_path.exists() else str(EXPORT_DIR) + ".zip")
except ImportError:
    print("Not Colab; files remain under", EXPORT_DIR)


ModuleNotFoundError: No module named 'onnxscript'

*File Operation*: `ls` on `content`

*File Operation*: `ls` on `/content/export/opportunity_10m`

**Session Created**: 2026-10-08 15:31:05
- Endpoint: `m-s-kkb-usw1b2-11o7v1809qz8k`
- Hardware: `NONE`

In [ ]:
# CELL: ENV
# Colab ships pandas/numpy; install tabular model extras.
%pip install -q --upgrade-strategy only-if-needed pyarrow requests scikit-learn xgboost lightgbm catboost

import json
import math
import pickle
import random
from concurrent.futures import ThreadPoolExecutor, as_completed
from datetime import datetime, timedelta, timezone
from pathlib import Path
from typing import Any, Dict, List, Optional, Sequence, Tuple

import numpy as np
import pandas as pd
from catboost import CatBoostClassifier, CatBoostRegressor
from lightgbm import LGBMClassifier, LGBMRegressor
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.linear_model import LogisticRegression, Ridge
from xgboost import XGBClassifier, XGBRegressor

print("sklearn / xgboost / lightgbm / catboost ready")


   ━━━━━━━━━━━╺━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.0/97.1 MB 194.0 MB/s eta 0:00:01

   ━━━━━━━━━━━━━━━━━━━━━━━╸━━━━━━━━━━━━━━━━ 57.2/97.1 MB 193.6 MB/s eta 0:00:01

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╸━━━━ 86.8/97.1 MB 117.9 MB/s eta 0:00:01

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╸ 97.1/97.1 MB 123.5 MB/s eta 0:00:01

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╸ 97.1/97.1 MB 123.5 MB/s eta 0:00:01

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╸ 97.1/97.1 MB 123.5 MB/s eta 0:00:01

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╸ 97.1/97.1 MB 123.5 MB/s eta 0:00:01

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╸ 97.1/97.1 MB 123.5 MB/s eta 0:00:01

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 8.3 MB/s eta 0:00:00


sklearn / xgboost / lightgbm / catboost ready


In [ ]:
# CELL: CONFIG
CONFIG: Dict[str, Any] = {
    "symbol": "BTCUSD",
    "base_url": "https://api.india.delta.exchange",
    "history_days": 900,
    "resolution_5m": "5m",
    "seed": 42,
    "tp_atr_mult": 1.5,
    "sl_atr_mult": 1.0,
    "horizon_bars": 3,
    "atr_period": 14,
    "window_len": 60,
    "round_trip_cost_pct": 0.0016,
    "minority_rate": 0.15,
    "ambiguous_win_max": 0.50,
    "train_fraction": 0.70,
    "val_fraction": 0.15,
    "purge_bars": 3,
    "n_estimators": 200,
    "max_depth": 8,
    "learning_rate": 0.08,
    "min_leaf": 40,
    "ensemble": ["LR", "RF", "XGB", "LGBM", "CatBoost"],
    "export_dir": "export/opportunity_10m",
    "ready_to_promote": False,
}

FEATURE_COLS: Tuple[str, ...] = (
    "ret_1",
    "atr_pct",
    "rsi",
    "ema_dist_atr",
    "candle_body",
    "volume_z",
)
LABEL_NAMES: Tuple[str, ...] = ("LONG", "SHORT", "NO_TRADE")
LABEL_LONG, LABEL_SHORT, LABEL_NO_TRADE = 0, 1, 2
print(json.dumps(CONFIG, indent=2))


{
  "symbol": "BTCUSD",
  "base_url": "https://api.india.delta.exchange",
  "history_days": 900,
  "resolution_5m": "5m",
  "seed": 42,
  "tp_atr_mult": 1.5,
  "sl_atr_mult": 1.0,
  "horizon_bars": 3,
  "atr_period": 14,
  "window_len": 60,
  "round_trip_cost_pct": 0.0016,
  "minority_rate": 0.15,
  "ambiguous_win_max": 0.5,
  "train_fraction": 0.7,
  "val_fraction": 0.15,
  "purge_bars": 3,
  "n_estimators": 200,
  "max_depth": 8,
  "learning_rate": 0.08,
  "min_leaf": 40,
  "ensemble": [
    "LR",
    "RF",
    "XGB",
    "LGBM",
    "CatBoost"
  ],
  "export_dir": "export/opportunity_10m",
  "ready_to_promote": false
}


In [ ]:
# CELL: SEED
def set_seed(seed: int) -> None:
    random.seed(int(seed))
    np.random.seed(int(seed))


set_seed(int(CONFIG["seed"]))
print("seed", CONFIG["seed"])


seed 42


In [ ]:
# CELL: FETCH
DELTA_CANDLE_PAGE_BARS = 2000
REQUEST_DELAY_SECONDS = 0.2
EMPTY_PAGE_RETRIES = 3
MIN_OHLCV_COMPLETENESS = 0.95
_RESOLUTION_MINUTES = {"5m": 5, "10m": 10}


def _bar_seconds(resolution: str) -> int:
    if resolution not in _RESOLUTION_MINUTES:
        raise ValueError(f"Unsupported resolution {resolution!r}")
    return int(_RESOLUTION_MINUTES[resolution]) * 60


def _parse_candle_rows(payload: Any) -> List[Dict[str, Any]]:
    if not isinstance(payload, dict):
        return []
    result = payload.get("result", [])
    if isinstance(result, dict):
        result = result.get("candles", []) or []
    if not isinstance(result, list):
        return []
    return [row for row in result if isinstance(row, dict) and "time" in row]


def _candle_epoch(row: Dict[str, Any]) -> Optional[int]:
    try:
        return int(row["time"])
    except (KeyError, TypeError, ValueError):
        return None


def _request_candle_page(
    url: str,
    *,
    symbol: str,
    resolution: str,
    start_ts: int,
    end_ts: int,
) -> List[Dict[str, Any]]:
    import requests

    last_error: Optional[Exception] = None
    for attempt in range(EMPTY_PAGE_RETRIES):
        try:
            response = requests.get(
                url,
                params={
                    "symbol": symbol,
                    "resolution": resolution,
                    "start": int(start_ts),
                    "end": int(end_ts),
                },
                timeout=30,
            )
            response.raise_for_status()
            payload = response.json()
        except Exception as exc:
            last_error = exc
            time.sleep(REQUEST_DELAY_SECONDS * (attempt + 1))
            continue
        if payload.get("success") is False:
            last_error = RuntimeError(f"Delta candles error for {symbol}: {payload}")
            time.sleep(REQUEST_DELAY_SECONDS * (attempt + 1))
            continue
        rows = _parse_candle_rows(payload)
        if rows:
            return rows
        last_error = None
        time.sleep(REQUEST_DELAY_SECONDS * (attempt + 1))
    if last_error is not None:
        raise last_error
    return []


def _rows_to_frame(all_rows: List[Dict[str, Any]]) -> pd.DataFrame:
    df = pd.DataFrame(all_rows)
    expected_cols = ["time", "open", "high", "low", "close", "volume"]
    df = df[[c for c in expected_cols if c in df.columns]]
    df["time"] = pd.to_datetime(df["time"], unit="s", utc=True)
    df = df.drop_duplicates(subset="time").sort_values("time").reset_index(drop=True)
    for col in ["open", "high", "low", "close", "volume"]:
        if col in df.columns:
            df[col] = df[col].astype(float)
    return df


def fetch_candles(
    symbol: str,
    resolution: str,
    start_ts: int,
    end_ts: int,
    base_url: str,
    *,
    page_bars: int = DELTA_CANDLE_PAGE_BARS,
) -> pd.DataFrame:
    url = f"{base_url.rstrip('/')}/v2/history/candles"
    bar_seconds = _bar_seconds(resolution)
    page_bars = max(1, int(page_bars))
    start_ts, end_ts = int(start_ts), int(end_ts)
    if end_ts <= start_ts:
        raise ValueError("end_ts must be greater than start_ts")
    all_rows: List[Dict[str, Any]] = []
    cursor_end = end_ts
    seen_oldest: Optional[int] = None
    max_pages = max(2, (end_ts - start_ts) // bar_seconds + 5)
    for _ in range(max_pages):
        if cursor_end <= start_ts:
            break
        page_start = max(start_ts, cursor_end - page_bars * bar_seconds)
        if page_start >= cursor_end:
            break
        rows = _request_candle_page(
            url,
            symbol=symbol,
            resolution=resolution,
            start_ts=page_start,
            end_ts=cursor_end,
        )
        if not rows:
            break
        epochs = [e for e in (_candle_epoch(r) for r in rows) if e is not None]
        if not epochs:
            break
        oldest = min(epochs)
        if seen_oldest is not None and oldest >= seen_oldest:
            break
        seen_oldest = oldest
        all_rows.extend(rows)
        cursor_end = oldest - 1
        time.sleep(REQUEST_DELAY_SECONDS)
    if not all_rows:
        raise RuntimeError(f"No candle data returned for symbol={symbol}")
    return _rows_to_frame(all_rows)


def validate_ohlcv_completeness(
    df: pd.DataFrame,
    resolution: str,
    *,
    min_completeness: float = MIN_OHLCV_COMPLETENESS,
    symbol: str = "",
) -> Dict[str, Any]:
    if df is None or df.empty or "time" not in df.columns:
        raise ValueError("Cannot validate completeness on an empty OHLCV frame")
    bar_seconds = _bar_seconds(resolution)
    time_col = df["time"]
    epochs = (pd.to_datetime(time_col, utc=True).astype("int64") // 10**9).tolist()
    if len(epochs) < 2:
        raise ValueError("Need at least 2 OHLCV bars to validate completeness")
    gaps = 0
    for prev, cur in zip(epochs, epochs[1:]):
        if int(cur) - int(prev) > bar_seconds * 1.5:
            gaps += 1
    completeness = 1.0 - (gaps / max(1, len(epochs) - 1))
    report = {
        "rows": len(epochs),
        "gaps": int(gaps),
        "completeness": round(float(completeness), 4),
    }
    if completeness < min_completeness:
        label = f"{symbol} {resolution}".strip()
        raise ValueError(
            f"OHLCV completeness {completeness:.1%} below {min_completeness:.0%} "
            f"for {label or 'candles'}"
        )
    return report


end_dt = datetime.now(timezone.utc)
start_dt = end_dt - timedelta(days=int(CONFIG["history_days"]))
print(
    f"Fetching {CONFIG['symbol']} 5m OHLCV for {CONFIG['history_days']}d "
    f"from {CONFIG['base_url']}..."
)
df5m = fetch_candles(
    str(CONFIG["symbol"]),
    str(CONFIG["resolution_5m"]),
    int(start_dt.timestamp()),
    int(end_dt.timestamp()),
    str(CONFIG["base_url"]),
)
ohlcv_report = validate_ohlcv_completeness(
    df5m, "5m", symbol=str(CONFIG["symbol"])
)
print(
    f"OHLCV {CONFIG['symbol']} 5m: {ohlcv_report['rows']} bars, "
    f"completeness {ohlcv_report['completeness']:.1%}, gaps={ohlcv_report['gaps']}"
)


Fetching BTCUSD 5m OHLCV for 900d from https://api.india.delta.exchange...


NameError: name 'time' is not defined

In [ ]:
# CELL: BARS_10M
def build_10m_ohlcv_from_5m(df5m: pd.DataFrame) -> pd.DataFrame:
    """Exchange-aligned 10m bars from two closed 5m candles. Drop a partial last bucket."""
    src = df5m.copy()
    src["time"] = pd.to_datetime(src["time"], utc=True)
    required = ("open", "high", "low", "close", "volume")
    missing = [c for c in required if c not in src.columns]
    if missing:
        raise ValueError(f"5m OHLCV missing columns: {missing}")
    if src.empty:
        return src.iloc[0:0].copy()
    grouped = src.set_index("time").sort_index()
    agg = grouped.resample("10min", label="left", closed="left").agg(
        {
            "open": "first",
            "high": "max",
            "low": "min",
            "close": "last",
            "volume": "sum",
        }
    )
    counts = grouped["close"].resample("10min", label="left", closed="left").count()
    complete = counts >= 2
    agg = agg.loc[complete].dropna(subset=["open", "high", "low", "close"])
    out = agg.reset_index()
    return out.sort_values("time").reset_index(drop=True)


bars_10m = build_10m_ohlcv_from_5m(df5m)
print(f"10m bars={len(bars_10m)} span={bars_10m['time'].iloc[0]} → {bars_10m['time'].iloc[-1]}")


NameError: name 'df5m' is not defined

In [ ]:
# CELL: FEATURES
def _wilder_rsi(close: pd.Series, period: int = 14) -> pd.Series:
    delta = close.diff()
    gain = delta.clip(lower=0.0)
    loss = -delta.clip(upper=0.0)
    avg_gain = gain.ewm(alpha=1.0 / period, min_periods=period, adjust=False).mean()
    avg_loss = loss.ewm(alpha=1.0 / period, min_periods=period, adjust=False).mean()
    rs = avg_gain / avg_loss.replace(0.0, np.nan)
    return 100.0 - (100.0 / (1.0 + rs))


def add_opportunity_features(
    df: pd.DataFrame,
    *,
    atr_period: int = 14,
    rsi_period: int = 14,
    ema_span: int = 20,
    vol_window: int = 20,
) -> pd.DataFrame:
    """Causal 10m features. ATR(14) is also stored for the label path."""
    out = df.copy()
    close = out["close"].astype(np.float64)
    high = out["high"].astype(np.float64)
    low = out["low"].astype(np.float64)
    open_ = out["open"].astype(np.float64)
    volume = out["volume"].astype(np.float64)
    prev_close = close.shift(1)
    tr = pd.concat(
        [
            high - low,
            (high - prev_close).abs(),
            (low - prev_close).abs(),
        ],
        axis=1,
    ).max(axis=1)
    atr = tr.rolling(int(atr_period), min_periods=int(atr_period)).mean()
    out["atr"] = atr
    out["ret_1"] = close.pct_change()
    out["atr_pct"] = atr / close.replace(0.0, np.nan)
    out["rsi"] = _wilder_rsi(close, period=int(rsi_period))
    ema = close.ewm(span=int(ema_span), adjust=False, min_periods=int(ema_span)).mean()
    out["ema_dist_atr"] = (close - ema) / atr.replace(0.0, np.nan)
    out["candle_body"] = (close - open_) / atr.replace(0.0, np.nan)
    vol_mean = volume.rolling(int(vol_window), min_periods=int(vol_window)).mean()
    vol_std = volume.rolling(int(vol_window), min_periods=int(vol_window)).std()
    out["volume_z"] = (volume - vol_mean) / vol_std.replace(0.0, np.nan)
    return out


featured = add_opportunity_features(
    bars_10m, atr_period=int(CONFIG["atr_period"])
)
print("feature null rates")
print(featured[list(FEATURE_COLS)].isna().mean().round(4))


NameError: name 'bars_10m' is not defined

In [ ]:
# CELL: LABELS
# Opportunity labels on a 10-minute series. Path is bars t+1 .. t+horizon only.

import numpy as np
import pandas as pd

_EPS = 1e-9
HORIZON_BARS_DEFAULT = 3
ATR_PERIOD_DEFAULT = 14
TP_ATR_MULT_DEFAULT = 1.5
SL_ATR_MULT_DEFAULT = 1.0

TP_FIRST = 0
SL_FIRST = 1
NEITHER = 2
AMBIGUOUS = 3

LABEL_LONG = 0
LABEL_SHORT = 1
LABEL_NO_TRADE = 2
LABEL_UNLABELED = -1
LABEL_NAMES = ("LONG", "SHORT", "NO_TRADE")


def tp_sl_from_path(
    fwd_high: np.ndarray,
    fwd_low: np.ndarray,
    entry: float,
    atr: float,
    tp_mult: float,
    sl_mult: float,
    side: str,
) -> int:
    """Walk future highs/lows. Same-bar TP and SL is AMBIGUOUS, not a win."""
    high = np.asarray(fwd_high, dtype=np.float64).ravel()
    low = np.asarray(fwd_low, dtype=np.float64).ravel()
    if high.size == 0 or high.size != low.size:
        return int(NEITHER)
    denom = max(float(atr), _EPS)
    entry_f = float(entry)
    if str(side).strip().upper() == "SHORT":
        tp_level = entry_f - float(tp_mult) * denom
        sl_level = entry_f + float(sl_mult) * denom
        tp_hit = low <= tp_level
        sl_hit = high >= sl_level
    else:
        tp_level = entry_f + float(tp_mult) * denom
        sl_level = entry_f - float(sl_mult) * denom
        tp_hit = high >= tp_level
        sl_hit = low <= sl_level
    for j in range(high.size):
        hit_tp = bool(tp_hit[j])
        hit_sl = bool(sl_hit[j])
        if hit_tp and hit_sl:
            return int(AMBIGUOUS)
        if hit_tp:
            return int(TP_FIRST)
        if hit_sl:
            return int(SL_FIRST)
    return int(NEITHER)


def _side_win(code: int) -> int:
    return 1 if int(code) == int(TP_FIRST) else 0


def _causal_atr(df: pd.DataFrame, period: int = ATR_PERIOD_DEFAULT) -> np.ndarray:
    if "atr" in df.columns:
        atr = df["atr"].to_numpy(dtype=np.float64)
        if np.isfinite(atr).any():
            return atr
    close = df["close"].to_numpy(dtype=np.float64)
    high = df["high"].to_numpy(dtype=np.float64)
    low = df["low"].to_numpy(dtype=np.float64)
    prev = np.concatenate([[close[0]], close[:-1]])
    tr = np.maximum(high - low, np.maximum(np.abs(high - prev), np.abs(low - prev)))
    return pd.Series(tr).rolling(int(period), min_periods=int(period)).mean().to_numpy(
        dtype=np.float64
    )


def compute_opportunity_labels(
    df: pd.DataFrame,
    *,
    tp_mult: float = TP_ATR_MULT_DEFAULT,
    sl_mult: float = SL_ATR_MULT_DEFAULT,
    horizon_bars: int = HORIZON_BARS_DEFAULT,
    atr_period: int = ATR_PERIOD_DEFAULT,
) -> pd.DataFrame:
    """Label each closed 10m bar. Last ``horizon_bars`` rows stay unlabeled."""
    out = df.copy().reset_index(drop=True)
    n = len(out)
    k = int(horizon_bars)
    close = out["close"].to_numpy(dtype=np.float64)
    high = out["high"].to_numpy(dtype=np.float64)
    low = out["low"].to_numpy(dtype=np.float64)
    atr = _causal_atr(out, period=int(atr_period))
    n_valid = max(n - k, 0)

    trade_label = np.full(n, LABEL_UNLABELED, dtype=np.int64)
    long_result = np.full(n, np.nan, dtype=np.float64)
    short_result = np.full(n, np.nan, dtype=np.float64)
    long_code = np.full(n, np.nan, dtype=np.float64)
    short_code = np.full(n, np.nan, dtype=np.float64)
    upside = np.full(n, np.nan, dtype=np.float64)
    downside = np.full(n, np.nan, dtype=np.float64)
    timeout_ret = np.full(n, np.nan, dtype=np.float64)
    long_pnl_pct = np.full(n, np.nan, dtype=np.float64)
    short_pnl_pct = np.full(n, np.nan, dtype=np.float64)

    for i in range(n_valid):
        atr_i = float(atr[i]) if i < len(atr) else float("nan")
        if not np.isfinite(atr_i) or atr_i <= _EPS:
            continue
        entry = float(close[i])
        if not np.isfinite(entry) or abs(entry) <= _EPS:
            continue
        fwd_high = high[i + 1 : i + 1 + k]
        fwd_low = low[i + 1 : i + 1 + k]
        fwd_close = close[i + 1 : i + 1 + k]
        if fwd_high.size < k:
            continue
        l_code = tp_sl_from_path(fwd_high, fwd_low, entry, atr_i, tp_mult, sl_mult, "LONG")
        s_code = tp_sl_from_path(fwd_high, fwd_low, entry, atr_i, tp_mult, sl_mult, "SHORT")
        l_win = _side_win(l_code)
        s_win = _side_win(s_code)
        if l_win == 1 and s_win == 0:
            label = LABEL_LONG
        elif s_win == 1 and l_win == 0:
            label = LABEL_SHORT
        else:
            label = LABEL_NO_TRADE
        trade_label[i] = label
        long_result[i] = float(l_win)
        short_result[i] = float(s_win)
        long_code[i] = float(l_code)
        short_code[i] = float(s_code)
        denom = max(atr_i, _EPS)
        upside[i] = max((float(np.max(fwd_high)) - entry) / denom, 0.0)
        downside[i] = max((entry - float(np.min(fwd_low))) / denom, 0.0)
        end_close = float(fwd_close[-1])
        timeout_ret[i] = (end_close - entry) / entry
        tp_pct = float(tp_mult) * denom / entry
        sl_pct = float(sl_mult) * denom / entry
        if l_code == TP_FIRST:
            long_pnl_pct[i] = tp_pct
        elif l_code == SL_FIRST:
            long_pnl_pct[i] = -sl_pct
        elif l_code == NEITHER:
            long_pnl_pct[i] = timeout_ret[i]
        if s_code == TP_FIRST:
            short_pnl_pct[i] = tp_pct
        elif s_code == SL_FIRST:
            short_pnl_pct[i] = -sl_pct
        elif s_code == NEITHER:
            short_pnl_pct[i] = -timeout_ret[i]

    out["atr"] = atr
    out["trade_label"] = trade_label
    out["long_result"] = long_result
    out["short_result"] = short_result
    out["long_code"] = long_code
    out["short_code"] = short_code
    out["upside"] = upside
    out["downside"] = downside
    out["timeout_ret"] = timeout_ret
    out["long_pnl_pct"] = long_pnl_pct
    out["short_pnl_pct"] = short_pnl_pct
    names = np.array(["UNLABELED"] * n, dtype=object)
    names[trade_label == LABEL_LONG] = "LONG"
    names[trade_label == LABEL_SHORT] = "SHORT"
    names[trade_label == LABEL_NO_TRADE] = "NO_TRADE"
    out["trade_name"] = names
    return out


In [ ]:
# CELL: APPLY_LABELS
labeled = compute_opportunity_labels(
    featured,
    tp_mult=float(CONFIG["tp_atr_mult"]),
    sl_mult=float(CONFIG["sl_atr_mult"]),
    horizon_bars=int(CONFIG["horizon_bars"]),
    atr_period=int(CONFIG["atr_period"]),
)
valid = labeled["trade_label"].to_numpy() >= 0
print(f"labeled rows={int(valid.sum())} / {len(labeled)}")
print(labeled.loc[valid, "trade_name"].value_counts(normalize=True).round(4))


NameError: name 'featured' is not defined

In [ ]:
# CELL: GATE
def train_period_mask(n: int, train_fraction: float) -> np.ndarray:
    cut = int(n * float(train_fraction))
    mask = np.zeros(n, dtype=bool)
    mask[:cut] = True
    return mask


def _path_code_counts(codes: np.ndarray, usable: np.ndarray) -> Dict[str, int]:
    finite = usable & np.isfinite(codes)
    return {
        "TP_FIRST": int(np.sum(finite & (codes == TP_FIRST))),
        "SL_FIRST": int(np.sum(finite & (codes == SL_FIRST))),
        "NEITHER": int(np.sum(finite & (codes == NEITHER))),
        "AMBIGUOUS": int(np.sum(finite & (codes == AMBIGUOUS))),
    }


def summarize_label_gate(
    df: pd.DataFrame,
    *,
    train_fraction: float,
    round_trip_cost_pct: float,
    minority_rate: float,
    ambiguous_win_max: float,
) -> Dict[str, Any]:
    n = len(df)
    in_train = train_period_mask(n, train_fraction)
    labels = df["trade_label"].to_numpy(dtype=np.int64)
    usable = in_train & (labels >= 0)
    n_use = int(usable.sum())
    if n_use == 0:
        raise RuntimeError("Train-period gate has no labeled rows")
    mix = {}
    mix_ok = True
    for cid, name in enumerate(LABEL_NAMES):
        rate = float(np.mean(labels[usable] == cid))
        mix[name] = rate
        if rate < float(minority_rate):
            mix_ok = False
    long_code = df["long_code"].to_numpy(dtype=np.float64)
    short_code = df["short_code"].to_numpy(dtype=np.float64)
    cost = float(round_trip_cost_pct)
    long_pnl = df["long_pnl_pct"].to_numpy(dtype=np.float64) - cost
    short_pnl = df["short_pnl_pct"].to_numpy(dtype=np.float64) - cost

    always_long_mask = usable & np.isfinite(long_pnl)
    always_short_mask = usable & np.isfinite(short_pnl)
    always_in_long = (
        float(np.mean(long_pnl[always_long_mask])) if always_long_mask.any() else float("nan")
    )
    always_in_short = (
        float(np.mean(short_pnl[always_short_mask])) if always_short_mask.any() else float("nan")
    )

    excl_long_mask = usable & (labels == LABEL_LONG) & np.isfinite(long_pnl)
    excl_short_mask = usable & (labels == LABEL_SHORT) & np.isfinite(short_pnl)
    exclusive_long = (
        float(np.mean(long_pnl[excl_long_mask])) if excl_long_mask.any() else float("nan")
    )
    exclusive_short = (
        float(np.mean(short_pnl[excl_short_mask])) if excl_short_mask.any() else float("nan")
    )

    def _ambig_share(codes: np.ndarray) -> float:
        finite = usable & np.isfinite(codes)
        tp = finite & (codes == TP_FIRST)
        amb = finite & (codes == AMBIGUOUS)
        denom = int(tp.sum() + amb.sum())
        if denom == 0:
            return 0.0
        return float(amb.sum()) / float(denom)

    long_ambig = _ambig_share(long_code)
    short_ambig = _ambig_share(short_code)
    reasons: List[str] = []
    if not mix_ok:
        reasons.append("class_mix_below_floor")
    if not (np.isfinite(exclusive_long) and exclusive_long > 0.0):
        reasons.append("exclusive_long_expectancy_not_positive")
    if not (np.isfinite(exclusive_short) and exclusive_short > 0.0):
        reasons.append("exclusive_short_expectancy_not_positive")
    if long_ambig >= float(ambiguous_win_max):
        reasons.append("long_ambiguous_majority")
    if short_ambig >= float(ambiguous_win_max):
        reasons.append("short_ambiguous_majority")
    passed = len(reasons) == 0
    return {
        "passed": passed,
        "n_train_labeled": n_use,
        "class_mix": mix,
        "minority_rate": float(minority_rate),
        "exclusive_long_expectancy_pct": exclusive_long,
        "exclusive_short_expectancy_pct": exclusive_short,
        "always_in_long_expectancy_pct": always_in_long,
        "always_in_short_expectancy_pct": always_in_short,
        "round_trip_cost_pct": cost,
        "long_ambiguous_share": long_ambig,
        "short_ambiguous_share": short_ambig,
        "long_path_codes": _path_code_counts(long_code, usable),
        "short_path_codes": _path_code_counts(short_code, usable),
        "tp_atr_mult": float(CONFIG["tp_atr_mult"]),
        "sl_atr_mult": float(CONFIG["sl_atr_mult"]),
        "reasons": reasons,
    }


EXPORT_DIR = Path(str(CONFIG["export_dir"]))
EXPORT_DIR.mkdir(parents=True, exist_ok=True)
label_report = summarize_label_gate(
    labeled,
    train_fraction=float(CONFIG["train_fraction"]),
    round_trip_cost_pct=float(CONFIG["round_trip_cost_pct"]),
    minority_rate=float(CONFIG["minority_rate"]),
    ambiguous_win_max=float(CONFIG["ambiguous_win_max"]),
)
report_path = EXPORT_DIR / "label_report.json"
report_path.write_text(json.dumps(label_report, indent=2), encoding="utf-8")
print(json.dumps(label_report, indent=2))
print("wrote", report_path)
if not label_report["passed"]:
    raise RuntimeError(
        "Train-period label gate failed: "
        f"{label_report['reasons']}. Change CONFIG tp/sl ATR multiples and re-run "
        "from the top. Do not fit models."
    )
print("Gate passed. Training may start.")


NameError: name 'labeled' is not defined

In [ ]:
# CELL: SPLIT
def purged_index_splits(
    n: int,
    *,
    train_fraction: float,
    val_fraction: float,
    purge_bars: int,
) -> Dict[str, np.ndarray]:
    """Time-ordered split. Purge ``purge_bars`` on each side of every cut."""
    n = int(n)
    purge = int(purge_bars)
    train_end = int(n * float(train_fraction))
    val_len = int(n * float(val_fraction))
    train_idx = np.arange(0, max(train_end - purge, 0), dtype=np.int64)
    val_start = min(train_end + purge, n)
    val_end = min(val_start + val_len, n)
    val_idx = np.arange(val_start, max(val_end - purge, val_start), dtype=np.int64)
    test_start = min(val_end + purge, n)
    test_idx = np.arange(test_start, n, dtype=np.int64)
    return {"train": train_idx, "val": val_idx, "test": test_idx}


def development_walk_forward(
    n_dev: int,
    *,
    folds: int = 4,
    purge_bars: int,
) -> List[Tuple[np.ndarray, np.ndarray]]:
    """Expanding-window folds inside the development prefix only."""
    folds = max(2, int(folds))
    purge = int(purge_bars)
    edges = [int(n_dev * (i + 1) / folds) for i in range(folds)]
    out: List[Tuple[np.ndarray, np.ndarray]] = []
    for i in range(1, folds):
        train_end = edges[i - 1]
        val_end = edges[i]
        tr = np.arange(0, max(train_end - purge, 0), dtype=np.int64)
        va = np.arange(min(train_end + purge, n_dev), val_end, dtype=np.int64)
        if len(tr) == 0 or len(va) == 0:
            continue
        out.append((tr, va))
    return out


splits = purged_index_splits(
    len(labeled),
    train_fraction=float(CONFIG["train_fraction"]),
    val_fraction=float(CONFIG["val_fraction"]),
    purge_bars=int(CONFIG["purge_bars"]),
)
for key, idx in splits.items():
    print(key, "n", len(idx), "range", int(idx[0]) if len(idx) else None, int(idx[-1]) if len(idx) else None)
dev_end = int(splits["test"][0]) if len(splits["test"]) else len(labeled)
wf_folds = development_walk_forward(
    dev_end, folds=4, purge_bars=int(CONFIG["purge_bars"])
)
print("walk-forward folds", len(wf_folds), "dev_end", dev_end)


NameError: name 'labeled' is not defined

In [ ]:
# CELL: DATASET
def build_windows(
    frame: pd.DataFrame,
    *,
    window_len: int,
    feature_cols: Sequence[str],
) -> Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    feats = frame.loc[:, list(feature_cols)].to_numpy(dtype=np.float64)
    y_cls = frame["trade_label"].to_numpy(dtype=np.int64)
    y_reg = frame.loc[:, ["upside", "downside"]].to_numpy(dtype=np.float64)
    w = int(window_len)
    n = len(frame)
    xs: List[np.ndarray] = []
    cs: List[int] = []
    rs: List[np.ndarray] = []
    ends: List[int] = []
    for end in range(w - 1, n):
        if y_cls[end] < 0:
            continue
        if not np.isfinite(y_reg[end]).all():
            continue
        window = feats[end - w + 1 : end + 1]
        if not np.isfinite(window).all():
            continue
        xs.append(window)
        cs.append(int(y_cls[end]))
        rs.append(y_reg[end])
        ends.append(int(end))
    if not xs:
        empty_x = np.zeros((0, w, len(feature_cols)), dtype=np.float32)
        return (
            empty_x,
            np.zeros((0,), dtype=np.int64),
            np.zeros((0, 2), dtype=np.float32),
            np.zeros((0,), dtype=np.int64),
        )
    return (
        np.stack(xs, axis=0).astype(np.float32),
        np.asarray(cs, dtype=np.int64),
        np.stack(rs, axis=0).astype(np.float32),
        np.asarray(ends, dtype=np.int64),
    )


all_x, all_y, all_r, all_ends = build_windows(
    labeled, window_len=int(CONFIG["window_len"]), feature_cols=FEATURE_COLS
)
print(
    "windows",
    all_x.shape,
    "class mix",
    {n: float(np.mean(all_y == i)) for i, n in enumerate(LABEL_NAMES)},
)


def _split_windows(split_idx: np.ndarray) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    allowed = set(int(i) for i in split_idx.tolist())
    keep = np.array([int(e) in allowed for e in all_ends], dtype=bool)
    return all_x[keep], all_y[keep], all_r[keep]


x_train, y_train, r_train = _split_windows(splits["train"])
x_val, y_val, r_val = _split_windows(splits["val"])
x_test, y_test, r_test = _split_windows(splits["test"])
print("train", x_train.shape, "val", x_val.shape, "test", x_test.shape)

feat_mean = x_train.mean(axis=(0, 1), keepdims=True)
feat_std = x_train.std(axis=(0, 1), keepdims=True)
feat_std = np.where(feat_std < 1e-6, 1.0, feat_std)
reg_mean = r_train.mean(axis=0, keepdims=True)
reg_std = r_train.std(axis=0, keepdims=True)
reg_std = np.where(reg_std < 1e-6, 1.0, reg_std)


def _scale_x(x: np.ndarray) -> np.ndarray:
    return ((x - feat_mean) / feat_std).astype(np.float32)


def _scale_r(r: np.ndarray) -> np.ndarray:
    return ((r - reg_mean) / reg_std).astype(np.float32)


def flatten(x: np.ndarray) -> np.ndarray:
    scaled = _scale_x(x)
    return scaled.reshape(len(scaled), -1)


X_train = flatten(x_train)
X_val = flatten(x_val)
X_test = flatten(x_test)
R_train = _scale_r(r_train)
R_val = _scale_r(r_val)
R_test = _scale_r(r_test)
print("flat train", X_train.shape)


NameError: name 'labeled' is not defined

In [ ]:
# CELL: MODEL
def make_family(name: str, seed: int) -> Dict[str, Any]:
    """Return unfitted classifier + two regressors for one family."""
    est = int(CONFIG["n_estimators"])
    depth = int(CONFIG["max_depth"])
    lr = float(CONFIG["learning_rate"])
    leaf = int(CONFIG["min_leaf"])
    if name == "LR":
        clf = LogisticRegression(
            max_iter=400,
            C=0.5,
            class_weight="balanced",
            solver="lbfgs",
        )
        up = Ridge(alpha=1.0)
        dn = Ridge(alpha=1.0)
    elif name == "RF":
        clf = RandomForestClassifier(
            n_estimators=est,
            max_depth=depth,
            min_samples_leaf=leaf,
            class_weight="balanced_subsample",
            n_jobs=1,
            random_state=seed,
        )
        up = RandomForestRegressor(
            n_estimators=est,
            max_depth=depth,
            min_samples_leaf=leaf,
            n_jobs=1,
            random_state=seed,
        )
        dn = RandomForestRegressor(
            n_estimators=est,
            max_depth=depth,
            min_samples_leaf=leaf,
            n_jobs=1,
            random_state=seed + 1,
        )
    elif name == "XGB":
        clf = XGBClassifier(
            n_estimators=est,
            max_depth=depth,
            learning_rate=lr,
            objective="multi:softprob",
            eval_metric="mlogloss",
            tree_method="hist",
            n_jobs=1,
            random_state=seed,
            verbosity=0,
        )
        up = XGBRegressor(
            n_estimators=est,
            max_depth=depth,
            learning_rate=lr,
            tree_method="hist",
            n_jobs=1,
            random_state=seed,
            verbosity=0,
        )
        dn = XGBRegressor(
            n_estimators=est,
            max_depth=depth,
            learning_rate=lr,
            tree_method="hist",
            n_jobs=1,
            random_state=seed + 1,
            verbosity=0,
        )
    elif name == "LGBM":
        clf = LGBMClassifier(
            n_estimators=est,
            max_depth=depth,
            learning_rate=lr,
            min_child_samples=leaf,
            n_jobs=1,
            random_state=seed,
            verbosity=-1,
        )
        up = LGBMRegressor(
            n_estimators=est,
            max_depth=depth,
            learning_rate=lr,
            min_child_samples=leaf,
            n_jobs=1,
            random_state=seed,
            verbosity=-1,
        )
        dn = LGBMRegressor(
            n_estimators=est,
            max_depth=depth,
            learning_rate=lr,
            min_child_samples=leaf,
            n_jobs=1,
            random_state=seed + 1,
            verbosity=-1,
        )
    elif name == "CatBoost":
        clf = CatBoostClassifier(
            iterations=est,
            depth=min(depth, 8),
            learning_rate=lr,
            loss_function="MultiClass",
            auto_class_weights="Balanced",
            random_seed=seed,
            verbose=False,
            thread_count=1,
            allow_writing_files=False,
        )
        up = CatBoostRegressor(
            iterations=est,
            depth=min(depth, 8),
            learning_rate=lr,
            random_seed=seed,
            verbose=False,
            thread_count=1,
            allow_writing_files=False,
        )
        dn = CatBoostRegressor(
            iterations=est,
            depth=min(depth, 8),
            learning_rate=lr,
            random_seed=seed + 1,
            verbose=False,
            thread_count=1,
            allow_writing_files=False,
        )
    else:
        raise ValueError(name)
    return {"name": name, "clf": clf, "up": up, "dn": dn}


print("families", CONFIG["ensemble"])


families ['LR', 'RF', 'XGB', 'LGBM', 'CatBoost']


In [ ]:
# CELL: TRAIN
def confusion(y_true: np.ndarray, y_pred: np.ndarray, n_classes: int = 3) -> np.ndarray:
    mat = np.zeros((n_classes, n_classes), dtype=np.int64)
    for t, p in zip(y_true.tolist(), y_pred.tolist()):
        if 0 <= int(t) < n_classes and 0 <= int(p) < n_classes:
            mat[int(t), int(p)] += 1
    return mat


def class_precision(mat: np.ndarray, cls: int) -> float:
    col = mat[:, cls]
    denom = float(col.sum())
    if denom <= 0:
        return float("nan")
    return float(mat[cls, cls]) / denom


def balanced_accuracy(mat: np.ndarray) -> float:
    recalls = []
    for i in range(mat.shape[0]):
        denom = float(mat[i].sum())
        recalls.append(float(mat[i, i]) / denom if denom else float("nan"))
    finite = [r for r in recalls if np.isfinite(r)]
    return float(np.mean(finite)) if finite else float("nan")


def corr(a: np.ndarray, b: np.ndarray) -> float:
    if a.size < 3:
        return float("nan")
    if float(np.std(a)) < 1e-12 or float(np.std(b)) < 1e-12:
        return float("nan")
    return float(np.corrcoef(a, b)[0, 1])


def as_cls(arr: np.ndarray) -> np.ndarray:
    return np.asarray(arr).reshape(-1).astype(np.int64)


def as_1d(arr: np.ndarray) -> np.ndarray:
    return np.asarray(arr, dtype=np.float64).reshape(-1)


def classifier_report(y_true: np.ndarray, y_pred: np.ndarray) -> Dict[str, float]:
    cm = confusion(y_true, y_pred)
    p_long = class_precision(cm, LABEL_LONG)
    p_short = class_precision(cm, LABEL_SHORT)
    n_long = int(np.sum(y_pred == LABEL_LONG))
    n_short = int(np.sum(y_pred == LABEL_SHORT))
    bacc = balanced_accuracy(cm)
    return {
        "precision_LONG": p_long,
        "precision_SHORT": p_short,
        "n_LONG": float(n_long),
        "n_SHORT": float(n_short),
        "balanced_accuracy": bacc,
        "confusion": cm.tolist(),
    }


def signal_expectancy(
    pred_cls: np.ndarray,
    split_idx: np.ndarray,
) -> Dict[str, float]:
    allowed = set(int(i) for i in split_idx.tolist())
    keep = np.array([int(e) in allowed for e in all_ends], dtype=bool)
    row_idx = all_ends[keep]
    cost = float(CONFIG["round_trip_cost_pct"])
    out: Dict[str, float] = {}
    for cls, name, pnl_col in (
        (LABEL_LONG, "LONG", "long_pnl_pct"),
        (LABEL_SHORT, "SHORT", "short_pnl_pct"),
    ):
        take = pred_cls == cls
        if not bool(take.any()):
            out[f"{name}_n"] = 0.0
            out[f"{name}_expectancy"] = float("nan")
            continue
        pnl = labeled.loc[row_idx[take], pnl_col].to_numpy(dtype=np.float64) - cost
        finite = np.isfinite(pnl)
        out[f"{name}_n"] = float(finite.sum())
        out[f"{name}_expectancy"] = float(np.mean(pnl[finite])) if finite.any() else float("nan")
    return out


def expectancy_rank_score(exp_report: Dict[str, float]) -> float:
    """Mean of sides with at least 20 predicted trades; else the available side."""
    parts: List[float] = []
    n_long = float(exp_report.get("LONG_n", 0.0))
    n_short = float(exp_report.get("SHORT_n", 0.0))
    e_long = float(exp_report.get("LONG_expectancy", float("nan")))
    e_short = float(exp_report.get("SHORT_expectancy", float("nan")))
    if n_long >= 20 and np.isfinite(e_long):
        parts.append(e_long)
    if n_short >= 20 and np.isfinite(e_short):
        parts.append(e_short)
    if parts:
        return float(np.mean(parts))
    return float("-inf")


def fit_family(name: str) -> Dict[str, Any]:
    pack = make_family(name, int(CONFIG["seed"]))
    pack["clf"].fit(X_train, y_train)
    pack["up"].fit(X_train, R_train[:, 0])
    pack["dn"].fit(X_train, R_train[:, 1])
    pred = as_cls(pack["clf"].predict(X_val))
    cls_rep = classifier_report(y_val, pred)
    exp_rep = signal_expectancy(pred, splits["val"])
    rank = expectancy_rank_score(exp_rep)
    cls_rep["score"] = rank
    up_hat = as_1d(pack["up"].predict(X_val))
    dn_hat = as_1d(pack["dn"].predict(X_val))
    up_c = corr(up_hat, R_val[:, 0])
    dn_c = corr(dn_hat, R_val[:, 1])
    reg_score = float(np.nanmean([up_c, dn_c]))
    print(
        f"{name} val  exp_score={rank:.6f}  "
        f"E_LONG={exp_rep['LONG_expectancy']} nL={exp_rep['LONG_n']:.0f}  "
        f"E_SHORT={exp_rep['SHORT_expectancy']} nS={exp_rep['SHORT_n']:.0f}  "
        f"P_LONG={cls_rep['precision_LONG']:.3f}  P_SHORT={cls_rep['precision_SHORT']:.3f}  "
        f"reg={reg_score:.3f}"
    )
    return {
        "name": name,
        "pack": pack,
        "cls": cls_rep,
        "exp": exp_rep,
        "upside_corr": up_c,
        "downside_corr": dn_c,
        "reg_score": reg_score,
    }


print("Fitting", CONFIG["ensemble"], "in parallel...")
fitted: Dict[str, Dict[str, Any]] = {}
with ThreadPoolExecutor(max_workers=len(CONFIG["ensemble"])) as pool:
    futs = {pool.submit(fit_family, name): name for name in CONFIG["ensemble"]}
    for fut in as_completed(futs):
        name = futs[fut]
        fitted[name] = fut.result()

cls_winner = max(fitted.values(), key=lambda r: float(r["cls"]["score"]))
reg_winner = max(fitted.values(), key=lambda r: float(r["reg_score"]))
print("best classifier", cls_winner["name"], "score", cls_winner["cls"]["score"], cls_winner["exp"])
print("best regressor family", reg_winner["name"], "reg_score", reg_winner["reg_score"])
leaderboard = {
    name: {
        "cls": {k: v for k, v in row["cls"].items() if k != "confusion"},
        "exp": row["exp"],
        "upside_corr": row["upside_corr"],
        "downside_corr": row["downside_corr"],
        "reg_score": row["reg_score"],
    }
    for name, row in fitted.items()
}
print(json.dumps(leaderboard, indent=2, default=float))


Fitting ['LR', 'RF', 'XGB', 'LGBM', 'CatBoost'] in parallel...


NameError: name 'X_train' is not defined

In [ ]:
# CELL: EVAL
def _unscale_reg(pred: np.ndarray, col: int) -> np.ndarray:
    return as_1d(pred) * float(reg_std[0, col]) + float(reg_mean[0, col])


best_clf = fitted[cls_winner["name"]]["pack"]["clf"]
best_up = fitted[reg_winner["name"]]["pack"]["up"]
best_dn = fitted[reg_winner["name"]]["pack"]["dn"]
pred_cls = as_cls(best_clf.predict(X_test))
up_hat = _unscale_reg(best_up.predict(X_test), 0)
dn_hat = _unscale_reg(best_dn.predict(X_test), 1)
r_true = r_test
cm = confusion(y_test, pred_cls)
print("TEST winner clf=", cls_winner["name"], "reg=", reg_winner["name"])
print("test confusion rows=true cols=pred LONG/SHORT/NO_TRADE")
print(cm)
member_test = {}
for name, row in fitted.items():
    p = as_cls(row["pack"]["clf"].predict(X_test))
    member_test[name] = classifier_report(y_test, p)
    member_test[name]["exp"] = signal_expectancy(p, splits["test"])
    member_test[name]["score"] = expectancy_rank_score(member_test[name]["exp"])
    member_test[name]["upside_corr"] = corr(
        _unscale_reg(row["pack"]["up"].predict(X_test), 0), r_true[:, 0]
    )
    member_test[name]["downside_corr"] = corr(
        _unscale_reg(row["pack"]["dn"].predict(X_test), 1), r_true[:, 1]
    )
    print(
        name,
        "test",
        {
            "score": member_test[name]["score"],
            "exp": member_test[name]["exp"],
            "precision_LONG": member_test[name]["precision_LONG"],
            "precision_SHORT": member_test[name]["precision_SHORT"],
        },
    )

cls_rep = classifier_report(y_test, pred_cls)
exp_report = signal_expectancy(pred_cls, splits["test"])
eval_report = {
    "n_test": int(len(y_test)),
    "best_classifier": cls_winner["name"],
    "best_regressor_family": reg_winner["name"],
    "val_leaderboard": leaderboard,
    "test_members": {
        k: {kk: vv for kk, vv in v.items() if kk != "confusion"}
        for k, v in member_test.items()
    },
    "confusion": cm.tolist(),
    "precision_LONG": cls_rep["precision_LONG"],
    "precision_SHORT": cls_rep["precision_SHORT"],
    "n_LONG": cls_rep["n_LONG"],
    "n_SHORT": cls_rep["n_SHORT"],
    "balanced_accuracy": cls_rep["balanced_accuracy"],
    "upside_corr": corr(up_hat, r_true[:, 0]),
    "downside_corr": corr(dn_hat, r_true[:, 1]),
    "signal_expectancy": exp_report,
    "score": expectancy_rank_score(exp_report),
    "note": (
        "Best classifier is chosen on validation model-selected LONG/SHORT "
        "expectancy (min 20 predicted trades per side when both exist). "
        "Test is scored once. Balanced accuracy is diagnostic only."
    ),
}
print(json.dumps(eval_report, indent=2, default=float))


NameError: name 'cls_winner' is not defined

In [ ]:
# CELL: WALK_FORWARD
wf_rows: List[Dict[str, Any]] = []
winner_name = cls_winner["name"]
for fold_i, (tr_idx, va_idx) in enumerate(wf_folds, start=1):
    x_tr, y_tr, r_tr = _split_windows(tr_idx)
    x_va, y_va, r_va = _split_windows(va_idx)
    if len(x_tr) < 32 or len(x_va) < 8:
        continue
    mean = x_tr.mean(axis=(0, 1), keepdims=True)
    std = x_tr.std(axis=(0, 1), keepdims=True)
    std = np.where(std < 1e-6, 1.0, std)
    X_tr = ((x_tr - mean) / std).reshape(len(x_tr), -1)
    X_va = ((x_va - mean) / std).reshape(len(x_va), -1)
    pack = make_family(winner_name, int(CONFIG["seed"]) + fold_i)
    pack["clf"].fit(X_tr, y_tr)
    pred = as_cls(pack["clf"].predict(X_va))
    rep = classifier_report(y_va, pred)
    exp_rep = signal_expectancy(pred, va_idx)
    score = expectancy_rank_score(exp_rep)
    wf_rows.append(
        {
            "fold": fold_i,
            "family": winner_name,
            "n_train": int(len(x_tr)),
            "n_val": int(len(x_va)),
            "score": score,
            "precision_LONG": rep["precision_LONG"],
            "precision_SHORT": rep["precision_SHORT"],
            "n_LONG": rep["n_LONG"],
            "n_SHORT": rep["n_SHORT"],
            "balanced_accuracy": rep["balanced_accuracy"],
            "signal_expectancy": exp_rep,
        }
    )
    print("fold", fold_i, wf_rows[-1])
print(json.dumps({"walk_forward": wf_rows}, indent=2, default=float))


NameError: name 'cls_winner' is not defined

In [ ]:
# CELL: EXPORT
EXPORT_DIR.mkdir(parents=True, exist_ok=True)
bundle = {
    "best_classifier": cls_winner["name"],
    "best_regressor_family": reg_winner["name"],
    "clf": best_clf,
    "up": best_up,
    "dn": best_dn,
    "feat_mean": feat_mean,
    "feat_std": feat_std,
    "reg_mean": reg_mean,
    "reg_std": reg_std,
    "feature_cols": list(FEATURE_COLS),
    "window_len": int(CONFIG["window_len"]),
}
with (EXPORT_DIR / "best_models.pkl").open("wb") as handle:
    pickle.dump(bundle, handle)
(EXPORT_DIR / "leaderboard.json").write_text(
    json.dumps(leaderboard, indent=2, default=float), encoding="utf-8"
)
metadata = {
    "model_family": "jacksparrow_opportunity_10m_tabular_ensemble",
    "ensemble": list(CONFIG["ensemble"]),
    "best_classifier": cls_winner["name"],
    "best_regressor_family": reg_winner["name"],
    "timeframe": "10m",
    "horizon_bars": int(CONFIG["horizon_bars"]),
    "horizon_minutes": 30,
    "tp_atr_mult": float(CONFIG["tp_atr_mult"]),
    "sl_atr_mult": float(CONFIG["sl_atr_mult"]),
    "round_trip_cost_pct": float(CONFIG["round_trip_cost_pct"]),
    "window_len": int(CONFIG["window_len"]),
    "feature_cols": list(FEATURE_COLS),
    "label_names": list(LABEL_NAMES),
    "ready_to_promote": False,
    "promote_reason": "research_opportunity_10m_not_live",
    "eval": eval_report,
    "label_gate": label_report,
    "walk_forward": wf_rows,
}
(EXPORT_DIR / "metadata.json").write_text(
    json.dumps(metadata, indent=2, default=float), encoding="utf-8"
)
(EXPORT_DIR / "eval_report.json").write_text(
    json.dumps(eval_report, indent=2, default=float), encoding="utf-8"
)
print("wrote", EXPORT_DIR / "best_models.pkl")
print("best classifier", cls_winner["name"], "best regressor", reg_winner["name"])
print("ready_to_promote=false")
try:
    from google.colab import files
    import shutil

    shutil.make_archive(str(EXPORT_DIR), "zip", root_dir=str(EXPORT_DIR))
    files.download(str(EXPORT_DIR) + ".zip")
except ImportError:
    print("Not Colab; files remain under", EXPORT_DIR)


NameError: name 'cls_winner' is not defined

*File Operation*: `ls` on `content`

*File Operation*: `ls` on `/content/export/opportunity_10m`

**Session Created**: 2026-10-08 15:32:39
- Endpoint: `m-s-kkb-usw1b1-1a6jexbsgpoxl`
- Hardware: `NONE`

In [ ]:
# CELL: ENV
# Colab ships pandas/numpy; install tabular model extras.
%pip install -q --upgrade-strategy only-if-needed pyarrow requests scikit-learn xgboost lightgbm catboost

import json
import math
import pickle
import random
import time
from concurrent.futures import ThreadPoolExecutor, as_completed
from datetime import datetime, timedelta, timezone
from pathlib import Path
from typing import Any, Dict, List, Optional, Sequence, Tuple

import numpy as np
import pandas as pd
from catboost import CatBoostClassifier, CatBoostRegressor
from lightgbm import LGBMClassifier, LGBMRegressor
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.linear_model import LogisticRegression, Ridge
from xgboost import XGBClassifier, XGBRegressor

print("sklearn / xgboost / lightgbm / catboost ready")


   ━━━━━━━╺━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.9/97.1 MB 153.4 MB/s eta 0:00:01

   ━━━━━━━━━━━━━━━━━━━━━━━━━━╺━━━━━━━━━━━━━ 63.2/97.1 MB 137.3 MB/s eta 0:00:01

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╸━━━━━━━ 79.8/97.1 MB 81.1 MB/s eta 0:00:01

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╺ 95.7/97.1 MB 88.9 MB/s eta 0:00:01

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╸ 97.1/97.1 MB 94.0 MB/s eta 0:00:01

...

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╸ 97.1/97.1 MB 94.0 MB/s eta 0:00:01

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╸ 97.1/97.1 MB 94.0 MB/s eta 0:00:01

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╸ 97.1/97.1 MB 94.0 MB/s eta 0:00:01

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 7.7 MB/s eta 0:00:00


sklearn / xgboost / lightgbm / catboost ready


In [ ]:
# CELL: CONFIG
CONFIG: Dict[str, Any] = {
    "symbol": "BTCUSD",
    "base_url": "https://api.india.delta.exchange",
    "history_days": 900,
    "resolution_5m": "5m",
    "seed": 42,
    "tp_atr_mult": 1.5,
    "sl_atr_mult": 1.0,
    "horizon_bars": 3,
    "atr_period": 14,
    "window_len": 60,
    "round_trip_cost_pct": 0.0016,
    "minority_rate": 0.15,
    "ambiguous_win_max": 0.50,
    "train_fraction": 0.70,
    "val_fraction": 0.15,
    "purge_bars": 3,
    "n_estimators": 200,
    "max_depth": 8,
    "learning_rate": 0.08,
    "min_leaf": 40,
    "ensemble": ["LR", "RF", "XGB", "LGBM", "CatBoost"],
    "export_dir": "export/opportunity_10m",
    "ready_to_promote": False,
}

FEATURE_COLS: Tuple[str, ...] = (
    "ret_1",
    "atr_pct",
    "rsi",
    "ema_dist_atr",
    "candle_body",
    "volume_z",
)
LABEL_NAMES: Tuple[str, ...] = ("LONG", "SHORT", "NO_TRADE")
LABEL_LONG, LABEL_SHORT, LABEL_NO_TRADE = 0, 1, 2
print(json.dumps(CONFIG, indent=2))


{
  "symbol": "BTCUSD",
  "base_url": "https://api.india.delta.exchange",
  "history_days": 900,
  "resolution_5m": "5m",
  "seed": 42,
  "tp_atr_mult": 1.5,
  "sl_atr_mult": 1.0,
  "horizon_bars": 3,
  "atr_period": 14,
  "window_len": 60,
  "round_trip_cost_pct": 0.0016,
  "minority_rate": 0.15,
  "ambiguous_win_max": 0.5,
  "train_fraction": 0.7,
  "val_fraction": 0.15,
  "purge_bars": 3,
  "n_estimators": 200,
  "max_depth": 8,
  "learning_rate": 0.08,
  "min_leaf": 40,
  "ensemble": [
    "LR",
    "RF",
    "XGB",
    "LGBM",
    "CatBoost"
  ],
  "export_dir": "export/opportunity_10m",
  "ready_to_promote": false
}


In [ ]:
# CELL: SEED
def set_seed(seed: int) -> None:
    random.seed(int(seed))
    np.random.seed(int(seed))


set_seed(int(CONFIG["seed"]))
print("seed", CONFIG["seed"])


seed 42


In [ ]:
# CELL: FETCH
DELTA_CANDLE_PAGE_BARS = 2000
REQUEST_DELAY_SECONDS = 0.2
EMPTY_PAGE_RETRIES = 3
MIN_OHLCV_COMPLETENESS = 0.95
_RESOLUTION_MINUTES = {"5m": 5, "10m": 10}


def _bar_seconds(resolution: str) -> int:
    if resolution not in _RESOLUTION_MINUTES:
        raise ValueError(f"Unsupported resolution {resolution!r}")
    return int(_RESOLUTION_MINUTES[resolution]) * 60


def _parse_candle_rows(payload: Any) -> List[Dict[str, Any]]:
    if not isinstance(payload, dict):
        return []
    result = payload.get("result", [])
    if isinstance(result, dict):
        result = result.get("candles", []) or []
    if not isinstance(result, list):
        return []
    return [row for row in result if isinstance(row, dict) and "time" in row]


def _candle_epoch(row: Dict[str, Any]) -> Optional[int]:
    try:
        return int(row["time"])
    except (KeyError, TypeError, ValueError):
        return None


def _request_candle_page(
    url: str,
    *,
    symbol: str,
    resolution: str,
    start_ts: int,
    end_ts: int,
) -> List[Dict[str, Any]]:
    import requests

    last_error: Optional[Exception] = None
    for attempt in range(EMPTY_PAGE_RETRIES):
        try:
            response = requests.get(
                url,
                params={
                    "symbol": symbol,
                    "resolution": resolution,
                    "start": int(start_ts),
                    "end": int(end_ts),
                },
                timeout=30,
            )
            response.raise_for_status()
            payload = response.json()
        except Exception as exc:
            last_error = exc
            time.sleep(REQUEST_DELAY_SECONDS * (attempt + 1))
            continue
        if payload.get("success") is False:
            last_error = RuntimeError(f"Delta candles error for {symbol}: {payload}")
            time.sleep(REQUEST_DELAY_SECONDS * (attempt + 1))
            continue
        rows = _parse_candle_rows(payload)
        if rows:
            return rows
        last_error = None
        time.sleep(REQUEST_DELAY_SECONDS * (attempt + 1))
    if last_error is not None:
        raise last_error
    return []


def _rows_to_frame(all_rows: List[Dict[str, Any]]) -> pd.DataFrame:
    df = pd.DataFrame(all_rows)
    expected_cols = ["time", "open", "high", "low", "close", "volume"]
    df = df[[c for c in expected_cols if c in df.columns]]
    df["time"] = pd.to_datetime(df["time"], unit="s", utc=True)
    df = df.drop_duplicates(subset="time").sort_values("time").reset_index(drop=True)
    for col in ["open", "high", "low", "close", "volume"]:
        if col in df.columns:
            df[col] = df[col].astype(float)
    return df


def fetch_candles(
    symbol: str,
    resolution: str,
    start_ts: int,
    end_ts: int,
    base_url: str,
    *,
    page_bars: int = DELTA_CANDLE_PAGE_BARS,
) -> pd.DataFrame:
    url = f"{base_url.rstrip('/')}/v2/history/candles"
    bar_seconds = _bar_seconds(resolution)
    page_bars = max(1, int(page_bars))
    start_ts, end_ts = int(start_ts), int(end_ts)
    if end_ts <= start_ts:
        raise ValueError("end_ts must be greater than start_ts")
    all_rows: List[Dict[str, Any]] = []
    cursor_end = end_ts
    seen_oldest: Optional[int] = None
    max_pages = max(2, (end_ts - start_ts) // bar_seconds + 5)
    for _ in range(max_pages):
        if cursor_end <= start_ts:
            break
        page_start = max(start_ts, cursor_end - page_bars * bar_seconds)
        if page_start >= cursor_end:
            break
        rows = _request_candle_page(
            url,
            symbol=symbol,
            resolution=resolution,
            start_ts=page_start,
            end_ts=cursor_end,
        )
        if not rows:
            break
        epochs = [e for e in (_candle_epoch(r) for r in rows) if e is not None]
        if not epochs:
            break
        oldest = min(epochs)
        if seen_oldest is not None and oldest >= seen_oldest:
            break
        seen_oldest = oldest
        all_rows.extend(rows)
        cursor_end = oldest - 1
        time.sleep(REQUEST_DELAY_SECONDS)
    if not all_rows:
        raise RuntimeError(f"No candle data returned for symbol={symbol}")
    return _rows_to_frame(all_rows)


def validate_ohlcv_completeness(
    df: pd.DataFrame,
    resolution: str,
    *,
    min_completeness: float = MIN_OHLCV_COMPLETENESS,
    symbol: str = "",
) -> Dict[str, Any]:
    if df is None or df.empty or "time" not in df.columns:
        raise ValueError("Cannot validate completeness on an empty OHLCV frame")
    bar_seconds = _bar_seconds(resolution)
    time_col = df["time"]
    epochs = (pd.to_datetime(time_col, utc=True).astype("int64") // 10**9).tolist()
    if len(epochs) < 2:
        raise ValueError("Need at least 2 OHLCV bars to validate completeness")
    gaps = 0
    for prev, cur in zip(epochs, epochs[1:]):
        if int(cur) - int(prev) > bar_seconds * 1.5:
            gaps += 1
    completeness = 1.0 - (gaps / max(1, len(epochs) - 1))
    report = {
        "rows": len(epochs),
        "gaps": int(gaps),
        "completeness": round(float(completeness), 4),
    }
    if completeness < min_completeness:
        label = f"{symbol} {resolution}".strip()
        raise ValueError(
            f"OHLCV completeness {completeness:.1%} below {min_completeness:.0%} "
            f"for {label or 'candles'}"
        )
    return report


end_dt = datetime.now(timezone.utc)
start_dt = end_dt - timedelta(days=int(CONFIG["history_days"]))
print(
    f"Fetching {CONFIG['symbol']} 5m OHLCV for {CONFIG['history_days']}d "
    f"from {CONFIG['base_url']}..."
)
df5m = fetch_candles(
    str(CONFIG["symbol"]),
    str(CONFIG["resolution_5m"]),
    int(start_dt.timestamp()),
    int(end_dt.timestamp()),
    str(CONFIG["base_url"]),
)
ohlcv_report = validate_ohlcv_completeness(
    df5m, "5m", symbol=str(CONFIG["symbol"])
)
print(
    f"OHLCV {CONFIG['symbol']} 5m: {ohlcv_report['rows']} bars, "
    f"completeness {ohlcv_report['completeness']:.1%}, gaps={ohlcv_report['gaps']}"
)


Fetching BTCUSD 5m OHLCV for 900d from https://api.india.delta.exchange...


OHLCV BTCUSD 5m: 259200 bars, completeness 100.0%, gaps=0


In [ ]:
# CELL: BARS_10M
def build_10m_ohlcv_from_5m(df5m: pd.DataFrame) -> pd.DataFrame:
    """Exchange-aligned 10m bars from two closed 5m candles. Drop a partial last bucket."""
    src = df5m.copy()
    src["time"] = pd.to_datetime(src["time"], utc=True)
    required = ("open", "high", "low", "close", "volume")
    missing = [c for c in required if c not in src.columns]
    if missing:
        raise ValueError(f"5m OHLCV missing columns: {missing}")
    if src.empty:
        return src.iloc[0:0].copy()
    grouped = src.set_index("time").sort_index()
    agg = grouped.resample("10min", label="left", closed="left").agg(
        {
            "open": "first",
            "high": "max",
            "low": "min",
            "close": "last",
            "volume": "sum",
        }
    )
    counts = grouped["close"].resample("10min", label="left", closed="left").count()
    complete = counts >= 2
    agg = agg.loc[complete].dropna(subset=["open", "high", "low", "close"])
    out = agg.reset_index()
    return out.sort_values("time").reset_index(drop=True)


bars_10m = build_10m_ohlcv_from_5m(df5m)
print(f"10m bars={len(bars_10m)} span={bars_10m['time'].iloc[0]} → {bars_10m['time'].iloc[-1]}")


10m bars=129599 span=2024-04-21 15:40:00+00:00 → 2026-10-08 15:20:00+00:00


In [ ]:
# CELL: FEATURES
def _wilder_rsi(close: pd.Series, period: int = 14) -> pd.Series:
    delta = close.diff()
    gain = delta.clip(lower=0.0)
    loss = -delta.clip(upper=0.0)
    avg_gain = gain.ewm(alpha=1.0 / period, min_periods=period, adjust=False).mean()
    avg_loss = loss.ewm(alpha=1.0 / period, min_periods=period, adjust=False).mean()
    rs = avg_gain / avg_loss.replace(0.0, np.nan)
    return 100.0 - (100.0 / (1.0 + rs))


def add_opportunity_features(
    df: pd.DataFrame,
    *,
    atr_period: int = 14,
    rsi_period: int = 14,
    ema_span: int = 20,
    vol_window: int = 20,
) -> pd.DataFrame:
    """Causal 10m features. ATR(14) is also stored for the label path."""
    out = df.copy()
    close = out["close"].astype(np.float64)
    high = out["high"].astype(np.float64)
    low = out["low"].astype(np.float64)
    open_ = out["open"].astype(np.float64)
    volume = out["volume"].astype(np.float64)
    prev_close = close.shift(1)
    tr = pd.concat(
        [
            high - low,
            (high - prev_close).abs(),
            (low - prev_close).abs(),
        ],
        axis=1,
    ).max(axis=1)
    atr = tr.rolling(int(atr_period), min_periods=int(atr_period)).mean()
    out["atr"] = atr
    out["ret_1"] = close.pct_change()
    out["atr_pct"] = atr / close.replace(0.0, np.nan)
    out["rsi"] = _wilder_rsi(close, period=int(rsi_period))
    ema = close.ewm(span=int(ema_span), adjust=False, min_periods=int(ema_span)).mean()
    out["ema_dist_atr"] = (close - ema) / atr.replace(0.0, np.nan)
    out["candle_body"] = (close - open_) / atr.replace(0.0, np.nan)
    vol_mean = volume.rolling(int(vol_window), min_periods=int(vol_window)).mean()
    vol_std = volume.rolling(int(vol_window), min_periods=int(vol_window)).std()
    out["volume_z"] = (volume - vol_mean) / vol_std.replace(0.0, np.nan)
    return out


featured = add_opportunity_features(
    bars_10m, atr_period=int(CONFIG["atr_period"])
)
print("feature null rates")
print(featured[list(FEATURE_COLS)].isna().mean().round(4))


feature null rates
ret_1           0.0000
atr_pct         0.0001
rsi             0.0001
ema_dist_atr    0.0001
candle_body     0.0001
volume_z        0.0001
dtype: float64


In [ ]:
# CELL: LABELS
# Opportunity labels on a 10-minute series. Path is bars t+1 .. t+horizon only.

import numpy as np
import pandas as pd

_EPS = 1e-9
HORIZON_BARS_DEFAULT = 3
ATR_PERIOD_DEFAULT = 14
TP_ATR_MULT_DEFAULT = 1.5
SL_ATR_MULT_DEFAULT = 1.0

TP_FIRST = 0
SL_FIRST = 1
NEITHER = 2
AMBIGUOUS = 3

LABEL_LONG = 0
LABEL_SHORT = 1
LABEL_NO_TRADE = 2
LABEL_UNLABELED = -1
LABEL_NAMES = ("LONG", "SHORT", "NO_TRADE")


def tp_sl_from_path(
    fwd_high: np.ndarray,
    fwd_low: np.ndarray,
    entry: float,
    atr: float,
    tp_mult: float,
    sl_mult: float,
    side: str,
) -> int:
    """Walk future highs/lows. Same-bar TP and SL is AMBIGUOUS, not a win."""
    high = np.asarray(fwd_high, dtype=np.float64).ravel()
    low = np.asarray(fwd_low, dtype=np.float64).ravel()
    if high.size == 0 or high.size != low.size:
        return int(NEITHER)
    denom = max(float(atr), _EPS)
    entry_f = float(entry)
    if str(side).strip().upper() == "SHORT":
        tp_level = entry_f - float(tp_mult) * denom
        sl_level = entry_f + float(sl_mult) * denom
        tp_hit = low <= tp_level
        sl_hit = high >= sl_level
    else:
        tp_level = entry_f + float(tp_mult) * denom
        sl_level = entry_f - float(sl_mult) * denom
        tp_hit = high >= tp_level
        sl_hit = low <= sl_level
    for j in range(high.size):
        hit_tp = bool(tp_hit[j])
        hit_sl = bool(sl_hit[j])
        if hit_tp and hit_sl:
            return int(AMBIGUOUS)
        if hit_tp:
            return int(TP_FIRST)
        if hit_sl:
            return int(SL_FIRST)
    return int(NEITHER)


def _side_win(code: int) -> int:
    return 1 if int(code) == int(TP_FIRST) else 0


def _causal_atr(df: pd.DataFrame, period: int = ATR_PERIOD_DEFAULT) -> np.ndarray:
    if "atr" in df.columns:
        atr = df["atr"].to_numpy(dtype=np.float64)
        if np.isfinite(atr).any():
            return atr
    close = df["close"].to_numpy(dtype=np.float64)
    high = df["high"].to_numpy(dtype=np.float64)
    low = df["low"].to_numpy(dtype=np.float64)
    prev = np.concatenate([[close[0]], close[:-1]])
    tr = np.maximum(high - low, np.maximum(np.abs(high - prev), np.abs(low - prev)))
    return pd.Series(tr).rolling(int(period), min_periods=int(period)).mean().to_numpy(
        dtype=np.float64
    )


def compute_opportunity_labels(
    df: pd.DataFrame,
    *,
    tp_mult: float = TP_ATR_MULT_DEFAULT,
    sl_mult: float = SL_ATR_MULT_DEFAULT,
    horizon_bars: int = HORIZON_BARS_DEFAULT,
    atr_period: int = ATR_PERIOD_DEFAULT,
) -> pd.DataFrame:
    """Label each closed 10m bar. Last ``horizon_bars`` rows stay unlabeled."""
    out = df.copy().reset_index(drop=True)
    n = len(out)
    k = int(horizon_bars)
    close = out["close"].to_numpy(dtype=np.float64)
    high = out["high"].to_numpy(dtype=np.float64)
    low = out["low"].to_numpy(dtype=np.float64)
    atr = _causal_atr(out, period=int(atr_period))
    n_valid = max(n - k, 0)

    trade_label = np.full(n, LABEL_UNLABELED, dtype=np.int64)
    long_result = np.full(n, np.nan, dtype=np.float64)
    short_result = np.full(n, np.nan, dtype=np.float64)
    long_code = np.full(n, np.nan, dtype=np.float64)
    short_code = np.full(n, np.nan, dtype=np.float64)
    upside = np.full(n, np.nan, dtype=np.float64)
    downside = np.full(n, np.nan, dtype=np.float64)
    timeout_ret = np.full(n, np.nan, dtype=np.float64)
    long_pnl_pct = np.full(n, np.nan, dtype=np.float64)
    short_pnl_pct = np.full(n, np.nan, dtype=np.float64)

    for i in range(n_valid):
        atr_i = float(atr[i]) if i < len(atr) else float("nan")
        if not np.isfinite(atr_i) or atr_i <= _EPS:
            continue
        entry = float(close[i])
        if not np.isfinite(entry) or abs(entry) <= _EPS:
            continue
        fwd_high = high[i + 1 : i + 1 + k]
        fwd_low = low[i + 1 : i + 1 + k]
        fwd_close = close[i + 1 : i + 1 + k]
        if fwd_high.size < k:
            continue
        l_code = tp_sl_from_path(fwd_high, fwd_low, entry, atr_i, tp_mult, sl_mult, "LONG")
        s_code = tp_sl_from_path(fwd_high, fwd_low, entry, atr_i, tp_mult, sl_mult, "SHORT")
        l_win = _side_win(l_code)
        s_win = _side_win(s_code)
        if l_win == 1 and s_win == 0:
            label = LABEL_LONG
        elif s_win == 1 and l_win == 0:
            label = LABEL_SHORT
        else:
            label = LABEL_NO_TRADE
        trade_label[i] = label
        long_result[i] = float(l_win)
        short_result[i] = float(s_win)
        long_code[i] = float(l_code)
        short_code[i] = float(s_code)
        denom = max(atr_i, _EPS)
        upside[i] = max((float(np.max(fwd_high)) - entry) / denom, 0.0)
        downside[i] = max((entry - float(np.min(fwd_low))) / denom, 0.0)
        end_close = float(fwd_close[-1])
        timeout_ret[i] = (end_close - entry) / entry
        tp_pct = float(tp_mult) * denom / entry
        sl_pct = float(sl_mult) * denom / entry
        if l_code == TP_FIRST:
            long_pnl_pct[i] = tp_pct
        elif l_code == SL_FIRST:
            long_pnl_pct[i] = -sl_pct
        elif l_code == NEITHER:
            long_pnl_pct[i] = timeout_ret[i]
        if s_code == TP_FIRST:
            short_pnl_pct[i] = tp_pct
        elif s_code == SL_FIRST:
            short_pnl_pct[i] = -sl_pct
        elif s_code == NEITHER:
            short_pnl_pct[i] = -timeout_ret[i]

    out["atr"] = atr
    out["trade_label"] = trade_label
    out["long_result"] = long_result
    out["short_result"] = short_result
    out["long_code"] = long_code
    out["short_code"] = short_code
    out["upside"] = upside
    out["downside"] = downside
    out["timeout_ret"] = timeout_ret
    out["long_pnl_pct"] = long_pnl_pct
    out["short_pnl_pct"] = short_pnl_pct
    names = np.array(["UNLABELED"] * n, dtype=object)
    names[trade_label == LABEL_LONG] = "LONG"
    names[trade_label == LABEL_SHORT] = "SHORT"
    names[trade_label == LABEL_NO_TRADE] = "NO_TRADE"
    out["trade_name"] = names
    return out


In [ ]:
# CELL: APPLY_LABELS
labeled = compute_opportunity_labels(
    featured,
    tp_mult=float(CONFIG["tp_atr_mult"]),
    sl_mult=float(CONFIG["sl_atr_mult"]),
    horizon_bars=int(CONFIG["horizon_bars"]),
    atr_period=int(CONFIG["atr_period"]),
)
valid = labeled["trade_label"].to_numpy() >= 0
print(f"labeled rows={int(valid.sum())} / {len(labeled)}")
print(labeled.loc[valid, "trade_name"].value_counts(normalize=True).round(4))


labeled rows=129583 / 129599
trade_name
NO_TRADE    0.6528
SHORT       0.1757
LONG        0.1715
Name: proportion, dtype: float64


In [ ]:
# CELL: GATE
def train_period_mask(n: int, train_fraction: float) -> np.ndarray:
    cut = int(n * float(train_fraction))
    mask = np.zeros(n, dtype=bool)
    mask[:cut] = True
    return mask


def _path_code_counts(codes: np.ndarray, usable: np.ndarray) -> Dict[str, int]:
    finite = usable & np.isfinite(codes)
    return {
        "TP_FIRST": int(np.sum(finite & (codes == TP_FIRST))),
        "SL_FIRST": int(np.sum(finite & (codes == SL_FIRST))),
        "NEITHER": int(np.sum(finite & (codes == NEITHER))),
        "AMBIGUOUS": int(np.sum(finite & (codes == AMBIGUOUS))),
    }


def summarize_label_gate(
    df: pd.DataFrame,
    *,
    train_fraction: float,
    round_trip_cost_pct: float,
    minority_rate: float,
    ambiguous_win_max: float,
) -> Dict[str, Any]:
    n = len(df)
    in_train = train_period_mask(n, train_fraction)
    labels = df["trade_label"].to_numpy(dtype=np.int64)
    usable = in_train & (labels >= 0)
    n_use = int(usable.sum())
    if n_use == 0:
        raise RuntimeError("Train-period gate has no labeled rows")
    mix = {}
    mix_ok = True
    for cid, name in enumerate(LABEL_NAMES):
        rate = float(np.mean(labels[usable] == cid))
        mix[name] = rate
        if rate < float(minority_rate):
            mix_ok = False
    long_code = df["long_code"].to_numpy(dtype=np.float64)
    short_code = df["short_code"].to_numpy(dtype=np.float64)
    cost = float(round_trip_cost_pct)
    long_pnl = df["long_pnl_pct"].to_numpy(dtype=np.float64) - cost
    short_pnl = df["short_pnl_pct"].to_numpy(dtype=np.float64) - cost

    always_long_mask = usable & np.isfinite(long_pnl)
    always_short_mask = usable & np.isfinite(short_pnl)
    always_in_long = (
        float(np.mean(long_pnl[always_long_mask])) if always_long_mask.any() else float("nan")
    )
    always_in_short = (
        float(np.mean(short_pnl[always_short_mask])) if always_short_mask.any() else float("nan")
    )

    excl_long_mask = usable & (labels == LABEL_LONG) & np.isfinite(long_pnl)
    excl_short_mask = usable & (labels == LABEL_SHORT) & np.isfinite(short_pnl)
    exclusive_long = (
        float(np.mean(long_pnl[excl_long_mask])) if excl_long_mask.any() else float("nan")
    )
    exclusive_short = (
        float(np.mean(short_pnl[excl_short_mask])) if excl_short_mask.any() else float("nan")
    )

    def _ambig_share(codes: np.ndarray) -> float:
        finite = usable & np.isfinite(codes)
        tp = finite & (codes == TP_FIRST)
        amb = finite & (codes == AMBIGUOUS)
        denom = int(tp.sum() + amb.sum())
        if denom == 0:
            return 0.0
        return float(amb.sum()) / float(denom)

    long_ambig = _ambig_share(long_code)
    short_ambig = _ambig_share(short_code)
    reasons: List[str] = []
    if not mix_ok:
        reasons.append("class_mix_below_floor")
    if not (np.isfinite(exclusive_long) and exclusive_long > 0.0):
        reasons.append("exclusive_long_expectancy_not_positive")
    if not (np.isfinite(exclusive_short) and exclusive_short > 0.0):
        reasons.append("exclusive_short_expectancy_not_positive")
    if long_ambig >= float(ambiguous_win_max):
        reasons.append("long_ambiguous_majority")
    if short_ambig >= float(ambiguous_win_max):
        reasons.append("short_ambiguous_majority")
    passed = len(reasons) == 0
    return {
        "passed": passed,
        "n_train_labeled": n_use,
        "class_mix": mix,
        "minority_rate": float(minority_rate),
        "exclusive_long_expectancy_pct": exclusive_long,
        "exclusive_short_expectancy_pct": exclusive_short,
        "always_in_long_expectancy_pct": always_in_long,
        "always_in_short_expectancy_pct": always_in_short,
        "round_trip_cost_pct": cost,
        "long_ambiguous_share": long_ambig,
        "short_ambiguous_share": short_ambig,
        "long_path_codes": _path_code_counts(long_code, usable),
        "short_path_codes": _path_code_counts(short_code, usable),
        "tp_atr_mult": float(CONFIG["tp_atr_mult"]),
        "sl_atr_mult": float(CONFIG["sl_atr_mult"]),
        "reasons": reasons,
    }


EXPORT_DIR = Path(str(CONFIG["export_dir"]))
EXPORT_DIR.mkdir(parents=True, exist_ok=True)
label_report = summarize_label_gate(
    labeled,
    train_fraction=float(CONFIG["train_fraction"]),
    round_trip_cost_pct=float(CONFIG["round_trip_cost_pct"]),
    minority_rate=float(CONFIG["minority_rate"]),
    ambiguous_win_max=float(CONFIG["ambiguous_win_max"]),
)
report_path = EXPORT_DIR / "label_report.json"
report_path.write_text(json.dumps(label_report, indent=2), encoding="utf-8")
print(json.dumps(label_report, indent=2))
print("wrote", report_path)
if not label_report["passed"]:
    raise RuntimeError(
        "Train-period label gate failed: "
        f"{label_report['reasons']}. Change CONFIG tp/sl ATR multiples and re-run "
        "from the top. Do not fit models."
    )
print("Gate passed. Training may start.")


{
  "passed": true,
  "n_train_labeled": 90706,
  "class_mix": {
    "LONG": 0.17139990739311622,
    "SHORT": 0.17497188719599585,
    "NO_TRADE": 0.653628205410888
  },
  "minority_rate": 0.15,
  "exclusive_long_expectancy_pct": 0.0018420287979210466,
  "exclusive_short_expectancy_pct": 0.0019101443342636204,
  "always_in_long_expectancy_pct": -0.0015948438214699052,
  "always_in_short_expectancy_pct": -0.0015955657442523604,
  "round_trip_cost_pct": 0.0016,
  "long_ambiguous_share": 0.017380862090759702,
  "short_ambiguous_share": 0.018126701311556546,
  "long_path_codes": {
    "TP_FIRST": 15547,
    "SL_FIRST": 31167,
    "NEITHER": 43717,
    "AMBIGUOUS": 275
  },
  "short_path_codes": {
    "TP_FIRST": 15871,
    "SL_FIRST": 30896,
    "NEITHER": 43646,
    "AMBIGUOUS": 293
  },
  "tp_atr_mult": 1.5,
  "sl_atr_mult": 1.0,
  "reasons": []
}
wrote export/opportunity_10m/label_report.json
Gate passed. Training may start.


In [ ]:
# CELL: SPLIT
def purged_index_splits(
    n: int,
    *,
    train_fraction: float,
    val_fraction: float,
    purge_bars: int,
) -> Dict[str, np.ndarray]:
    """Time-ordered split. Purge ``purge_bars`` on each side of every cut."""
    n = int(n)
    purge = int(purge_bars)
    train_end = int(n * float(train_fraction))
    val_len = int(n * float(val_fraction))
    train_idx = np.arange(0, max(train_end - purge, 0), dtype=np.int64)
    val_start = min(train_end + purge, n)
    val_end = min(val_start + val_len, n)
    val_idx = np.arange(val_start, max(val_end - purge, val_start), dtype=np.int64)
    test_start = min(val_end + purge, n)
    test_idx = np.arange(test_start, n, dtype=np.int64)
    return {"train": train_idx, "val": val_idx, "test": test_idx}


def development_walk_forward(
    n_dev: int,
    *,
    folds: int = 4,
    purge_bars: int,
) -> List[Tuple[np.ndarray, np.ndarray]]:
    """Expanding-window folds inside the development prefix only."""
    folds = max(2, int(folds))
    purge = int(purge_bars)
    edges = [int(n_dev * (i + 1) / folds) for i in range(folds)]
    out: List[Tuple[np.ndarray, np.ndarray]] = []
    for i in range(1, folds):
        train_end = edges[i - 1]
        val_end = edges[i]
        tr = np.arange(0, max(train_end - purge, 0), dtype=np.int64)
        va = np.arange(min(train_end + purge, n_dev), val_end, dtype=np.int64)
        if len(tr) == 0 or len(va) == 0:
            continue
        out.append((tr, va))
    return out


splits = purged_index_splits(
    len(labeled),
    train_fraction=float(CONFIG["train_fraction"]),
    val_fraction=float(CONFIG["val_fraction"]),
    purge_bars=int(CONFIG["purge_bars"]),
)
for key, idx in splits.items():
    print(key, "n", len(idx), "range", int(idx[0]) if len(idx) else None, int(idx[-1]) if len(idx) else None)
dev_end = int(splits["test"][0]) if len(splits["test"]) else len(labeled)
wf_folds = development_walk_forward(
    dev_end, folds=4, purge_bars=int(CONFIG["purge_bars"])
)
print("walk-forward folds", len(wf_folds), "dev_end", dev_end)


train n 90716 range 0 90715
val n 19436 range 90722 110157
test n 19435 range 110164 129598
walk-forward folds 3 dev_end 110164


In [ ]:
# CELL: DATASET
def build_windows(
    frame: pd.DataFrame,
    *,
    window_len: int,
    feature_cols: Sequence[str],
) -> Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    feats = frame.loc[:, list(feature_cols)].to_numpy(dtype=np.float64)
    y_cls = frame["trade_label"].to_numpy(dtype=np.int64)
    y_reg = frame.loc[:, ["upside", "downside"]].to_numpy(dtype=np.float64)
    w = int(window_len)
    n = len(frame)
    xs: List[np.ndarray] = []
    cs: List[int] = []
    rs: List[np.ndarray] = []
    ends: List[int] = []
    for end in range(w - 1, n):
        if y_cls[end] < 0:
            continue
        if not np.isfinite(y_reg[end]).all():
            continue
        window = feats[end - w + 1 : end + 1]
        if not np.isfinite(window).all():
            continue
        xs.append(window)
        cs.append(int(y_cls[end]))
        rs.append(y_reg[end])
        ends.append(int(end))
    if not xs:
        empty_x = np.zeros((0, w, len(feature_cols)), dtype=np.float32)
        return (
            empty_x,
            np.zeros((0,), dtype=np.int64),
            np.zeros((0, 2), dtype=np.float32),
            np.zeros((0,), dtype=np.int64),
        )
    return (
        np.stack(xs, axis=0).astype(np.float32),
        np.asarray(cs, dtype=np.int64),
        np.stack(rs, axis=0).astype(np.float32),
        np.asarray(ends, dtype=np.int64),
    )


all_x, all_y, all_r, all_ends = build_windows(
    labeled, window_len=int(CONFIG["window_len"]), feature_cols=FEATURE_COLS
)
print(
    "windows",
    all_x.shape,
    "class mix",
    {n: float(np.mean(all_y == i)) for i, n in enumerate(LABEL_NAMES)},
)


def _split_windows(split_idx: np.ndarray) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    allowed = set(int(i) for i in split_idx.tolist())
    keep = np.array([int(e) in allowed for e in all_ends], dtype=bool)
    return all_x[keep], all_y[keep], all_r[keep]


x_train, y_train, r_train = _split_windows(splits["train"])
x_val, y_val, r_val = _split_windows(splits["val"])
x_test, y_test, r_test = _split_windows(splits["test"])
print("train", x_train.shape, "val", x_val.shape, "test", x_test.shape)

feat_mean = x_train.mean(axis=(0, 1), keepdims=True)
feat_std = x_train.std(axis=(0, 1), keepdims=True)
feat_std = np.where(feat_std < 1e-6, 1.0, feat_std)
reg_mean = r_train.mean(axis=0, keepdims=True)
reg_std = r_train.std(axis=0, keepdims=True)
reg_std = np.where(reg_std < 1e-6, 1.0, reg_std)


def _scale_x(x: np.ndarray) -> np.ndarray:
    return ((x - feat_mean) / feat_std).astype(np.float32)


def _scale_r(r: np.ndarray) -> np.ndarray:
    return ((r - reg_mean) / reg_std).astype(np.float32)


def flatten(x: np.ndarray) -> np.ndarray:
    scaled = _scale_x(x)
    return scaled.reshape(len(scaled), -1)


X_train = flatten(x_train)
X_val = flatten(x_val)
X_test = flatten(x_test)
R_train = _scale_r(r_train)
R_val = _scale_r(r_val)
R_test = _scale_r(r_test)
print("flat train", X_train.shape)


windows (129518, 60, 6) class mix {'LONG': 0.1714665143068917, 'SHORT': 0.1756975864358622, 'NO_TRADE': 0.652835899257246}
train (90638, 60, 6) val (19436, 60, 6) test (19432, 60, 6)


flat train (90638, 360)


In [ ]:
# CELL: MODEL
def make_family(name: str, seed: int) -> Dict[str, Any]:
    """Return unfitted classifier + two regressors for one family."""
    est = int(CONFIG["n_estimators"])
    depth = int(CONFIG["max_depth"])
    lr = float(CONFIG["learning_rate"])
    leaf = int(CONFIG["min_leaf"])
    if name == "LR":
        clf = LogisticRegression(
            max_iter=400,
            C=0.5,
            class_weight="balanced",
            solver="lbfgs",
        )
        up = Ridge(alpha=1.0)
        dn = Ridge(alpha=1.0)
    elif name == "RF":
        clf = RandomForestClassifier(
            n_estimators=est,
            max_depth=depth,
            min_samples_leaf=leaf,
            class_weight="balanced_subsample",
            n_jobs=1,
            random_state=seed,
        )
        up = RandomForestRegressor(
            n_estimators=est,
            max_depth=depth,
            min_samples_leaf=leaf,
            n_jobs=1,
            random_state=seed,
        )
        dn = RandomForestRegressor(
            n_estimators=est,
            max_depth=depth,
            min_samples_leaf=leaf,
            n_jobs=1,
            random_state=seed + 1,
        )
    elif name == "XGB":
        clf = XGBClassifier(
            n_estimators=est,
            max_depth=depth,
            learning_rate=lr,
            objective="multi:softprob",
            eval_metric="mlogloss",
            tree_method="hist",
            n_jobs=1,
            random_state=seed,
            verbosity=0,
        )
        up = XGBRegressor(
            n_estimators=est,
            max_depth=depth,
            learning_rate=lr,
            tree_method="hist",
            n_jobs=1,
            random_state=seed,
            verbosity=0,
        )
        dn = XGBRegressor(
            n_estimators=est,
            max_depth=depth,
            learning_rate=lr,
            tree_method="hist",
            n_jobs=1,
            random_state=seed + 1,
            verbosity=0,
        )
    elif name == "LGBM":
        clf = LGBMClassifier(
            n_estimators=est,
            max_depth=depth,
            learning_rate=lr,
            min_child_samples=leaf,
            n_jobs=1,
            random_state=seed,
            verbosity=-1,
        )
        up = LGBMRegressor(
            n_estimators=est,
            max_depth=depth,
            learning_rate=lr,
            min_child_samples=leaf,
            n_jobs=1,
            random_state=seed,
            verbosity=-1,
        )
        dn = LGBMRegressor(
            n_estimators=est,
            max_depth=depth,
            learning_rate=lr,
            min_child_samples=leaf,
            n_jobs=1,
            random_state=seed + 1,
            verbosity=-1,
        )
    elif name == "CatBoost":
        clf = CatBoostClassifier(
            iterations=est,
            depth=min(depth, 8),
            learning_rate=lr,
            loss_function="MultiClass",
            auto_class_weights="Balanced",
            random_seed=seed,
            verbose=False,
            thread_count=1,
            allow_writing_files=False,
        )
        up = CatBoostRegressor(
            iterations=est,
            depth=min(depth, 8),
            learning_rate=lr,
            random_seed=seed,
            verbose=False,
            thread_count=1,
            allow_writing_files=False,
        )
        dn = CatBoostRegressor(
            iterations=est,
            depth=min(depth, 8),
            learning_rate=lr,
            random_seed=seed + 1,
            verbose=False,
            thread_count=1,
            allow_writing_files=False,
        )
    else:
        raise ValueError(name)
    return {"name": name, "clf": clf, "up": up, "dn": dn}


print("families", CONFIG["ensemble"])


families ['LR', 'RF', 'XGB', 'LGBM', 'CatBoost']


**Session Created**: 2026-10-08 18:20:26
- Endpoint: `m-s-kkb-usc1c0-3vwh3hy3vuq8f`
- Hardware: `NONE`

In [ ]:
# CELL: ENV
# Colab ships pandas/numpy; install tabular model extras.
%pip install -q --upgrade-strategy only-if-needed pyarrow requests scikit-learn xgboost lightgbm catboost

import json
import math
import pickle
import random
import time
from concurrent.futures import ThreadPoolExecutor, as_completed
from datetime import datetime, timedelta, timezone
from pathlib import Path
from typing import Any, Dict, List, Optional, Sequence, Tuple

import numpy as np
import pandas as pd
from catboost import CatBoostClassifier, CatBoostRegressor
from lightgbm import LGBMClassifier, LGBMRegressor
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.linear_model import LogisticRegression, Ridge
from xgboost import XGBClassifier, XGBRegressor

print("sklearn / xgboost / lightgbm / catboost ready")


   ━━━━━━╺━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.2/97.1 MB 154.2 MB/s eta 0:00:01

   ━━━━━━━━━━━━━━━━━━━╺━━━━━━━━━━━━━━━━━━━━ 46.9/97.1 MB 111.4 MB/s eta 0:00:01

   ━━━━━━━━━━━━━━━━━━━━━━━━━━╸━━━━━━━━━━━━━ 64.7/97.1 MB 77.4 MB/s eta 0:00:01

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╺━━━━━ 83.3/97.1 MB 111.2 MB/s eta 0:00:01

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╸ 97.1/97.1 MB 125.1 MB/s eta 0:00:01

...

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╸ 97.1/97.1 MB 125.1 MB/s eta 0:00:01

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╸ 97.1/97.1 MB 125.1 MB/s eta 0:00:01

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╸ 97.1/97.1 MB 125.1 MB/s eta 0:00:01

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 8.3 MB/s eta 0:00:00


sklearn / xgboost / lightgbm / catboost ready


In [ ]:
# CELL: CONFIG
CONFIG: Dict[str, Any] = {
    "symbol": "BTCUSD",
    "base_url": "https://api.india.delta.exchange",
    "history_days": 900,
    "resolution_5m": "5m",
    "seed": 42,
    "tp_atr_mult": 1.5,
    "sl_atr_mult": 1.0,
    "horizon_bars": 3,
    "atr_period": 14,
    "window_len": 60,
    "round_trip_cost_pct": 0.0016,
    "minority_rate": 0.15,
    "ambiguous_win_max": 0.50,
    "train_fraction": 0.70,
    "val_fraction": 0.15,
    "purge_bars": 3,
    "n_estimators": 200,
    "max_depth": 8,
    "learning_rate": 0.08,
    "min_leaf": 40,
    "ensemble": ["LR", "RF", "XGB", "LGBM", "CatBoost"],
    "export_dir": "export/opportunity_10m",
    "ready_to_promote": False,
}

FEATURE_COLS: Tuple[str, ...] = (
    "ret_1",
    "atr_pct",
    "rsi",
    "ema_dist_atr",
    "candle_body",
    "volume_z",
)
LABEL_NAMES: Tuple[str, ...] = ("LONG", "SHORT", "NO_TRADE")
LABEL_LONG, LABEL_SHORT, LABEL_NO_TRADE = 0, 1, 2
print(json.dumps(CONFIG, indent=2))


{
  "symbol": "BTCUSD",
  "base_url": "https://api.india.delta.exchange",
  "history_days": 900,
  "resolution_5m": "5m",
  "seed": 42,
  "tp_atr_mult": 1.5,
  "sl_atr_mult": 1.0,
  "horizon_bars": 3,
  "atr_period": 14,
  "window_len": 60,
  "round_trip_cost_pct": 0.0016,
  "minority_rate": 0.15,
  "ambiguous_win_max": 0.5,
  "train_fraction": 0.7,
  "val_fraction": 0.15,
  "purge_bars": 3,
  "n_estimators": 200,
  "max_depth": 8,
  "learning_rate": 0.08,
  "min_leaf": 40,
  "ensemble": [
    "LR",
    "RF",
    "XGB",
    "LGBM",
    "CatBoost"
  ],
  "export_dir": "export/opportunity_10m",
  "ready_to_promote": false
}


In [ ]:
# CELL: SEED
def set_seed(seed: int) -> None:
    random.seed(int(seed))
    np.random.seed(int(seed))


set_seed(int(CONFIG["seed"]))
print("seed", CONFIG["seed"])


seed 42


In [ ]:
# CELL: FETCH
DELTA_CANDLE_PAGE_BARS = 2000
REQUEST_DELAY_SECONDS = 0.2
EMPTY_PAGE_RETRIES = 3
MIN_OHLCV_COMPLETENESS = 0.95
_RESOLUTION_MINUTES = {"5m": 5, "10m": 10}


def _bar_seconds(resolution: str) -> int:
    if resolution not in _RESOLUTION_MINUTES:
        raise ValueError(f"Unsupported resolution {resolution!r}")
    return int(_RESOLUTION_MINUTES[resolution]) * 60


def _parse_candle_rows(payload: Any) -> List[Dict[str, Any]]:
    if not isinstance(payload, dict):
        return []
    result = payload.get("result", [])
    if isinstance(result, dict):
        result = result.get("candles", []) or []
    if not isinstance(result, list):
        return []
    return [row for row in result if isinstance(row, dict) and "time" in row]


def _candle_epoch(row: Dict[str, Any]) -> Optional[int]:
    try:
        return int(row["time"])
    except (KeyError, TypeError, ValueError):
        return None


def _request_candle_page(
    url: str,
    *,
    symbol: str,
    resolution: str,
    start_ts: int,
    end_ts: int,
) -> List[Dict[str, Any]]:
    import requests

    last_error: Optional[Exception] = None
    for attempt in range(EMPTY_PAGE_RETRIES):
        try:
            response = requests.get(
                url,
                params={
                    "symbol": symbol,
                    "resolution": resolution,
                    "start": int(start_ts),
                    "end": int(end_ts),
                },
                timeout=30,
            )
            response.raise_for_status()
            payload = response.json()
        except Exception as exc:
            last_error = exc
            time.sleep(REQUEST_DELAY_SECONDS * (attempt + 1))
            continue
        if payload.get("success") is False:
            last_error = RuntimeError(f"Delta candles error for {symbol}: {payload}")
            time.sleep(REQUEST_DELAY_SECONDS * (attempt + 1))
            continue
        rows = _parse_candle_rows(payload)
        if rows:
            return rows
        last_error = None
        time.sleep(REQUEST_DELAY_SECONDS * (attempt + 1))
    if last_error is not None:
        raise last_error
    return []


def _rows_to_frame(all_rows: List[Dict[str, Any]]) -> pd.DataFrame:
    df = pd.DataFrame(all_rows)
    expected_cols = ["time", "open", "high", "low", "close", "volume"]
    df = df[[c for c in expected_cols if c in df.columns]]
    df["time"] = pd.to_datetime(df["time"], unit="s", utc=True)
    df = df.drop_duplicates(subset="time").sort_values("time").reset_index(drop=True)
    for col in ["open", "high", "low", "close", "volume"]:
        if col in df.columns:
            df[col] = df[col].astype(float)
    return df


def fetch_candles(
    symbol: str,
    resolution: str,
    start_ts: int,
    end_ts: int,
    base_url: str,
    *,
    page_bars: int = DELTA_CANDLE_PAGE_BARS,
) -> pd.DataFrame:
    url = f"{base_url.rstrip('/')}/v2/history/candles"
    bar_seconds = _bar_seconds(resolution)
    page_bars = max(1, int(page_bars))
    start_ts, end_ts = int(start_ts), int(end_ts)
    if end_ts <= start_ts:
        raise ValueError("end_ts must be greater than start_ts")
    all_rows: List[Dict[str, Any]] = []
    cursor_end = end_ts
    seen_oldest: Optional[int] = None
    max_pages = max(2, (end_ts - start_ts) // bar_seconds + 5)
    for _ in range(max_pages):
        if cursor_end <= start_ts:
            break
        page_start = max(start_ts, cursor_end - page_bars * bar_seconds)
        if page_start >= cursor_end:
            break
        rows = _request_candle_page(
            url,
            symbol=symbol,
            resolution=resolution,
            start_ts=page_start,
            end_ts=cursor_end,
        )
        if not rows:
            break
        epochs = [e for e in (_candle_epoch(r) for r in rows) if e is not None]
        if not epochs:
            break
        oldest = min(epochs)
        if seen_oldest is not None and oldest >= seen_oldest:
            break
        seen_oldest = oldest
        all_rows.extend(rows)
        cursor_end = oldest - 1
        time.sleep(REQUEST_DELAY_SECONDS)
    if not all_rows:
        raise RuntimeError(f"No candle data returned for symbol={symbol}")
    return _rows_to_frame(all_rows)


def validate_ohlcv_completeness(
    df: pd.DataFrame,
    resolution: str,
    *,
    min_completeness: float = MIN_OHLCV_COMPLETENESS,
    symbol: str = "",
) -> Dict[str, Any]:
    if df is None or df.empty or "time" not in df.columns:
        raise ValueError("Cannot validate completeness on an empty OHLCV frame")
    bar_seconds = _bar_seconds(resolution)
    time_col = df["time"]
    epochs = (pd.to_datetime(time_col, utc=True).astype("int64") // 10**9).tolist()
    if len(epochs) < 2:
        raise ValueError("Need at least 2 OHLCV bars to validate completeness")
    gaps = 0
    for prev, cur in zip(epochs, epochs[1:]):
        if int(cur) - int(prev) > bar_seconds * 1.5:
            gaps += 1
    completeness = 1.0 - (gaps / max(1, len(epochs) - 1))
    report = {
        "rows": len(epochs),
        "gaps": int(gaps),
        "completeness": round(float(completeness), 4),
    }
    if completeness < min_completeness:
        label = f"{symbol} {resolution}".strip()
        raise ValueError(
            f"OHLCV completeness {completeness:.1%} below {min_completeness:.0%} "
            f"for {label or 'candles'}"
        )
    return report


end_dt = datetime.now(timezone.utc)
start_dt = end_dt - timedelta(days=int(CONFIG["history_days"]))
print(
    f"Fetching {CONFIG['symbol']} 5m OHLCV for {CONFIG['history_days']}d "
    f"from {CONFIG['base_url']}..."
)
df5m = fetch_candles(
    str(CONFIG["symbol"]),
    str(CONFIG["resolution_5m"]),
    int(start_dt.timestamp()),
    int(end_dt.timestamp()),
    str(CONFIG["base_url"]),
)
ohlcv_report = validate_ohlcv_completeness(
    df5m, "5m", symbol=str(CONFIG["symbol"])
)
print(
    f"OHLCV {CONFIG['symbol']} 5m: {ohlcv_report['rows']} bars, "
    f"completeness {ohlcv_report['completeness']:.1%}, gaps={ohlcv_report['gaps']}"
)


Fetching BTCUSD 5m OHLCV for 900d from https://api.india.delta.exchange...


OHLCV BTCUSD 5m: 259200 bars, completeness 100.0%, gaps=0


In [ ]:
# CELL: BARS_10M
def build_10m_ohlcv_from_5m(df5m: pd.DataFrame) -> pd.DataFrame:
    """Exchange-aligned 10m bars from two closed 5m candles. Drop a partial last bucket."""
    src = df5m.copy()
    src["time"] = pd.to_datetime(src["time"], utc=True)
    required = ("open", "high", "low", "close", "volume")
    missing = [c for c in required if c not in src.columns]
    if missing:
        raise ValueError(f"5m OHLCV missing columns: {missing}")
    if src.empty:
        return src.iloc[0:0].copy()
    grouped = src.set_index("time").sort_index()
    agg = grouped.resample("10min", label="left", closed="left").agg(
        {
            "open": "first",
            "high": "max",
            "low": "min",
            "close": "last",
            "volume": "sum",
        }
    )
    counts = grouped["close"].resample("10min", label="left", closed="left").count()
    complete = counts >= 2
    agg = agg.loc[complete].dropna(subset=["open", "high", "low", "close"])
    out = agg.reset_index()
    return out.sort_values("time").reset_index(drop=True)


bars_10m = build_10m_ohlcv_from_5m(df5m)
print(f"10m bars={len(bars_10m)} span={bars_10m['time'].iloc[0]} → {bars_10m['time'].iloc[-1]}")


10m bars=129599 span=2024-04-21 18:30:00+00:00 → 2026-10-08 18:10:00+00:00


In [ ]:
# CELL: FEATURES
def _wilder_rsi(close: pd.Series, period: int = 14) -> pd.Series:
    delta = close.diff()
    gain = delta.clip(lower=0.0)
    loss = -delta.clip(upper=0.0)
    avg_gain = gain.ewm(alpha=1.0 / period, min_periods=period, adjust=False).mean()
    avg_loss = loss.ewm(alpha=1.0 / period, min_periods=period, adjust=False).mean()
    rs = avg_gain / avg_loss.replace(0.0, np.nan)
    return 100.0 - (100.0 / (1.0 + rs))


def add_opportunity_features(
    df: pd.DataFrame,
    *,
    atr_period: int = 14,
    rsi_period: int = 14,
    ema_span: int = 20,
    vol_window: int = 20,
) -> pd.DataFrame:
    """Causal 10m features. ATR(14) is also stored for the label path."""
    out = df.copy()
    close = out["close"].astype(np.float64)
    high = out["high"].astype(np.float64)
    low = out["low"].astype(np.float64)
    open_ = out["open"].astype(np.float64)
    volume = out["volume"].astype(np.float64)
    prev_close = close.shift(1)
    tr = pd.concat(
        [
            high - low,
            (high - prev_close).abs(),
            (low - prev_close).abs(),
        ],
        axis=1,
    ).max(axis=1)
    atr = tr.rolling(int(atr_period), min_periods=int(atr_period)).mean()
    out["atr"] = atr
    out["ret_1"] = close.pct_change()
    out["atr_pct"] = atr / close.replace(0.0, np.nan)
    out["rsi"] = _wilder_rsi(close, period=int(rsi_period))
    ema = close.ewm(span=int(ema_span), adjust=False, min_periods=int(ema_span)).mean()
    out["ema_dist_atr"] = (close - ema) / atr.replace(0.0, np.nan)
    out["candle_body"] = (close - open_) / atr.replace(0.0, np.nan)
    vol_mean = volume.rolling(int(vol_window), min_periods=int(vol_window)).mean()
    vol_std = volume.rolling(int(vol_window), min_periods=int(vol_window)).std()
    out["volume_z"] = (volume - vol_mean) / vol_std.replace(0.0, np.nan)
    return out


featured = add_opportunity_features(
    bars_10m, atr_period=int(CONFIG["atr_period"])
)
print("feature null rates")
print(featured[list(FEATURE_COLS)].isna().mean().round(4))


feature null rates
ret_1           0.0000
atr_pct         0.0001
rsi             0.0001
ema_dist_atr    0.0001
candle_body     0.0001
volume_z        0.0001
dtype: float64


In [ ]:
# CELL: LABELS
# Opportunity labels on a 10-minute series. Path is bars t+1 .. t+horizon only.

import numpy as np
import pandas as pd

_EPS = 1e-9
HORIZON_BARS_DEFAULT = 3
ATR_PERIOD_DEFAULT = 14
TP_ATR_MULT_DEFAULT = 1.5
SL_ATR_MULT_DEFAULT = 1.0

TP_FIRST = 0
SL_FIRST = 1
NEITHER = 2
AMBIGUOUS = 3

LABEL_LONG = 0
LABEL_SHORT = 1
LABEL_NO_TRADE = 2
LABEL_UNLABELED = -1
LABEL_NAMES = ("LONG", "SHORT", "NO_TRADE")


def tp_sl_from_path(
    fwd_high: np.ndarray,
    fwd_low: np.ndarray,
    entry: float,
    atr: float,
    tp_mult: float,
    sl_mult: float,
    side: str,
) -> int:
    """Walk future highs/lows. Same-bar TP and SL is AMBIGUOUS, not a win."""
    high = np.asarray(fwd_high, dtype=np.float64).ravel()
    low = np.asarray(fwd_low, dtype=np.float64).ravel()
    if high.size == 0 or high.size != low.size:
        return int(NEITHER)
    denom = max(float(atr), _EPS)
    entry_f = float(entry)
    if str(side).strip().upper() == "SHORT":
        tp_level = entry_f - float(tp_mult) * denom
        sl_level = entry_f + float(sl_mult) * denom
        tp_hit = low <= tp_level
        sl_hit = high >= sl_level
    else:
        tp_level = entry_f + float(tp_mult) * denom
        sl_level = entry_f - float(sl_mult) * denom
        tp_hit = high >= tp_level
        sl_hit = low <= sl_level
    for j in range(high.size):
        hit_tp = bool(tp_hit[j])
        hit_sl = bool(sl_hit[j])
        if hit_tp and hit_sl:
            return int(AMBIGUOUS)
        if hit_tp:
            return int(TP_FIRST)
        if hit_sl:
            return int(SL_FIRST)
    return int(NEITHER)


def _side_win(code: int) -> int:
    return 1 if int(code) == int(TP_FIRST) else 0


def _causal_atr(df: pd.DataFrame, period: int = ATR_PERIOD_DEFAULT) -> np.ndarray:
    if "atr" in df.columns:
        atr = df["atr"].to_numpy(dtype=np.float64)
        if np.isfinite(atr).any():
            return atr
    close = df["close"].to_numpy(dtype=np.float64)
    high = df["high"].to_numpy(dtype=np.float64)
    low = df["low"].to_numpy(dtype=np.float64)
    prev = np.concatenate([[close[0]], close[:-1]])
    tr = np.maximum(high - low, np.maximum(np.abs(high - prev), np.abs(low - prev)))
    return pd.Series(tr).rolling(int(period), min_periods=int(period)).mean().to_numpy(
        dtype=np.float64
    )


def compute_opportunity_labels(
    df: pd.DataFrame,
    *,
    tp_mult: float = TP_ATR_MULT_DEFAULT,
    sl_mult: float = SL_ATR_MULT_DEFAULT,
    horizon_bars: int = HORIZON_BARS_DEFAULT,
    atr_period: int = ATR_PERIOD_DEFAULT,
) -> pd.DataFrame:
    """Label each closed 10m bar. Last ``horizon_bars`` rows stay unlabeled."""
    out = df.copy().reset_index(drop=True)
    n = len(out)
    k = int(horizon_bars)
    close = out["close"].to_numpy(dtype=np.float64)
    high = out["high"].to_numpy(dtype=np.float64)
    low = out["low"].to_numpy(dtype=np.float64)
    atr = _causal_atr(out, period=int(atr_period))
    n_valid = max(n - k, 0)

    trade_label = np.full(n, LABEL_UNLABELED, dtype=np.int64)
    long_result = np.full(n, np.nan, dtype=np.float64)
    short_result = np.full(n, np.nan, dtype=np.float64)
    long_code = np.full(n, np.nan, dtype=np.float64)
    short_code = np.full(n, np.nan, dtype=np.float64)
    upside = np.full(n, np.nan, dtype=np.float64)
    downside = np.full(n, np.nan, dtype=np.float64)
    timeout_ret = np.full(n, np.nan, dtype=np.float64)
    long_pnl_pct = np.full(n, np.nan, dtype=np.float64)
    short_pnl_pct = np.full(n, np.nan, dtype=np.float64)

    for i in range(n_valid):
        atr_i = float(atr[i]) if i < len(atr) else float("nan")
        if not np.isfinite(atr_i) or atr_i <= _EPS:
            continue
        entry = float(close[i])
        if not np.isfinite(entry) or abs(entry) <= _EPS:
            continue
        fwd_high = high[i + 1 : i + 1 + k]
        fwd_low = low[i + 1 : i + 1 + k]
        fwd_close = close[i + 1 : i + 1 + k]
        if fwd_high.size < k:
            continue
        l_code = tp_sl_from_path(fwd_high, fwd_low, entry, atr_i, tp_mult, sl_mult, "LONG")
        s_code = tp_sl_from_path(fwd_high, fwd_low, entry, atr_i, tp_mult, sl_mult, "SHORT")
        l_win = _side_win(l_code)
        s_win = _side_win(s_code)
        if l_win == 1 and s_win == 0:
            label = LABEL_LONG
        elif s_win == 1 and l_win == 0:
            label = LABEL_SHORT
        else:
            label = LABEL_NO_TRADE
        trade_label[i] = label
        long_result[i] = float(l_win)
        short_result[i] = float(s_win)
        long_code[i] = float(l_code)
        short_code[i] = float(s_code)
        denom = max(atr_i, _EPS)
        upside[i] = max((float(np.max(fwd_high)) - entry) / denom, 0.0)
        downside[i] = max((entry - float(np.min(fwd_low))) / denom, 0.0)
        end_close = float(fwd_close[-1])
        timeout_ret[i] = (end_close - entry) / entry
        tp_pct = float(tp_mult) * denom / entry
        sl_pct = float(sl_mult) * denom / entry
        if l_code == TP_FIRST:
            long_pnl_pct[i] = tp_pct
        elif l_code == SL_FIRST:
            long_pnl_pct[i] = -sl_pct
        elif l_code == NEITHER:
            long_pnl_pct[i] = timeout_ret[i]
        if s_code == TP_FIRST:
            short_pnl_pct[i] = tp_pct
        elif s_code == SL_FIRST:
            short_pnl_pct[i] = -sl_pct
        elif s_code == NEITHER:
            short_pnl_pct[i] = -timeout_ret[i]

    out["atr"] = atr
    out["trade_label"] = trade_label
    out["long_result"] = long_result
    out["short_result"] = short_result
    out["long_code"] = long_code
    out["short_code"] = short_code
    out["upside"] = upside
    out["downside"] = downside
    out["timeout_ret"] = timeout_ret
    out["long_pnl_pct"] = long_pnl_pct
    out["short_pnl_pct"] = short_pnl_pct
    names = np.array(["UNLABELED"] * n, dtype=object)
    names[trade_label == LABEL_LONG] = "LONG"
    names[trade_label == LABEL_SHORT] = "SHORT"
    names[trade_label == LABEL_NO_TRADE] = "NO_TRADE"
    out["trade_name"] = names
    return out


In [ ]:
# CELL: APPLY_LABELS
labeled = compute_opportunity_labels(
    featured,
    tp_mult=float(CONFIG["tp_atr_mult"]),
    sl_mult=float(CONFIG["sl_atr_mult"]),
    horizon_bars=int(CONFIG["horizon_bars"]),
    atr_period=int(CONFIG["atr_period"]),
)
valid = labeled["trade_label"].to_numpy() >= 0
print(f"labeled rows={int(valid.sum())} / {len(labeled)}")
print(labeled.loc[valid, "trade_name"].value_counts(normalize=True).round(4))


labeled rows=129583 / 129599
trade_name
NO_TRADE    0.6528
SHORT       0.1757
LONG        0.1715
Name: proportion, dtype: float64


In [ ]:
# CELL: GATE
def train_period_mask(n: int, train_fraction: float) -> np.ndarray:
    cut = int(n * float(train_fraction))
    mask = np.zeros(n, dtype=bool)
    mask[:cut] = True
    return mask


def _path_code_counts(codes: np.ndarray, usable: np.ndarray) -> Dict[str, int]:
    finite = usable & np.isfinite(codes)
    return {
        "TP_FIRST": int(np.sum(finite & (codes == TP_FIRST))),
        "SL_FIRST": int(np.sum(finite & (codes == SL_FIRST))),
        "NEITHER": int(np.sum(finite & (codes == NEITHER))),
        "AMBIGUOUS": int(np.sum(finite & (codes == AMBIGUOUS))),
    }


def summarize_label_gate(
    df: pd.DataFrame,
    *,
    train_fraction: float,
    round_trip_cost_pct: float,
    minority_rate: float,
    ambiguous_win_max: float,
) -> Dict[str, Any]:
    n = len(df)
    in_train = train_period_mask(n, train_fraction)
    labels = df["trade_label"].to_numpy(dtype=np.int64)
    usable = in_train & (labels >= 0)
    n_use = int(usable.sum())
    if n_use == 0:
        raise RuntimeError("Train-period gate has no labeled rows")
    mix = {}
    mix_ok = True
    for cid, name in enumerate(LABEL_NAMES):
        rate = float(np.mean(labels[usable] == cid))
        mix[name] = rate
        if rate < float(minority_rate):
            mix_ok = False
    long_code = df["long_code"].to_numpy(dtype=np.float64)
    short_code = df["short_code"].to_numpy(dtype=np.float64)
    cost = float(round_trip_cost_pct)
    long_pnl = df["long_pnl_pct"].to_numpy(dtype=np.float64) - cost
    short_pnl = df["short_pnl_pct"].to_numpy(dtype=np.float64) - cost

    always_long_mask = usable & np.isfinite(long_pnl)
    always_short_mask = usable & np.isfinite(short_pnl)
    always_in_long = (
        float(np.mean(long_pnl[always_long_mask])) if always_long_mask.any() else float("nan")
    )
    always_in_short = (
        float(np.mean(short_pnl[always_short_mask])) if always_short_mask.any() else float("nan")
    )

    excl_long_mask = usable & (labels == LABEL_LONG) & np.isfinite(long_pnl)
    excl_short_mask = usable & (labels == LABEL_SHORT) & np.isfinite(short_pnl)
    exclusive_long = (
        float(np.mean(long_pnl[excl_long_mask])) if excl_long_mask.any() else float("nan")
    )
    exclusive_short = (
        float(np.mean(short_pnl[excl_short_mask])) if excl_short_mask.any() else float("nan")
    )

    def _ambig_share(codes: np.ndarray) -> float:
        finite = usable & np.isfinite(codes)
        tp = finite & (codes == TP_FIRST)
        amb = finite & (codes == AMBIGUOUS)
        denom = int(tp.sum() + amb.sum())
        if denom == 0:
            return 0.0
        return float(amb.sum()) / float(denom)

    long_ambig = _ambig_share(long_code)
    short_ambig = _ambig_share(short_code)
    reasons: List[str] = []
    if not mix_ok:
        reasons.append("class_mix_below_floor")
    if not (np.isfinite(exclusive_long) and exclusive_long > 0.0):
        reasons.append("exclusive_long_expectancy_not_positive")
    if not (np.isfinite(exclusive_short) and exclusive_short > 0.0):
        reasons.append("exclusive_short_expectancy_not_positive")
    if long_ambig >= float(ambiguous_win_max):
        reasons.append("long_ambiguous_majority")
    if short_ambig >= float(ambiguous_win_max):
        reasons.append("short_ambiguous_majority")
    passed = len(reasons) == 0
    return {
        "passed": passed,
        "n_train_labeled": n_use,
        "class_mix": mix,
        "minority_rate": float(minority_rate),
        "exclusive_long_expectancy_pct": exclusive_long,
        "exclusive_short_expectancy_pct": exclusive_short,
        "always_in_long_expectancy_pct": always_in_long,
        "always_in_short_expectancy_pct": always_in_short,
        "round_trip_cost_pct": cost,
        "long_ambiguous_share": long_ambig,
        "short_ambiguous_share": short_ambig,
        "long_path_codes": _path_code_counts(long_code, usable),
        "short_path_codes": _path_code_counts(short_code, usable),
        "tp_atr_mult": float(CONFIG["tp_atr_mult"]),
        "sl_atr_mult": float(CONFIG["sl_atr_mult"]),
        "reasons": reasons,
    }


EXPORT_DIR = Path(str(CONFIG["export_dir"]))
EXPORT_DIR.mkdir(parents=True, exist_ok=True)
label_report = summarize_label_gate(
    labeled,
    train_fraction=float(CONFIG["train_fraction"]),
    round_trip_cost_pct=float(CONFIG["round_trip_cost_pct"]),
    minority_rate=float(CONFIG["minority_rate"]),
    ambiguous_win_max=float(CONFIG["ambiguous_win_max"]),
)
report_path = EXPORT_DIR / "label_report.json"
report_path.write_text(json.dumps(label_report, indent=2), encoding="utf-8")
print(json.dumps(label_report, indent=2))
print("wrote", report_path)
if not label_report["passed"]:
    raise RuntimeError(
        "Train-period label gate failed: "
        f"{label_report['reasons']}. Change CONFIG tp/sl ATR multiples and re-run "
        "from the top. Do not fit models."
    )
print("Gate passed. Training may start.")


{
  "passed": true,
  "n_train_labeled": 90706,
  "class_mix": {
    "LONG": 0.17144400590920117,
    "SHORT": 0.17492778867991093,
    "NO_TRADE": 0.653628205410888
  },
  "minority_rate": 0.15,
  "exclusive_long_expectancy_pct": 0.0018415821469474946,
  "exclusive_short_expectancy_pct": 0.0019102740710709631,
  "always_in_long_expectancy_pct": -0.001594682949987454,
  "always_in_short_expectancy_pct": -0.00159570815764566,
  "round_trip_cost_pct": 0.0016,
  "long_ambiguous_share": 0.01737646910147858,
  "short_ambiguous_share": 0.01813118811881188,
  "long_path_codes": {
    "TP_FIRST": 15551,
    "SL_FIRST": 31162,
    "NEITHER": 43718,
    "AMBIGUOUS": 275
  },
  "short_path_codes": {
    "TP_FIRST": 15867,
    "SL_FIRST": 30900,
    "NEITHER": 43646,
    "AMBIGUOUS": 293
  },
  "tp_atr_mult": 1.5,
  "sl_atr_mult": 1.0,
  "reasons": []
}
wrote export/opportunity_10m/label_report.json
Gate passed. Training may start.


In [ ]:
# CELL: SPLIT
def purged_index_splits(
    n: int,
    *,
    train_fraction: float,
    val_fraction: float,
    purge_bars: int,
) -> Dict[str, np.ndarray]:
    """Time-ordered split. Purge ``purge_bars`` on each side of every cut."""
    n = int(n)
    purge = int(purge_bars)
    train_end = int(n * float(train_fraction))
    val_len = int(n * float(val_fraction))
    train_idx = np.arange(0, max(train_end - purge, 0), dtype=np.int64)
    val_start = min(train_end + purge, n)
    val_end = min(val_start + val_len, n)
    val_idx = np.arange(val_start, max(val_end - purge, val_start), dtype=np.int64)
    test_start = min(val_end + purge, n)
    test_idx = np.arange(test_start, n, dtype=np.int64)
    return {"train": train_idx, "val": val_idx, "test": test_idx}


def development_walk_forward(
    n_dev: int,
    *,
    folds: int = 4,
    purge_bars: int,
) -> List[Tuple[np.ndarray, np.ndarray]]:
    """Expanding-window folds inside the development prefix only."""
    folds = max(2, int(folds))
    purge = int(purge_bars)
    edges = [int(n_dev * (i + 1) / folds) for i in range(folds)]
    out: List[Tuple[np.ndarray, np.ndarray]] = []
    for i in range(1, folds):
        train_end = edges[i - 1]
        val_end = edges[i]
        tr = np.arange(0, max(train_end - purge, 0), dtype=np.int64)
        va = np.arange(min(train_end + purge, n_dev), val_end, dtype=np.int64)
        if len(tr) == 0 or len(va) == 0:
            continue
        out.append((tr, va))
    return out


splits = purged_index_splits(
    len(labeled),
    train_fraction=float(CONFIG["train_fraction"]),
    val_fraction=float(CONFIG["val_fraction"]),
    purge_bars=int(CONFIG["purge_bars"]),
)
for key, idx in splits.items():
    print(key, "n", len(idx), "range", int(idx[0]) if len(idx) else None, int(idx[-1]) if len(idx) else None)
dev_end = int(splits["test"][0]) if len(splits["test"]) else len(labeled)
wf_folds = development_walk_forward(
    dev_end, folds=4, purge_bars=int(CONFIG["purge_bars"])
)
print("walk-forward folds", len(wf_folds), "dev_end", dev_end)


train n 90716 range 0 90715
val n 19436 range 90722 110157
test n 19435 range 110164 129598
walk-forward folds 3 dev_end 110164


In [ ]:
# CELL: DATASET
def build_windows(
    frame: pd.DataFrame,
    *,
    window_len: int,
    feature_cols: Sequence[str],
) -> Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    feats = frame.loc[:, list(feature_cols)].to_numpy(dtype=np.float64)
    y_cls = frame["trade_label"].to_numpy(dtype=np.int64)
    y_reg = frame.loc[:, ["upside", "downside"]].to_numpy(dtype=np.float64)
    w = int(window_len)
    n = len(frame)
    xs: List[np.ndarray] = []
    cs: List[int] = []
    rs: List[np.ndarray] = []
    ends: List[int] = []
    for end in range(w - 1, n):
        if y_cls[end] < 0:
            continue
        if not np.isfinite(y_reg[end]).all():
            continue
        window = feats[end - w + 1 : end + 1]
        if not np.isfinite(window).all():
            continue
        xs.append(window)
        cs.append(int(y_cls[end]))
        rs.append(y_reg[end])
        ends.append(int(end))
    if not xs:
        empty_x = np.zeros((0, w, len(feature_cols)), dtype=np.float32)
        return (
            empty_x,
            np.zeros((0,), dtype=np.int64),
            np.zeros((0, 2), dtype=np.float32),
            np.zeros((0,), dtype=np.int64),
        )
    return (
        np.stack(xs, axis=0).astype(np.float32),
        np.asarray(cs, dtype=np.int64),
        np.stack(rs, axis=0).astype(np.float32),
        np.asarray(ends, dtype=np.int64),
    )


all_x, all_y, all_r, all_ends = build_windows(
    labeled, window_len=int(CONFIG["window_len"]), feature_cols=FEATURE_COLS
)
print(
    "windows",
    all_x.shape,
    "class mix",
    {n: float(np.mean(all_y == i)) for i, n in enumerate(LABEL_NAMES)},
)


def _split_windows(split_idx: np.ndarray) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    allowed = set(int(i) for i in split_idx.tolist())
    keep = np.array([int(e) in allowed for e in all_ends], dtype=bool)
    return all_x[keep], all_y[keep], all_r[keep]


x_train, y_train, r_train = _split_windows(splits["train"])
x_val, y_val, r_val = _split_windows(splits["val"])
x_test, y_test, r_test = _split_windows(splits["test"])
print("train", x_train.shape, "val", x_val.shape, "test", x_test.shape)

feat_mean = x_train.mean(axis=(0, 1), keepdims=True)
feat_std = x_train.std(axis=(0, 1), keepdims=True)
feat_std = np.where(feat_std < 1e-6, 1.0, feat_std)
reg_mean = r_train.mean(axis=0, keepdims=True)
reg_std = r_train.std(axis=0, keepdims=True)
reg_std = np.where(reg_std < 1e-6, 1.0, reg_std)


def _scale_x(x: np.ndarray) -> np.ndarray:
    return ((x - feat_mean) / feat_std).astype(np.float32)


def _scale_r(r: np.ndarray) -> np.ndarray:
    return ((r - reg_mean) / reg_std).astype(np.float32)


def flatten(x: np.ndarray) -> np.ndarray:
    scaled = _scale_x(x)
    return scaled.reshape(len(scaled), -1)


X_train = flatten(x_train)
X_val = flatten(x_val)
X_test = flatten(x_test)
R_train = _scale_r(r_train)
R_val = _scale_r(r_val)
R_test = _scale_r(r_test)
print("flat train", X_train.shape)


windows (129518, 60, 6) class mix {'LONG': 0.17145107243780788, 'SHORT': 0.17572074923948794, 'NO_TRADE': 0.6528281783227042}
train (90638, 60, 6) val (19436, 60, 6) test (19432, 60, 6)


flat train (90638, 360)


In [ ]:
# CELL: MODEL
def make_family(name: str, seed: int) -> Dict[str, Any]:
    """Return unfitted classifier + two regressors for one family."""
    est = int(CONFIG["n_estimators"])
    depth = int(CONFIG["max_depth"])
    lr = float(CONFIG["learning_rate"])
    leaf = int(CONFIG["min_leaf"])
    if name == "LR":
        clf = LogisticRegression(
            max_iter=400,
            C=0.5,
            class_weight="balanced",
            solver="lbfgs",
        )
        up = Ridge(alpha=1.0)
        dn = Ridge(alpha=1.0)
    elif name == "RF":
        clf = RandomForestClassifier(
            n_estimators=est,
            max_depth=depth,
            min_samples_leaf=leaf,
            class_weight="balanced_subsample",
            n_jobs=-1,
            random_state=seed,
        )
        up = RandomForestRegressor(
            n_estimators=est,
            max_depth=depth,
            min_samples_leaf=leaf,
            n_jobs=-1,
            random_state=seed,
        )
        dn = RandomForestRegressor(
            n_estimators=est,
            max_depth=depth,
            min_samples_leaf=leaf,
            n_jobs=-1,
            random_state=seed + 1,
        )
    elif name == "XGB":
        clf = XGBClassifier(
            n_estimators=est,
            max_depth=depth,
            learning_rate=lr,
            objective="multi:softprob",
            eval_metric="mlogloss",
            tree_method="hist",
            n_jobs=1,
            random_state=seed,
            verbosity=0,
        )
        up = XGBRegressor(
            n_estimators=est,
            max_depth=depth,
            learning_rate=lr,
            tree_method="hist",
            n_jobs=1,
            random_state=seed,
            verbosity=0,
        )
        dn = XGBRegressor(
            n_estimators=est,
            max_depth=depth,
            learning_rate=lr,
            tree_method="hist",
            n_jobs=1,
            random_state=seed + 1,
            verbosity=0,
        )
    elif name == "LGBM":
        clf = LGBMClassifier(
            n_estimators=est,
            max_depth=depth,
            learning_rate=lr,
            min_child_samples=leaf,
            n_jobs=1,
            random_state=seed,
            verbosity=-1,
        )
        up = LGBMRegressor(
            n_estimators=est,
            max_depth=depth,
            learning_rate=lr,
            min_child_samples=leaf,
            n_jobs=1,
            random_state=seed,
            verbosity=-1,
        )
        dn = LGBMRegressor(
            n_estimators=est,
            max_depth=depth,
            learning_rate=lr,
            min_child_samples=leaf,
            n_jobs=1,
            random_state=seed + 1,
            verbosity=-1,
        )
    elif name == "CatBoost":
        clf = CatBoostClassifier(
            iterations=est,
            depth=min(depth, 8),
            learning_rate=lr,
            loss_function="MultiClass",
            auto_class_weights="Balanced",
            random_seed=seed,
            verbose=False,
            thread_count=1,
            allow_writing_files=False,
        )
        up = CatBoostRegressor(
            iterations=est,
            depth=min(depth, 8),
            learning_rate=lr,
            random_seed=seed,
            verbose=False,
            thread_count=1,
            allow_writing_files=False,
        )
        dn = CatBoostRegressor(
            iterations=est,
            depth=min(depth, 8),
            learning_rate=lr,
            random_seed=seed + 1,
            verbose=False,
            thread_count=1,
            allow_writing_files=False,
        )
    else:
        raise ValueError(name)
    return {"name": name, "clf": clf, "up": up, "dn": dn}


print("families", CONFIG["ensemble"])


families ['LR', 'RF', 'XGB', 'LGBM', 'CatBoost']


In [ ]:
# CELL: TRAIN
def confusion(y_true: np.ndarray, y_pred: np.ndarray, n_classes: int = 3) -> np.ndarray:
    mat = np.zeros((n_classes, n_classes), dtype=np.int64)
    for t, p in zip(y_true.tolist(), y_pred.tolist()):
        if 0 <= int(t) < n_classes and 0 <= int(p) < n_classes:
            mat[int(t), int(p)] += 1
    return mat


def class_precision(mat: np.ndarray, cls: int) -> float:
    col = mat[:, cls]
    denom = float(col.sum())
    if denom <= 0:
        return float("nan")
    return float(mat[cls, cls]) / denom


def balanced_accuracy(mat: np.ndarray) -> float:
    recalls = []
    for i in range(mat.shape[0]):
        denom = float(mat[i].sum())
        recalls.append(float(mat[i, i]) / denom if denom else float("nan"))
    finite = [r for r in recalls if np.isfinite(r)]
    return float(np.mean(finite)) if finite else float("nan")


def corr(a: np.ndarray, b: np.ndarray) -> float:
    if a.size < 3:
        return float("nan")
    if float(np.std(a)) < 1e-12 or float(np.std(b)) < 1e-12:
        return float("nan")
    return float(np.corrcoef(a, b)[0, 1])


def as_cls(arr: np.ndarray) -> np.ndarray:
    return np.asarray(arr).reshape(-1).astype(np.int64)


def as_1d(arr: np.ndarray) -> np.ndarray:
    return np.asarray(arr, dtype=np.float64).reshape(-1)


def classifier_report(y_true: np.ndarray, y_pred: np.ndarray) -> Dict[str, float]:
    cm = confusion(y_true, y_pred)
    p_long = class_precision(cm, LABEL_LONG)
    p_short = class_precision(cm, LABEL_SHORT)
    n_long = int(np.sum(y_pred == LABEL_LONG))
    n_short = int(np.sum(y_pred == LABEL_SHORT))
    bacc = balanced_accuracy(cm)
    return {
        "precision_LONG": p_long,
        "precision_SHORT": p_short,
        "n_LONG": float(n_long),
        "n_SHORT": float(n_short),
        "balanced_accuracy": bacc,
        "confusion": cm.tolist(),
    }


def signal_expectancy(
    pred_cls: np.ndarray,
    split_idx: np.ndarray,
) -> Dict[str, float]:
    allowed = set(int(i) for i in split_idx.tolist())
    keep = np.array([int(e) in allowed for e in all_ends], dtype=bool)
    row_idx = all_ends[keep]
    cost = float(CONFIG["round_trip_cost_pct"])
    out: Dict[str, float] = {}
    for cls, name, pnl_col in (
        (LABEL_LONG, "LONG", "long_pnl_pct"),
        (LABEL_SHORT, "SHORT", "short_pnl_pct"),
    ):
        take = pred_cls == cls
        if not bool(take.any()):
            out[f"{name}_n"] = 0.0
            out[f"{name}_expectancy"] = float("nan")
            continue
        pnl = labeled.loc[row_idx[take], pnl_col].to_numpy(dtype=np.float64) - cost
        finite = np.isfinite(pnl)
        out[f"{name}_n"] = float(finite.sum())
        out[f"{name}_expectancy"] = float(np.mean(pnl[finite])) if finite.any() else float("nan")
    return out


def expectancy_rank_score(exp_report: Dict[str, float]) -> float:
    """Mean of sides with at least 20 predicted trades; else the available side."""
    parts: List[float] = []
    n_long = float(exp_report.get("LONG_n", 0.0))
    n_short = float(exp_report.get("SHORT_n", 0.0))
    e_long = float(exp_report.get("LONG_expectancy", float("nan")))
    e_short = float(exp_report.get("SHORT_expectancy", float("nan")))
    if n_long >= 20 and np.isfinite(e_long):
        parts.append(e_long)
    if n_short >= 20 and np.isfinite(e_short):
        parts.append(e_short)
    if parts:
        return float(np.mean(parts))
    return float("-inf")


def fit_family(name: str) -> Dict[str, Any]:
    pack = make_family(name, int(CONFIG["seed"]))
    pack["clf"].fit(X_train, y_train)
    pack["up"].fit(X_train, R_train[:, 0])
    pack["dn"].fit(X_train, R_train[:, 1])
    pred = as_cls(pack["clf"].predict(X_val))
    cls_rep = classifier_report(y_val, pred)
    exp_rep = signal_expectancy(pred, splits["val"])
    rank = expectancy_rank_score(exp_rep)
    cls_rep["score"] = rank
    up_hat = as_1d(pack["up"].predict(X_val))
    dn_hat = as_1d(pack["dn"].predict(X_val))
    up_c = corr(up_hat, R_val[:, 0])
    dn_c = corr(dn_hat, R_val[:, 1])
    reg_score = float(np.nanmean([up_c, dn_c]))
    print(
        f"{name} val  exp_score={rank:.6f}  "
        f"E_LONG={exp_rep['LONG_expectancy']} nL={exp_rep['LONG_n']:.0f}  "
        f"E_SHORT={exp_rep['SHORT_expectancy']} nS={exp_rep['SHORT_n']:.0f}  "
        f"P_LONG={cls_rep['precision_LONG']:.3f}  P_SHORT={cls_rep['precision_SHORT']:.3f}  "
        f"reg={reg_score:.3f}"
    )
    return {
        "name": name,
        "pack": pack,
        "cls": cls_rep,
        "exp": exp_rep,
        "upside_corr": up_c,
        "downside_corr": dn_c,
        "reg_score": reg_score,
    }


print("Fitting", CONFIG["ensemble"], "in parallel...")
fitted: Dict[str, Dict[str, Any]] = {}
with ThreadPoolExecutor(max_workers=len(CONFIG["ensemble"])) as pool:
    futs = {pool.submit(fit_family, name): name for name in CONFIG["ensemble"]}
    for fut in as_completed(futs):
        name = futs[fut]
        fitted[name] = fut.result()

cls_winner = max(fitted.values(), key=lambda r: float(r["cls"]["score"]))
reg_winner = max(fitted.values(), key=lambda r: float(r["reg_score"]))
print("best classifier", cls_winner["name"], "score", cls_winner["cls"]["score"], cls_winner["exp"])
print("best regressor family", reg_winner["name"], "reg_score", reg_winner["reg_score"])
leaderboard = {
    name: {
        "cls": {k: v for k, v in row["cls"].items() if k != "confusion"},
        "exp": row["exp"],
        "upside_corr": row["upside_corr"],
        "downside_corr": row["downside_corr"],
        "reg_score": row["reg_score"],
    }
    for name, row in fitted.items()
}
print(json.dumps(leaderboard, indent=2, default=float))


Fitting ['LR', 'RF', 'XGB', 'LGBM', 'CatBoost'] in parallel...


/usr/local/lib/python3.13/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


LR val  exp_score=-0.001598  E_LONG=-0.001586893223607647 nL=5295  E_SHORT=-0.0016092386956595465 nS=5321  P_LONG=0.200  P_SHORT=0.225  reg=0.094


/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


LGBM val  exp_score=-0.001714  E_LONG=-0.001450114501507271 nL=145  E_SHORT=-0.001977275839392658 nS=131  P_LONG=0.336  P_SHORT=0.301  reg=0.221


XGB val  exp_score=-0.001607  E_LONG=-0.0014156224205640966 nL=172  E_SHORT=-0.0017984090172824302 nS=177  P_LONG=0.316  P_SHORT=0.309  reg=0.147


CatBoost val  exp_score=-0.001589  E_LONG=-0.0015852568320530583 nL=4862  E_SHORT=-0.0015920553669938353 nS=4632  P_LONG=0.211  P_SHORT=0.236  reg=0.176


RF val  exp_score=-0.001592  E_LONG=-0.00159767898726567 nL=5617  E_SHORT=-0.0015872031222569159 nS=4302  P_LONG=0.207  P_SHORT=0.231  reg=0.189
best classifier CatBoost score -0.0015886560995234467 {'LONG_n': 4862.0, 'LONG_expectancy': -0.0015852568320530583, 'SHORT_n': 4632.0, 'SHORT_expectancy': -0.0015920553669938353}
best regressor family LGBM reg_score 0.22112526907671093
{
  "LR": {
    "cls": {
      "precision_LONG": 0.19988728160811572,
      "precision_SHORT": 0.22533632286995517,
      "n_LONG": 5323.0,
      "n_SHORT": 5352.0,
      "balanced_accuracy": 0.3949204963576038,
      "score": -0.0015980659596335966
    },
    "exp": {
      "LONG_n": 5295.0,
      "LONG_expectancy": -0.001586893223607647,
      "SHORT_n": 5321.0,
      "SHORT_expectancy": -0.0016092386956595465
    },
    "upside_corr": 0.02284949562034056,
    "downside_corr": 0.164371276854251,
    "reg_score": 0.09361038623729578
  },
  "LGBM": {
    "cls": {
      "precision_LONG": 0.3356164383561644,
     

In [ ]:
# CELL: EVAL
def _unscale_reg(pred: np.ndarray, col: int) -> np.ndarray:
    return as_1d(pred) * float(reg_std[0, col]) + float(reg_mean[0, col])


best_clf = fitted[cls_winner["name"]]["pack"]["clf"]
best_up = fitted[reg_winner["name"]]["pack"]["up"]
best_dn = fitted[reg_winner["name"]]["pack"]["dn"]
pred_cls = as_cls(best_clf.predict(X_test))
up_hat = _unscale_reg(best_up.predict(X_test), 0)
dn_hat = _unscale_reg(best_dn.predict(X_test), 1)
r_true = r_test
cm = confusion(y_test, pred_cls)
print("TEST winner clf=", cls_winner["name"], "reg=", reg_winner["name"])
print("test confusion rows=true cols=pred LONG/SHORT/NO_TRADE")
print(cm)
member_test = {}
for name, row in fitted.items():
    p = as_cls(row["pack"]["clf"].predict(X_test))
    member_test[name] = classifier_report(y_test, p)
    member_test[name]["exp"] = signal_expectancy(p, splits["test"])
    member_test[name]["score"] = expectancy_rank_score(member_test[name]["exp"])
    member_test[name]["upside_corr"] = corr(
        _unscale_reg(row["pack"]["up"].predict(X_test), 0), r_true[:, 0]
    )
    member_test[name]["downside_corr"] = corr(
        _unscale_reg(row["pack"]["dn"].predict(X_test), 1), r_true[:, 1]
    )
    print(
        name,
        "test",
        {
            "score": member_test[name]["score"],
            "exp": member_test[name]["exp"],
            "precision_LONG": member_test[name]["precision_LONG"],
            "precision_SHORT": member_test[name]["precision_SHORT"],
        },
    )

cls_rep = classifier_report(y_test, pred_cls)
exp_report = signal_expectancy(pred_cls, splits["test"])
eval_report = {
    "n_test": int(len(y_test)),
    "best_classifier": cls_winner["name"],
    "best_regressor_family": reg_winner["name"],
    "val_leaderboard": leaderboard,
    "test_members": {
        k: {kk: vv for kk, vv in v.items() if kk != "confusion"}
        for k, v in member_test.items()
    },
    "confusion": cm.tolist(),
    "precision_LONG": cls_rep["precision_LONG"],
    "precision_SHORT": cls_rep["precision_SHORT"],
    "n_LONG": cls_rep["n_LONG"],
    "n_SHORT": cls_rep["n_SHORT"],
    "balanced_accuracy": cls_rep["balanced_accuracy"],
    "upside_corr": corr(up_hat, r_true[:, 0]),
    "downside_corr": corr(dn_hat, r_true[:, 1]),
    "signal_expectancy": exp_report,
    "score": expectancy_rank_score(exp_report),
    "note": (
        "Best classifier is chosen on validation model-selected LONG/SHORT "
        "expectancy (min 20 predicted trades per side when both exist). "
        "Test is scored once. Balanced accuracy is diagnostic only."
    ),
}
print(json.dumps(eval_report, indent=2, default=float))


/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


TEST winner clf= CatBoost reg= LGBM
test confusion rows=true cols=pred LONG/SHORT/NO_TRADE
[[1372 1049 1018]
 [1252 1185 1014]
 [3549 2993 6000]]


LR test {'score': -0.0015894477837699464, 'exp': {'LONG_n': 6258.0, 'LONG_expectancy': -0.0016158911637463571, 'SHORT_n': 5533.0, 'SHORT_expectancy': -0.0015630044037935355}, 'precision_LONG': 0.2113885796087164, 'precision_SHORT': 0.22402013664149587}


/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


...

/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


LGBM test {'score': -0.001517581674092551, 'exp': {'LONG_n': 316.0, 'LONG_expectancy': -0.0015147655859918375, 'SHORT_n': 168.0, 'SHORT_expectancy': -0.0015203977621932642}, 'precision_LONG': 0.29283489096573206, 'precision_SHORT': 0.3236994219653179}


XGB test {'score': -0.0016322131627059286, 'exp': {'LONG_n': 340.0, 'LONG_expectancy': -0.0016205327310305257, 'SHORT_n': 245.0, 'SHORT_expectancy': -0.0016438935943813317}, 'precision_LONG': 0.26011560693641617, 'precision_SHORT': 0.2857142857142857}


CatBoost test {'score': -0.0015781009800271634, 'exp': {'LONG_n': 6144.0, 'LONG_expectancy': -0.0015908915881215804, 'SHORT_n': 5200.0, 'SHORT_expectancy': -0.0015653103719327464}, 'precision_LONG': 0.22225822128624656, 'precision_SHORT': 0.22670748039028124}


RF test {'score': -0.0015844716504328173, 'exp': {'LONG_n': 7642.0, 'LONG_expectancy': -0.0015794372897853198, 'SHORT_n': 4850.0, 'SHORT_expectancy': -0.0015895060110803147}, 'precision_LONG': 0.2144440099074436, 'precision_SHORT': 0.22005742411812962}
{
  "n_test": 19432,
  "best_classifier": "CatBoost",
  "best_regressor_family": "LGBM",
  "val_leaderboard": {
    "LR": {
      "cls": {
        "precision_LONG": 0.19988728160811572,
        "precision_SHORT": 0.22533632286995517,
        "n_LONG": 5323.0,
        "n_SHORT": 5352.0,
        "balanced_accuracy": 0.3949204963576038,
        "score": -0.0015980659596335966
      },
      "exp": {
        "LONG_n": 5295.0,
        "LONG_expectancy": -0.001586893223607647,
        "SHORT_n": 5321.0,
        "SHORT_expectancy": -0.0016092386956595465
      },
      "upside_corr": 0.02284949562034056,
      "downside_corr": 0.164371276854251,
      "reg_score": 0.09361038623729578
    },
    "LGBM": {
      "cls": {
        "precision_LONG":

In [ ]:
# CELL: WALK_FORWARD
wf_rows: List[Dict[str, Any]] = []
winner_name = cls_winner["name"]
for fold_i, (tr_idx, va_idx) in enumerate(wf_folds, start=1):
    x_tr, y_tr, r_tr = _split_windows(tr_idx)
    x_va, y_va, r_va = _split_windows(va_idx)
    if len(x_tr) < 32 or len(x_va) < 8:
        continue
    mean = x_tr.mean(axis=(0, 1), keepdims=True)
    std = x_tr.std(axis=(0, 1), keepdims=True)
    std = np.where(std < 1e-6, 1.0, std)
    X_tr = ((x_tr - mean) / std).reshape(len(x_tr), -1)
    X_va = ((x_va - mean) / std).reshape(len(x_va), -1)
    pack = make_family(winner_name, int(CONFIG["seed"]) + fold_i)
    pack["clf"].fit(X_tr, y_tr)
    pred = as_cls(pack["clf"].predict(X_va))
    rep = classifier_report(y_va, pred)
    exp_rep = signal_expectancy(pred, va_idx)
    score = expectancy_rank_score(exp_rep)
    wf_rows.append(
        {
            "fold": fold_i,
            "family": winner_name,
            "n_train": int(len(x_tr)),
            "n_val": int(len(x_va)),
            "score": score,
            "precision_LONG": rep["precision_LONG"],
            "precision_SHORT": rep["precision_SHORT"],
            "n_LONG": rep["n_LONG"],
            "n_SHORT": rep["n_SHORT"],
            "balanced_accuracy": rep["balanced_accuracy"],
            "signal_expectancy": exp_rep,
        }
    )
    print("fold", fold_i, wf_rows[-1])
print(json.dumps({"walk_forward": wf_rows}, indent=2, default=float))


fold 1 {'fold': 1, 'family': 'CatBoost', 'n_train': 27460, 'n_val': 27538, 'score': -0.0016219113415702298, 'precision_LONG': 0.20938680616099972, 'precision_SHORT': 0.2151876534549281, 'n_LONG': 6882.0, 'n_SHORT': 5702.0, 'balanced_accuracy': 0.3918537384406209, 'signal_expectancy': {'LONG_n': 6851.0, 'LONG_expectancy': -0.0015715002331843668, 'SHORT_n': 5683.0, 'SHORT_expectancy': -0.001672322449956093}}


fold 2 {'fold': 2, 'family': 'CatBoost', 'n_train': 55001, 'n_val': 27538, 'score': -0.0016116224530173379, 'precision_LONG': 0.20844944822204883, 'precision_SHORT': 0.22124508215690813, 'n_LONG': 8971.0, 'n_SHORT': 8642.0, 'balanced_accuracy': 0.3969817948139962, 'signal_expectancy': {'LONG_n': 8941.0, 'LONG_expectancy': -0.0016438027008999858, 'SHORT_n': 8606.0, 'SHORT_expectancy': -0.00157944220513469}}


fold 3 {'fold': 3, 'family': 'CatBoost', 'n_train': 82542, 'n_val': 27538, 'score': -0.0016004329754505634, 'precision_LONG': 0.2124982327159621, 'precision_SHORT': 0.21928391026593375, 'n_LONG': 7073.0, 'n_SHORT': 6731.0, 'balanced_accuracy': 0.3973101113123905, 'signal_expectancy': {'LONG_n': 7029.0, 'LONG_expectancy': -0.0015741570805789592, 'SHORT_n': 6683.0, 'SHORT_expectancy': -0.0016267088703221673}}
{
  "walk_forward": [
    {
      "fold": 1,
      "family": "CatBoost",
      "n_train": 27460,
      "n_val": 27538,
      "score": -0.0016219113415702298,
      "precision_LONG": 0.20938680616099972,
      "precision_SHORT": 0.2151876534549281,
      "n_LONG": 6882.0,
      "n_SHORT": 5702.0,
      "balanced_accuracy": 0.3918537384406209,
      "signal_expectancy": {
        "LONG_n": 6851.0,
        "LONG_expectancy": -0.0015715002331843668,
        "SHORT_n": 5683.0,
        "SHORT_expectancy": -0.001672322449956093
      }
    },
    {
      "fold": 2,
      "family": "CatBoos

In [ ]:
# CELL: EXPORT
EXPORT_DIR.mkdir(parents=True, exist_ok=True)
bundle = {
    "best_classifier": cls_winner["name"],
    "best_regressor_family": reg_winner["name"],
    "clf": best_clf,
    "up": best_up,
    "dn": best_dn,
    "feat_mean": feat_mean,
    "feat_std": feat_std,
    "reg_mean": reg_mean,
    "reg_std": reg_std,
    "feature_cols": list(FEATURE_COLS),
    "window_len": int(CONFIG["window_len"]),
}
with (EXPORT_DIR / "best_models.pkl").open("wb") as handle:
    pickle.dump(bundle, handle)
(EXPORT_DIR / "leaderboard.json").write_text(
    json.dumps(leaderboard, indent=2, default=float), encoding="utf-8"
)
metadata = {
    "model_family": "jacksparrow_opportunity_10m_tabular_ensemble",
    "ensemble": list(CONFIG["ensemble"]),
    "best_classifier": cls_winner["name"],
    "best_regressor_family": reg_winner["name"],
    "timeframe": "10m",
    "horizon_bars": int(CONFIG["horizon_bars"]),
    "horizon_minutes": 30,
    "tp_atr_mult": float(CONFIG["tp_atr_mult"]),
    "sl_atr_mult": float(CONFIG["sl_atr_mult"]),
    "round_trip_cost_pct": float(CONFIG["round_trip_cost_pct"]),
    "window_len": int(CONFIG["window_len"]),
    "feature_cols": list(FEATURE_COLS),
    "label_names": list(LABEL_NAMES),
    "ready_to_promote": False,
    "promote_reason": "research_opportunity_10m_not_live",
    "eval": eval_report,
    "label_gate": label_report,
    "walk_forward": wf_rows,
}
(EXPORT_DIR / "metadata.json").write_text(
    json.dumps(metadata, indent=2, default=float), encoding="utf-8"
)
(EXPORT_DIR / "eval_report.json").write_text(
    json.dumps(eval_report, indent=2, default=float), encoding="utf-8"
)
print("wrote", EXPORT_DIR / "best_models.pkl")
print("best classifier", cls_winner["name"], "best regressor", reg_winner["name"])
print("ready_to_promote=false")
try:
    from google.colab import files
    import shutil

    shutil.make_archive(str(EXPORT_DIR), "zip", root_dir=str(EXPORT_DIR))
    files.download(str(EXPORT_DIR) + ".zip")
except ImportError:
    print("Not Colab; files remain under", EXPORT_DIR)


wrote export/opportunity_10m/best_models.pkl
best classifier CatBoost best regressor LGBM
ready_to_promote=false


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

**Session Created**: 2026-10-09 09:51:47
- Endpoint: `m-s-kkb-usc1b1-tk3a03fydy1l`
- Hardware: `NONE`

In [ ]:
# CELL: ENV
# Colab ships pandas/numpy; install tabular model extras.
%pip install -q --upgrade-strategy only-if-needed pyarrow requests scikit-learn xgboost lightgbm catboost

import json
import math
import pickle
import random
import time
from concurrent.futures import ThreadPoolExecutor, as_completed
from datetime import datetime, timedelta, timezone
from pathlib import Path
from typing import Any, Dict, List, Optional, Sequence, Tuple

import numpy as np
import pandas as pd
from catboost import CatBoostClassifier, CatBoostRegressor
from lightgbm import LGBMClassifier, LGBMRegressor
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.linear_model import LogisticRegression, Ridge
from xgboost import XGBClassifier, XGBRegressor

print("sklearn / xgboost / lightgbm / catboost ready")


   ━╺━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/97.1 MB 45.8 MB/s eta 0:00:03

   ━━━━━━━━━━━━━━╺━━━━━━━━━━━━━━━━━━━━━━━━━ 34.8/97.1 MB 67.3 MB/s eta 0:00:01

   ━━━━━━━━━━━━━━━━━━━━━╸━━━━━━━━━━━━━━━━━━ 53.3/97.1 MB 90.5 MB/s eta 0:00:01

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━╸━━━━━━━━━━━ 69.8/97.1 MB 55.5 MB/s eta 0:00:01

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╸━━━━━━ 82.0/97.1 MB 53.5 MB/s eta 0:00:01

...

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╸ 97.1/97.1 MB 50.2 MB/s eta 0:00:01

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╸ 97.1/97.1 MB 50.2 MB/s eta 0:00:01

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╸ 97.1/97.1 MB 50.2 MB/s eta 0:00:01

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 6.6 MB/s eta 0:00:00


sklearn / xgboost / lightgbm / catboost ready


In [ ]:
# CELL: CONFIG
CONFIG: Dict[str, Any] = {
    "symbol": "BTCUSD",
    "base_url": "https://api.india.delta.exchange",
    "history_days": 900,
    "resolution_5m": "5m",
    "seed": 42,
    # Bracket. TP must clear the round-trip cost by a wide margin, otherwise no
    # classifier can be profitable (see the notebook header).
    "tp_atr_mult": 4.0,
    "sl_atr_mult": 2.0,
    "horizon_bars": 24,
    "atr_period": 14,
    "round_trip_cost_pct": 0.0016,
    # Gate: median TP distance >= ratio x cost. Signal rule: a candle is only
    # eligible when its own TP distance >= ratio x cost (quiet-market guard).
    "min_tp_cost_ratio": 4.0,
    "minority_rate": 0.15,
    "ambiguous_win_max": 0.50,
    "train_fraction": 0.70,
    "val_fraction": 0.15,
    # Decision rule: trade only when P(side) clears a threshold chosen on
    # calibration rows. Candidates are probability quantiles of the calibration set.
    "calib_fraction": 0.15,
    "min_signals": 100,
    "signal_quantiles": [0.99, 0.98, 0.97, 0.95, 0.92, 0.90, 0.85, 0.80],
    "walk_forward_folds": 4,
    # Promotion gate thresholds (held-out test + walk-forward).
    "promote_min_tstat": 2.0,
    "promote_min_wf_positive_share": 0.67,
    "cost_sensitivity_pct": [0.0010, 0.0016, 0.0020],
    # Model capacity. Deliberately regularised: ~90k noisy rows, overlapping labels.
    "n_estimators": 400,
    "max_depth": 6,
    "learning_rate": 0.03,
    "min_leaf": 200,
    "threads_per_model": 2,
    "ensemble": ["LR", "RF", "XGB", "LGBM", "CatBoost"],
    "export_dir": "export/opportunity_10m",
}
# Purge must cover the label look-ahead so no training label peeks into validation.
CONFIG["purge_bars"] = int(CONFIG["horizon_bars"]) + 1

LABEL_NAMES: Tuple[str, ...] = ("LONG", "SHORT", "NO_TRADE")
LABEL_LONG, LABEL_SHORT, LABEL_NO_TRADE = 0, 1, 2
ENSEMBLE_NAME = "ENSEMBLE"
print(json.dumps(CONFIG, indent=2))


{
  "symbol": "BTCUSD",
  "base_url": "https://api.india.delta.exchange",
  "history_days": 900,
  "resolution_5m": "5m",
  "seed": 42,
  "tp_atr_mult": 4.0,
  "sl_atr_mult": 2.0,
  "horizon_bars": 24,
  "atr_period": 14,
  "round_trip_cost_pct": 0.0016,
  "min_tp_cost_ratio": 4.0,
  "minority_rate": 0.15,
  "ambiguous_win_max": 0.5,
  "train_fraction": 0.7,
  "val_fraction": 0.15,
  "calib_fraction": 0.15,
  "min_signals": 100,
  "signal_quantiles": [
    0.99,
    0.98,
    0.97,
    0.95,
    0.92,
    0.9,
    0.85,
    0.8
  ],
  "walk_forward_folds": 4,
  "promote_min_tstat": 2.0,
  "promote_min_wf_positive_share": 0.67,
  "cost_sensitivity_pct": [
    0.001,
    0.0016,
    0.002
  ],
  "n_estimators": 400,
  "max_depth": 6,
  "learning_rate": 0.03,
  "min_leaf": 200,
  "threads_per_model": 2,
  "ensemble": [
    "LR",
    "RF",
    "XGB",
    "LGBM",
    "CatBoost"
  ],
  "export_dir": "export/opportunity_10m",
  "purge_bars": 25
}


In [ ]:
# CELL: SEED
def set_seed(seed: int) -> None:
    random.seed(int(seed))
    np.random.seed(int(seed))


set_seed(int(CONFIG["seed"]))
print("seed", CONFIG["seed"])


seed 42


In [ ]:
# CELL: FETCH
DELTA_CANDLE_PAGE_BARS = 2000
REQUEST_DELAY_SECONDS = 0.2
EMPTY_PAGE_RETRIES = 3
MIN_OHLCV_COMPLETENESS = 0.95
_RESOLUTION_MINUTES = {"5m": 5, "10m": 10}


def _bar_seconds(resolution: str) -> int:
    if resolution not in _RESOLUTION_MINUTES:
        raise ValueError(f"Unsupported resolution {resolution!r}")
    return int(_RESOLUTION_MINUTES[resolution]) * 60


def _parse_candle_rows(payload: Any) -> List[Dict[str, Any]]:
    if not isinstance(payload, dict):
        return []
    result = payload.get("result", [])
    if isinstance(result, dict):
        result = result.get("candles", []) or []
    if not isinstance(result, list):
        return []
    return [row for row in result if isinstance(row, dict) and "time" in row]


def _candle_epoch(row: Dict[str, Any]) -> Optional[int]:
    try:
        return int(row["time"])
    except (KeyError, TypeError, ValueError):
        return None


def _request_candle_page(
    url: str,
    *,
    symbol: str,
    resolution: str,
    start_ts: int,
    end_ts: int,
) -> List[Dict[str, Any]]:
    import requests

    last_error: Optional[Exception] = None
    for attempt in range(EMPTY_PAGE_RETRIES):
        try:
            response = requests.get(
                url,
                params={
                    "symbol": symbol,
                    "resolution": resolution,
                    "start": int(start_ts),
                    "end": int(end_ts),
                },
                timeout=30,
            )
            response.raise_for_status()
            payload = response.json()
        except Exception as exc:
            last_error = exc
            time.sleep(REQUEST_DELAY_SECONDS * (attempt + 1))
            continue
        if payload.get("success") is False:
            last_error = RuntimeError(f"Delta candles error for {symbol}: {payload}")
            time.sleep(REQUEST_DELAY_SECONDS * (attempt + 1))
            continue
        rows = _parse_candle_rows(payload)
        if rows:
            return rows
        last_error = None
        time.sleep(REQUEST_DELAY_SECONDS * (attempt + 1))
    if last_error is not None:
        raise last_error
    return []


def _rows_to_frame(all_rows: List[Dict[str, Any]]) -> pd.DataFrame:
    df = pd.DataFrame(all_rows)
    expected_cols = ["time", "open", "high", "low", "close", "volume"]
    df = df[[c for c in expected_cols if c in df.columns]]
    df["time"] = pd.to_datetime(df["time"], unit="s", utc=True)
    df = df.drop_duplicates(subset="time").sort_values("time").reset_index(drop=True)
    for col in ["open", "high", "low", "close", "volume"]:
        if col in df.columns:
            df[col] = df[col].astype(float)
    return df


def fetch_candles(
    symbol: str,
    resolution: str,
    start_ts: int,
    end_ts: int,
    base_url: str,
    *,
    page_bars: int = DELTA_CANDLE_PAGE_BARS,
) -> pd.DataFrame:
    url = f"{base_url.rstrip('/')}/v2/history/candles"
    bar_seconds = _bar_seconds(resolution)
    page_bars = max(1, int(page_bars))
    start_ts, end_ts = int(start_ts), int(end_ts)
    if end_ts <= start_ts:
        raise ValueError("end_ts must be greater than start_ts")
    all_rows: List[Dict[str, Any]] = []
    cursor_end = end_ts
    seen_oldest: Optional[int] = None
    max_pages = max(2, (end_ts - start_ts) // bar_seconds + 5)
    for _ in range(max_pages):
        if cursor_end <= start_ts:
            break
        page_start = max(start_ts, cursor_end - page_bars * bar_seconds)
        if page_start >= cursor_end:
            break
        rows = _request_candle_page(
            url,
            symbol=symbol,
            resolution=resolution,
            start_ts=page_start,
            end_ts=cursor_end,
        )
        if not rows:
            break
        epochs = [e for e in (_candle_epoch(r) for r in rows) if e is not None]
        if not epochs:
            break
        oldest = min(epochs)
        if seen_oldest is not None and oldest >= seen_oldest:
            break
        seen_oldest = oldest
        all_rows.extend(rows)
        cursor_end = oldest - 1
        time.sleep(REQUEST_DELAY_SECONDS)
    if not all_rows:
        raise RuntimeError(f"No candle data returned for symbol={symbol}")
    return _rows_to_frame(all_rows)


def validate_ohlcv_completeness(
    df: pd.DataFrame,
    resolution: str,
    *,
    min_completeness: float = MIN_OHLCV_COMPLETENESS,
    symbol: str = "",
) -> Dict[str, Any]:
    if df is None or df.empty or "time" not in df.columns:
        raise ValueError("Cannot validate completeness on an empty OHLCV frame")
    bar_seconds = _bar_seconds(resolution)
    time_col = df["time"]
    epochs = (pd.to_datetime(time_col, utc=True).astype("int64") // 10**9).tolist()
    if len(epochs) < 2:
        raise ValueError("Need at least 2 OHLCV bars to validate completeness")
    gaps = 0
    for prev, cur in zip(epochs, epochs[1:]):
        if int(cur) - int(prev) > bar_seconds * 1.5:
            gaps += 1
    completeness = 1.0 - (gaps / max(1, len(epochs) - 1))
    report = {
        "rows": len(epochs),
        "gaps": int(gaps),
        "completeness": round(float(completeness), 4),
    }
    if completeness < min_completeness:
        label = f"{symbol} {resolution}".strip()
        raise ValueError(
            f"OHLCV completeness {completeness:.1%} below {min_completeness:.0%} "
            f"for {label or 'candles'}"
        )
    return report


end_dt = datetime.now(timezone.utc)
start_dt = end_dt - timedelta(days=int(CONFIG["history_days"]))
print(
    f"Fetching {CONFIG['symbol']} 5m OHLCV for {CONFIG['history_days']}d "
    f"from {CONFIG['base_url']}..."
)
df5m = fetch_candles(
    str(CONFIG["symbol"]),
    str(CONFIG["resolution_5m"]),
    int(start_dt.timestamp()),
    int(end_dt.timestamp()),
    str(CONFIG["base_url"]),
)
ohlcv_report = validate_ohlcv_completeness(
    df5m, "5m", symbol=str(CONFIG["symbol"])
)
print(
    f"OHLCV {CONFIG['symbol']} 5m: {ohlcv_report['rows']} bars, "
    f"completeness {ohlcv_report['completeness']:.1%}, gaps={ohlcv_report['gaps']}"
)


Fetching BTCUSD 5m OHLCV for 900d from https://api.india.delta.exchange...


OHLCV BTCUSD 5m: 259200 bars, completeness 100.0%, gaps=0


In [ ]:
# CELL: BARS_10M
def build_10m_ohlcv_from_5m(df5m: pd.DataFrame) -> pd.DataFrame:
    """Exchange-aligned 10m bars from two closed 5m candles. Drop a partial last bucket."""
    src = df5m.copy()
    src["time"] = pd.to_datetime(src["time"], utc=True)
    required = ("open", "high", "low", "close", "volume")
    missing = [c for c in required if c not in src.columns]
    if missing:
        raise ValueError(f"5m OHLCV missing columns: {missing}")
    if src.empty:
        return src.iloc[0:0].copy()
    grouped = src.set_index("time").sort_index()
    agg = grouped.resample("10min", label="left", closed="left").agg(
        {
            "open": "first",
            "high": "max",
            "low": "min",
            "close": "last",
            "volume": "sum",
        }
    )
    counts = grouped["close"].resample("10min", label="left", closed="left").count()
    complete = counts >= 2
    agg = agg.loc[complete].dropna(subset=["open", "high", "low", "close"])
    out = agg.reset_index()
    return out.sort_values("time").reset_index(drop=True)


bars_10m = build_10m_ohlcv_from_5m(df5m)
print(f"10m bars={len(bars_10m)} span={bars_10m['time'].iloc[0]} → {bars_10m['time'].iloc[-1]}")


10m bars=129599 span=2024-04-22 10:00:00+00:00 → 2026-10-09 09:40:00+00:00


In [ ]:
# CELL: FEATURES
OHLCV_COLS: Tuple[str, ...] = ("time", "open", "high", "low", "close", "volume")


def _wilder_rsi(close: pd.Series, period: int = 14) -> pd.Series:
    delta = close.diff()
    gain = delta.clip(lower=0.0)
    loss = -delta.clip(upper=0.0)
    avg_gain = gain.ewm(alpha=1.0 / period, min_periods=period, adjust=False).mean()
    avg_loss = loss.ewm(alpha=1.0 / period, min_periods=period, adjust=False).mean()
    rs = avg_gain / avg_loss.replace(0.0, np.nan)
    return 100.0 - (100.0 / (1.0 + rs))


def add_opportunity_features(df: pd.DataFrame, *, atr_period: int = 14) -> pd.DataFrame:
    """Causal signal-candle features from the 10m series only.

    Everything is expressed in ATR units or as a ratio so the model is scale
    invariant across the 900-day price range. ATR(14) is also stored for the
    label path. Nothing here looks past the signal candle.
    """
    out = df.copy()
    close = out["close"].astype(np.float64)
    high = out["high"].astype(np.float64)
    low = out["low"].astype(np.float64)
    open_ = out["open"].astype(np.float64)
    volume = out["volume"].astype(np.float64)
    prev_close = close.shift(1)
    tr = pd.concat(
        [high - low, (high - prev_close).abs(), (low - prev_close).abs()], axis=1
    ).max(axis=1)
    atr = tr.rolling(int(atr_period), min_periods=int(atr_period)).mean()
    atr_safe = atr.replace(0.0, np.nan)
    out["atr"] = atr

    feats: Dict[str, pd.Series] = {}
    # Momentum at several lookbacks, in ATR units.
    feats["ret_1"] = close.pct_change()
    for k in (3, 6, 12, 24, 48, 144):
        feats[f"ret_{k}_atr"] = (close - close.shift(k)) / atr_safe
    # Volatility level and regime.
    feats["atr_pct"] = atr / close.replace(0.0, np.nan)
    atr_long = tr.rolling(int(atr_period) * 6, min_periods=int(atr_period) * 6).mean()
    feats["atr_ratio_84"] = atr / atr_long.replace(0.0, np.nan)
    log_ret = np.log(close).diff()
    rv_12 = log_ret.rolling(12, min_periods=12).std()
    rv_48 = log_ret.rolling(48, min_periods=48).std()
    rv_144 = log_ret.rolling(144, min_periods=144).std()
    feats["rv_12_48"] = rv_12 / rv_48.replace(0.0, np.nan)
    feats["rv_48_144"] = rv_48 / rv_144.replace(0.0, np.nan)
    # Oscillators and trend.
    for period in (7, 14, 28):
        feats[f"rsi_{period}"] = _wilder_rsi(close, period=period)
    for span in (20, 50, 200):
        ema = close.ewm(span=span, adjust=False, min_periods=span).mean()
        feats[f"ema_dist_{span}_atr"] = (close - ema) / atr_safe
        feats[f"ema_slope_{span}_atr"] = (ema - ema.shift(6)) / atr_safe
    # Candle anatomy.
    feats["candle_body"] = (close - open_) / atr_safe
    feats["upper_wick"] = (high - np.maximum(close, open_)) / atr_safe
    feats["lower_wick"] = (np.minimum(close, open_) - low) / atr_safe
    feats["range_atr"] = (high - low) / atr_safe
    # Volume anomalies.
    for window in (20, 100):
        vol_mean = volume.rolling(window, min_periods=window).mean()
        vol_std = volume.rolling(window, min_periods=window).std()
        feats[f"volume_z_{window}"] = (volume - vol_mean) / vol_std.replace(0.0, np.nan)
    # Position inside recent ranges and distance to swing extremes.
    for window in (24, 72, 144):
        hh = high.rolling(window, min_periods=window).max()
        ll = low.rolling(window, min_periods=window).min()
        feats[f"range_pos_{window}"] = (close - ll) / (hh - ll).replace(0.0, np.nan)
        feats[f"dist_high_{window}_atr"] = (hh - close) / atr_safe
        feats[f"dist_low_{window}_atr"] = (close - ll) / atr_safe
    # Consecutive same-direction closes (signed).
    sign = np.sign(close.diff()).fillna(0.0)
    groups = (sign != sign.shift()).cumsum()
    feats["streak"] = sign * (sign.groupby(groups).cumcount() + 1)
    # Intraday / weekly seasonality.
    ts = pd.to_datetime(out["time"], utc=True)
    hour = ts.dt.hour + ts.dt.minute / 60.0
    feats["hour_sin"] = np.sin(2.0 * np.pi * hour / 24.0)
    feats["hour_cos"] = np.cos(2.0 * np.pi * hour / 24.0)
    feats["dow_sin"] = np.sin(2.0 * np.pi * ts.dt.dayofweek / 7.0)
    feats["dow_cos"] = np.cos(2.0 * np.pi * ts.dt.dayofweek / 7.0)
    # Short raw history of the last six candle returns (ATR units).
    for lag in range(1, 7):
        feats[f"ret_lag{lag}_atr"] = (close.shift(lag - 1) - close.shift(lag)) / atr_safe
    return pd.concat([out, pd.DataFrame(feats, index=out.index)], axis=1)


featured = add_opportunity_features(bars_10m, atr_period=int(CONFIG["atr_period"]))
FEATURE_COLS: Tuple[str, ...] = tuple(
    c for c in featured.columns if c not in OHLCV_COLS and c != "atr"
)
print(len(FEATURE_COLS), "features")
null_rates = featured[list(FEATURE_COLS)].isna().mean()
print("max feature null rate", round(float(null_rates.max()), 4), "at", null_rates.idxmax())


46 features
max feature null rate 0.0016 at ema_slope_200_atr


In [ ]:
# CELL: LABELS
# Opportunity labels on a 10-minute series. Path is bars t+1 .. t+horizon only.
# Vectorised over bars; semantics are identical to walking each path bar by bar.

import numpy as np
import pandas as pd

_EPS = 1e-9
HORIZON_BARS_DEFAULT = 24
ATR_PERIOD_DEFAULT = 14
TP_ATR_MULT_DEFAULT = 4.0
SL_ATR_MULT_DEFAULT = 2.0

TP_FIRST = 0
SL_FIRST = 1
NEITHER = 2
AMBIGUOUS = 3

LABEL_LONG = 0
LABEL_SHORT = 1
LABEL_NO_TRADE = 2
LABEL_UNLABELED = -1
LABEL_NAMES = ("LONG", "SHORT", "NO_TRADE")


def tp_sl_from_path(
    fwd_high: np.ndarray,
    fwd_low: np.ndarray,
    entry: float,
    atr: float,
    tp_mult: float,
    sl_mult: float,
    side: str,
) -> int:
    """Walk future highs/lows. Same-bar TP and SL is AMBIGUOUS, not a win."""
    high = np.asarray(fwd_high, dtype=np.float64).ravel()
    low = np.asarray(fwd_low, dtype=np.float64).ravel()
    if high.size == 0 or high.size != low.size:
        return int(NEITHER)
    denom = max(float(atr), _EPS)
    entry_f = float(entry)
    if str(side).strip().upper() == "SHORT":
        tp_level = entry_f - float(tp_mult) * denom
        sl_level = entry_f + float(sl_mult) * denom
        tp_hit = low <= tp_level
        sl_hit = high >= sl_level
    else:
        tp_level = entry_f + float(tp_mult) * denom
        sl_level = entry_f - float(sl_mult) * denom
        tp_hit = high >= tp_level
        sl_hit = low <= sl_level
    for j in range(high.size):
        hit_tp = bool(tp_hit[j])
        hit_sl = bool(sl_hit[j])
        if hit_tp and hit_sl:
            return int(AMBIGUOUS)
        if hit_tp:
            return int(TP_FIRST)
        if hit_sl:
            return int(SL_FIRST)
    return int(NEITHER)


def _forward_stack(values: np.ndarray, k: int) -> np.ndarray:
    """(k, n) array where row j-1 holds values shifted back by j bars (NaN padded)."""
    n = values.shape[0]
    out = np.full((k, n), np.nan, dtype=np.float64)
    for j in range(1, k + 1):
        if n - j > 0:
            out[j - 1, : n - j] = values[j:]
    return out


def _side_codes_vectorised(
    fwd_high: np.ndarray,
    fwd_low: np.ndarray,
    entry: np.ndarray,
    atr: np.ndarray,
    tp_mult: float,
    sl_mult: float,
    side: str,
) -> np.ndarray:
    """Vectorised twin of ``tp_sl_from_path`` for every bar at once."""
    denom = np.maximum(atr, _EPS)
    if str(side).strip().upper() == "SHORT":
        tp_level = entry - float(tp_mult) * denom
        sl_level = entry + float(sl_mult) * denom
        tp_hits = fwd_low <= tp_level[None, :]
        sl_hits = fwd_high >= sl_level[None, :]
    else:
        tp_level = entry + float(tp_mult) * denom
        sl_level = entry - float(sl_mult) * denom
        tp_hits = fwd_high >= tp_level[None, :]
        sl_hits = fwd_low <= sl_level[None, :]
    any_hit = tp_hits | sl_hits
    has_hit = any_hit.any(axis=0)
    first = np.argmax(any_hit, axis=0)
    cols = np.arange(entry.shape[0])
    tp_first = tp_hits[first, cols]
    sl_first = sl_hits[first, cols]
    codes = np.where(
        ~has_hit,
        NEITHER,
        np.where(tp_first & sl_first, AMBIGUOUS, np.where(tp_first, TP_FIRST, SL_FIRST)),
    )
    return codes.astype(np.int64)


def _causal_atr(df: pd.DataFrame, period: int = ATR_PERIOD_DEFAULT) -> np.ndarray:
    if "atr" in df.columns:
        atr = df["atr"].to_numpy(dtype=np.float64)
        if np.isfinite(atr).any():
            return atr
    close = df["close"].to_numpy(dtype=np.float64)
    high = df["high"].to_numpy(dtype=np.float64)
    low = df["low"].to_numpy(dtype=np.float64)
    prev = np.concatenate([[close[0]], close[:-1]])
    tr = np.maximum(high - low, np.maximum(np.abs(high - prev), np.abs(low - prev)))
    return pd.Series(tr).rolling(int(period), min_periods=int(period)).mean().to_numpy(
        dtype=np.float64
    )


def compute_opportunity_labels(
    df: pd.DataFrame,
    *,
    tp_mult: float = TP_ATR_MULT_DEFAULT,
    sl_mult: float = SL_ATR_MULT_DEFAULT,
    horizon_bars: int = HORIZON_BARS_DEFAULT,
    atr_period: int = ATR_PERIOD_DEFAULT,
) -> pd.DataFrame:
    """Label each closed 10m bar. Last ``horizon_bars`` rows stay unlabeled.

    Realised P&L columns (``long_pnl_pct`` / ``short_pnl_pct``) are gross of
    cost: +TP distance on TP_FIRST, −SL distance on SL_FIRST **and** AMBIGUOUS
    (a same-bar double touch is scored as a stop), timeout return on NEITHER.
    """
    out = df.copy().reset_index(drop=True)
    n = len(out)
    k = int(horizon_bars)
    close = out["close"].to_numpy(dtype=np.float64)
    high = out["high"].to_numpy(dtype=np.float64)
    low = out["low"].to_numpy(dtype=np.float64)
    atr = np.asarray(_causal_atr(out, period=int(atr_period)), dtype=np.float64)
    if atr.shape[0] != n:
        atr = np.resize(atr, n)

    fwd_high = _forward_stack(high, k)
    fwd_low = _forward_stack(low, k)
    fwd_close_end = np.full(n, np.nan, dtype=np.float64)
    if n - k > 0:
        fwd_close_end[: n - k] = close[k:]

    index = np.arange(n)
    valid = (
        (index < n - k)
        & np.isfinite(atr)
        & (atr > _EPS)
        & np.isfinite(close)
        & (np.abs(close) > _EPS)
    )
    atr_safe = np.where(valid, atr, np.nan)
    entry_safe = np.where(valid, close, np.nan)

    l_code = _side_codes_vectorised(fwd_high, fwd_low, entry_safe, atr_safe, tp_mult, sl_mult, "LONG")
    s_code = _side_codes_vectorised(fwd_high, fwd_low, entry_safe, atr_safe, tp_mult, sl_mult, "SHORT")
    l_win = l_code == TP_FIRST
    s_win = s_code == TP_FIRST
    label = np.where(l_win & ~s_win, LABEL_LONG, np.where(s_win & ~l_win, LABEL_SHORT, LABEL_NO_TRADE))

    with np.errstate(invalid="ignore", divide="ignore"):
        max_high = np.nanmax(np.where(np.isfinite(fwd_high), fwd_high, -np.inf), axis=0)
        min_low = np.nanmin(np.where(np.isfinite(fwd_low), fwd_low, np.inf), axis=0)
        upside = np.maximum((max_high - entry_safe) / atr_safe, 0.0)
        downside = np.maximum((entry_safe - min_low) / atr_safe, 0.0)
        timeout_ret = (fwd_close_end - entry_safe) / entry_safe
        tp_pct = float(tp_mult) * atr_safe / entry_safe
        sl_pct = float(sl_mult) * atr_safe / entry_safe
        long_pnl = np.where(
            l_code == TP_FIRST, tp_pct, np.where(l_code == NEITHER, timeout_ret, -sl_pct)
        )
        short_pnl = np.where(
            s_code == TP_FIRST, tp_pct, np.where(s_code == NEITHER, -timeout_ret, -sl_pct)
        )

    def _masked(values: np.ndarray) -> np.ndarray:
        return np.where(valid, values, np.nan)

    out["atr"] = atr
    out["trade_label"] = np.where(valid, label, LABEL_UNLABELED).astype(np.int64)
    out["long_result"] = _masked(l_win.astype(np.float64))
    out["short_result"] = _masked(s_win.astype(np.float64))
    out["long_code"] = _masked(l_code.astype(np.float64))
    out["short_code"] = _masked(s_code.astype(np.float64))
    out["upside"] = _masked(upside)
    out["downside"] = _masked(downside)
    out["timeout_ret"] = _masked(timeout_ret)
    out["tp_pct"] = _masked(tp_pct)
    out["sl_pct"] = _masked(sl_pct)
    out["long_pnl_pct"] = _masked(long_pnl)
    out["short_pnl_pct"] = _masked(short_pnl)
    names = np.array(["UNLABELED"] * n, dtype=object)
    names[out["trade_label"].to_numpy() == LABEL_LONG] = "LONG"
    names[out["trade_label"].to_numpy() == LABEL_SHORT] = "SHORT"
    names[out["trade_label"].to_numpy() == LABEL_NO_TRADE] = "NO_TRADE"
    out["trade_name"] = names
    return out


In [ ]:
# CELL: APPLY_LABELS
labeled = compute_opportunity_labels(
    featured,
    tp_mult=float(CONFIG["tp_atr_mult"]),
    sl_mult=float(CONFIG["sl_atr_mult"]),
    horizon_bars=int(CONFIG["horizon_bars"]),
    atr_period=int(CONFIG["atr_period"]),
)
valid = labeled["trade_label"].to_numpy() >= 0
print(f"labeled rows={int(valid.sum())} / {len(labeled)}")
print(labeled.loc[valid, "trade_name"].value_counts(normalize=True).round(4))
print(
    "median TP distance",
    f"{labeled.loc[valid, 'tp_pct'].median():.4%}",
    "median SL distance",
    f"{labeled.loc[valid, 'sl_pct'].median():.4%}",
    "round trip cost",
    f"{float(CONFIG['round_trip_cost_pct']):.4%}",
)


labeled rows=129562 / 129599
trade_name
NO_TRADE    0.6133
SHORT       0.1979
LONG        0.1889
Name: proportion, dtype: float64
median TP distance 0.8592% median SL distance 0.4296% round trip cost 0.1600%


In [ ]:
# CELL: GATE
def train_period_mask(n: int, train_fraction: float) -> np.ndarray:
    cut = int(n * float(train_fraction))
    mask = np.zeros(n, dtype=bool)
    mask[:cut] = True
    return mask


def _path_code_counts(codes: np.ndarray, usable: np.ndarray) -> Dict[str, int]:
    finite = usable & np.isfinite(codes)
    return {
        "TP_FIRST": int(np.sum(finite & (codes == TP_FIRST))),
        "SL_FIRST": int(np.sum(finite & (codes == SL_FIRST))),
        "NEITHER": int(np.sum(finite & (codes == NEITHER))),
        "AMBIGUOUS": int(np.sum(finite & (codes == AMBIGUOUS))),
    }


def summarize_label_gate(
    df: pd.DataFrame,
    *,
    train_fraction: float,
    round_trip_cost_pct: float,
    minority_rate: float,
    ambiguous_win_max: float,
    min_tp_cost_ratio: float = 4.0,
) -> Dict[str, Any]:
    """Decide whether the bracket is worth training on, using train rows only.

    Besides class mix and oracle exclusive expectancy, the gate checks the
    bracket economics: the median take-profit distance must be at least
    ``min_tp_cost_ratio`` times the round-trip cost, and the implied
    break-even win rate ``(sl + cost) / (tp + sl)`` is reported.
    """
    n = len(df)
    in_train = train_period_mask(n, train_fraction)
    labels = df["trade_label"].to_numpy(dtype=np.int64)
    usable = in_train & (labels >= 0)
    n_use = int(usable.sum())
    if n_use == 0:
        raise RuntimeError("Train-period gate has no labeled rows")
    mix = {}
    mix_ok = True
    for cid, name in enumerate(LABEL_NAMES):
        rate = float(np.mean(labels[usable] == cid))
        mix[name] = rate
        if rate < float(minority_rate):
            mix_ok = False
    long_code = df["long_code"].to_numpy(dtype=np.float64)
    short_code = df["short_code"].to_numpy(dtype=np.float64)
    cost = float(round_trip_cost_pct)
    long_pnl = df["long_pnl_pct"].to_numpy(dtype=np.float64) - cost
    short_pnl = df["short_pnl_pct"].to_numpy(dtype=np.float64) - cost

    always_long_mask = usable & np.isfinite(long_pnl)
    always_short_mask = usable & np.isfinite(short_pnl)
    always_in_long = (
        float(np.mean(long_pnl[always_long_mask])) if always_long_mask.any() else float("nan")
    )
    always_in_short = (
        float(np.mean(short_pnl[always_short_mask])) if always_short_mask.any() else float("nan")
    )

    excl_long_mask = usable & (labels == LABEL_LONG) & np.isfinite(long_pnl)
    excl_short_mask = usable & (labels == LABEL_SHORT) & np.isfinite(short_pnl)
    exclusive_long = (
        float(np.mean(long_pnl[excl_long_mask])) if excl_long_mask.any() else float("nan")
    )
    exclusive_short = (
        float(np.mean(short_pnl[excl_short_mask])) if excl_short_mask.any() else float("nan")
    )

    def _ambig_share(codes: np.ndarray) -> float:
        finite = usable & np.isfinite(codes)
        tp = finite & (codes == TP_FIRST)
        amb = finite & (codes == AMBIGUOUS)
        denom = int(tp.sum() + amb.sum())
        if denom == 0:
            return 0.0
        return float(amb.sum()) / float(denom)

    long_ambig = _ambig_share(long_code)
    short_ambig = _ambig_share(short_code)

    if "tp_pct" in df.columns and "sl_pct" in df.columns:
        tp_pct = df["tp_pct"].to_numpy(dtype=np.float64)
        sl_pct = df["sl_pct"].to_numpy(dtype=np.float64)
    else:
        atr_pct = df["atr"].to_numpy(dtype=np.float64) / df["close"].to_numpy(dtype=np.float64)
        tp_pct = float(CONFIG["tp_atr_mult"]) * atr_pct
        sl_pct = float(CONFIG["sl_atr_mult"]) * atr_pct
    econ_mask = usable & np.isfinite(tp_pct) & np.isfinite(sl_pct)
    median_tp = float(np.median(tp_pct[econ_mask])) if econ_mask.any() else float("nan")
    median_sl = float(np.median(sl_pct[econ_mask])) if econ_mask.any() else float("nan")
    tp_cost_ratio = median_tp / cost if cost > 0 and np.isfinite(median_tp) else float("inf")
    breakeven_win_rate = (
        (median_sl + cost) / (median_tp + median_sl)
        if np.isfinite(median_tp) and np.isfinite(median_sl) and (median_tp + median_sl) > 0
        else float("nan")
    )
    long_tp_rate = float(np.mean(long_code[usable & np.isfinite(long_code)] == TP_FIRST))
    short_tp_rate = float(np.mean(short_code[usable & np.isfinite(short_code)] == TP_FIRST))

    reasons: List[str] = []
    if not mix_ok:
        reasons.append("class_mix_below_floor")
    if not (np.isfinite(exclusive_long) and exclusive_long > 0.0):
        reasons.append("exclusive_long_expectancy_not_positive")
    if not (np.isfinite(exclusive_short) and exclusive_short > 0.0):
        reasons.append("exclusive_short_expectancy_not_positive")
    if long_ambig >= float(ambiguous_win_max):
        reasons.append("long_ambiguous_majority")
    if short_ambig >= float(ambiguous_win_max):
        reasons.append("short_ambiguous_majority")
    if tp_cost_ratio < float(min_tp_cost_ratio):
        reasons.append("tp_distance_below_cost_ratio")
    passed = len(reasons) == 0
    return {
        "passed": passed,
        "n_train_labeled": n_use,
        "class_mix": mix,
        "minority_rate": float(minority_rate),
        "exclusive_long_expectancy_pct": exclusive_long,
        "exclusive_short_expectancy_pct": exclusive_short,
        "always_in_long_expectancy_pct": always_in_long,
        "always_in_short_expectancy_pct": always_in_short,
        "round_trip_cost_pct": cost,
        "median_tp_pct": median_tp,
        "median_sl_pct": median_sl,
        "tp_cost_ratio": tp_cost_ratio,
        "min_tp_cost_ratio": float(min_tp_cost_ratio),
        "breakeven_win_rate": breakeven_win_rate,
        "base_long_tp_rate": long_tp_rate,
        "base_short_tp_rate": short_tp_rate,
        "long_ambiguous_share": long_ambig,
        "short_ambiguous_share": short_ambig,
        "long_path_codes": _path_code_counts(long_code, usable),
        "short_path_codes": _path_code_counts(short_code, usable),
        "tp_atr_mult": float(CONFIG["tp_atr_mult"]),
        "sl_atr_mult": float(CONFIG["sl_atr_mult"]),
        "reasons": reasons,
    }


EXPORT_DIR = Path(str(CONFIG["export_dir"]))
EXPORT_DIR.mkdir(parents=True, exist_ok=True)
label_report = summarize_label_gate(
    labeled,
    train_fraction=float(CONFIG["train_fraction"]),
    round_trip_cost_pct=float(CONFIG["round_trip_cost_pct"]),
    minority_rate=float(CONFIG["minority_rate"]),
    ambiguous_win_max=float(CONFIG["ambiguous_win_max"]),
    min_tp_cost_ratio=float(CONFIG["min_tp_cost_ratio"]),
)
report_path = EXPORT_DIR / "label_report.json"
report_path.write_text(json.dumps(label_report, indent=2), encoding="utf-8")
print(json.dumps(label_report, indent=2))
print("wrote", report_path)
if not label_report["passed"]:
    raise RuntimeError(
        "Train-period label gate failed: "
        f"{label_report['reasons']}. Change CONFIG tp/sl ATR multiples or horizon and "
        "re-run from the top. Do not fit models."
    )
print(
    f"Gate passed. Break-even win rate {label_report['breakeven_win_rate']:.1%} vs base "
    f"TP rates long {label_report['base_long_tp_rate']:.1%} / short "
    f"{label_report['base_short_tp_rate']:.1%}. Training may start."
)


{
  "passed": true,
  "n_train_labeled": 90706,
  "class_mix": {
    "LONG": 0.1886644764403678,
    "SHORT": 0.19655811081957092,
    "NO_TRADE": 0.6147774127400613
  },
  "minority_rate": 0.15,
  "exclusive_long_expectancy_pct": 0.007118231299915457,
  "exclusive_short_expectancy_pct": 0.0075778794381298,
  "always_in_long_expectancy_pct": -0.0015982457106803246,
  "always_in_short_expectancy_pct": -0.0016349279903534648,
  "round_trip_cost_pct": 0.0016,
  "median_tp_pct": 0.008727318334842016,
  "median_sl_pct": 0.004363659167421008,
  "tp_cost_ratio": 5.45457395927626,
  "min_tp_cost_ratio": 4.0,
  "breakeven_win_rate": 0.4555549168417772,
  "base_long_tp_rate": 0.1886644764403678,
  "base_short_tp_rate": 0.19655811081957092,
  "long_ambiguous_share": 0.002273787313432836,
  "short_ambiguous_share": 0.0011205109529945655,
  "long_path_codes": {
    "TP_FIRST": 17113,
    "SL_FIRST": 45107,
    "NEITHER": 28447,
    "AMBIGUOUS": 39
  },
  "short_path_codes": {
    "TP_FIRST": 17829,

In [ ]:
# CELL: SPLIT
def purged_index_splits(
    n: int,
    *,
    train_fraction: float,
    val_fraction: float,
    purge_bars: int,
) -> Dict[str, np.ndarray]:
    """Time-ordered split. Purge ``purge_bars`` on each side of every cut."""
    n = int(n)
    purge = int(purge_bars)
    train_end = int(n * float(train_fraction))
    val_len = int(n * float(val_fraction))
    train_idx = np.arange(0, max(train_end - purge, 0), dtype=np.int64)
    val_start = min(train_end + purge, n)
    val_end = min(val_start + val_len, n)
    val_idx = np.arange(val_start, max(val_end - purge, val_start), dtype=np.int64)
    test_start = min(val_end + purge, n)
    test_idx = np.arange(test_start, n, dtype=np.int64)
    return {"train": train_idx, "val": val_idx, "test": test_idx}


def development_walk_forward(
    n_dev: int,
    *,
    folds: int = 4,
    purge_bars: int,
) -> List[Tuple[np.ndarray, np.ndarray]]:
    """Expanding-window folds inside the development prefix only."""
    folds = max(2, int(folds))
    purge = int(purge_bars)
    edges = [int(n_dev * (i + 1) / folds) for i in range(folds)]
    out: List[Tuple[np.ndarray, np.ndarray]] = []
    for i in range(1, folds):
        train_end = edges[i - 1]
        val_end = edges[i]
        tr = np.arange(0, max(train_end - purge, 0), dtype=np.int64)
        va = np.arange(min(train_end + purge, n_dev), val_end, dtype=np.int64)
        if len(tr) == 0 or len(va) == 0:
            continue
        out.append((tr, va))
    return out


def inner_calibration_split(
    train_idx: np.ndarray, *, calib_fraction: float, purge_bars: int
) -> Tuple[np.ndarray, np.ndarray]:
    """Reserve the purged tail of a training window for threshold selection."""
    n = len(train_idx)
    cut = int(n * (1.0 - float(calib_fraction)))
    fit_idx = train_idx[: max(cut - int(purge_bars), 0)]
    calib_idx = train_idx[cut:]
    return fit_idx, calib_idx


splits = purged_index_splits(
    len(labeled),
    train_fraction=float(CONFIG["train_fraction"]),
    val_fraction=float(CONFIG["val_fraction"]),
    purge_bars=int(CONFIG["purge_bars"]),
)
for key, idx in splits.items():
    print(key, "n", len(idx), "range", int(idx[0]) if len(idx) else None, int(idx[-1]) if len(idx) else None)
dev_end = int(splits["test"][0]) if len(splits["test"]) else len(labeled)
wf_folds = development_walk_forward(
    dev_end, folds=int(CONFIG["walk_forward_folds"]), purge_bars=int(CONFIG["purge_bars"])
)
print("walk-forward folds", len(wf_folds), "dev_end", dev_end)


train n 90694 range 0 90693
val n 19414 range 90744 110157
test n 19391 range 110208 129598
walk-forward folds 3 dev_end 110208


In [ ]:
# CELL: DATASET
def build_rows(
    frame: pd.DataFrame, *, feature_cols: Sequence[str]
) -> Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    """One tabular row per labeled bar with finite features and finite path targets."""
    feats = frame.loc[:, list(feature_cols)].to_numpy(dtype=np.float64)
    y_cls = frame["trade_label"].to_numpy(dtype=np.int64)
    y_reg = frame.loc[:, ["upside", "downside"]].to_numpy(dtype=np.float64)
    ok = (y_cls >= 0) & np.isfinite(feats).all(axis=1) & np.isfinite(y_reg).all(axis=1)
    rows = np.flatnonzero(ok).astype(np.int64)
    return (
        feats[rows].astype(np.float32),
        y_cls[rows],
        y_reg[rows].astype(np.float32),
        rows,
    )


all_x, all_y, all_r, all_rows = build_rows(labeled, feature_cols=FEATURE_COLS)
print(
    "rows",
    all_x.shape,
    "class mix",
    {n: round(float(np.mean(all_y == i)), 4) for i, n in enumerate(LABEL_NAMES)},
)


def rows_in(split_idx: np.ndarray) -> np.ndarray:
    """Boolean selector over ``all_rows`` for a set of frame indices."""
    allowed = np.zeros(len(labeled), dtype=bool)
    allowed[np.asarray(split_idx, dtype=np.int64)] = True
    return allowed[all_rows]


MIN_SIGNAL_TP_PCT = float(CONFIG["min_tp_cost_ratio"]) * float(CONFIG["round_trip_cost_pct"])


def signal_eligible(rows: np.ndarray) -> np.ndarray:
    """Quiet-market guard: a candle may carry a signal only if its TP distance pays for cost."""
    tp_pct = labeled["tp_pct"].to_numpy(dtype=np.float64)[rows]
    return np.isfinite(tp_pct) & (tp_pct >= MIN_SIGNAL_TP_PCT)


print(
    f"signal eligibility: TP distance >= {MIN_SIGNAL_TP_PCT:.4%} -> "
    f"{float(signal_eligible(all_rows).mean()):.1%} of labeled candles"
)


def fit_scaler(x: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:
    mean = x.mean(axis=0, keepdims=True)
    std = x.std(axis=0, keepdims=True)
    std = np.where(std < 1e-6, 1.0, std)
    return mean.astype(np.float32), std.astype(np.float32)


def scale_x(x: np.ndarray, mean: np.ndarray, std: np.ndarray) -> np.ndarray:
    return ((x - mean) / std).astype(np.float32)


sel_train = rows_in(splits["train"])
sel_val = rows_in(splits["val"])
sel_test = rows_in(splits["test"])
feat_mean, feat_std = fit_scaler(all_x[sel_train])
reg_mean = all_r[sel_train].mean(axis=0, keepdims=True)
reg_std = all_r[sel_train].std(axis=0, keepdims=True)
reg_std = np.where(reg_std < 1e-6, 1.0, reg_std)


def scale_r(r: np.ndarray) -> np.ndarray:
    return ((r - reg_mean) / reg_std).astype(np.float32)


X_train = scale_x(all_x[sel_train], feat_mean, feat_std)
X_val = scale_x(all_x[sel_val], feat_mean, feat_std)
X_test = scale_x(all_x[sel_test], feat_mean, feat_std)
y_train, y_val, y_test = all_y[sel_train], all_y[sel_val], all_y[sel_test]
r_train, r_val, r_test = all_r[sel_train], all_r[sel_val], all_r[sel_test]
R_train, R_val, R_test = scale_r(r_train), scale_r(r_val), scale_r(r_test)
print("train", X_train.shape, "val", X_val.shape, "test", X_test.shape)


rows (129370, 46) class mix {'LONG': 0.1888, 'SHORT': 0.1979, 'NO_TRADE': 0.6133}
signal eligibility: TP distance >= 0.6400% -> 71.0% of labeled candles
train (90489, 46) val (19414, 46) test (19367, 46)


In [ ]:
# CELL: MODEL
def make_family(name: str, seed: int) -> Dict[str, Any]:
    """Return unfitted classifier + two regressors for one family.

    No class re-weighting anywhere: the decision rule thresholds calibrated-ish
    probabilities, and ``balanced`` weights distort them.
    """
    est = int(CONFIG["n_estimators"])
    depth = int(CONFIG["max_depth"])
    lr = float(CONFIG["learning_rate"])
    leaf = int(CONFIG["min_leaf"])
    threads = int(CONFIG["threads_per_model"])
    if name == "LR":
        clf = LogisticRegression(max_iter=3000, C=0.1, solver="lbfgs")
        up = Ridge(alpha=10.0)
        dn = Ridge(alpha=10.0)
    elif name == "RF":
        rf_kwargs = dict(
            n_estimators=est,
            max_depth=depth + 2,
            min_samples_leaf=leaf,
            max_features="sqrt",
            n_jobs=threads,
        )
        clf = RandomForestClassifier(random_state=seed, **rf_kwargs)
        up = RandomForestRegressor(random_state=seed, **rf_kwargs)
        dn = RandomForestRegressor(random_state=seed + 1, **rf_kwargs)
    elif name == "XGB":
        xgb_kwargs = dict(
            n_estimators=est,
            max_depth=depth,
            learning_rate=lr,
            min_child_weight=leaf,
            subsample=0.8,
            colsample_bytree=0.7,
            reg_lambda=5.0,
            tree_method="hist",
            n_jobs=threads,
            verbosity=0,
        )
        clf = XGBClassifier(
            objective="multi:softprob", eval_metric="mlogloss", random_state=seed, **xgb_kwargs
        )
        up = XGBRegressor(random_state=seed, **xgb_kwargs)
        dn = XGBRegressor(random_state=seed + 1, **xgb_kwargs)
    elif name == "LGBM":
        lgb_kwargs = dict(
            n_estimators=est,
            num_leaves=31,
            max_depth=-1,
            learning_rate=lr,
            min_child_samples=leaf,
            subsample=0.8,
            subsample_freq=1,
            colsample_bytree=0.7,
            reg_lambda=5.0,
            n_jobs=threads,
            verbosity=-1,
        )
        clf = LGBMClassifier(random_state=seed, **lgb_kwargs)
        up = LGBMRegressor(random_state=seed, **lgb_kwargs)
        dn = LGBMRegressor(random_state=seed + 1, **lgb_kwargs)
    elif name == "CatBoost":
        cb_kwargs = dict(
            iterations=est,
            depth=depth,
            learning_rate=lr,
            l2_leaf_reg=10.0,
            verbose=False,
            thread_count=threads,
            allow_writing_files=False,
        )
        clf = CatBoostClassifier(loss_function="MultiClass", random_seed=seed, **cb_kwargs)
        up = CatBoostRegressor(random_seed=seed, **cb_kwargs)
        dn = CatBoostRegressor(random_seed=seed + 1, **cb_kwargs)
    else:
        raise ValueError(name)
    return {"name": name, "clf": clf, "up": up, "dn": dn}


print("families", CONFIG["ensemble"], "+", ENSEMBLE_NAME)


families ['LR', 'RF', 'XGB', 'LGBM', 'CatBoost'] + ENSEMBLE


In [ ]:
# CELL: TRAIN
def confusion(y_true: np.ndarray, y_pred: np.ndarray, n_classes: int = 3) -> np.ndarray:
    mat = np.zeros((n_classes, n_classes), dtype=np.int64)
    for t, p in zip(y_true.tolist(), y_pred.tolist()):
        if 0 <= int(t) < n_classes and 0 <= int(p) < n_classes:
            mat[int(t), int(p)] += 1
    return mat


def class_precision(mat: np.ndarray, cls: int) -> float:
    col = mat[:, cls]
    denom = float(col.sum())
    if denom <= 0:
        return float("nan")
    return float(mat[cls, cls]) / denom


def balanced_accuracy(mat: np.ndarray) -> float:
    recalls = []
    for i in range(mat.shape[0]):
        denom = float(mat[i].sum())
        recalls.append(float(mat[i, i]) / denom if denom else float("nan"))
    finite = [r for r in recalls if np.isfinite(r)]
    return float(np.mean(finite)) if finite else float("nan")


def corr(a: np.ndarray, b: np.ndarray) -> float:
    if a.size < 3:
        return float("nan")
    if float(np.std(a)) < 1e-12 or float(np.std(b)) < 1e-12:
        return float("nan")
    return float(np.corrcoef(a, b)[0, 1])


def as_cls(arr: np.ndarray) -> np.ndarray:
    return np.asarray(arr).reshape(-1).astype(np.int64)


def as_1d(arr: np.ndarray) -> np.ndarray:
    return np.asarray(arr, dtype=np.float64).reshape(-1)


def proba_of(clf: Any, X: np.ndarray) -> np.ndarray:
    """(n, 3) class probabilities in LONG/SHORT/NO_TRADE order regardless of estimator."""
    p = np.asarray(clf.predict_proba(X), dtype=np.float64)
    if p.ndim == 1:
        p = p.reshape(len(X), -1)
    classes = getattr(clf, "classes_", None)
    if classes is None:
        return p
    full = np.zeros((p.shape[0], len(LABEL_NAMES)), dtype=np.float64)
    for j, c in enumerate(np.asarray(classes).reshape(-1).astype(int)):
        full[:, c] = p[:, j]
    return full


def classifier_report(y_true: np.ndarray, y_pred: np.ndarray) -> Dict[str, Any]:
    cm = confusion(y_true, y_pred)
    return {
        "precision_LONG": class_precision(cm, LABEL_LONG),
        "precision_SHORT": class_precision(cm, LABEL_SHORT),
        "n_LONG": float(np.sum(y_pred == LABEL_LONG)),
        "n_SHORT": float(np.sum(y_pred == LABEL_SHORT)),
        "balanced_accuracy": balanced_accuracy(cm),
        "confusion": cm.tolist(),
    }


_PNL_COL = {LABEL_LONG: "long_pnl_pct", LABEL_SHORT: "short_pnl_pct"}
_SIDE_NAME = {LABEL_LONG: "LONG", LABEL_SHORT: "SHORT"}


def net_pnl(cls: int, rows: np.ndarray, cost: Optional[float] = None) -> np.ndarray:
    """Net-of-cost realised P&L (fraction of price) for taking ``cls`` on ``rows``."""
    c = float(CONFIG["round_trip_cost_pct"]) if cost is None else float(cost)
    return labeled[_PNL_COL[int(cls)]].to_numpy(dtype=np.float64)[rows] - c


def signal_expectancy(
    pred_cls: np.ndarray, rows: np.ndarray, cost: Optional[float] = None
) -> Dict[str, float]:
    """Per-side expectancy, t-stat, profit factor and total for the predicted signals."""
    out: Dict[str, float] = {}
    for cls, name in _SIDE_NAME.items():
        take = pred_cls == cls
        pnl = net_pnl(cls, rows[take], cost) if take.any() else np.zeros(0)
        pnl = pnl[np.isfinite(pnl)]
        n = int(pnl.size)
        out[f"{name}_n"] = float(n)
        out[f"{name}_expectancy"] = float(pnl.mean()) if n else float("nan")
        std = float(pnl.std(ddof=1)) if n > 1 else float("nan")
        out[f"{name}_tstat"] = (
            float(pnl.mean() / (std / math.sqrt(n))) if n > 1 and std > 0 else float("nan")
        )
        wins = float(pnl[pnl > 0].sum())
        losses = float(-pnl[pnl < 0].sum())
        out[f"{name}_profit_factor"] = (
            wins / losses if losses > 0 else (float("inf") if wins > 0 else float("nan"))
        )
        out[f"{name}_total_pct"] = float(pnl.sum())
    return out


def expectancy_rank_score(exp_report: Dict[str, float]) -> float:
    """Mean expectancy over sides with >= ``min_signals`` trades; -inf when none qualify."""
    min_n = int(CONFIG["min_signals"])
    parts: List[float] = []
    for name in ("LONG", "SHORT"):
        n = float(exp_report.get(f"{name}_n", 0.0))
        e = float(exp_report.get(f"{name}_expectancy", float("nan")))
        if n >= min_n and np.isfinite(e):
            parts.append(e)
    return float(np.mean(parts)) if parts else float("-inf")


def choose_thresholds(proba: np.ndarray, rows: np.ndarray) -> Dict[str, Optional[float]]:
    """Per side, the probability threshold that maximises calibration expectancy.

    Candidates are quantiles of the side's probability on the calibration rows.
    A side with no candidate reaching ``min_signals`` trades is disabled (None).
    """
    min_n = int(CONFIG["min_signals"])
    eligible = signal_eligible(rows)
    out: Dict[str, Optional[float]] = {}
    for cls, name in _SIDE_NAME.items():
        other = LABEL_SHORT if cls == LABEL_LONG else LABEL_LONG
        p_side = proba[:, cls]
        p_other = proba[:, other]
        pnl_all = net_pnl(cls, rows)
        best_thr: Optional[float] = None
        best_e = float("-inf")
        if not eligible.any():
            out[name] = None
            continue
        for q in CONFIG["signal_quantiles"]:
            thr = float(np.quantile(p_side[eligible], float(q)))
            take = eligible & (p_side >= thr) & (p_side > p_other) & np.isfinite(pnl_all)
            n = int(take.sum())
            if n < min_n:
                continue
            e = float(pnl_all[take].mean())
            if e > best_e:
                best_thr, best_e = thr, e
        out[name] = best_thr
    return out


def apply_thresholds(
    proba: np.ndarray, thr: Dict[str, Optional[float]], eligible: Optional[np.ndarray] = None
) -> np.ndarray:
    """Live decision rule.

    Abstain unless the candle is eligible (TP distance pays for cost), P(side)
    clears the side's threshold and beats the other side.
    """
    pred = np.full(proba.shape[0], LABEL_NO_TRADE, dtype=np.int64)
    ok = np.ones(proba.shape[0], dtype=bool) if eligible is None else np.asarray(eligible, dtype=bool)
    p_long, p_short = proba[:, LABEL_LONG], proba[:, LABEL_SHORT]
    if thr.get("LONG") is not None:
        pred[ok & (p_long >= float(thr["LONG"])) & (p_long > p_short)] = LABEL_LONG
    if thr.get("SHORT") is not None:
        pred[ok & (p_short >= float(thr["SHORT"])) & (p_short > p_long)] = LABEL_SHORT
    return pred


def score_probas(
    p_calib: np.ndarray,
    rows_calib: np.ndarray,
    p_eval: np.ndarray,
    rows_eval: np.ndarray,
    y_eval: np.ndarray,
) -> Dict[str, Any]:
    thr = choose_thresholds(p_calib, rows_calib)
    pred = apply_thresholds(p_eval, thr, signal_eligible(rows_eval))
    exp_rep = signal_expectancy(pred, rows_eval)
    cls_rep = classifier_report(y_eval, pred)
    return {
        "thresholds": thr,
        "cls": {k: v for k, v in cls_rep.items() if k != "confusion"},
        "confusion": cls_rep["confusion"],
        "exp": exp_rep,
        "score": expectancy_rank_score(exp_rep),
        "pred": pred,
    }


def fit_fold(
    name: str, fold: str, fit_sel: np.ndarray, calib_sel: np.ndarray, eval_sel: np.ndarray, *, seed: int
) -> Dict[str, Any]:
    """Fit one family on ``fit_sel`` rows, return probabilities on calibration and eval rows."""
    mean, std = fit_scaler(all_x[fit_sel])
    pack = make_family(name, seed)
    pack["clf"].fit(scale_x(all_x[fit_sel], mean, std), all_y[fit_sel])
    result = {
        "name": name,
        "fold": fold,
        "clf": pack["clf"],
        "scaler": (mean, std),
        "p_calib": proba_of(pack["clf"], scale_x(all_x[calib_sel], mean, std)),
        "p_eval": proba_of(pack["clf"], scale_x(all_x[eval_sel], mean, std)),
    }
    if fold == "final":
        pack["up"].fit(X_train, R_train[:, 0])
        pack["dn"].fit(X_train, R_train[:, 1])
        result["up"] = pack["up"]
        result["dn"] = pack["dn"]
    return result


# Fold plan: expanding walk-forward folds inside development, then the final split.
# Every fold has disjoint fit / calibration / evaluation rows; thresholds are chosen
# on calibration rows only, so evaluation expectancy is out-of-sample for both the
# model and the decision rule.
fold_plan: List[Tuple[str, np.ndarray, np.ndarray, np.ndarray]] = []
for fold_i, (tr_idx, va_idx) in enumerate(wf_folds, start=1):
    fit_idx, calib_idx = inner_calibration_split(
        tr_idx, calib_fraction=float(CONFIG["calib_fraction"]), purge_bars=int(CONFIG["purge_bars"])
    )
    fold_plan.append((f"wf{fold_i}", rows_in(fit_idx), rows_in(calib_idx), rows_in(va_idx)))
fold_plan.append(("final", sel_train, sel_val, sel_test))
for fold, f_sel, c_sel, e_sel in fold_plan:
    print(f"{fold}: fit={int(f_sel.sum())} calib={int(c_sel.sum())} eval={int(e_sel.sum())}")

FAMILIES: List[str] = list(CONFIG["ensemble"])
runs: Dict[str, Dict[str, Dict[str, Any]]] = {fold: {} for fold, *_ in fold_plan}
print("Fitting", FAMILIES, "x", [f for f, *_ in fold_plan], "in parallel...")
t_start = time.time()
with ThreadPoolExecutor(max_workers=len(FAMILIES)) as pool:
    futs = {}
    for fold_i, (fold, f_sel, c_sel, e_sel) in enumerate(fold_plan):
        for name in FAMILIES:
            fut = pool.submit(
                fit_fold, name, fold, f_sel, c_sel, e_sel, seed=int(CONFIG["seed"]) + fold_i
            )
            futs[fut] = (fold, name)
    for fut in as_completed(futs):
        fold, name = futs[fut]
        try:
            runs[fold][name] = fut.result()
            print(f"  done {name:9s} {fold}  ({time.time() - t_start:.0f}s)")
        except Exception as exc:  # keep the bake-off alive if one family fails
            print(f"  FAILED {name} {fold}: {exc!r}")

# Soft-vote ensemble over the members that succeeded in a fold.
for fold, f_sel, c_sel, e_sel in fold_plan:
    members = [runs[fold][n] for n in FAMILIES if n in runs[fold]]
    if len(members) >= 2:
        runs[fold][ENSEMBLE_NAME] = {
            "name": ENSEMBLE_NAME,
            "fold": fold,
            "members": [m["name"] for m in members],
            "p_calib": np.mean([m["p_calib"] for m in members], axis=0),
            "p_eval": np.mean([m["p_eval"] for m in members], axis=0),
        }

CANDIDATES: List[str] = [n for n in FAMILIES + [ENSEMBLE_NAME] if n in runs["final"]]
scores: Dict[str, Dict[str, Dict[str, Any]]] = {name: {} for name in CANDIDATES}
for fold, f_sel, c_sel, e_sel in fold_plan:
    rows_c, rows_e, y_e = all_rows[c_sel], all_rows[e_sel], all_y[e_sel]
    for name in CANDIDATES:
        if name not in runs[fold]:
            continue
        r = runs[fold][name]
        scores[name][fold] = score_probas(r["p_calib"], rows_c, r["p_eval"], rows_e, y_e)

WF_FOLDS: List[str] = [fold for fold, *_ in fold_plan if fold != "final"]


def walk_forward_summary(name: str) -> Dict[str, float]:
    vals = [scores[name][f]["score"] for f in WF_FOLDS if f in scores[name]]
    finite = [v for v in vals if np.isfinite(v)]
    return {
        "wf_folds_scored": float(len(finite)),
        "wf_mean_score": float(np.mean(finite)) if finite else float("-inf"),
        "wf_positive_share": float(np.mean([v > 0 for v in finite])) if finite else 0.0,
        "wf_min_score": float(np.min(finite)) if finite else float("-inf"),
    }


leaderboard: Dict[str, Dict[str, Any]] = {}
for name in CANDIDATES:
    wf = walk_forward_summary(name)
    fin = scores[name].get("final")
    row: Dict[str, Any] = {
        **wf,
        "val_thresholds": fin["thresholds"] if fin else None,
        "wf_folds": {
            f: {"score": scores[name][f]["score"], "exp": scores[name][f]["exp"], "thresholds": scores[name][f]["thresholds"]}
            for f in WF_FOLDS
            if f in scores[name]
        },
    }
    if name != ENSEMBLE_NAME and "up" in runs["final"].get(name, {}):
        r = runs["final"][name]
        up_c = corr(as_1d(r["up"].predict(X_val)), R_val[:, 0])
        dn_c = corr(as_1d(r["dn"].predict(X_val)), R_val[:, 1])
        row["upside_corr"] = up_c
        row["downside_corr"] = dn_c
        row["reg_score"] = float(np.nanmean([up_c, dn_c]))
    leaderboard[name] = row
    print(
        f"{name:9s} WF mean={wf['wf_mean_score']:+.6f} min={wf['wf_min_score']:+.6f} "
        f"pos_share={wf['wf_positive_share']:.2f} folds={wf['wf_folds_scored']:.0f} "
        f"val_thr={row['val_thresholds']} reg={row.get('reg_score', float('nan')):.3f}"
    )


def _selection_key(name: str) -> Tuple[float, float]:
    wf = leaderboard[name]
    return (wf["wf_mean_score"], wf["wf_min_score"])


cls_winner_name = max(CANDIDATES, key=_selection_key)
reg_candidates = [n for n in CANDIDATES if "reg_score" in leaderboard[n]]
reg_winner_name = max(reg_candidates, key=lambda n: float(leaderboard[n]["reg_score"]))
cls_winner = {"name": cls_winner_name, **leaderboard[cls_winner_name]}
reg_winner = {"name": reg_winner_name, "reg_score": leaderboard[reg_winner_name]["reg_score"]}
print("best classifier (walk-forward):", cls_winner_name, "| best regressor family (val):", reg_winner_name)


wf1: fit=23167 calib=4130 eval=27527
wf2: fit=46587 calib=8262 eval=27527
wf3: fit=70006 calib=12395 eval=27527
final: fit=90489 calib=19414 eval=19367
Fitting ['LR', 'RF', 'XGB', 'LGBM', 'CatBoost'] x ['wf1', 'wf2', 'wf3', 'final'] in parallel...


  done LR        wf1  (18s)


  done LR        wf2  (51s)


  done CatBoost  wf1  (140s)


  done RF        wf1  (167s)


...

RF        WF mean=-0.001711 min=-0.002286 pos_share=0.00 folds=3 val_thr={'LONG': 0.24718577477491863, 'SHORT': 0.29663462662933154} reg=0.272


XGB       WF mean=-0.001649 min=-0.001841 pos_share=0.00 folds=3 val_thr={'LONG': 0.38238136738538736, 'SHORT': 0.41037094652652734} reg=0.298


/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


LGBM      WF mean=-0.001415 min=-0.001608 pos_share=0.00 folds=3 val_thr={'LONG': 0.3526476648420395, 'SHORT': 0.4361746067305746} reg=0.317
CatBoost  WF mean=-0.001495 min=-0.002003 pos_share=0.00 folds=3 val_thr={'LONG': 0.3193166767726072, 'SHORT': 0.366312650399112} reg=0.192
ENSEMBLE  WF mean=-0.001502 min=-0.001706 pos_share=0.00 folds=3 val_thr={'LONG': 0.29968136692580943, 'SHORT': 0.3498481838122219} reg=nan
best classifier (walk-forward): LGBM | best regressor family (val): LGBM


In [ ]:
# CELL: EVAL
def _unscale_reg(pred: np.ndarray, col: int) -> np.ndarray:
    return as_1d(pred) * float(reg_std[0, col]) + float(reg_mean[0, col])


rows_test = all_rows[sel_test]
final_scores = {name: scores[name]["final"] for name in CANDIDATES}
win = final_scores[cls_winner_name]
pred_cls = win["pred"]
cm = np.asarray(win["confusion"])
print("TEST winner clf=", cls_winner_name, "thresholds=", win["thresholds"], "reg=", reg_winner_name)
print(
    f"test candles eligible for a signal (TP distance >= {MIN_SIGNAL_TP_PCT:.4%}): "
    f"{float(signal_eligible(rows_test).mean()):.1%}"
)
print("test confusion rows=true cols=pred LONG/SHORT/NO_TRADE")
print(cm)
print("test signals", {k: v for k, v in win["exp"].items()})

# Cost sensitivity of the winner's test signals (same decisions, different cost).
cost_sensitivity = {}
for c in CONFIG["cost_sensitivity_pct"]:
    rep = signal_expectancy(pred_cls, rows_test, cost=float(c))
    cost_sensitivity[f"{float(c):.4f}"] = {
        "score": expectancy_rank_score(rep),
        "LONG_expectancy": rep["LONG_expectancy"],
        "SHORT_expectancy": rep["SHORT_expectancy"],
    }
    print(f"cost {float(c):.4%}: score={cost_sensitivity[f'{float(c):.4f}']['score']:+.6f}")

member_test: Dict[str, Dict[str, Any]] = {}
for name in CANDIDATES:
    s = final_scores[name]
    member_test[name] = {**s["cls"], "exp": s["exp"], "score": s["score"], "thresholds": s["thresholds"]}
    if name != ENSEMBLE_NAME and "up" in runs["final"][name]:
        r = runs["final"][name]
        member_test[name]["upside_corr"] = corr(_unscale_reg(r["up"].predict(X_test), 0), r_test[:, 0])
        member_test[name]["downside_corr"] = corr(_unscale_reg(r["dn"].predict(X_test), 1), r_test[:, 1])
    print(
        name,
        "test",
        {
            "score": round(s["score"], 6) if np.isfinite(s["score"]) else s["score"],
            "LONG": (int(s["exp"]["LONG_n"]), round(s["exp"]["LONG_expectancy"], 6) if np.isfinite(s["exp"]["LONG_expectancy"]) else None),
            "SHORT": (int(s["exp"]["SHORT_n"]), round(s["exp"]["SHORT_expectancy"], 6) if np.isfinite(s["exp"]["SHORT_expectancy"]) else None),
            "P_LONG": round(s["cls"]["precision_LONG"], 3) if np.isfinite(s["cls"]["precision_LONG"]) else None,
            "P_SHORT": round(s["cls"]["precision_SHORT"], 3) if np.isfinite(s["cls"]["precision_SHORT"]) else None,
        },
    )

best_up = runs["final"][reg_winner_name]["up"]
best_dn = runs["final"][reg_winner_name]["dn"]
up_hat = _unscale_reg(best_up.predict(X_test), 0)
dn_hat = _unscale_reg(best_dn.predict(X_test), 1)
naive_up_mae = float(np.mean(np.abs(r_test[:, 0] - float(reg_mean[0, 0]))))
naive_dn_mae = float(np.mean(np.abs(r_test[:, 1] - float(reg_mean[0, 1]))))
reg_eval = {
    "upside_corr": corr(up_hat, r_test[:, 0]),
    "downside_corr": corr(dn_hat, r_test[:, 1]),
    "upside_mae_skill": 1.0 - float(np.mean(np.abs(r_test[:, 0] - up_hat))) / naive_up_mae,
    "downside_mae_skill": 1.0 - float(np.mean(np.abs(r_test[:, 1] - dn_hat))) / naive_dn_mae,
}
print("regression test", reg_eval)

eval_report = {
    "n_test": int(len(y_test)),
    "best_classifier": cls_winner_name,
    "best_regressor_family": reg_winner_name,
    "decision_rule": {
        "type": "probability_threshold_abstain",
        "thresholds": win["thresholds"],
        "min_signals": int(CONFIG["min_signals"]),
        "min_signal_tp_pct": MIN_SIGNAL_TP_PCT,
        "test_eligible_share": float(signal_eligible(rows_test).mean()),
    },
    "val_leaderboard": leaderboard,
    "test_members": member_test,
    "confusion": cm.tolist(),
    "precision_LONG": win["cls"]["precision_LONG"],
    "precision_SHORT": win["cls"]["precision_SHORT"],
    "n_LONG": win["cls"]["n_LONG"],
    "n_SHORT": win["cls"]["n_SHORT"],
    "balanced_accuracy": win["cls"]["balanced_accuracy"],
    "signal_expectancy": win["exp"],
    "score": win["score"],
    "cost_sensitivity": cost_sensitivity,
    "regression": reg_eval,
    "note": (
        "Classifier chosen by mean walk-forward thresholded expectancy (both sides, "
        f"min {int(CONFIG['min_signals'])} signals per side). Thresholds for the test come from "
        "validation only. Test is scored once. Balanced accuracy is diagnostic only."
    ),
}
print(json.dumps({k: v for k, v in eval_report.items() if k not in ("val_leaderboard", "test_members")}, indent=2, default=float))


TEST winner clf= LGBM thresholds= {'LONG': 0.3526476648420395, 'SHORT': 0.4361746067305746} reg= LGBM
test candles eligible for a signal (TP distance >= 0.6400%): 61.8%
test confusion rows=true cols=pred LONG/SHORT/NO_TRADE
[[  116    18  3690]
 [  100    40  3717]
 [  194    48 11444]]
test signals {'LONG_n': 410.0, 'LONG_expectancy': -0.0009927591039856842, 'LONG_tstat': -3.2684214119319894, 'LONG_profit_factor': 0.7003632403045161, 'LONG_total_pct': -0.40703123263413055, 'SHORT_n': 106.0, 'SHORT_expectancy': -9.72727994427298e-05, 'SHORT_tstat': -0.18451016588306657, 'SHORT_profit_factor': 0.9621794571625465, 'SHORT_total_pct': -0.01031091674092936}
cost 0.1000%: score=+0.000055
cost 0.1600%: score=-0.000545
cost 0.2000%: score=-0.000945
LR test {'score': -0.001187, 'LONG': (92, -0.000369), 'SHORT': (2397, -0.001187), 'P_LONG': 0.402, 'P_SHORT': 0.263}


RF test {'score': -0.001132, 'LONG': (680, -0.002213), 'SHORT': (195, -5.1e-05), 'P_LONG': 0.251, 'P_SHORT': 0.462}


XGB test {'score': -0.000733, 'LONG': (100, -0.00136), 'SHORT': (115, -0.000106), 'P_LONG': 0.27, 'P_SHORT': 0.417}


/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


LGBM test {'score': -0.000545, 'LONG': (410, -0.000993), 'SHORT': (106, -9.7e-05), 'P_LONG': 0.283, 'P_SHORT': 0.377}
CatBoost test {'score': -0.00159, 'LONG': (225, -0.00159), 'SHORT': (88, 0.000502), 'P_LONG': 0.298, 'P_SHORT': 0.511}
ENSEMBLE test {'score': -0.000289, 'LONG': (350, -0.001543), 'SHORT': (141, 0.000964), 'P_LONG': 0.291, 'P_SHORT': 0.539}


/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


regression test {'upside_corr': 0.16708646138519, 'downside_corr': 0.1363087149297014, 'upside_mae_skill': -0.12398556060657806, 'downside_mae_skill': -0.09151037788829819}
{
  "n_test": 19367,
  "best_classifier": "LGBM",
  "best_regressor_family": "LGBM",
  "decision_rule": {
    "type": "probability_threshold_abstain",
    "thresholds": {
      "LONG": 0.3526476648420395,
      "SHORT": 0.4361746067305746
    },
    "min_signals": 100,
    "min_signal_tp_pct": 0.0064,
    "test_eligible_share": 0.6182681881551092
  },
  "confusion": [
    [
      116,
      18,
      3690
    ],
    [
      100,
      40,
      3717
    ],
    [
      194,
      48,
      11444
    ]
  ],
  "precision_LONG": 0.28292682926829266,
  "precision_SHORT": 0.37735849056603776,
  "n_LONG": 410.0,
  "n_SHORT": 106.0,
  "balanced_accuracy": 0.3399989807907892,
  "signal_expectancy": {
    "LONG_n": 410.0,
    "LONG_expectancy": -0.0009927591039856842,
    "LONG_tstat": -3.2684214119319894,
    "LONG_profit_fa

In [ ]:
# CELL: PROMOTE
def promotion_gate(eval_rep: Dict[str, Any], lb_row: Dict[str, Any]) -> Dict[str, Any]:
    """Decide ``ready_to_promote`` from held-out evidence only.

    Requirements: positive thresholded expectancy on the single held-out test,
    every active side has >= ``min_signals`` trades and t-stat >= ``promote_min_tstat``,
    expectancy stays positive at the highest cost in ``cost_sensitivity_pct``, and the
    walk-forward positive share is >= ``promote_min_wf_positive_share``.
    """
    reasons: List[str] = []
    min_n = int(CONFIG["min_signals"])
    min_t = float(CONFIG["promote_min_tstat"])
    exp = eval_rep["signal_expectancy"]
    active_sides = [s for s in ("LONG", "SHORT") if float(exp.get(f"{s}_n", 0.0)) > 0]
    if not active_sides:
        reasons.append("no_signals_on_test")
    if not (np.isfinite(eval_rep["score"]) and eval_rep["score"] > 0.0):
        reasons.append("test_expectancy_not_positive")
    for side in active_sides:
        if float(exp[f"{side}_n"]) < min_n:
            reasons.append(f"{side.lower()}_too_few_test_signals")
        t = float(exp.get(f"{side}_tstat", float("nan")))
        if not (np.isfinite(t) and t >= min_t):
            reasons.append(f"{side.lower()}_tstat_below_{min_t:g}")
        if not (np.isfinite(exp[f"{side}_expectancy"]) and exp[f"{side}_expectancy"] > 0.0):
            reasons.append(f"{side.lower()}_expectancy_not_positive")
    worst_cost = max(float(c) for c in CONFIG["cost_sensitivity_pct"])
    worst = eval_rep["cost_sensitivity"][f"{worst_cost:.4f}"]["score"]
    if not (np.isfinite(worst) and worst > 0.0):
        reasons.append("fragile_to_higher_cost")
    if float(lb_row.get("wf_positive_share", 0.0)) < float(CONFIG["promote_min_wf_positive_share"]):
        reasons.append("walk_forward_not_consistently_positive")
    if float(lb_row.get("wf_folds_scored", 0.0)) < 2:
        reasons.append("walk_forward_insufficient_folds")
    return {
        "ready_to_promote": len(reasons) == 0,
        "reasons": reasons,
        "test_score": eval_rep["score"],
        "test_signals": {s: exp.get(f"{s}_n") for s in ("LONG", "SHORT")},
        "test_tstats": {s: exp.get(f"{s}_tstat") for s in ("LONG", "SHORT")},
        "wf_positive_share": lb_row.get("wf_positive_share"),
        "wf_mean_score": lb_row.get("wf_mean_score"),
        "worst_cost_score": worst,
    }


promotion = promotion_gate(eval_report, leaderboard[cls_winner_name])
ready_to_promote = bool(promotion["ready_to_promote"])
print(json.dumps(promotion, indent=2, default=float))
if ready_to_promote:
    print("VERDICT: held-out evidence supports paper-trading this bundle (still research-only here).")
else:
    print("VERDICT: NOT ready. Do not trade this bundle live. Reasons:", promotion["reasons"])


{
  "ready_to_promote": false,
  "reasons": [
    "test_expectancy_not_positive",
    "long_tstat_below_2",
    "long_expectancy_not_positive",
    "short_tstat_below_2",
    "short_expectancy_not_positive",
    "fragile_to_higher_cost",
    "walk_forward_not_consistently_positive"
  ],
  "test_score": -0.0005450159517142069,
  "test_signals": {
    "LONG": 410.0,
    "SHORT": 106.0
  },
  "test_tstats": {
    "LONG": -3.2684214119319894,
    "SHORT": -0.18451016588306657
  },
  "wf_positive_share": 0.0,
  "wf_mean_score": -0.001414534818372775,
  "worst_cost_score": -0.000945015951714207
}
VERDICT: NOT ready. Do not trade this bundle live. Reasons: ['test_expectancy_not_positive', 'long_tstat_below_2', 'long_expectancy_not_positive', 'short_tstat_below_2', 'short_expectancy_not_positive', 'fragile_to_higher_cost', 'walk_forward_not_consistently_positive']


In [ ]:
# CELL: EXPORT
EXPORT_DIR.mkdir(parents=True, exist_ok=True)
final_run = runs["final"][cls_winner_name]
if cls_winner_name == ENSEMBLE_NAME:
    clf_payload: Any = {
        "members": {m: runs["final"][m]["clf"] for m in final_run["members"]},
        "member_scalers": {m: runs["final"][m]["scaler"] for m in final_run["members"]},
    }
else:
    clf_payload = {"members": {cls_winner_name: final_run["clf"]}, "member_scalers": {cls_winner_name: final_run["scaler"]}}
bundle = {
    "best_classifier": cls_winner_name,
    "best_regressor_family": reg_winner_name,
    "clf": clf_payload,
    "thresholds": eval_report["decision_rule"]["thresholds"],
    "up": best_up,
    "dn": best_dn,
    "feat_mean": feat_mean,
    "feat_std": feat_std,
    "reg_mean": reg_mean,
    "reg_std": reg_std,
    "feature_cols": list(FEATURE_COLS),
    "label_names": list(LABEL_NAMES),
}
with (EXPORT_DIR / "best_models.pkl").open("wb") as handle:
    pickle.dump(bundle, handle)
(EXPORT_DIR / "leaderboard.json").write_text(
    json.dumps(leaderboard, indent=2, default=float), encoding="utf-8"
)
metadata = {
    "model_family": "jacksparrow_opportunity_10m_tabular_ensemble",
    "ensemble": list(CONFIG["ensemble"]),
    "candidates": CANDIDATES,
    "best_classifier": cls_winner_name,
    "best_regressor_family": reg_winner_name,
    "timeframe": "10m",
    "horizon_bars": int(CONFIG["horizon_bars"]),
    "horizon_minutes": int(CONFIG["horizon_bars"]) * 10,
    "tp_atr_mult": float(CONFIG["tp_atr_mult"]),
    "sl_atr_mult": float(CONFIG["sl_atr_mult"]),
    "round_trip_cost_pct": float(CONFIG["round_trip_cost_pct"]),
    "feature_cols": list(FEATURE_COLS),
    "label_names": list(LABEL_NAMES),
    "decision_rule": {
        **eval_report["decision_rule"],
        "description": (
            "At a closed 10m candle: if tp_atr_mult * ATR(14) / close < min_signal_tp_pct emit "
            "NO_TRADE (quiet-market guard). Otherwise scale features with the member scaler(s), "
            "average member predict_proba when best_classifier == ENSEMBLE, then emit LONG when "
            "P(LONG) >= thresholds.LONG and P(LONG) > P(SHORT); SHORT symmetrically; else NO_TRADE. "
            "A null threshold disables that side."
        ),
    },
    "ready_to_promote": ready_to_promote,
    "promote_reason": (
        "held_out_gate_passed_paper_trade_first" if ready_to_promote else ";".join(promotion["reasons"])
    ),
    "promotion": promotion,
    "eval": eval_report,
    "label_gate": label_report,
    "walk_forward": {name: leaderboard[name]["wf_folds"] for name in CANDIDATES},
    "config": CONFIG,
}
(EXPORT_DIR / "metadata.json").write_text(
    json.dumps(metadata, indent=2, default=float), encoding="utf-8"
)
(EXPORT_DIR / "eval_report.json").write_text(
    json.dumps(eval_report, indent=2, default=float), encoding="utf-8"
)
print("wrote", EXPORT_DIR / "best_models.pkl")
print("best classifier", cls_winner_name, "best regressor", reg_winner_name)
print("ready_to_promote=", ready_to_promote)
try:
    from google.colab import files
    import shutil

    shutil.make_archive(str(EXPORT_DIR), "zip", root_dir=str(EXPORT_DIR))
    files.download(str(EXPORT_DIR) + ".zip")
except ImportError:
    print("Not Colab; files remain under", EXPORT_DIR)


wrote export/opportunity_10m/best_models.pkl
best classifier LGBM best regressor LGBM
ready_to_promote= False


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

*File Operation*: `download` on `/content/export/opportunity_10m.zip`

**Session Created**: 2026-10-09 17:27:38
- Endpoint: `m-s-kkb-usw3b2-38pl9pq4ake1h`
- Hardware: `NONE`

In [ ]:
# CELL: ENV
# Colab ships pandas/numpy; install tabular model extras.
%pip install -q --upgrade-strategy only-if-needed pyarrow requests scikit-learn xgboost lightgbm catboost

import json
import math
import pickle
import random
import time
from concurrent.futures import ThreadPoolExecutor, as_completed
from datetime import datetime, timedelta, timezone
from pathlib import Path
from typing import Any, Dict, List, Optional, Sequence, Tuple

import numpy as np
import pandas as pd
from catboost import CatBoostClassifier, CatBoostRegressor
from lightgbm import LGBMClassifier, LGBMRegressor
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.linear_model import LogisticRegression, Ridge
from xgboost import XGBClassifier, XGBRegressor

print("sklearn / xgboost / lightgbm / catboost ready")


   ━━━━╸━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.1/97.1 MB 92.2 MB/s eta 0:00:01

   ━━━━━━━━━━━━━━━╺━━━━━━━━━━━━━━━━━━━━━━━━ 36.7/97.1 MB 128.8 MB/s eta 0:00:01

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━╸━━━━━━━━━━━━ 67.2/97.1 MB 80.5 MB/s eta 0:00:01

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╺━━━━ 85.2/97.1 MB 85.4 MB/s eta 0:00:01

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╸ 97.1/97.1 MB 102.4 MB/s eta 0:00:01

...

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╸ 97.1/97.1 MB 102.4 MB/s eta 0:00:01

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╸ 97.1/97.1 MB 102.4 MB/s eta 0:00:01

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╸ 97.1/97.1 MB 102.4 MB/s eta 0:00:01

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 7.8 MB/s eta 0:00:00


sklearn / xgboost / lightgbm / catboost ready


In [ ]:
# CELL: CONFIG
from pathlib import Path

RESOLUTION_MINUTES: Dict[str, int] = {
    "5m": 5,
    "10m": 10,
    "15m": 15,
    "30m": 30,
    "1h": 60,
}

CONFIG: Dict[str, Any] = {
    "symbol": "BTCUSD",
    "base_url": "https://api.india.delta.exchange",
    "history_days": 900,
    "resolution": "10m",
    "horizon_minutes": 240,
    "seed": 42,
    # Bracket. TP must clear the round-trip cost by a wide margin, otherwise no
    # classifier can be profitable (see the notebook header).
    "tp_atr_mult": 4.0,
    "sl_atr_mult": 2.0,
    "atr_period": 14,
    "round_trip_cost_pct": 0.0016,
    # Gate: median TP distance >= ratio x cost. Signal rule: a candle is only
    # eligible when its own TP distance >= ratio x cost (quiet-market guard).
    "min_tp_cost_ratio": 4.0,
    "minority_rate": 0.15,
    "ambiguous_win_max": 0.50,
    "train_fraction": 0.70,
    "val_fraction": 0.15,
    # Decision rule: trade only when P(side) clears a threshold chosen on
    # calibration rows. Candidates are probability quantiles of the calibration set.
    "calib_fraction": 0.15,
    "min_signals": 100,
    "signal_quantiles": [0.99, 0.98, 0.97, 0.95, 0.92, 0.90, 0.85, 0.80],
    "walk_forward_folds": 4,
    # Promotion gate thresholds (held-out test + walk-forward).
    "promote_min_tstat": 2.0,
    "promote_min_wf_positive_share": 0.67,
    "cost_sensitivity_pct": [0.0010, 0.0016, 0.0020],
    # Model capacity. Deliberately regularised: ~90k noisy rows, overlapping labels.
    "n_estimators": 400,
    "max_depth": 6,
    "learning_rate": 0.03,
    "min_leaf": 200,
    "threads_per_model": 2,
    "ensemble": ["LR", "RF", "XGB", "LGBM", "CatBoost"],
}


def apply_resolution_config(cfg: Dict[str, Any]) -> None:
    """Derive bar minutes, horizon_bars, purge, and export_dir from resolution."""
    res = str(cfg["resolution"])
    if res not in RESOLUTION_MINUTES:
        raise ValueError(
            f"Unsupported resolution {res!r}; use one of {sorted(RESOLUTION_MINUTES)}"
        )
    bar_min = int(RESOLUTION_MINUTES[res])
    horizon_min = int(cfg["horizon_minutes"])
    if bar_min <= 0 or horizon_min <= 0 or horizon_min % bar_min != 0:
        raise ValueError(
            f"horizon_minutes={horizon_min} must be a positive multiple of "
            f"{res} bar minutes ({bar_min})"
        )
    cfg["bar_minutes"] = bar_min
    cfg["horizon_bars"] = horizon_min // bar_min
    # Purge must cover the label look-ahead so no training label peeks into val.
    cfg["purge_bars"] = int(cfg["horizon_bars"]) + 1
    cfg["export_dir"] = f"export/opportunity_{res}"


_override = Path("opportunity_run_config.json")
if _override.is_file():
    CONFIG.update(json.loads(_override.read_text(encoding="utf-8")))
apply_resolution_config(CONFIG)

LABEL_NAMES: Tuple[str, ...] = ("LONG", "SHORT", "NO_TRADE")
LABEL_LONG, LABEL_SHORT, LABEL_NO_TRADE = 0, 1, 2
ENSEMBLE_NAME = "ENSEMBLE"
print(json.dumps(CONFIG, indent=2))


{
  "symbol": "BTCUSD",
  "base_url": "https://api.india.delta.exchange",
  "history_days": 900,
  "resolution": "10m",
  "horizon_minutes": 240,
  "seed": 42,
  "tp_atr_mult": 4.0,
  "sl_atr_mult": 2.0,
  "atr_period": 14,
  "round_trip_cost_pct": 0.0016,
  "min_tp_cost_ratio": 4.0,
  "minority_rate": 0.15,
  "ambiguous_win_max": 0.5,
  "train_fraction": 0.7,
  "val_fraction": 0.15,
  "calib_fraction": 0.15,
  "min_signals": 100,
  "signal_quantiles": [
    0.99,
    0.98,
    0.97,
    0.95,
    0.92,
    0.9,
    0.85,
    0.8
  ],
  "walk_forward_folds": 4,
  "promote_min_tstat": 2.0,
  "promote_min_wf_positive_share": 0.67,
  "cost_sensitivity_pct": [
    0.001,
    0.0016,
    0.002
  ],
  "n_estimators": 400,
  "max_depth": 6,
  "learning_rate": 0.03,
  "min_leaf": 200,
  "threads_per_model": 2,
  "ensemble": [
    "LR",
    "RF",
    "XGB",
    "LGBM",
    "CatBoost"
  ],
  "bar_minutes": 10,
  "horizon_bars": 24,
  "purge_bars": 25,
  "export_dir": "export/opportunity_10m"
}


In [ ]:
# CELL: SEED
def set_seed(seed: int) -> None:
    random.seed(int(seed))
    np.random.seed(int(seed))


set_seed(int(CONFIG["seed"]))
print("seed", CONFIG["seed"])


seed 42


In [ ]:
# CELL: FETCH
DELTA_CANDLE_PAGE_BARS = 2000
REQUEST_DELAY_SECONDS = 0.2
EMPTY_PAGE_RETRIES = 3
MIN_OHLCV_COMPLETENESS = 0.95
_RESOLUTION_MINUTES = {
    "1m": 1,
    "3m": 3,
    "5m": 5,
    "10m": 10,
    "15m": 15,
    "30m": 30,
    "1h": 60,
    "2h": 120,
    "4h": 240,
    "1d": 1440,
}


def _bar_seconds(resolution: str) -> int:
    if resolution not in _RESOLUTION_MINUTES:
        raise ValueError(f"Unsupported resolution {resolution!r}")
    return int(_RESOLUTION_MINUTES[resolution]) * 60


def _parse_candle_rows(payload: Any) -> List[Dict[str, Any]]:
    if not isinstance(payload, dict):
        return []
    result = payload.get("result", [])
    if isinstance(result, dict):
        result = result.get("candles", []) or []
    if not isinstance(result, list):
        return []
    return [row for row in result if isinstance(row, dict) and "time" in row]


def _candle_epoch(row: Dict[str, Any]) -> Optional[int]:
    try:
        return int(row["time"])
    except (KeyError, TypeError, ValueError):
        return None


def _request_candle_page(
    url: str,
    *,
    symbol: str,
    resolution: str,
    start_ts: int,
    end_ts: int,
) -> List[Dict[str, Any]]:
    import requests

    last_error: Optional[Exception] = None
    for attempt in range(EMPTY_PAGE_RETRIES):
        try:
            response = requests.get(
                url,
                params={
                    "symbol": symbol,
                    "resolution": resolution,
                    "start": int(start_ts),
                    "end": int(end_ts),
                },
                timeout=30,
            )
            response.raise_for_status()
            payload = response.json()
        except Exception as exc:
            last_error = exc
            time.sleep(REQUEST_DELAY_SECONDS * (attempt + 1))
            continue
        if payload.get("success") is False:
            last_error = RuntimeError(f"Delta candles error for {symbol}: {payload}")
            time.sleep(REQUEST_DELAY_SECONDS * (attempt + 1))
            continue
        rows = _parse_candle_rows(payload)
        if rows:
            return rows
        last_error = None
        time.sleep(REQUEST_DELAY_SECONDS * (attempt + 1))
    if last_error is not None:
        raise last_error
    return []


def _rows_to_frame(all_rows: List[Dict[str, Any]]) -> pd.DataFrame:
    df = pd.DataFrame(all_rows)
    expected_cols = ["time", "open", "high", "low", "close", "volume"]
    df = df[[c for c in expected_cols if c in df.columns]]
    df["time"] = pd.to_datetime(df["time"], unit="s", utc=True)
    df = df.drop_duplicates(subset="time").sort_values("time").reset_index(drop=True)
    for col in ["open", "high", "low", "close", "volume"]:
        if col in df.columns:
            df[col] = df[col].astype(float)
    return df


def fetch_candles(
    symbol: str,
    resolution: str,
    start_ts: int,
    end_ts: int,
    base_url: str,
    *,
    page_bars: int = DELTA_CANDLE_PAGE_BARS,
) -> pd.DataFrame:
    url = f"{base_url.rstrip('/')}/v2/history/candles"
    bar_seconds = _bar_seconds(resolution)
    page_bars = max(1, int(page_bars))
    start_ts, end_ts = int(start_ts), int(end_ts)
    if end_ts <= start_ts:
        raise ValueError("end_ts must be greater than start_ts")
    all_rows: List[Dict[str, Any]] = []
    cursor_end = end_ts
    seen_oldest: Optional[int] = None
    max_pages = max(2, (end_ts - start_ts) // bar_seconds + 5)
    for _ in range(max_pages):
        if cursor_end <= start_ts:
            break
        page_start = max(start_ts, cursor_end - page_bars * bar_seconds)
        if page_start >= cursor_end:
            break
        rows = _request_candle_page(
            url,
            symbol=symbol,
            resolution=resolution,
            start_ts=page_start,
            end_ts=cursor_end,
        )
        if not rows:
            break
        epochs = [e for e in (_candle_epoch(r) for r in rows) if e is not None]
        if not epochs:
            break
        oldest = min(epochs)
        if seen_oldest is not None and oldest >= seen_oldest:
            break
        seen_oldest = oldest
        all_rows.extend(rows)
        cursor_end = oldest - 1
        time.sleep(REQUEST_DELAY_SECONDS)
    if not all_rows:
        raise RuntimeError(f"No candle data returned for symbol={symbol}")
    return _rows_to_frame(all_rows)


def validate_ohlcv_completeness(
    df: pd.DataFrame,
    resolution: str,
    *,
    min_completeness: float = MIN_OHLCV_COMPLETENESS,
    symbol: str = "",
) -> Dict[str, Any]:
    if df is None or df.empty or "time" not in df.columns:
        raise ValueError("Cannot validate completeness on an empty OHLCV frame")
    bar_seconds = _bar_seconds(resolution)
    time_col = df["time"]
    epochs = (pd.to_datetime(time_col, utc=True).astype("int64") // 10**9).tolist()
    if len(epochs) < 2:
        raise ValueError("Need at least 2 OHLCV bars to validate completeness")
    gaps = 0
    for prev, cur in zip(epochs, epochs[1:]):
        if int(cur) - int(prev) > bar_seconds * 1.5:
            gaps += 1
    completeness = 1.0 - (gaps / max(1, len(epochs) - 1))
    report = {
        "rows": len(epochs),
        "gaps": int(gaps),
        "completeness": round(float(completeness), 4),
    }
    if completeness < min_completeness:
        label = f"{symbol} {resolution}".strip()
        raise ValueError(
            f"OHLCV completeness {completeness:.1%} below {min_completeness:.0%} "
            f"for {label or 'candles'}"
        )
    return report


end_dt = datetime.now(timezone.utc)
start_dt = end_dt - timedelta(days=int(CONFIG["history_days"]))
_resolution = str(CONFIG["resolution"])
print(
    f"Fetching {CONFIG['symbol']} {_resolution} OHLCV for {CONFIG['history_days']}d "
    f"from {CONFIG['base_url']}..."
)
bars = fetch_candles(
    str(CONFIG["symbol"]),
    _resolution,
    int(start_dt.timestamp()),
    int(end_dt.timestamp()),
    str(CONFIG["base_url"]),
)
ohlcv_report = validate_ohlcv_completeness(
    bars, _resolution, symbol=str(CONFIG["symbol"])
)
print(
    f"OHLCV {CONFIG['symbol']} {_resolution}: {ohlcv_report['rows']} bars, "
    f"completeness {ohlcv_report['completeness']:.1%}, gaps={ohlcv_report['gaps']}"
)


Fetching BTCUSD 10m OHLCV for 900d from https://api.india.delta.exchange...


HTTPError: 400 Client Error: Bad Request for url: https://api.india.delta.exchange/v2/history/candles?symbol=BTCUSD&resolution=10m&start=1790366884&end=1791566884

In [ ]:
# CELL: BARS
if bars is None or getattr(bars, "empty", True):
    raise ValueError(f"No OHLCV bars for {CONFIG['resolution']}")
print(
    f"{CONFIG['resolution']} bars={len(bars)} "
    f"span={bars['time'].iloc[0]} → {bars['time'].iloc[-1]} "
    f"horizon_bars={CONFIG['horizon_bars']} ({CONFIG['horizon_minutes']}m)"
)


NameError: name 'bars' is not defined

In [ ]:
# CELL: FEATURES
OHLCV_COLS: Tuple[str, ...] = ("time", "open", "high", "low", "close", "volume")


def _wilder_rsi(close: pd.Series, period: int = 14) -> pd.Series:
    delta = close.diff()
    gain = delta.clip(lower=0.0)
    loss = -delta.clip(upper=0.0)
    avg_gain = gain.ewm(alpha=1.0 / period, min_periods=period, adjust=False).mean()
    avg_loss = loss.ewm(alpha=1.0 / period, min_periods=period, adjust=False).mean()
    rs = avg_gain / avg_loss.replace(0.0, np.nan)
    return 100.0 - (100.0 / (1.0 + rs))


def add_opportunity_features(df: pd.DataFrame, *, atr_period: int = 14) -> pd.DataFrame:
    """Causal signal-candle features from the selected timeframe series only.

    Everything is expressed in ATR units or as a ratio so the model is scale
    invariant across the 900-day price range. ATR(14) is also stored for the
    label path. Nothing here looks past the signal candle.
    """
    out = df.copy()
    close = out["close"].astype(np.float64)
    high = out["high"].astype(np.float64)
    low = out["low"].astype(np.float64)
    open_ = out["open"].astype(np.float64)
    volume = out["volume"].astype(np.float64)
    prev_close = close.shift(1)
    tr = pd.concat(
        [high - low, (high - prev_close).abs(), (low - prev_close).abs()], axis=1
    ).max(axis=1)
    atr = tr.rolling(int(atr_period), min_periods=int(atr_period)).mean()
    atr_safe = atr.replace(0.0, np.nan)
    out["atr"] = atr

    feats: Dict[str, pd.Series] = {}
    # Momentum at several lookbacks, in ATR units.
    feats["ret_1"] = close.pct_change()
    for k in (3, 6, 12, 24, 48, 144):
        feats[f"ret_{k}_atr"] = (close - close.shift(k)) / atr_safe
    # Volatility level and regime.
    feats["atr_pct"] = atr / close.replace(0.0, np.nan)
    atr_long = tr.rolling(int(atr_period) * 6, min_periods=int(atr_period) * 6).mean()
    feats["atr_ratio_84"] = atr / atr_long.replace(0.0, np.nan)
    log_ret = np.log(close).diff()
    rv_12 = log_ret.rolling(12, min_periods=12).std()
    rv_48 = log_ret.rolling(48, min_periods=48).std()
    rv_144 = log_ret.rolling(144, min_periods=144).std()
    feats["rv_12_48"] = rv_12 / rv_48.replace(0.0, np.nan)
    feats["rv_48_144"] = rv_48 / rv_144.replace(0.0, np.nan)
    # Oscillators and trend.
    for period in (7, 14, 28):
        feats[f"rsi_{period}"] = _wilder_rsi(close, period=period)
    for span in (20, 50, 200):
        ema = close.ewm(span=span, adjust=False, min_periods=span).mean()
        feats[f"ema_dist_{span}_atr"] = (close - ema) / atr_safe
        feats[f"ema_slope_{span}_atr"] = (ema - ema.shift(6)) / atr_safe
    # Candle anatomy.
    feats["candle_body"] = (close - open_) / atr_safe
    feats["upper_wick"] = (high - np.maximum(close, open_)) / atr_safe
    feats["lower_wick"] = (np.minimum(close, open_) - low) / atr_safe
    feats["range_atr"] = (high - low) / atr_safe
    # Volume anomalies.
    for window in (20, 100):
        vol_mean = volume.rolling(window, min_periods=window).mean()
        vol_std = volume.rolling(window, min_periods=window).std()
        feats[f"volume_z_{window}"] = (volume - vol_mean) / vol_std.replace(0.0, np.nan)
    # Position inside recent ranges and distance to swing extremes.
    for window in (24, 72, 144):
        hh = high.rolling(window, min_periods=window).max()
        ll = low.rolling(window, min_periods=window).min()
        feats[f"range_pos_{window}"] = (close - ll) / (hh - ll).replace(0.0, np.nan)
        feats[f"dist_high_{window}_atr"] = (hh - close) / atr_safe
        feats[f"dist_low_{window}_atr"] = (close - ll) / atr_safe
    # Consecutive same-direction closes (signed).
    sign = np.sign(close.diff()).fillna(0.0)
    groups = (sign != sign.shift()).cumsum()
    feats["streak"] = sign * (sign.groupby(groups).cumcount() + 1)
    # Intraday / weekly seasonality.
    ts = pd.to_datetime(out["time"], utc=True)
    hour = ts.dt.hour + ts.dt.minute / 60.0
    feats["hour_sin"] = np.sin(2.0 * np.pi * hour / 24.0)
    feats["hour_cos"] = np.cos(2.0 * np.pi * hour / 24.0)
    feats["dow_sin"] = np.sin(2.0 * np.pi * ts.dt.dayofweek / 7.0)
    feats["dow_cos"] = np.cos(2.0 * np.pi * ts.dt.dayofweek / 7.0)
    # Short raw history of the last six candle returns (ATR units).
    for lag in range(1, 7):
        feats[f"ret_lag{lag}_atr"] = (close.shift(lag - 1) - close.shift(lag)) / atr_safe
    return pd.concat([out, pd.DataFrame(feats, index=out.index)], axis=1)


featured = add_opportunity_features(bars, atr_period=int(CONFIG["atr_period"]))
FEATURE_COLS: Tuple[str, ...] = tuple(
    c for c in featured.columns if c not in OHLCV_COLS and c != "atr"
)
print(len(FEATURE_COLS), "features")
null_rates = featured[list(FEATURE_COLS)].isna().mean()
print("max feature null rate", round(float(null_rates.max()), 4), "at", null_rates.idxmax())


NameError: name 'bars' is not defined

In [ ]:
# CELL: LABELS
# Opportunity labels on a closed-bar series. Path is bars t+1 .. t+horizon only.
# Vectorised over bars; semantics are identical to walking each path bar by bar.

import numpy as np
import pandas as pd

_EPS = 1e-9
HORIZON_BARS_DEFAULT = 24
ATR_PERIOD_DEFAULT = 14
TP_ATR_MULT_DEFAULT = 4.0
SL_ATR_MULT_DEFAULT = 2.0

TP_FIRST = 0
SL_FIRST = 1
NEITHER = 2
AMBIGUOUS = 3

LABEL_LONG = 0
LABEL_SHORT = 1
LABEL_NO_TRADE = 2
LABEL_UNLABELED = -1
LABEL_NAMES = ("LONG", "SHORT", "NO_TRADE")


def tp_sl_from_path(
    fwd_high: np.ndarray,
    fwd_low: np.ndarray,
    entry: float,
    atr: float,
    tp_mult: float,
    sl_mult: float,
    side: str,
) -> int:
    """Walk future highs/lows. Same-bar TP and SL is AMBIGUOUS, not a win."""
    high = np.asarray(fwd_high, dtype=np.float64).ravel()
    low = np.asarray(fwd_low, dtype=np.float64).ravel()
    if high.size == 0 or high.size != low.size:
        return int(NEITHER)
    denom = max(float(atr), _EPS)
    entry_f = float(entry)
    if str(side).strip().upper() == "SHORT":
        tp_level = entry_f - float(tp_mult) * denom
        sl_level = entry_f + float(sl_mult) * denom
        tp_hit = low <= tp_level
        sl_hit = high >= sl_level
    else:
        tp_level = entry_f + float(tp_mult) * denom
        sl_level = entry_f - float(sl_mult) * denom
        tp_hit = high >= tp_level
        sl_hit = low <= sl_level
    for j in range(high.size):
        hit_tp = bool(tp_hit[j])
        hit_sl = bool(sl_hit[j])
        if hit_tp and hit_sl:
            return int(AMBIGUOUS)
        if hit_tp:
            return int(TP_FIRST)
        if hit_sl:
            return int(SL_FIRST)
    return int(NEITHER)


def _forward_stack(values: np.ndarray, k: int) -> np.ndarray:
    """(k, n) array where row j-1 holds values shifted back by j bars (NaN padded)."""
    n = values.shape[0]
    out = np.full((k, n), np.nan, dtype=np.float64)
    for j in range(1, k + 1):
        if n - j > 0:
            out[j - 1, : n - j] = values[j:]
    return out


def _side_codes_vectorised(
    fwd_high: np.ndarray,
    fwd_low: np.ndarray,
    entry: np.ndarray,
    atr: np.ndarray,
    tp_mult: float,
    sl_mult: float,
    side: str,
) -> np.ndarray:
    """Vectorised twin of ``tp_sl_from_path`` for every bar at once."""
    denom = np.maximum(atr, _EPS)
    if str(side).strip().upper() == "SHORT":
        tp_level = entry - float(tp_mult) * denom
        sl_level = entry + float(sl_mult) * denom
        tp_hits = fwd_low <= tp_level[None, :]
        sl_hits = fwd_high >= sl_level[None, :]
    else:
        tp_level = entry + float(tp_mult) * denom
        sl_level = entry - float(sl_mult) * denom
        tp_hits = fwd_high >= tp_level[None, :]
        sl_hits = fwd_low <= sl_level[None, :]
    any_hit = tp_hits | sl_hits
    has_hit = any_hit.any(axis=0)
    first = np.argmax(any_hit, axis=0)
    cols = np.arange(entry.shape[0])
    tp_first = tp_hits[first, cols]
    sl_first = sl_hits[first, cols]
    codes = np.where(
        ~has_hit,
        NEITHER,
        np.where(tp_first & sl_first, AMBIGUOUS, np.where(tp_first, TP_FIRST, SL_FIRST)),
    )
    return codes.astype(np.int64)


def _causal_atr(df: pd.DataFrame, period: int = ATR_PERIOD_DEFAULT) -> np.ndarray:
    if "atr" in df.columns:
        atr = df["atr"].to_numpy(dtype=np.float64)
        if np.isfinite(atr).any():
            return atr
    close = df["close"].to_numpy(dtype=np.float64)
    high = df["high"].to_numpy(dtype=np.float64)
    low = df["low"].to_numpy(dtype=np.float64)
    prev = np.concatenate([[close[0]], close[:-1]])
    tr = np.maximum(high - low, np.maximum(np.abs(high - prev), np.abs(low - prev)))
    return pd.Series(tr).rolling(int(period), min_periods=int(period)).mean().to_numpy(
        dtype=np.float64
    )


def compute_opportunity_labels(
    df: pd.DataFrame,
    *,
    tp_mult: float = TP_ATR_MULT_DEFAULT,
    sl_mult: float = SL_ATR_MULT_DEFAULT,
    horizon_bars: int = HORIZON_BARS_DEFAULT,
    atr_period: int = ATR_PERIOD_DEFAULT,
) -> pd.DataFrame:
    """Label each closed bar. Last ``horizon_bars`` rows stay unlabeled.

    Realised P&L columns (``long_pnl_pct`` / ``short_pnl_pct``) are gross of
    cost: +TP distance on TP_FIRST, −SL distance on SL_FIRST **and** AMBIGUOUS
    (a same-bar double touch is scored as a stop), timeout return on NEITHER.
    """
    out = df.copy().reset_index(drop=True)
    n = len(out)
    k = int(horizon_bars)
    close = out["close"].to_numpy(dtype=np.float64)
    high = out["high"].to_numpy(dtype=np.float64)
    low = out["low"].to_numpy(dtype=np.float64)
    atr = np.asarray(_causal_atr(out, period=int(atr_period)), dtype=np.float64)
    if atr.shape[0] != n:
        atr = np.resize(atr, n)

    fwd_high = _forward_stack(high, k)
    fwd_low = _forward_stack(low, k)
    fwd_close_end = np.full(n, np.nan, dtype=np.float64)
    if n - k > 0:
        fwd_close_end[: n - k] = close[k:]

    index = np.arange(n)
    valid = (
        (index < n - k)
        & np.isfinite(atr)
        & (atr > _EPS)
        & np.isfinite(close)
        & (np.abs(close) > _EPS)
    )
    atr_safe = np.where(valid, atr, np.nan)
    entry_safe = np.where(valid, close, np.nan)

    l_code = _side_codes_vectorised(fwd_high, fwd_low, entry_safe, atr_safe, tp_mult, sl_mult, "LONG")
    s_code = _side_codes_vectorised(fwd_high, fwd_low, entry_safe, atr_safe, tp_mult, sl_mult, "SHORT")
    l_win = l_code == TP_FIRST
    s_win = s_code == TP_FIRST
    label = np.where(l_win & ~s_win, LABEL_LONG, np.where(s_win & ~l_win, LABEL_SHORT, LABEL_NO_TRADE))

    with np.errstate(invalid="ignore", divide="ignore"):
        max_high = np.nanmax(np.where(np.isfinite(fwd_high), fwd_high, -np.inf), axis=0)
        min_low = np.nanmin(np.where(np.isfinite(fwd_low), fwd_low, np.inf), axis=0)
        upside = np.maximum((max_high - entry_safe) / atr_safe, 0.0)
        downside = np.maximum((entry_safe - min_low) / atr_safe, 0.0)
        timeout_ret = (fwd_close_end - entry_safe) / entry_safe
        tp_pct = float(tp_mult) * atr_safe / entry_safe
        sl_pct = float(sl_mult) * atr_safe / entry_safe
        long_pnl = np.where(
            l_code == TP_FIRST, tp_pct, np.where(l_code == NEITHER, timeout_ret, -sl_pct)
        )
        short_pnl = np.where(
            s_code == TP_FIRST, tp_pct, np.where(s_code == NEITHER, -timeout_ret, -sl_pct)
        )

    def _masked(values: np.ndarray) -> np.ndarray:
        return np.where(valid, values, np.nan)

    out["atr"] = atr
    out["trade_label"] = np.where(valid, label, LABEL_UNLABELED).astype(np.int64)
    out["long_result"] = _masked(l_win.astype(np.float64))
    out["short_result"] = _masked(s_win.astype(np.float64))
    out["long_code"] = _masked(l_code.astype(np.float64))
    out["short_code"] = _masked(s_code.astype(np.float64))
    out["upside"] = _masked(upside)
    out["downside"] = _masked(downside)
    out["timeout_ret"] = _masked(timeout_ret)
    out["tp_pct"] = _masked(tp_pct)
    out["sl_pct"] = _masked(sl_pct)
    out["long_pnl_pct"] = _masked(long_pnl)
    out["short_pnl_pct"] = _masked(short_pnl)
    names = np.array(["UNLABELED"] * n, dtype=object)
    names[out["trade_label"].to_numpy() == LABEL_LONG] = "LONG"
    names[out["trade_label"].to_numpy() == LABEL_SHORT] = "SHORT"
    names[out["trade_label"].to_numpy() == LABEL_NO_TRADE] = "NO_TRADE"
    out["trade_name"] = names
    return out


In [ ]:
# CELL: APPLY_LABELS
labeled = compute_opportunity_labels(
    featured,
    tp_mult=float(CONFIG["tp_atr_mult"]),
    sl_mult=float(CONFIG["sl_atr_mult"]),
    horizon_bars=int(CONFIG["horizon_bars"]),
    atr_period=int(CONFIG["atr_period"]),
)
valid = labeled["trade_label"].to_numpy() >= 0
print(f"labeled rows={int(valid.sum())} / {len(labeled)}")
print(labeled.loc[valid, "trade_name"].value_counts(normalize=True).round(4))
print(
    "median TP distance",
    f"{labeled.loc[valid, 'tp_pct'].median():.4%}",
    "median SL distance",
    f"{labeled.loc[valid, 'sl_pct'].median():.4%}",
    "round trip cost",
    f"{float(CONFIG['round_trip_cost_pct']):.4%}",
)


NameError: name 'featured' is not defined

In [ ]:
# CELL: GATE
def train_period_mask(n: int, train_fraction: float) -> np.ndarray:
    cut = int(n * float(train_fraction))
    mask = np.zeros(n, dtype=bool)
    mask[:cut] = True
    return mask


def _path_code_counts(codes: np.ndarray, usable: np.ndarray) -> Dict[str, int]:
    finite = usable & np.isfinite(codes)
    return {
        "TP_FIRST": int(np.sum(finite & (codes == TP_FIRST))),
        "SL_FIRST": int(np.sum(finite & (codes == SL_FIRST))),
        "NEITHER": int(np.sum(finite & (codes == NEITHER))),
        "AMBIGUOUS": int(np.sum(finite & (codes == AMBIGUOUS))),
    }


def summarize_label_gate(
    df: pd.DataFrame,
    *,
    train_fraction: float,
    round_trip_cost_pct: float,
    minority_rate: float,
    ambiguous_win_max: float,
    min_tp_cost_ratio: float = 4.0,
) -> Dict[str, Any]:
    """Decide whether the bracket is worth training on, using train rows only.

    Besides class mix and oracle exclusive expectancy, the gate checks the
    bracket economics: the median take-profit distance must be at least
    ``min_tp_cost_ratio`` times the round-trip cost, and the implied
    break-even win rate ``(sl + cost) / (tp + sl)`` is reported.
    """
    n = len(df)
    in_train = train_period_mask(n, train_fraction)
    labels = df["trade_label"].to_numpy(dtype=np.int64)
    usable = in_train & (labels >= 0)
    n_use = int(usable.sum())
    if n_use == 0:
        raise RuntimeError("Train-period gate has no labeled rows")
    mix = {}
    mix_ok = True
    for cid, name in enumerate(LABEL_NAMES):
        rate = float(np.mean(labels[usable] == cid))
        mix[name] = rate
        if rate < float(minority_rate):
            mix_ok = False
    long_code = df["long_code"].to_numpy(dtype=np.float64)
    short_code = df["short_code"].to_numpy(dtype=np.float64)
    cost = float(round_trip_cost_pct)
    long_pnl = df["long_pnl_pct"].to_numpy(dtype=np.float64) - cost
    short_pnl = df["short_pnl_pct"].to_numpy(dtype=np.float64) - cost

    always_long_mask = usable & np.isfinite(long_pnl)
    always_short_mask = usable & np.isfinite(short_pnl)
    always_in_long = (
        float(np.mean(long_pnl[always_long_mask])) if always_long_mask.any() else float("nan")
    )
    always_in_short = (
        float(np.mean(short_pnl[always_short_mask])) if always_short_mask.any() else float("nan")
    )

    excl_long_mask = usable & (labels == LABEL_LONG) & np.isfinite(long_pnl)
    excl_short_mask = usable & (labels == LABEL_SHORT) & np.isfinite(short_pnl)
    exclusive_long = (
        float(np.mean(long_pnl[excl_long_mask])) if excl_long_mask.any() else float("nan")
    )
    exclusive_short = (
        float(np.mean(short_pnl[excl_short_mask])) if excl_short_mask.any() else float("nan")
    )

    def _ambig_share(codes: np.ndarray) -> float:
        finite = usable & np.isfinite(codes)
        tp = finite & (codes == TP_FIRST)
        amb = finite & (codes == AMBIGUOUS)
        denom = int(tp.sum() + amb.sum())
        if denom == 0:
            return 0.0
        return float(amb.sum()) / float(denom)

    long_ambig = _ambig_share(long_code)
    short_ambig = _ambig_share(short_code)

    if "tp_pct" in df.columns and "sl_pct" in df.columns:
        tp_pct = df["tp_pct"].to_numpy(dtype=np.float64)
        sl_pct = df["sl_pct"].to_numpy(dtype=np.float64)
    else:
        atr_pct = df["atr"].to_numpy(dtype=np.float64) / df["close"].to_numpy(dtype=np.float64)
        tp_pct = float(CONFIG["tp_atr_mult"]) * atr_pct
        sl_pct = float(CONFIG["sl_atr_mult"]) * atr_pct
    econ_mask = usable & np.isfinite(tp_pct) & np.isfinite(sl_pct)
    median_tp = float(np.median(tp_pct[econ_mask])) if econ_mask.any() else float("nan")
    median_sl = float(np.median(sl_pct[econ_mask])) if econ_mask.any() else float("nan")
    tp_cost_ratio = median_tp / cost if cost > 0 and np.isfinite(median_tp) else float("inf")
    breakeven_win_rate = (
        (median_sl + cost) / (median_tp + median_sl)
        if np.isfinite(median_tp) and np.isfinite(median_sl) and (median_tp + median_sl) > 0
        else float("nan")
    )
    long_tp_rate = float(np.mean(long_code[usable & np.isfinite(long_code)] == TP_FIRST))
    short_tp_rate = float(np.mean(short_code[usable & np.isfinite(short_code)] == TP_FIRST))

    reasons: List[str] = []
    if not mix_ok:
        reasons.append("class_mix_below_floor")
    if not (np.isfinite(exclusive_long) and exclusive_long > 0.0):
        reasons.append("exclusive_long_expectancy_not_positive")
    if not (np.isfinite(exclusive_short) and exclusive_short > 0.0):
        reasons.append("exclusive_short_expectancy_not_positive")
    if long_ambig >= float(ambiguous_win_max):
        reasons.append("long_ambiguous_majority")
    if short_ambig >= float(ambiguous_win_max):
        reasons.append("short_ambiguous_majority")
    if tp_cost_ratio < float(min_tp_cost_ratio):
        reasons.append("tp_distance_below_cost_ratio")
    passed = len(reasons) == 0
    return {
        "passed": passed,
        "n_train_labeled": n_use,
        "class_mix": mix,
        "minority_rate": float(minority_rate),
        "exclusive_long_expectancy_pct": exclusive_long,
        "exclusive_short_expectancy_pct": exclusive_short,
        "always_in_long_expectancy_pct": always_in_long,
        "always_in_short_expectancy_pct": always_in_short,
        "round_trip_cost_pct": cost,
        "median_tp_pct": median_tp,
        "median_sl_pct": median_sl,
        "tp_cost_ratio": tp_cost_ratio,
        "min_tp_cost_ratio": float(min_tp_cost_ratio),
        "breakeven_win_rate": breakeven_win_rate,
        "base_long_tp_rate": long_tp_rate,
        "base_short_tp_rate": short_tp_rate,
        "long_ambiguous_share": long_ambig,
        "short_ambiguous_share": short_ambig,
        "long_path_codes": _path_code_counts(long_code, usable),
        "short_path_codes": _path_code_counts(short_code, usable),
        "tp_atr_mult": float(CONFIG["tp_atr_mult"]),
        "sl_atr_mult": float(CONFIG["sl_atr_mult"]),
        "reasons": reasons,
    }


EXPORT_DIR = Path(str(CONFIG["export_dir"]))
EXPORT_DIR.mkdir(parents=True, exist_ok=True)
label_report = summarize_label_gate(
    labeled,
    train_fraction=float(CONFIG["train_fraction"]),
    round_trip_cost_pct=float(CONFIG["round_trip_cost_pct"]),
    minority_rate=float(CONFIG["minority_rate"]),
    ambiguous_win_max=float(CONFIG["ambiguous_win_max"]),
    min_tp_cost_ratio=float(CONFIG["min_tp_cost_ratio"]),
)
report_path = EXPORT_DIR / "label_report.json"
report_path.write_text(json.dumps(label_report, indent=2), encoding="utf-8")
print(json.dumps(label_report, indent=2))
print("wrote", report_path)
if not label_report["passed"]:
    raise RuntimeError(
        "Train-period label gate failed: "
        f"{label_report['reasons']}. Change CONFIG tp/sl ATR multiples or horizon and "
        "re-run from the top. Do not fit models."
    )
print(
    f"Gate passed. Break-even win rate {label_report['breakeven_win_rate']:.1%} vs base "
    f"TP rates long {label_report['base_long_tp_rate']:.1%} / short "
    f"{label_report['base_short_tp_rate']:.1%}. Training may start."
)


NameError: name 'labeled' is not defined

In [ ]:
# CELL: SPLIT
def purged_index_splits(
    n: int,
    *,
    train_fraction: float,
    val_fraction: float,
    purge_bars: int,
) -> Dict[str, np.ndarray]:
    """Time-ordered split. Purge ``purge_bars`` on each side of every cut."""
    n = int(n)
    purge = int(purge_bars)
    train_end = int(n * float(train_fraction))
    val_len = int(n * float(val_fraction))
    train_idx = np.arange(0, max(train_end - purge, 0), dtype=np.int64)
    val_start = min(train_end + purge, n)
    val_end = min(val_start + val_len, n)
    val_idx = np.arange(val_start, max(val_end - purge, val_start), dtype=np.int64)
    test_start = min(val_end + purge, n)
    test_idx = np.arange(test_start, n, dtype=np.int64)
    return {"train": train_idx, "val": val_idx, "test": test_idx}


def development_walk_forward(
    n_dev: int,
    *,
    folds: int = 4,
    purge_bars: int,
) -> List[Tuple[np.ndarray, np.ndarray]]:
    """Expanding-window folds inside the development prefix only."""
    folds = max(2, int(folds))
    purge = int(purge_bars)
    edges = [int(n_dev * (i + 1) / folds) for i in range(folds)]
    out: List[Tuple[np.ndarray, np.ndarray]] = []
    for i in range(1, folds):
        train_end = edges[i - 1]
        val_end = edges[i]
        tr = np.arange(0, max(train_end - purge, 0), dtype=np.int64)
        va = np.arange(min(train_end + purge, n_dev), val_end, dtype=np.int64)
        if len(tr) == 0 or len(va) == 0:
            continue
        out.append((tr, va))
    return out


def inner_calibration_split(
    train_idx: np.ndarray, *, calib_fraction: float, purge_bars: int
) -> Tuple[np.ndarray, np.ndarray]:
    """Reserve the purged tail of a training window for threshold selection."""
    n = len(train_idx)
    cut = int(n * (1.0 - float(calib_fraction)))
    fit_idx = train_idx[: max(cut - int(purge_bars), 0)]
    calib_idx = train_idx[cut:]
    return fit_idx, calib_idx


splits = purged_index_splits(
    len(labeled),
    train_fraction=float(CONFIG["train_fraction"]),
    val_fraction=float(CONFIG["val_fraction"]),
    purge_bars=int(CONFIG["purge_bars"]),
)
for key, idx in splits.items():
    print(key, "n", len(idx), "range", int(idx[0]) if len(idx) else None, int(idx[-1]) if len(idx) else None)
dev_end = int(splits["test"][0]) if len(splits["test"]) else len(labeled)
wf_folds = development_walk_forward(
    dev_end, folds=int(CONFIG["walk_forward_folds"]), purge_bars=int(CONFIG["purge_bars"])
)
print("walk-forward folds", len(wf_folds), "dev_end", dev_end)


NameError: name 'labeled' is not defined

In [ ]:
# CELL: DATASET
def build_rows(
    frame: pd.DataFrame, *, feature_cols: Sequence[str]
) -> Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    """One tabular row per labeled bar with finite features and finite path targets."""
    feats = frame.loc[:, list(feature_cols)].to_numpy(dtype=np.float64)
    y_cls = frame["trade_label"].to_numpy(dtype=np.int64)
    y_reg = frame.loc[:, ["upside", "downside"]].to_numpy(dtype=np.float64)
    ok = (y_cls >= 0) & np.isfinite(feats).all(axis=1) & np.isfinite(y_reg).all(axis=1)
    rows = np.flatnonzero(ok).astype(np.int64)
    return (
        feats[rows].astype(np.float32),
        y_cls[rows],
        y_reg[rows].astype(np.float32),
        rows,
    )


all_x, all_y, all_r, all_rows = build_rows(labeled, feature_cols=FEATURE_COLS)
print(
    "rows",
    all_x.shape,
    "class mix",
    {n: round(float(np.mean(all_y == i)), 4) for i, n in enumerate(LABEL_NAMES)},
)


def rows_in(split_idx: np.ndarray) -> np.ndarray:
    """Boolean selector over ``all_rows`` for a set of frame indices."""
    allowed = np.zeros(len(labeled), dtype=bool)
    allowed[np.asarray(split_idx, dtype=np.int64)] = True
    return allowed[all_rows]


MIN_SIGNAL_TP_PCT = float(CONFIG["min_tp_cost_ratio"]) * float(CONFIG["round_trip_cost_pct"])


def signal_eligible(rows: np.ndarray) -> np.ndarray:
    """Quiet-market guard: a candle may carry a signal only if its TP distance pays for cost."""
    tp_pct = labeled["tp_pct"].to_numpy(dtype=np.float64)[rows]
    return np.isfinite(tp_pct) & (tp_pct >= MIN_SIGNAL_TP_PCT)


print(
    f"signal eligibility: TP distance >= {MIN_SIGNAL_TP_PCT:.4%} -> "
    f"{float(signal_eligible(all_rows).mean()):.1%} of labeled candles"
)


def fit_scaler(x: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:
    mean = x.mean(axis=0, keepdims=True)
    std = x.std(axis=0, keepdims=True)
    std = np.where(std < 1e-6, 1.0, std)
    return mean.astype(np.float32), std.astype(np.float32)


def scale_x(x: np.ndarray, mean: np.ndarray, std: np.ndarray) -> np.ndarray:
    return ((x - mean) / std).astype(np.float32)


sel_train = rows_in(splits["train"])
sel_val = rows_in(splits["val"])
sel_test = rows_in(splits["test"])
feat_mean, feat_std = fit_scaler(all_x[sel_train])
reg_mean = all_r[sel_train].mean(axis=0, keepdims=True)
reg_std = all_r[sel_train].std(axis=0, keepdims=True)
reg_std = np.where(reg_std < 1e-6, 1.0, reg_std)


def scale_r(r: np.ndarray) -> np.ndarray:
    return ((r - reg_mean) / reg_std).astype(np.float32)


X_train = scale_x(all_x[sel_train], feat_mean, feat_std)
X_val = scale_x(all_x[sel_val], feat_mean, feat_std)
X_test = scale_x(all_x[sel_test], feat_mean, feat_std)
y_train, y_val, y_test = all_y[sel_train], all_y[sel_val], all_y[sel_test]
r_train, r_val, r_test = all_r[sel_train], all_r[sel_val], all_r[sel_test]
R_train, R_val, R_test = scale_r(r_train), scale_r(r_val), scale_r(r_test)
print("train", X_train.shape, "val", X_val.shape, "test", X_test.shape)


NameError: name 'labeled' is not defined

In [ ]:
# CELL: MODEL
def make_family(name: str, seed: int) -> Dict[str, Any]:
    """Return unfitted classifier + two regressors for one family.

    No class re-weighting anywhere: the decision rule thresholds calibrated-ish
    probabilities, and ``balanced`` weights distort them.
    """
    est = int(CONFIG["n_estimators"])
    depth = int(CONFIG["max_depth"])
    lr = float(CONFIG["learning_rate"])
    leaf = int(CONFIG["min_leaf"])
    threads = int(CONFIG["threads_per_model"])
    if name == "LR":
        clf = LogisticRegression(max_iter=3000, C=0.1, solver="lbfgs")
        up = Ridge(alpha=10.0)
        dn = Ridge(alpha=10.0)
    elif name == "RF":
        rf_kwargs = dict(
            n_estimators=est,
            max_depth=depth + 2,
            min_samples_leaf=leaf,
            max_features="sqrt",
            n_jobs=threads,
        )
        clf = RandomForestClassifier(random_state=seed, **rf_kwargs)
        up = RandomForestRegressor(random_state=seed, **rf_kwargs)
        dn = RandomForestRegressor(random_state=seed + 1, **rf_kwargs)
    elif name == "XGB":
        xgb_kwargs = dict(
            n_estimators=est,
            max_depth=depth,
            learning_rate=lr,
            min_child_weight=leaf,
            subsample=0.8,
            colsample_bytree=0.7,
            reg_lambda=5.0,
            tree_method="hist",
            n_jobs=threads,
            verbosity=0,
        )
        clf = XGBClassifier(
            objective="multi:softprob", eval_metric="mlogloss", random_state=seed, **xgb_kwargs
        )
        up = XGBRegressor(random_state=seed, **xgb_kwargs)
        dn = XGBRegressor(random_state=seed + 1, **xgb_kwargs)
    elif name == "LGBM":
        lgb_kwargs = dict(
            n_estimators=est,
            num_leaves=31,
            max_depth=-1,
            learning_rate=lr,
            min_child_samples=leaf,
            subsample=0.8,
            subsample_freq=1,
            colsample_bytree=0.7,
            reg_lambda=5.0,
            n_jobs=threads,
            verbosity=-1,
        )
        clf = LGBMClassifier(random_state=seed, **lgb_kwargs)
        up = LGBMRegressor(random_state=seed, **lgb_kwargs)
        dn = LGBMRegressor(random_state=seed + 1, **lgb_kwargs)
    elif name == "CatBoost":
        cb_kwargs = dict(
            iterations=est,
            depth=depth,
            learning_rate=lr,
            l2_leaf_reg=10.0,
            verbose=False,
            thread_count=threads,
            allow_writing_files=False,
        )
        clf = CatBoostClassifier(loss_function="MultiClass", random_seed=seed, **cb_kwargs)
        up = CatBoostRegressor(random_seed=seed, **cb_kwargs)
        dn = CatBoostRegressor(random_seed=seed + 1, **cb_kwargs)
    else:
        raise ValueError(name)
    return {"name": name, "clf": clf, "up": up, "dn": dn}


print("families", CONFIG["ensemble"], "+", ENSEMBLE_NAME)


families ['LR', 'RF', 'XGB', 'LGBM', 'CatBoost'] + ENSEMBLE


In [ ]:
# CELL: TRAIN
def confusion(y_true: np.ndarray, y_pred: np.ndarray, n_classes: int = 3) -> np.ndarray:
    mat = np.zeros((n_classes, n_classes), dtype=np.int64)
    for t, p in zip(y_true.tolist(), y_pred.tolist()):
        if 0 <= int(t) < n_classes and 0 <= int(p) < n_classes:
            mat[int(t), int(p)] += 1
    return mat


def class_precision(mat: np.ndarray, cls: int) -> float:
    col = mat[:, cls]
    denom = float(col.sum())
    if denom <= 0:
        return float("nan")
    return float(mat[cls, cls]) / denom


def balanced_accuracy(mat: np.ndarray) -> float:
    recalls = []
    for i in range(mat.shape[0]):
        denom = float(mat[i].sum())
        recalls.append(float(mat[i, i]) / denom if denom else float("nan"))
    finite = [r for r in recalls if np.isfinite(r)]
    return float(np.mean(finite)) if finite else float("nan")


def corr(a: np.ndarray, b: np.ndarray) -> float:
    if a.size < 3:
        return float("nan")
    if float(np.std(a)) < 1e-12 or float(np.std(b)) < 1e-12:
        return float("nan")
    return float(np.corrcoef(a, b)[0, 1])


def as_cls(arr: np.ndarray) -> np.ndarray:
    return np.asarray(arr).reshape(-1).astype(np.int64)


def as_1d(arr: np.ndarray) -> np.ndarray:
    return np.asarray(arr, dtype=np.float64).reshape(-1)


def proba_of(clf: Any, X: np.ndarray) -> np.ndarray:
    """(n, 3) class probabilities in LONG/SHORT/NO_TRADE order regardless of estimator."""
    p = np.asarray(clf.predict_proba(X), dtype=np.float64)
    if p.ndim == 1:
        p = p.reshape(len(X), -1)
    classes = getattr(clf, "classes_", None)
    if classes is None:
        return p
    full = np.zeros((p.shape[0], len(LABEL_NAMES)), dtype=np.float64)
    for j, c in enumerate(np.asarray(classes).reshape(-1).astype(int)):
        full[:, c] = p[:, j]
    return full


def classifier_report(y_true: np.ndarray, y_pred: np.ndarray) -> Dict[str, Any]:
    cm = confusion(y_true, y_pred)
    return {
        "precision_LONG": class_precision(cm, LABEL_LONG),
        "precision_SHORT": class_precision(cm, LABEL_SHORT),
        "n_LONG": float(np.sum(y_pred == LABEL_LONG)),
        "n_SHORT": float(np.sum(y_pred == LABEL_SHORT)),
        "balanced_accuracy": balanced_accuracy(cm),
        "confusion": cm.tolist(),
    }


_PNL_COL = {LABEL_LONG: "long_pnl_pct", LABEL_SHORT: "short_pnl_pct"}
_SIDE_NAME = {LABEL_LONG: "LONG", LABEL_SHORT: "SHORT"}


def net_pnl(cls: int, rows: np.ndarray, cost: Optional[float] = None) -> np.ndarray:
    """Net-of-cost realised P&L (fraction of price) for taking ``cls`` on ``rows``."""
    c = float(CONFIG["round_trip_cost_pct"]) if cost is None else float(cost)
    return labeled[_PNL_COL[int(cls)]].to_numpy(dtype=np.float64)[rows] - c


def signal_expectancy(
    pred_cls: np.ndarray, rows: np.ndarray, cost: Optional[float] = None
) -> Dict[str, float]:
    """Per-side expectancy, t-stat, profit factor and total for the predicted signals."""
    out: Dict[str, float] = {}
    for cls, name in _SIDE_NAME.items():
        take = pred_cls == cls
        pnl = net_pnl(cls, rows[take], cost) if take.any() else np.zeros(0)
        pnl = pnl[np.isfinite(pnl)]
        n = int(pnl.size)
        out[f"{name}_n"] = float(n)
        out[f"{name}_expectancy"] = float(pnl.mean()) if n else float("nan")
        std = float(pnl.std(ddof=1)) if n > 1 else float("nan")
        out[f"{name}_tstat"] = (
            float(pnl.mean() / (std / math.sqrt(n))) if n > 1 and std > 0 else float("nan")
        )
        wins = float(pnl[pnl > 0].sum())
        losses = float(-pnl[pnl < 0].sum())
        out[f"{name}_profit_factor"] = (
            wins / losses if losses > 0 else (float("inf") if wins > 0 else float("nan"))
        )
        out[f"{name}_total_pct"] = float(pnl.sum())
    return out


def expectancy_rank_score(exp_report: Dict[str, float]) -> float:
    """Mean expectancy over sides with >= ``min_signals`` trades; -inf when none qualify."""
    min_n = int(CONFIG["min_signals"])
    parts: List[float] = []
    for name in ("LONG", "SHORT"):
        n = float(exp_report.get(f"{name}_n", 0.0))
        e = float(exp_report.get(f"{name}_expectancy", float("nan")))
        if n >= min_n and np.isfinite(e):
            parts.append(e)
    return float(np.mean(parts)) if parts else float("-inf")


def choose_thresholds(proba: np.ndarray, rows: np.ndarray) -> Dict[str, Optional[float]]:
    """Per side, the probability threshold that maximises calibration expectancy.

    Candidates are quantiles of the side's probability on the calibration rows.
    A side with no candidate reaching ``min_signals`` trades is disabled (None).
    """
    min_n = int(CONFIG["min_signals"])
    eligible = signal_eligible(rows)
    out: Dict[str, Optional[float]] = {}
    for cls, name in _SIDE_NAME.items():
        other = LABEL_SHORT if cls == LABEL_LONG else LABEL_LONG
        p_side = proba[:, cls]
        p_other = proba[:, other]
        pnl_all = net_pnl(cls, rows)
        best_thr: Optional[float] = None
        best_e = float("-inf")
        if not eligible.any():
            out[name] = None
            continue
        for q in CONFIG["signal_quantiles"]:
            thr = float(np.quantile(p_side[eligible], float(q)))
            take = eligible & (p_side >= thr) & (p_side > p_other) & np.isfinite(pnl_all)
            n = int(take.sum())
            if n < min_n:
                continue
            e = float(pnl_all[take].mean())
            if e > best_e:
                best_thr, best_e = thr, e
        out[name] = best_thr
    return out


def apply_thresholds(
    proba: np.ndarray, thr: Dict[str, Optional[float]], eligible: Optional[np.ndarray] = None
) -> np.ndarray:
    """Live decision rule.

    Abstain unless the candle is eligible (TP distance pays for cost), P(side)
    clears the side's threshold and beats the other side.
    """
    pred = np.full(proba.shape[0], LABEL_NO_TRADE, dtype=np.int64)
    ok = np.ones(proba.shape[0], dtype=bool) if eligible is None else np.asarray(eligible, dtype=bool)
    p_long, p_short = proba[:, LABEL_LONG], proba[:, LABEL_SHORT]
    if thr.get("LONG") is not None:
        pred[ok & (p_long >= float(thr["LONG"])) & (p_long > p_short)] = LABEL_LONG
    if thr.get("SHORT") is not None:
        pred[ok & (p_short >= float(thr["SHORT"])) & (p_short > p_long)] = LABEL_SHORT
    return pred


def score_probas(
    p_calib: np.ndarray,
    rows_calib: np.ndarray,
    p_eval: np.ndarray,
    rows_eval: np.ndarray,
    y_eval: np.ndarray,
) -> Dict[str, Any]:
    thr = choose_thresholds(p_calib, rows_calib)
    pred = apply_thresholds(p_eval, thr, signal_eligible(rows_eval))
    exp_rep = signal_expectancy(pred, rows_eval)
    cls_rep = classifier_report(y_eval, pred)
    return {
        "thresholds": thr,
        "cls": {k: v for k, v in cls_rep.items() if k != "confusion"},
        "confusion": cls_rep["confusion"],
        "exp": exp_rep,
        "score": expectancy_rank_score(exp_rep),
        "pred": pred,
    }


def fit_fold(
    name: str, fold: str, fit_sel: np.ndarray, calib_sel: np.ndarray, eval_sel: np.ndarray, *, seed: int
) -> Dict[str, Any]:
    """Fit one family on ``fit_sel`` rows, return probabilities on calibration and eval rows."""
    mean, std = fit_scaler(all_x[fit_sel])
    pack = make_family(name, seed)
    pack["clf"].fit(scale_x(all_x[fit_sel], mean, std), all_y[fit_sel])
    result = {
        "name": name,
        "fold": fold,
        "clf": pack["clf"],
        "scaler": (mean, std),
        "p_calib": proba_of(pack["clf"], scale_x(all_x[calib_sel], mean, std)),
        "p_eval": proba_of(pack["clf"], scale_x(all_x[eval_sel], mean, std)),
    }
    if fold == "final":
        pack["up"].fit(X_train, R_train[:, 0])
        pack["dn"].fit(X_train, R_train[:, 1])
        result["up"] = pack["up"]
        result["dn"] = pack["dn"]
    return result


# Fold plan: expanding walk-forward folds inside development, then the final split.
# Every fold has disjoint fit / calibration / evaluation rows; thresholds are chosen
# on calibration rows only, so evaluation expectancy is out-of-sample for both the
# model and the decision rule.
fold_plan: List[Tuple[str, np.ndarray, np.ndarray, np.ndarray]] = []
for fold_i, (tr_idx, va_idx) in enumerate(wf_folds, start=1):
    fit_idx, calib_idx = inner_calibration_split(
        tr_idx, calib_fraction=float(CONFIG["calib_fraction"]), purge_bars=int(CONFIG["purge_bars"])
    )
    fold_plan.append((f"wf{fold_i}", rows_in(fit_idx), rows_in(calib_idx), rows_in(va_idx)))
fold_plan.append(("final", sel_train, sel_val, sel_test))
for fold, f_sel, c_sel, e_sel in fold_plan:
    print(f"{fold}: fit={int(f_sel.sum())} calib={int(c_sel.sum())} eval={int(e_sel.sum())}")

FAMILIES: List[str] = list(CONFIG["ensemble"])
runs: Dict[str, Dict[str, Dict[str, Any]]] = {fold: {} for fold, *_ in fold_plan}
print("Fitting", FAMILIES, "x", [f for f, *_ in fold_plan], "in parallel...")
t_start = time.time()
with ThreadPoolExecutor(max_workers=len(FAMILIES)) as pool:
    futs = {}
    for fold_i, (fold, f_sel, c_sel, e_sel) in enumerate(fold_plan):
        for name in FAMILIES:
            fut = pool.submit(
                fit_fold, name, fold, f_sel, c_sel, e_sel, seed=int(CONFIG["seed"]) + fold_i
            )
            futs[fut] = (fold, name)
    for fut in as_completed(futs):
        fold, name = futs[fut]
        try:
            runs[fold][name] = fut.result()
            print(f"  done {name:9s} {fold}  ({time.time() - t_start:.0f}s)")
        except Exception as exc:  # keep the bake-off alive if one family fails
            print(f"  FAILED {name} {fold}: {exc!r}")

# Soft-vote ensemble over the members that succeeded in a fold.
for fold, f_sel, c_sel, e_sel in fold_plan:
    members = [runs[fold][n] for n in FAMILIES if n in runs[fold]]
    if len(members) >= 2:
        runs[fold][ENSEMBLE_NAME] = {
            "name": ENSEMBLE_NAME,
            "fold": fold,
            "members": [m["name"] for m in members],
            "p_calib": np.mean([m["p_calib"] for m in members], axis=0),
            "p_eval": np.mean([m["p_eval"] for m in members], axis=0),
        }

CANDIDATES: List[str] = [n for n in FAMILIES + [ENSEMBLE_NAME] if n in runs["final"]]
scores: Dict[str, Dict[str, Dict[str, Any]]] = {name: {} for name in CANDIDATES}
for fold, f_sel, c_sel, e_sel in fold_plan:
    rows_c, rows_e, y_e = all_rows[c_sel], all_rows[e_sel], all_y[e_sel]
    for name in CANDIDATES:
        if name not in runs[fold]:
            continue
        r = runs[fold][name]
        scores[name][fold] = score_probas(r["p_calib"], rows_c, r["p_eval"], rows_e, y_e)

WF_FOLDS: List[str] = [fold for fold, *_ in fold_plan if fold != "final"]


def walk_forward_summary(name: str) -> Dict[str, float]:
    vals = [scores[name][f]["score"] for f in WF_FOLDS if f in scores[name]]
    finite = [v for v in vals if np.isfinite(v)]
    return {
        "wf_folds_scored": float(len(finite)),
        "wf_mean_score": float(np.mean(finite)) if finite else float("-inf"),
        "wf_positive_share": float(np.mean([v > 0 for v in finite])) if finite else 0.0,
        "wf_min_score": float(np.min(finite)) if finite else float("-inf"),
    }


leaderboard: Dict[str, Dict[str, Any]] = {}
for name in CANDIDATES:
    wf = walk_forward_summary(name)
    fin = scores[name].get("final")
    row: Dict[str, Any] = {
        **wf,
        "val_thresholds": fin["thresholds"] if fin else None,
        "wf_folds": {
            f: {"score": scores[name][f]["score"], "exp": scores[name][f]["exp"], "thresholds": scores[name][f]["thresholds"]}
            for f in WF_FOLDS
            if f in scores[name]
        },
    }
    if name != ENSEMBLE_NAME and "up" in runs["final"].get(name, {}):
        r = runs["final"][name]
        up_c = corr(as_1d(r["up"].predict(X_val)), R_val[:, 0])
        dn_c = corr(as_1d(r["dn"].predict(X_val)), R_val[:, 1])
        row["upside_corr"] = up_c
        row["downside_corr"] = dn_c
        row["reg_score"] = float(np.nanmean([up_c, dn_c]))
    leaderboard[name] = row
    print(
        f"{name:9s} WF mean={wf['wf_mean_score']:+.6f} min={wf['wf_min_score']:+.6f} "
        f"pos_share={wf['wf_positive_share']:.2f} folds={wf['wf_folds_scored']:.0f} "
        f"val_thr={row['val_thresholds']} reg={row.get('reg_score', float('nan')):.3f}"
    )


def _selection_key(name: str) -> Tuple[float, float]:
    wf = leaderboard[name]
    return (wf["wf_mean_score"], wf["wf_min_score"])


cls_winner_name = max(CANDIDATES, key=_selection_key)
reg_candidates = [n for n in CANDIDATES if "reg_score" in leaderboard[n]]
reg_winner_name = max(reg_candidates, key=lambda n: float(leaderboard[n]["reg_score"]))
cls_winner = {"name": cls_winner_name, **leaderboard[cls_winner_name]}
reg_winner = {"name": reg_winner_name, "reg_score": leaderboard[reg_winner_name]["reg_score"]}
print("best classifier (walk-forward):", cls_winner_name, "| best regressor family (val):", reg_winner_name)


NameError: name 'wf_folds' is not defined

In [ ]:
# CELL: EVAL
def _unscale_reg(pred: np.ndarray, col: int) -> np.ndarray:
    return as_1d(pred) * float(reg_std[0, col]) + float(reg_mean[0, col])


rows_test = all_rows[sel_test]
final_scores = {name: scores[name]["final"] for name in CANDIDATES}
win = final_scores[cls_winner_name]
pred_cls = win["pred"]
cm = np.asarray(win["confusion"])
print("TEST winner clf=", cls_winner_name, "thresholds=", win["thresholds"], "reg=", reg_winner_name)
print(
    f"test candles eligible for a signal (TP distance >= {MIN_SIGNAL_TP_PCT:.4%}): "
    f"{float(signal_eligible(rows_test).mean()):.1%}"
)
print("test confusion rows=true cols=pred LONG/SHORT/NO_TRADE")
print(cm)
print("test signals", {k: v for k, v in win["exp"].items()})

# Cost sensitivity of the winner's test signals (same decisions, different cost).
cost_sensitivity = {}
for c in CONFIG["cost_sensitivity_pct"]:
    rep = signal_expectancy(pred_cls, rows_test, cost=float(c))
    cost_sensitivity[f"{float(c):.4f}"] = {
        "score": expectancy_rank_score(rep),
        "LONG_expectancy": rep["LONG_expectancy"],
        "SHORT_expectancy": rep["SHORT_expectancy"],
    }
    print(f"cost {float(c):.4%}: score={cost_sensitivity[f'{float(c):.4f}']['score']:+.6f}")

member_test: Dict[str, Dict[str, Any]] = {}
for name in CANDIDATES:
    s = final_scores[name]
    member_test[name] = {**s["cls"], "exp": s["exp"], "score": s["score"], "thresholds": s["thresholds"]}
    if name != ENSEMBLE_NAME and "up" in runs["final"][name]:
        r = runs["final"][name]
        member_test[name]["upside_corr"] = corr(_unscale_reg(r["up"].predict(X_test), 0), r_test[:, 0])
        member_test[name]["downside_corr"] = corr(_unscale_reg(r["dn"].predict(X_test), 1), r_test[:, 1])
    print(
        name,
        "test",
        {
            "score": round(s["score"], 6) if np.isfinite(s["score"]) else s["score"],
            "LONG": (int(s["exp"]["LONG_n"]), round(s["exp"]["LONG_expectancy"], 6) if np.isfinite(s["exp"]["LONG_expectancy"]) else None),
            "SHORT": (int(s["exp"]["SHORT_n"]), round(s["exp"]["SHORT_expectancy"], 6) if np.isfinite(s["exp"]["SHORT_expectancy"]) else None),
            "P_LONG": round(s["cls"]["precision_LONG"], 3) if np.isfinite(s["cls"]["precision_LONG"]) else None,
            "P_SHORT": round(s["cls"]["precision_SHORT"], 3) if np.isfinite(s["cls"]["precision_SHORT"]) else None,
        },
    )

best_up = runs["final"][reg_winner_name]["up"]
best_dn = runs["final"][reg_winner_name]["dn"]
up_hat = _unscale_reg(best_up.predict(X_test), 0)
dn_hat = _unscale_reg(best_dn.predict(X_test), 1)
naive_up_mae = float(np.mean(np.abs(r_test[:, 0] - float(reg_mean[0, 0]))))
naive_dn_mae = float(np.mean(np.abs(r_test[:, 1] - float(reg_mean[0, 1]))))
reg_eval = {
    "upside_corr": corr(up_hat, r_test[:, 0]),
    "downside_corr": corr(dn_hat, r_test[:, 1]),
    "upside_mae_skill": 1.0 - float(np.mean(np.abs(r_test[:, 0] - up_hat))) / naive_up_mae,
    "downside_mae_skill": 1.0 - float(np.mean(np.abs(r_test[:, 1] - dn_hat))) / naive_dn_mae,
}
print("regression test", reg_eval)

eval_report = {
    "n_test": int(len(y_test)),
    "best_classifier": cls_winner_name,
    "best_regressor_family": reg_winner_name,
    "decision_rule": {
        "type": "probability_threshold_abstain",
        "thresholds": win["thresholds"],
        "min_signals": int(CONFIG["min_signals"]),
        "min_signal_tp_pct": MIN_SIGNAL_TP_PCT,
        "test_eligible_share": float(signal_eligible(rows_test).mean()),
    },
    "val_leaderboard": leaderboard,
    "test_members": member_test,
    "confusion": cm.tolist(),
    "precision_LONG": win["cls"]["precision_LONG"],
    "precision_SHORT": win["cls"]["precision_SHORT"],
    "n_LONG": win["cls"]["n_LONG"],
    "n_SHORT": win["cls"]["n_SHORT"],
    "balanced_accuracy": win["cls"]["balanced_accuracy"],
    "signal_expectancy": win["exp"],
    "score": win["score"],
    "cost_sensitivity": cost_sensitivity,
    "regression": reg_eval,
    "note": (
        "Classifier chosen by mean walk-forward thresholded expectancy (both sides, "
        f"min {int(CONFIG['min_signals'])} signals per side). Thresholds for the test come from "
        "validation only. Test is scored once. Balanced accuracy is diagnostic only."
    ),
}
print(json.dumps({k: v for k, v in eval_report.items() if k not in ("val_leaderboard", "test_members")}, indent=2, default=float))


NameError: name 'all_rows' is not defined

In [ ]:
# CELL: PROMOTE
def promotion_gate(eval_rep: Dict[str, Any], lb_row: Dict[str, Any]) -> Dict[str, Any]:
    """Decide ``ready_to_promote`` from held-out evidence only.

    Requirements: positive thresholded expectancy on the single held-out test,
    every active side has >= ``min_signals`` trades and t-stat >= ``promote_min_tstat``,
    expectancy stays positive at the highest cost in ``cost_sensitivity_pct``, and the
    walk-forward positive share is >= ``promote_min_wf_positive_share``.
    """
    reasons: List[str] = []
    min_n = int(CONFIG["min_signals"])
    min_t = float(CONFIG["promote_min_tstat"])
    exp = eval_rep["signal_expectancy"]
    active_sides = [s for s in ("LONG", "SHORT") if float(exp.get(f"{s}_n", 0.0)) > 0]
    if not active_sides:
        reasons.append("no_signals_on_test")
    if not (np.isfinite(eval_rep["score"]) and eval_rep["score"] > 0.0):
        reasons.append("test_expectancy_not_positive")
    for side in active_sides:
        if float(exp[f"{side}_n"]) < min_n:
            reasons.append(f"{side.lower()}_too_few_test_signals")
        t = float(exp.get(f"{side}_tstat", float("nan")))
        if not (np.isfinite(t) and t >= min_t):
            reasons.append(f"{side.lower()}_tstat_below_{min_t:g}")
        if not (np.isfinite(exp[f"{side}_expectancy"]) and exp[f"{side}_expectancy"] > 0.0):
            reasons.append(f"{side.lower()}_expectancy_not_positive")
    worst_cost = max(float(c) for c in CONFIG["cost_sensitivity_pct"])
    worst = eval_rep["cost_sensitivity"][f"{worst_cost:.4f}"]["score"]
    if not (np.isfinite(worst) and worst > 0.0):
        reasons.append("fragile_to_higher_cost")
    if float(lb_row.get("wf_positive_share", 0.0)) < float(CONFIG["promote_min_wf_positive_share"]):
        reasons.append("walk_forward_not_consistently_positive")
    if float(lb_row.get("wf_folds_scored", 0.0)) < 2:
        reasons.append("walk_forward_insufficient_folds")
    return {
        "ready_to_promote": len(reasons) == 0,
        "reasons": reasons,
        "test_score": eval_rep["score"],
        "test_signals": {s: exp.get(f"{s}_n") for s in ("LONG", "SHORT")},
        "test_tstats": {s: exp.get(f"{s}_tstat") for s in ("LONG", "SHORT")},
        "wf_positive_share": lb_row.get("wf_positive_share"),
        "wf_mean_score": lb_row.get("wf_mean_score"),
        "worst_cost_score": worst,
    }


promotion = promotion_gate(eval_report, leaderboard[cls_winner_name])
ready_to_promote = bool(promotion["ready_to_promote"])
print(json.dumps(promotion, indent=2, default=float))
if ready_to_promote:
    print("VERDICT: held-out evidence supports paper-trading this bundle (still research-only here).")
else:
    print("VERDICT: NOT ready. Do not trade this bundle live. Reasons:", promotion["reasons"])


NameError: name 'eval_report' is not defined

In [ ]:
# CELL: EXPORT
EXPORT_DIR.mkdir(parents=True, exist_ok=True)
final_run = runs["final"][cls_winner_name]
if cls_winner_name == ENSEMBLE_NAME:
    clf_payload: Any = {
        "members": {m: runs["final"][m]["clf"] for m in final_run["members"]},
        "member_scalers": {m: runs["final"][m]["scaler"] for m in final_run["members"]},
    }
else:
    clf_payload = {"members": {cls_winner_name: final_run["clf"]}, "member_scalers": {cls_winner_name: final_run["scaler"]}}
bundle = {
    "best_classifier": cls_winner_name,
    "best_regressor_family": reg_winner_name,
    "clf": clf_payload,
    "thresholds": eval_report["decision_rule"]["thresholds"],
    "up": best_up,
    "dn": best_dn,
    "feat_mean": feat_mean,
    "feat_std": feat_std,
    "reg_mean": reg_mean,
    "reg_std": reg_std,
    "feature_cols": list(FEATURE_COLS),
    "label_names": list(LABEL_NAMES),
}
with (EXPORT_DIR / "best_models.pkl").open("wb") as handle:
    pickle.dump(bundle, handle)
(EXPORT_DIR / "leaderboard.json").write_text(
    json.dumps(leaderboard, indent=2, default=float), encoding="utf-8"
)
metadata = {
    "model_family": f"jacksparrow_opportunity_{CONFIG['resolution']}_tabular_ensemble",
    "ensemble": list(CONFIG["ensemble"]),
    "candidates": CANDIDATES,
    "best_classifier": cls_winner_name,
    "best_regressor_family": reg_winner_name,
    "timeframe": str(CONFIG["resolution"]),
    "resolution": str(CONFIG["resolution"]),
    "horizon_bars": int(CONFIG["horizon_bars"]),
    "horizon_minutes": int(CONFIG["horizon_minutes"]),
    "tp_atr_mult": float(CONFIG["tp_atr_mult"]),
    "sl_atr_mult": float(CONFIG["sl_atr_mult"]),
    "round_trip_cost_pct": float(CONFIG["round_trip_cost_pct"]),
    "feature_cols": list(FEATURE_COLS),
    "label_names": list(LABEL_NAMES),
    "decision_rule": {
        **eval_report["decision_rule"],
        "description": (
            f"At a closed {CONFIG['resolution']} candle: if tp_atr_mult * ATR(14) / close < min_signal_tp_pct emit "
            "NO_TRADE (quiet-market guard). Otherwise scale features with the member scaler(s), "
            "average member predict_proba when best_classifier == ENSEMBLE, then emit LONG when "
            "P(LONG) >= thresholds.LONG and P(LONG) > P(SHORT); SHORT symmetrically; else NO_TRADE. "
            "A null threshold disables that side."
        ),
    },
    "ready_to_promote": ready_to_promote,
    "promote_reason": (
        "held_out_gate_passed_paper_trade_first" if ready_to_promote else ";".join(promotion["reasons"])
    ),
    "promotion": promotion,
    "eval": eval_report,
    "label_gate": label_report,
    "walk_forward": {name: leaderboard[name]["wf_folds"] for name in CANDIDATES},
    "config": CONFIG,
}
(EXPORT_DIR / "metadata.json").write_text(
    json.dumps(metadata, indent=2, default=float), encoding="utf-8"
)
(EXPORT_DIR / "eval_report.json").write_text(
    json.dumps(eval_report, indent=2, default=float), encoding="utf-8"
)
print("wrote", EXPORT_DIR / "best_models.pkl")
print("best classifier", cls_winner_name, "best regressor", reg_winner_name)
print("ready_to_promote=", ready_to_promote)
try:
    from google.colab import files
    import shutil

    shutil.make_archive(str(EXPORT_DIR), "zip", root_dir=str(EXPORT_DIR))
    files.download(str(EXPORT_DIR) + ".zip")
except ImportError:
    print("Not Colab; files remain under", EXPORT_DIR)


NameError: name 'runs' is not defined

*File Operation*: `ls` on `content`

*File Operation*: `ls` on `/content/export/opportunity_10m`

**Session Created**: 2026-10-09 17:29:41
- Endpoint: `m-s-kkb-use1d2-1pfqji91qbmev`
- Hardware: `NONE`

In [ ]:
# CELL: ENV
# Colab ships pandas/numpy; install tabular model extras.
%pip install -q --upgrade-strategy only-if-needed pyarrow requests scikit-learn xgboost lightgbm catboost

import json
import math
import pickle
import random
import time
from concurrent.futures import ThreadPoolExecutor, as_completed
from datetime import datetime, timedelta, timezone
from pathlib import Path
from typing import Any, Dict, List, Optional, Sequence, Tuple

import numpy as np
import pandas as pd
from catboost import CatBoostClassifier, CatBoostRegressor
from lightgbm import LGBMClassifier, LGBMRegressor
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.linear_model import LogisticRegression, Ridge
from xgboost import XGBClassifier, XGBRegressor

print("sklearn / xgboost / lightgbm / catboost ready")


   ━━━━━━━━╸━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.2/97.1 MB 174.0 MB/s eta 0:00:01

   ━━━━━━━━━━━━━━━━━━━━━━━━━━╸━━━━━━━━━━━━━ 65.4/97.1 MB 129.3 MB/s eta 0:00:01

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╺━━━━ 85.7/97.1 MB 106.4 MB/s eta 0:00:01

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╸ 97.1/97.1 MB 133.7 MB/s eta 0:00:01

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╸ 97.1/97.1 MB 133.7 MB/s eta 0:00:01

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╸ 97.1/97.1 MB 133.7 MB/s eta 0:00:01

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╸ 97.1/97.1 MB 133.7 MB/s eta 0:00:01

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 8.4 MB/s eta 0:00:00


sklearn / xgboost / lightgbm / catboost ready


In [ ]:
# CELL: CONFIG
from pathlib import Path

RESOLUTION_MINUTES: Dict[str, int] = {
    "5m": 5,
    "10m": 10,
    "15m": 15,
    "30m": 30,
    "1h": 60,
}

CONFIG: Dict[str, Any] = {
    "symbol": "BTCUSD",
    "base_url": "https://api.india.delta.exchange",
    "history_days": 900,
    "resolution": "10m",
    "horizon_minutes": 240,
    "seed": 42,
    # Bracket. TP must clear the round-trip cost by a wide margin, otherwise no
    # classifier can be profitable (see the notebook header).
    "tp_atr_mult": 4.0,
    "sl_atr_mult": 2.0,
    "atr_period": 14,
    "round_trip_cost_pct": 0.0016,
    # Gate: median TP distance >= ratio x cost. Signal rule: a candle is only
    # eligible when its own TP distance >= ratio x cost (quiet-market guard).
    "min_tp_cost_ratio": 4.0,
    "minority_rate": 0.15,
    "ambiguous_win_max": 0.50,
    "train_fraction": 0.70,
    "val_fraction": 0.15,
    # Decision rule: trade only when P(side) clears a threshold chosen on
    # calibration rows. Candidates are probability quantiles of the calibration set.
    "calib_fraction": 0.15,
    "min_signals": 100,
    "signal_quantiles": [0.99, 0.98, 0.97, 0.95, 0.92, 0.90, 0.85, 0.80],
    "walk_forward_folds": 4,
    # Promotion gate thresholds (held-out test + walk-forward).
    "promote_min_tstat": 2.0,
    "promote_min_wf_positive_share": 0.67,
    "cost_sensitivity_pct": [0.0010, 0.0016, 0.0020],
    # Model capacity. Deliberately regularised: ~90k noisy rows, overlapping labels.
    "n_estimators": 400,
    "max_depth": 6,
    "learning_rate": 0.03,
    "min_leaf": 200,
    "threads_per_model": 2,
    "ensemble": ["LR", "RF", "XGB", "LGBM", "CatBoost"],
}


def apply_resolution_config(cfg: Dict[str, Any]) -> None:
    """Derive bar minutes, horizon_bars, purge, and export_dir from resolution."""
    res = str(cfg["resolution"])
    if res not in RESOLUTION_MINUTES:
        raise ValueError(
            f"Unsupported resolution {res!r}; use one of {sorted(RESOLUTION_MINUTES)}"
        )
    bar_min = int(RESOLUTION_MINUTES[res])
    horizon_min = int(cfg["horizon_minutes"])
    if bar_min <= 0 or horizon_min <= 0 or horizon_min % bar_min != 0:
        raise ValueError(
            f"horizon_minutes={horizon_min} must be a positive multiple of "
            f"{res} bar minutes ({bar_min})"
        )
    cfg["bar_minutes"] = bar_min
    cfg["horizon_bars"] = horizon_min // bar_min
    # Purge must cover the label look-ahead so no training label peeks into val.
    cfg["purge_bars"] = int(cfg["horizon_bars"]) + 1
    cfg["export_dir"] = f"export/opportunity_{res}"


_override = Path("opportunity_run_config.json")
if _override.is_file():
    CONFIG.update(json.loads(_override.read_text(encoding="utf-8")))
apply_resolution_config(CONFIG)

LABEL_NAMES: Tuple[str, ...] = ("LONG", "SHORT", "NO_TRADE")
LABEL_LONG, LABEL_SHORT, LABEL_NO_TRADE = 0, 1, 2
ENSEMBLE_NAME = "ENSEMBLE"
print(json.dumps(CONFIG, indent=2))


{
  "symbol": "BTCUSD",
  "base_url": "https://api.india.delta.exchange",
  "history_days": 900,
  "resolution": "10m",
  "horizon_minutes": 240,
  "seed": 42,
  "tp_atr_mult": 4.0,
  "sl_atr_mult": 2.0,
  "atr_period": 14,
  "round_trip_cost_pct": 0.0016,
  "min_tp_cost_ratio": 4.0,
  "minority_rate": 0.15,
  "ambiguous_win_max": 0.5,
  "train_fraction": 0.7,
  "val_fraction": 0.15,
  "calib_fraction": 0.15,
  "min_signals": 100,
  "signal_quantiles": [
    0.99,
    0.98,
    0.97,
    0.95,
    0.92,
    0.9,
    0.85,
    0.8
  ],
  "walk_forward_folds": 4,
  "promote_min_tstat": 2.0,
  "promote_min_wf_positive_share": 0.67,
  "cost_sensitivity_pct": [
    0.001,
    0.0016,
    0.002
  ],
  "n_estimators": 400,
  "max_depth": 6,
  "learning_rate": 0.03,
  "min_leaf": 200,
  "threads_per_model": 2,
  "ensemble": [
    "LR",
    "RF",
    "XGB",
    "LGBM",
    "CatBoost"
  ],
  "bar_minutes": 10,
  "horizon_bars": 24,
  "purge_bars": 25,
  "export_dir": "export/opportunity_10m"
}


In [ ]:
# CELL: SEED
def set_seed(seed: int) -> None:
    random.seed(int(seed))
    np.random.seed(int(seed))


set_seed(int(CONFIG["seed"]))
print("seed", CONFIG["seed"])


seed 42


In [ ]:
# CELL: FETCH
DELTA_CANDLE_PAGE_BARS = 2000
REQUEST_DELAY_SECONDS = 0.2
EMPTY_PAGE_RETRIES = 3
MIN_OHLCV_COMPLETENESS = 0.95
_RESOLUTION_MINUTES = {
    "1m": 1,
    "3m": 3,
    "5m": 5,
    "10m": 10,
    "15m": 15,
    "30m": 30,
    "1h": 60,
    "2h": 120,
    "4h": 240,
    "1d": 1440,
}


def _bar_seconds(resolution: str) -> int:
    if resolution not in _RESOLUTION_MINUTES:
        raise ValueError(f"Unsupported resolution {resolution!r}")
    return int(_RESOLUTION_MINUTES[resolution]) * 60


def _parse_candle_rows(payload: Any) -> List[Dict[str, Any]]:
    if not isinstance(payload, dict):
        return []
    result = payload.get("result", [])
    if isinstance(result, dict):
        result = result.get("candles", []) or []
    if not isinstance(result, list):
        return []
    return [row for row in result if isinstance(row, dict) and "time" in row]


def _candle_epoch(row: Dict[str, Any]) -> Optional[int]:
    try:
        return int(row["time"])
    except (KeyError, TypeError, ValueError):
        return None


def _request_candle_page(
    url: str,
    *,
    symbol: str,
    resolution: str,
    start_ts: int,
    end_ts: int,
) -> List[Dict[str, Any]]:
    import requests

    last_error: Optional[Exception] = None
    for attempt in range(EMPTY_PAGE_RETRIES):
        try:
            response = requests.get(
                url,
                params={
                    "symbol": symbol,
                    "resolution": resolution,
                    "start": int(start_ts),
                    "end": int(end_ts),
                },
                timeout=30,
            )
            response.raise_for_status()
            payload = response.json()
        except Exception as exc:
            last_error = exc
            time.sleep(REQUEST_DELAY_SECONDS * (attempt + 1))
            continue
        if payload.get("success") is False:
            last_error = RuntimeError(f"Delta candles error for {symbol}: {payload}")
            time.sleep(REQUEST_DELAY_SECONDS * (attempt + 1))
            continue
        rows = _parse_candle_rows(payload)
        if rows:
            return rows
        last_error = None
        time.sleep(REQUEST_DELAY_SECONDS * (attempt + 1))
    if last_error is not None:
        raise last_error
    return []


def _rows_to_frame(all_rows: List[Dict[str, Any]]) -> pd.DataFrame:
    df = pd.DataFrame(all_rows)
    expected_cols = ["time", "open", "high", "low", "close", "volume"]
    df = df[[c for c in expected_cols if c in df.columns]]
    df["time"] = pd.to_datetime(df["time"], unit="s", utc=True)
    df = df.drop_duplicates(subset="time").sort_values("time").reset_index(drop=True)
    for col in ["open", "high", "low", "close", "volume"]:
        if col in df.columns:
            df[col] = df[col].astype(float)
    return df


def fetch_candles(
    symbol: str,
    resolution: str,
    start_ts: int,
    end_ts: int,
    base_url: str,
    *,
    page_bars: int = DELTA_CANDLE_PAGE_BARS,
) -> pd.DataFrame:
    url = f"{base_url.rstrip('/')}/v2/history/candles"
    bar_seconds = _bar_seconds(resolution)
    page_bars = max(1, int(page_bars))
    start_ts, end_ts = int(start_ts), int(end_ts)
    if end_ts <= start_ts:
        raise ValueError("end_ts must be greater than start_ts")
    all_rows: List[Dict[str, Any]] = []
    cursor_end = end_ts
    seen_oldest: Optional[int] = None
    max_pages = max(2, (end_ts - start_ts) // bar_seconds + 5)
    for _ in range(max_pages):
        if cursor_end <= start_ts:
            break
        page_start = max(start_ts, cursor_end - page_bars * bar_seconds)
        if page_start >= cursor_end:
            break
        rows = _request_candle_page(
            url,
            symbol=symbol,
            resolution=resolution,
            start_ts=page_start,
            end_ts=cursor_end,
        )
        if not rows:
            break
        epochs = [e for e in (_candle_epoch(r) for r in rows) if e is not None]
        if not epochs:
            break
        oldest = min(epochs)
        if seen_oldest is not None and oldest >= seen_oldest:
            break
        seen_oldest = oldest
        all_rows.extend(rows)
        cursor_end = oldest - 1
        time.sleep(REQUEST_DELAY_SECONDS)
    if not all_rows:
        raise RuntimeError(f"No candle data returned for symbol={symbol}")
    return _rows_to_frame(all_rows)


def validate_ohlcv_completeness(
    df: pd.DataFrame,
    resolution: str,
    *,
    min_completeness: float = MIN_OHLCV_COMPLETENESS,
    symbol: str = "",
) -> Dict[str, Any]:
    if df is None or df.empty or "time" not in df.columns:
        raise ValueError("Cannot validate completeness on an empty OHLCV frame")
    bar_seconds = _bar_seconds(resolution)
    time_col = df["time"]
    epochs = (pd.to_datetime(time_col, utc=True).astype("int64") // 10**9).tolist()
    if len(epochs) < 2:
        raise ValueError("Need at least 2 OHLCV bars to validate completeness")
    gaps = 0
    for prev, cur in zip(epochs, epochs[1:]):
        if int(cur) - int(prev) > bar_seconds * 1.5:
            gaps += 1
    completeness = 1.0 - (gaps / max(1, len(epochs) - 1))
    report = {
        "rows": len(epochs),
        "gaps": int(gaps),
        "completeness": round(float(completeness), 4),
    }
    if completeness < min_completeness:
        label = f"{symbol} {resolution}".strip()
        raise ValueError(
            f"OHLCV completeness {completeness:.1%} below {min_completeness:.0%} "
            f"for {label or 'candles'}"
        )
    return report


end_dt = datetime.now(timezone.utc)
start_dt = end_dt - timedelta(days=int(CONFIG["history_days"]))
_resolution = str(CONFIG["resolution"])
# Delta has no native 10m candles; everything else is fetched at its own resolution.
_source_resolution = "5m" if _resolution == "10m" else _resolution
print(
    f"Fetching {CONFIG['symbol']} {_source_resolution} OHLCV for {CONFIG['history_days']}d "
    f"from {CONFIG['base_url']} (target {_resolution})..."
)
raw_bars = fetch_candles(
    str(CONFIG["symbol"]),
    _source_resolution,
    int(start_dt.timestamp()),
    int(end_dt.timestamp()),
    str(CONFIG["base_url"]),
)
ohlcv_report = validate_ohlcv_completeness(
    raw_bars, _source_resolution, symbol=str(CONFIG["symbol"])
)
print(
    f"OHLCV {CONFIG['symbol']} {_source_resolution}: {ohlcv_report['rows']} bars, "
    f"completeness {ohlcv_report['completeness']:.1%}, gaps={ohlcv_report['gaps']}"
)


Fetching BTCUSD 5m OHLCV for 900d from https://api.india.delta.exchange (target 10m)...


OHLCV BTCUSD 5m: 259200 bars, completeness 100.0%, gaps=0


In [ ]:
# CELL: BARS
def build_10m_ohlcv_from_5m(df5m: pd.DataFrame) -> pd.DataFrame:
    """Exchange-aligned 10m bars from two closed 5m candles. Drop a partial last bucket."""
    src = df5m.copy()
    src["time"] = pd.to_datetime(src["time"], utc=True)
    required = ("open", "high", "low", "close", "volume")
    missing = [c for c in required if c not in src.columns]
    if missing:
        raise ValueError(f"5m OHLCV missing columns: {missing}")
    if src.empty:
        return src.iloc[0:0].copy()
    grouped = src.set_index("time").sort_index()
    agg = grouped.resample("10min", label="left", closed="left").agg(
        {
            "open": "first",
            "high": "max",
            "low": "min",
            "close": "last",
            "volume": "sum",
        }
    )
    counts = grouped["close"].resample("10min", label="left", closed="left").count()
    complete = counts >= 2
    agg = agg.loc[complete].dropna(subset=["open", "high", "low", "close"])
    out = agg.reset_index()
    return out.sort_values("time").reset_index(drop=True)


if str(CONFIG["resolution"]) == "10m":
    bars = build_10m_ohlcv_from_5m(raw_bars)
else:
    bars = raw_bars
if bars is None or getattr(bars, "empty", True):
    raise ValueError(f"No OHLCV bars for {CONFIG['resolution']}")
print(
    f"{CONFIG['resolution']} bars={len(bars)} "
    f"span={bars['time'].iloc[0]} → {bars['time'].iloc[-1]} "
    f"horizon_bars={CONFIG['horizon_bars']} ({CONFIG['horizon_minutes']}m)"
)


10m bars=129599 span=2024-04-22 17:40:00+00:00 → 2026-10-09 17:20:00+00:00 horizon_bars=24 (240m)


In [ ]:
# CELL: FEATURES
OHLCV_COLS: Tuple[str, ...] = ("time", "open", "high", "low", "close", "volume")


def _wilder_rsi(close: pd.Series, period: int = 14) -> pd.Series:
    delta = close.diff()
    gain = delta.clip(lower=0.0)
    loss = -delta.clip(upper=0.0)
    avg_gain = gain.ewm(alpha=1.0 / period, min_periods=period, adjust=False).mean()
    avg_loss = loss.ewm(alpha=1.0 / period, min_periods=period, adjust=False).mean()
    rs = avg_gain / avg_loss.replace(0.0, np.nan)
    return 100.0 - (100.0 / (1.0 + rs))


def add_opportunity_features(df: pd.DataFrame, *, atr_period: int = 14) -> pd.DataFrame:
    """Causal signal-candle features from the selected timeframe series only.

    Everything is expressed in ATR units or as a ratio so the model is scale
    invariant across the 900-day price range. ATR(14) is also stored for the
    label path. Nothing here looks past the signal candle.
    """
    out = df.copy()
    close = out["close"].astype(np.float64)
    high = out["high"].astype(np.float64)
    low = out["low"].astype(np.float64)
    open_ = out["open"].astype(np.float64)
    volume = out["volume"].astype(np.float64)
    prev_close = close.shift(1)
    tr = pd.concat(
        [high - low, (high - prev_close).abs(), (low - prev_close).abs()], axis=1
    ).max(axis=1)
    atr = tr.rolling(int(atr_period), min_periods=int(atr_period)).mean()
    atr_safe = atr.replace(0.0, np.nan)
    out["atr"] = atr

    feats: Dict[str, pd.Series] = {}
    # Momentum at several lookbacks, in ATR units.
    feats["ret_1"] = close.pct_change()
    for k in (3, 6, 12, 24, 48, 144):
        feats[f"ret_{k}_atr"] = (close - close.shift(k)) / atr_safe
    # Volatility level and regime.
    feats["atr_pct"] = atr / close.replace(0.0, np.nan)
    atr_long = tr.rolling(int(atr_period) * 6, min_periods=int(atr_period) * 6).mean()
    feats["atr_ratio_84"] = atr / atr_long.replace(0.0, np.nan)
    log_ret = np.log(close).diff()
    rv_12 = log_ret.rolling(12, min_periods=12).std()
    rv_48 = log_ret.rolling(48, min_periods=48).std()
    rv_144 = log_ret.rolling(144, min_periods=144).std()
    feats["rv_12_48"] = rv_12 / rv_48.replace(0.0, np.nan)
    feats["rv_48_144"] = rv_48 / rv_144.replace(0.0, np.nan)
    # Oscillators and trend.
    for period in (7, 14, 28):
        feats[f"rsi_{period}"] = _wilder_rsi(close, period=period)
    for span in (20, 50, 200):
        ema = close.ewm(span=span, adjust=False, min_periods=span).mean()
        feats[f"ema_dist_{span}_atr"] = (close - ema) / atr_safe
        feats[f"ema_slope_{span}_atr"] = (ema - ema.shift(6)) / atr_safe
    # Candle anatomy.
    feats["candle_body"] = (close - open_) / atr_safe
    feats["upper_wick"] = (high - np.maximum(close, open_)) / atr_safe
    feats["lower_wick"] = (np.minimum(close, open_) - low) / atr_safe
    feats["range_atr"] = (high - low) / atr_safe
    # Volume anomalies.
    for window in (20, 100):
        vol_mean = volume.rolling(window, min_periods=window).mean()
        vol_std = volume.rolling(window, min_periods=window).std()
        feats[f"volume_z_{window}"] = (volume - vol_mean) / vol_std.replace(0.0, np.nan)
    # Position inside recent ranges and distance to swing extremes.
    for window in (24, 72, 144):
        hh = high.rolling(window, min_periods=window).max()
        ll = low.rolling(window, min_periods=window).min()
        feats[f"range_pos_{window}"] = (close - ll) / (hh - ll).replace(0.0, np.nan)
        feats[f"dist_high_{window}_atr"] = (hh - close) / atr_safe
        feats[f"dist_low_{window}_atr"] = (close - ll) / atr_safe
    # Consecutive same-direction closes (signed).
    sign = np.sign(close.diff()).fillna(0.0)
    groups = (sign != sign.shift()).cumsum()
    feats["streak"] = sign * (sign.groupby(groups).cumcount() + 1)
    # Intraday / weekly seasonality.
    ts = pd.to_datetime(out["time"], utc=True)
    hour = ts.dt.hour + ts.dt.minute / 60.0
    feats["hour_sin"] = np.sin(2.0 * np.pi * hour / 24.0)
    feats["hour_cos"] = np.cos(2.0 * np.pi * hour / 24.0)
    feats["dow_sin"] = np.sin(2.0 * np.pi * ts.dt.dayofweek / 7.0)
    feats["dow_cos"] = np.cos(2.0 * np.pi * ts.dt.dayofweek / 7.0)
    # Short raw history of the last six candle returns (ATR units).
    for lag in range(1, 7):
        feats[f"ret_lag{lag}_atr"] = (close.shift(lag - 1) - close.shift(lag)) / atr_safe
    return pd.concat([out, pd.DataFrame(feats, index=out.index)], axis=1)


featured = add_opportunity_features(bars, atr_period=int(CONFIG["atr_period"]))
FEATURE_COLS: Tuple[str, ...] = tuple(
    c for c in featured.columns if c not in OHLCV_COLS and c != "atr"
)
print(len(FEATURE_COLS), "features")
null_rates = featured[list(FEATURE_COLS)].isna().mean()
print("max feature null rate", round(float(null_rates.max()), 4), "at", null_rates.idxmax())


46 features
max feature null rate 0.0016 at ema_slope_200_atr


In [ ]:
# CELL: LABELS
# Opportunity labels on a closed-bar series. Path is bars t+1 .. t+horizon only.
# Vectorised over bars; semantics are identical to walking each path bar by bar.

import numpy as np
import pandas as pd

_EPS = 1e-9
HORIZON_BARS_DEFAULT = 24
ATR_PERIOD_DEFAULT = 14
TP_ATR_MULT_DEFAULT = 4.0
SL_ATR_MULT_DEFAULT = 2.0

TP_FIRST = 0
SL_FIRST = 1
NEITHER = 2
AMBIGUOUS = 3

LABEL_LONG = 0
LABEL_SHORT = 1
LABEL_NO_TRADE = 2
LABEL_UNLABELED = -1
LABEL_NAMES = ("LONG", "SHORT", "NO_TRADE")


def tp_sl_from_path(
    fwd_high: np.ndarray,
    fwd_low: np.ndarray,
    entry: float,
    atr: float,
    tp_mult: float,
    sl_mult: float,
    side: str,
) -> int:
    """Walk future highs/lows. Same-bar TP and SL is AMBIGUOUS, not a win."""
    high = np.asarray(fwd_high, dtype=np.float64).ravel()
    low = np.asarray(fwd_low, dtype=np.float64).ravel()
    if high.size == 0 or high.size != low.size:
        return int(NEITHER)
    denom = max(float(atr), _EPS)
    entry_f = float(entry)
    if str(side).strip().upper() == "SHORT":
        tp_level = entry_f - float(tp_mult) * denom
        sl_level = entry_f + float(sl_mult) * denom
        tp_hit = low <= tp_level
        sl_hit = high >= sl_level
    else:
        tp_level = entry_f + float(tp_mult) * denom
        sl_level = entry_f - float(sl_mult) * denom
        tp_hit = high >= tp_level
        sl_hit = low <= sl_level
    for j in range(high.size):
        hit_tp = bool(tp_hit[j])
        hit_sl = bool(sl_hit[j])
        if hit_tp and hit_sl:
            return int(AMBIGUOUS)
        if hit_tp:
            return int(TP_FIRST)
        if hit_sl:
            return int(SL_FIRST)
    return int(NEITHER)


def _forward_stack(values: np.ndarray, k: int) -> np.ndarray:
    """(k, n) array where row j-1 holds values shifted back by j bars (NaN padded)."""
    n = values.shape[0]
    out = np.full((k, n), np.nan, dtype=np.float64)
    for j in range(1, k + 1):
        if n - j > 0:
            out[j - 1, : n - j] = values[j:]
    return out


def _side_codes_vectorised(
    fwd_high: np.ndarray,
    fwd_low: np.ndarray,
    entry: np.ndarray,
    atr: np.ndarray,
    tp_mult: float,
    sl_mult: float,
    side: str,
) -> np.ndarray:
    """Vectorised twin of ``tp_sl_from_path`` for every bar at once."""
    denom = np.maximum(atr, _EPS)
    if str(side).strip().upper() == "SHORT":
        tp_level = entry - float(tp_mult) * denom
        sl_level = entry + float(sl_mult) * denom
        tp_hits = fwd_low <= tp_level[None, :]
        sl_hits = fwd_high >= sl_level[None, :]
    else:
        tp_level = entry + float(tp_mult) * denom
        sl_level = entry - float(sl_mult) * denom
        tp_hits = fwd_high >= tp_level[None, :]
        sl_hits = fwd_low <= sl_level[None, :]
    any_hit = tp_hits | sl_hits
    has_hit = any_hit.any(axis=0)
    first = np.argmax(any_hit, axis=0)
    cols = np.arange(entry.shape[0])
    tp_first = tp_hits[first, cols]
    sl_first = sl_hits[first, cols]
    codes = np.where(
        ~has_hit,
        NEITHER,
        np.where(tp_first & sl_first, AMBIGUOUS, np.where(tp_first, TP_FIRST, SL_FIRST)),
    )
    return codes.astype(np.int64)


def _causal_atr(df: pd.DataFrame, period: int = ATR_PERIOD_DEFAULT) -> np.ndarray:
    if "atr" in df.columns:
        atr = df["atr"].to_numpy(dtype=np.float64)
        if np.isfinite(atr).any():
            return atr
    close = df["close"].to_numpy(dtype=np.float64)
    high = df["high"].to_numpy(dtype=np.float64)
    low = df["low"].to_numpy(dtype=np.float64)
    prev = np.concatenate([[close[0]], close[:-1]])
    tr = np.maximum(high - low, np.maximum(np.abs(high - prev), np.abs(low - prev)))
    return pd.Series(tr).rolling(int(period), min_periods=int(period)).mean().to_numpy(
        dtype=np.float64
    )


def compute_opportunity_labels(
    df: pd.DataFrame,
    *,
    tp_mult: float = TP_ATR_MULT_DEFAULT,
    sl_mult: float = SL_ATR_MULT_DEFAULT,
    horizon_bars: int = HORIZON_BARS_DEFAULT,
    atr_period: int = ATR_PERIOD_DEFAULT,
) -> pd.DataFrame:
    """Label each closed bar. Last ``horizon_bars`` rows stay unlabeled.

    Realised P&L columns (``long_pnl_pct`` / ``short_pnl_pct``) are gross of
    cost: +TP distance on TP_FIRST, −SL distance on SL_FIRST **and** AMBIGUOUS
    (a same-bar double touch is scored as a stop), timeout return on NEITHER.
    """
    out = df.copy().reset_index(drop=True)
    n = len(out)
    k = int(horizon_bars)
    close = out["close"].to_numpy(dtype=np.float64)
    high = out["high"].to_numpy(dtype=np.float64)
    low = out["low"].to_numpy(dtype=np.float64)
    atr = np.asarray(_causal_atr(out, period=int(atr_period)), dtype=np.float64)
    if atr.shape[0] != n:
        atr = np.resize(atr, n)

    fwd_high = _forward_stack(high, k)
    fwd_low = _forward_stack(low, k)
    fwd_close_end = np.full(n, np.nan, dtype=np.float64)
    if n - k > 0:
        fwd_close_end[: n - k] = close[k:]

    index = np.arange(n)
    valid = (
        (index < n - k)
        & np.isfinite(atr)
        & (atr > _EPS)
        & np.isfinite(close)
        & (np.abs(close) > _EPS)
    )
    atr_safe = np.where(valid, atr, np.nan)
    entry_safe = np.where(valid, close, np.nan)

    l_code = _side_codes_vectorised(fwd_high, fwd_low, entry_safe, atr_safe, tp_mult, sl_mult, "LONG")
    s_code = _side_codes_vectorised(fwd_high, fwd_low, entry_safe, atr_safe, tp_mult, sl_mult, "SHORT")
    l_win = l_code == TP_FIRST
    s_win = s_code == TP_FIRST
    label = np.where(l_win & ~s_win, LABEL_LONG, np.where(s_win & ~l_win, LABEL_SHORT, LABEL_NO_TRADE))

    with np.errstate(invalid="ignore", divide="ignore"):
        max_high = np.nanmax(np.where(np.isfinite(fwd_high), fwd_high, -np.inf), axis=0)
        min_low = np.nanmin(np.where(np.isfinite(fwd_low), fwd_low, np.inf), axis=0)
        upside = np.maximum((max_high - entry_safe) / atr_safe, 0.0)
        downside = np.maximum((entry_safe - min_low) / atr_safe, 0.0)
        timeout_ret = (fwd_close_end - entry_safe) / entry_safe
        tp_pct = float(tp_mult) * atr_safe / entry_safe
        sl_pct = float(sl_mult) * atr_safe / entry_safe
        long_pnl = np.where(
            l_code == TP_FIRST, tp_pct, np.where(l_code == NEITHER, timeout_ret, -sl_pct)
        )
        short_pnl = np.where(
            s_code == TP_FIRST, tp_pct, np.where(s_code == NEITHER, -timeout_ret, -sl_pct)
        )

    def _masked(values: np.ndarray) -> np.ndarray:
        return np.where(valid, values, np.nan)

    out["atr"] = atr
    out["trade_label"] = np.where(valid, label, LABEL_UNLABELED).astype(np.int64)
    out["long_result"] = _masked(l_win.astype(np.float64))
    out["short_result"] = _masked(s_win.astype(np.float64))
    out["long_code"] = _masked(l_code.astype(np.float64))
    out["short_code"] = _masked(s_code.astype(np.float64))
    out["upside"] = _masked(upside)
    out["downside"] = _masked(downside)
    out["timeout_ret"] = _masked(timeout_ret)
    out["tp_pct"] = _masked(tp_pct)
    out["sl_pct"] = _masked(sl_pct)
    out["long_pnl_pct"] = _masked(long_pnl)
    out["short_pnl_pct"] = _masked(short_pnl)
    names = np.array(["UNLABELED"] * n, dtype=object)
    names[out["trade_label"].to_numpy() == LABEL_LONG] = "LONG"
    names[out["trade_label"].to_numpy() == LABEL_SHORT] = "SHORT"
    names[out["trade_label"].to_numpy() == LABEL_NO_TRADE] = "NO_TRADE"
    out["trade_name"] = names
    return out


In [ ]:
# CELL: APPLY_LABELS
labeled = compute_opportunity_labels(
    featured,
    tp_mult=float(CONFIG["tp_atr_mult"]),
    sl_mult=float(CONFIG["sl_atr_mult"]),
    horizon_bars=int(CONFIG["horizon_bars"]),
    atr_period=int(CONFIG["atr_period"]),
)
valid = labeled["trade_label"].to_numpy() >= 0
print(f"labeled rows={int(valid.sum())} / {len(labeled)}")
print(labeled.loc[valid, "trade_name"].value_counts(normalize=True).round(4))
print(
    "median TP distance",
    f"{labeled.loc[valid, 'tp_pct'].median():.4%}",
    "median SL distance",
    f"{labeled.loc[valid, 'sl_pct'].median():.4%}",
    "round trip cost",
    f"{float(CONFIG['round_trip_cost_pct']):.4%}",
)


labeled rows=129562 / 129599
trade_name
NO_TRADE    0.6132
SHORT       0.1979
LONG        0.1889
Name: proportion, dtype: float64
median TP distance 0.8590% median SL distance 0.4295% round trip cost 0.1600%


In [ ]:
# CELL: GATE
def train_period_mask(n: int, train_fraction: float) -> np.ndarray:
    cut = int(n * float(train_fraction))
    mask = np.zeros(n, dtype=bool)
    mask[:cut] = True
    return mask


def _path_code_counts(codes: np.ndarray, usable: np.ndarray) -> Dict[str, int]:
    finite = usable & np.isfinite(codes)
    return {
        "TP_FIRST": int(np.sum(finite & (codes == TP_FIRST))),
        "SL_FIRST": int(np.sum(finite & (codes == SL_FIRST))),
        "NEITHER": int(np.sum(finite & (codes == NEITHER))),
        "AMBIGUOUS": int(np.sum(finite & (codes == AMBIGUOUS))),
    }


def summarize_label_gate(
    df: pd.DataFrame,
    *,
    train_fraction: float,
    round_trip_cost_pct: float,
    minority_rate: float,
    ambiguous_win_max: float,
    min_tp_cost_ratio: float = 4.0,
) -> Dict[str, Any]:
    """Decide whether the bracket is worth training on, using train rows only.

    Besides class mix and oracle exclusive expectancy, the gate checks the
    bracket economics: the median take-profit distance must be at least
    ``min_tp_cost_ratio`` times the round-trip cost, and the implied
    break-even win rate ``(sl + cost) / (tp + sl)`` is reported.
    """
    n = len(df)
    in_train = train_period_mask(n, train_fraction)
    labels = df["trade_label"].to_numpy(dtype=np.int64)
    usable = in_train & (labels >= 0)
    n_use = int(usable.sum())
    if n_use == 0:
        raise RuntimeError("Train-period gate has no labeled rows")
    mix = {}
    mix_ok = True
    for cid, name in enumerate(LABEL_NAMES):
        rate = float(np.mean(labels[usable] == cid))
        mix[name] = rate
        if rate < float(minority_rate):
            mix_ok = False
    long_code = df["long_code"].to_numpy(dtype=np.float64)
    short_code = df["short_code"].to_numpy(dtype=np.float64)
    cost = float(round_trip_cost_pct)
    long_pnl = df["long_pnl_pct"].to_numpy(dtype=np.float64) - cost
    short_pnl = df["short_pnl_pct"].to_numpy(dtype=np.float64) - cost

    always_long_mask = usable & np.isfinite(long_pnl)
    always_short_mask = usable & np.isfinite(short_pnl)
    always_in_long = (
        float(np.mean(long_pnl[always_long_mask])) if always_long_mask.any() else float("nan")
    )
    always_in_short = (
        float(np.mean(short_pnl[always_short_mask])) if always_short_mask.any() else float("nan")
    )

    excl_long_mask = usable & (labels == LABEL_LONG) & np.isfinite(long_pnl)
    excl_short_mask = usable & (labels == LABEL_SHORT) & np.isfinite(short_pnl)
    exclusive_long = (
        float(np.mean(long_pnl[excl_long_mask])) if excl_long_mask.any() else float("nan")
    )
    exclusive_short = (
        float(np.mean(short_pnl[excl_short_mask])) if excl_short_mask.any() else float("nan")
    )

    def _ambig_share(codes: np.ndarray) -> float:
        finite = usable & np.isfinite(codes)
        tp = finite & (codes == TP_FIRST)
        amb = finite & (codes == AMBIGUOUS)
        denom = int(tp.sum() + amb.sum())
        if denom == 0:
            return 0.0
        return float(amb.sum()) / float(denom)

    long_ambig = _ambig_share(long_code)
    short_ambig = _ambig_share(short_code)

    if "tp_pct" in df.columns and "sl_pct" in df.columns:
        tp_pct = df["tp_pct"].to_numpy(dtype=np.float64)
        sl_pct = df["sl_pct"].to_numpy(dtype=np.float64)
    else:
        atr_pct = df["atr"].to_numpy(dtype=np.float64) / df["close"].to_numpy(dtype=np.float64)
        tp_pct = float(CONFIG["tp_atr_mult"]) * atr_pct
        sl_pct = float(CONFIG["sl_atr_mult"]) * atr_pct
    econ_mask = usable & np.isfinite(tp_pct) & np.isfinite(sl_pct)
    median_tp = float(np.median(tp_pct[econ_mask])) if econ_mask.any() else float("nan")
    median_sl = float(np.median(sl_pct[econ_mask])) if econ_mask.any() else float("nan")
    tp_cost_ratio = median_tp / cost if cost > 0 and np.isfinite(median_tp) else float("inf")
    breakeven_win_rate = (
        (median_sl + cost) / (median_tp + median_sl)
        if np.isfinite(median_tp) and np.isfinite(median_sl) and (median_tp + median_sl) > 0
        else float("nan")
    )
    long_tp_rate = float(np.mean(long_code[usable & np.isfinite(long_code)] == TP_FIRST))
    short_tp_rate = float(np.mean(short_code[usable & np.isfinite(short_code)] == TP_FIRST))

    reasons: List[str] = []
    if not mix_ok:
        reasons.append("class_mix_below_floor")
    if not (np.isfinite(exclusive_long) and exclusive_long > 0.0):
        reasons.append("exclusive_long_expectancy_not_positive")
    if not (np.isfinite(exclusive_short) and exclusive_short > 0.0):
        reasons.append("exclusive_short_expectancy_not_positive")
    if long_ambig >= float(ambiguous_win_max):
        reasons.append("long_ambiguous_majority")
    if short_ambig >= float(ambiguous_win_max):
        reasons.append("short_ambiguous_majority")
    if tp_cost_ratio < float(min_tp_cost_ratio):
        reasons.append("tp_distance_below_cost_ratio")
    passed = len(reasons) == 0
    return {
        "passed": passed,
        "n_train_labeled": n_use,
        "class_mix": mix,
        "minority_rate": float(minority_rate),
        "exclusive_long_expectancy_pct": exclusive_long,
        "exclusive_short_expectancy_pct": exclusive_short,
        "always_in_long_expectancy_pct": always_in_long,
        "always_in_short_expectancy_pct": always_in_short,
        "round_trip_cost_pct": cost,
        "median_tp_pct": median_tp,
        "median_sl_pct": median_sl,
        "tp_cost_ratio": tp_cost_ratio,
        "min_tp_cost_ratio": float(min_tp_cost_ratio),
        "breakeven_win_rate": breakeven_win_rate,
        "base_long_tp_rate": long_tp_rate,
        "base_short_tp_rate": short_tp_rate,
        "long_ambiguous_share": long_ambig,
        "short_ambiguous_share": short_ambig,
        "long_path_codes": _path_code_counts(long_code, usable),
        "short_path_codes": _path_code_counts(short_code, usable),
        "tp_atr_mult": float(CONFIG["tp_atr_mult"]),
        "sl_atr_mult": float(CONFIG["sl_atr_mult"]),
        "reasons": reasons,
    }


EXPORT_DIR = Path(str(CONFIG["export_dir"]))
EXPORT_DIR.mkdir(parents=True, exist_ok=True)
label_report = summarize_label_gate(
    labeled,
    train_fraction=float(CONFIG["train_fraction"]),
    round_trip_cost_pct=float(CONFIG["round_trip_cost_pct"]),
    minority_rate=float(CONFIG["minority_rate"]),
    ambiguous_win_max=float(CONFIG["ambiguous_win_max"]),
    min_tp_cost_ratio=float(CONFIG["min_tp_cost_ratio"]),
)
report_path = EXPORT_DIR / "label_report.json"
report_path.write_text(json.dumps(label_report, indent=2), encoding="utf-8")
print(json.dumps(label_report, indent=2))
print("wrote", report_path)
if not label_report["passed"]:
    raise RuntimeError(
        "Train-period label gate failed: "
        f"{label_report['reasons']}. Change CONFIG tp/sl ATR multiples or horizon and "
        "re-run from the top. Do not fit models."
    )
print(
    f"Gate passed. Break-even win rate {label_report['breakeven_win_rate']:.1%} vs base "
    f"TP rates long {label_report['base_long_tp_rate']:.1%} / short "
    f"{label_report['base_short_tp_rate']:.1%}. Training may start."
)


{
  "passed": true,
  "n_train_labeled": 90706,
  "class_mix": {
    "LONG": 0.18858730403721916,
    "SHORT": 0.19669040636782573,
    "NO_TRADE": 0.6147222895949551
  },
  "minority_rate": 0.15,
  "exclusive_long_expectancy_pct": 0.00711605929243632,
  "exclusive_short_expectancy_pct": 0.007575630454604985,
  "always_in_long_expectancy_pct": -0.0016005508375630513,
  "always_in_short_expectancy_pct": -0.0016322682072115523,
  "round_trip_cost_pct": 0.0016,
  "median_tp_pct": 0.008724346406829142,
  "median_sl_pct": 0.004362173203414571,
  "tp_cost_ratio": 5.4527165042682135,
  "min_tp_cost_ratio": 4.0,
  "breakeven_win_rate": 0.4555965513357402,
  "base_long_tp_rate": 0.18858730403721916,
  "base_short_tp_rate": 0.19669040636782573,
  "long_ambiguous_share": 0.002274715660542432,
  "short_ambiguous_share": 0.0011197581322434354,
  "long_path_codes": {
    "TP_FIRST": 17106,
    "SL_FIRST": 45122,
    "NEITHER": 28439,
    "AMBIGUOUS": 39
  },
  "short_path_codes": {
    "TP_FIRST": 1

In [ ]:
# CELL: SPLIT
def purged_index_splits(
    n: int,
    *,
    train_fraction: float,
    val_fraction: float,
    purge_bars: int,
) -> Dict[str, np.ndarray]:
    """Time-ordered split. Purge ``purge_bars`` on each side of every cut."""
    n = int(n)
    purge = int(purge_bars)
    train_end = int(n * float(train_fraction))
    val_len = int(n * float(val_fraction))
    train_idx = np.arange(0, max(train_end - purge, 0), dtype=np.int64)
    val_start = min(train_end + purge, n)
    val_end = min(val_start + val_len, n)
    val_idx = np.arange(val_start, max(val_end - purge, val_start), dtype=np.int64)
    test_start = min(val_end + purge, n)
    test_idx = np.arange(test_start, n, dtype=np.int64)
    return {"train": train_idx, "val": val_idx, "test": test_idx}


def development_walk_forward(
    n_dev: int,
    *,
    folds: int = 4,
    purge_bars: int,
) -> List[Tuple[np.ndarray, np.ndarray]]:
    """Expanding-window folds inside the development prefix only."""
    folds = max(2, int(folds))
    purge = int(purge_bars)
    edges = [int(n_dev * (i + 1) / folds) for i in range(folds)]
    out: List[Tuple[np.ndarray, np.ndarray]] = []
    for i in range(1, folds):
        train_end = edges[i - 1]
        val_end = edges[i]
        tr = np.arange(0, max(train_end - purge, 0), dtype=np.int64)
        va = np.arange(min(train_end + purge, n_dev), val_end, dtype=np.int64)
        if len(tr) == 0 or len(va) == 0:
            continue
        out.append((tr, va))
    return out


def inner_calibration_split(
    train_idx: np.ndarray, *, calib_fraction: float, purge_bars: int
) -> Tuple[np.ndarray, np.ndarray]:
    """Reserve the purged tail of a training window for threshold selection."""
    n = len(train_idx)
    cut = int(n * (1.0 - float(calib_fraction)))
    fit_idx = train_idx[: max(cut - int(purge_bars), 0)]
    calib_idx = train_idx[cut:]
    return fit_idx, calib_idx


splits = purged_index_splits(
    len(labeled),
    train_fraction=float(CONFIG["train_fraction"]),
    val_fraction=float(CONFIG["val_fraction"]),
    purge_bars=int(CONFIG["purge_bars"]),
)
for key, idx in splits.items():
    print(key, "n", len(idx), "range", int(idx[0]) if len(idx) else None, int(idx[-1]) if len(idx) else None)
dev_end = int(splits["test"][0]) if len(splits["test"]) else len(labeled)
wf_folds = development_walk_forward(
    dev_end, folds=int(CONFIG["walk_forward_folds"]), purge_bars=int(CONFIG["purge_bars"])
)
print("walk-forward folds", len(wf_folds), "dev_end", dev_end)


train n 90694 range 0 90693
val n 19414 range 90744 110157
test n 19391 range 110208 129598
walk-forward folds 3 dev_end 110208


In [ ]:
# CELL: DATASET
def build_rows(
    frame: pd.DataFrame, *, feature_cols: Sequence[str]
) -> Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    """One tabular row per labeled bar with finite features and finite path targets."""
    feats = frame.loc[:, list(feature_cols)].to_numpy(dtype=np.float64)
    y_cls = frame["trade_label"].to_numpy(dtype=np.int64)
    y_reg = frame.loc[:, ["upside", "downside"]].to_numpy(dtype=np.float64)
    ok = (y_cls >= 0) & np.isfinite(feats).all(axis=1) & np.isfinite(y_reg).all(axis=1)
    rows = np.flatnonzero(ok).astype(np.int64)
    return (
        feats[rows].astype(np.float32),
        y_cls[rows],
        y_reg[rows].astype(np.float32),
        rows,
    )


all_x, all_y, all_r, all_rows = build_rows(labeled, feature_cols=FEATURE_COLS)
print(
    "rows",
    all_x.shape,
    "class mix",
    {n: round(float(np.mean(all_y == i)), 4) for i, n in enumerate(LABEL_NAMES)},
)


def rows_in(split_idx: np.ndarray) -> np.ndarray:
    """Boolean selector over ``all_rows`` for a set of frame indices."""
    allowed = np.zeros(len(labeled), dtype=bool)
    allowed[np.asarray(split_idx, dtype=np.int64)] = True
    return allowed[all_rows]


MIN_SIGNAL_TP_PCT = float(CONFIG["min_tp_cost_ratio"]) * float(CONFIG["round_trip_cost_pct"])


def signal_eligible(rows: np.ndarray) -> np.ndarray:
    """Quiet-market guard: a candle may carry a signal only if its TP distance pays for cost."""
    tp_pct = labeled["tp_pct"].to_numpy(dtype=np.float64)[rows]
    return np.isfinite(tp_pct) & (tp_pct >= MIN_SIGNAL_TP_PCT)


print(
    f"signal eligibility: TP distance >= {MIN_SIGNAL_TP_PCT:.4%} -> "
    f"{float(signal_eligible(all_rows).mean()):.1%} of labeled candles"
)


def fit_scaler(x: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:
    mean = x.mean(axis=0, keepdims=True)
    std = x.std(axis=0, keepdims=True)
    std = np.where(std < 1e-6, 1.0, std)
    return mean.astype(np.float32), std.astype(np.float32)


def scale_x(x: np.ndarray, mean: np.ndarray, std: np.ndarray) -> np.ndarray:
    return ((x - mean) / std).astype(np.float32)


sel_train = rows_in(splits["train"])
sel_val = rows_in(splits["val"])
sel_test = rows_in(splits["test"])
feat_mean, feat_std = fit_scaler(all_x[sel_train])
reg_mean = all_r[sel_train].mean(axis=0, keepdims=True)
reg_std = all_r[sel_train].std(axis=0, keepdims=True)
reg_std = np.where(reg_std < 1e-6, 1.0, reg_std)


def scale_r(r: np.ndarray) -> np.ndarray:
    return ((r - reg_mean) / reg_std).astype(np.float32)


X_train = scale_x(all_x[sel_train], feat_mean, feat_std)
X_val = scale_x(all_x[sel_val], feat_mean, feat_std)
X_test = scale_x(all_x[sel_test], feat_mean, feat_std)
y_train, y_val, y_test = all_y[sel_train], all_y[sel_val], all_y[sel_test]
r_train, r_val, r_test = all_r[sel_train], all_r[sel_val], all_r[sel_test]
R_train, R_val, R_test = scale_r(r_train), scale_r(r_val), scale_r(r_test)
print("train", X_train.shape, "val", X_val.shape, "test", X_test.shape)


rows (129370, 46) class mix {'LONG': 0.1889, 'SHORT': 0.1979, 'NO_TRADE': 0.6132}
signal eligibility: TP distance >= 0.6400% -> 71.0% of labeled candles
train (90489, 46) val (19414, 46) test (19367, 46)


In [ ]:
# CELL: MODEL
def make_family(name: str, seed: int) -> Dict[str, Any]:
    """Return unfitted classifier + two regressors for one family.

    No class re-weighting anywhere: the decision rule thresholds calibrated-ish
    probabilities, and ``balanced`` weights distort them.
    """
    est = int(CONFIG["n_estimators"])
    depth = int(CONFIG["max_depth"])
    lr = float(CONFIG["learning_rate"])
    leaf = int(CONFIG["min_leaf"])
    threads = int(CONFIG["threads_per_model"])
    if name == "LR":
        clf = LogisticRegression(max_iter=3000, C=0.1, solver="lbfgs")
        up = Ridge(alpha=10.0)
        dn = Ridge(alpha=10.0)
    elif name == "RF":
        rf_kwargs = dict(
            n_estimators=est,
            max_depth=depth + 2,
            min_samples_leaf=leaf,
            max_features="sqrt",
            n_jobs=threads,
        )
        clf = RandomForestClassifier(random_state=seed, **rf_kwargs)
        up = RandomForestRegressor(random_state=seed, **rf_kwargs)
        dn = RandomForestRegressor(random_state=seed + 1, **rf_kwargs)
    elif name == "XGB":
        xgb_kwargs = dict(
            n_estimators=est,
            max_depth=depth,
            learning_rate=lr,
            min_child_weight=leaf,
            subsample=0.8,
            colsample_bytree=0.7,
            reg_lambda=5.0,
            tree_method="hist",
            n_jobs=threads,
            verbosity=0,
        )
        clf = XGBClassifier(
            objective="multi:softprob", eval_metric="mlogloss", random_state=seed, **xgb_kwargs
        )
        up = XGBRegressor(random_state=seed, **xgb_kwargs)
        dn = XGBRegressor(random_state=seed + 1, **xgb_kwargs)
    elif name == "LGBM":
        lgb_kwargs = dict(
            n_estimators=est,
            num_leaves=31,
            max_depth=-1,
            learning_rate=lr,
            min_child_samples=leaf,
            subsample=0.8,
            subsample_freq=1,
            colsample_bytree=0.7,
            reg_lambda=5.0,
            n_jobs=threads,
            verbosity=-1,
        )
        clf = LGBMClassifier(random_state=seed, **lgb_kwargs)
        up = LGBMRegressor(random_state=seed, **lgb_kwargs)
        dn = LGBMRegressor(random_state=seed + 1, **lgb_kwargs)
    elif name == "CatBoost":
        cb_kwargs = dict(
            iterations=est,
            depth=depth,
            learning_rate=lr,
            l2_leaf_reg=10.0,
            verbose=False,
            thread_count=threads,
            allow_writing_files=False,
        )
        clf = CatBoostClassifier(loss_function="MultiClass", random_seed=seed, **cb_kwargs)
        up = CatBoostRegressor(random_seed=seed, **cb_kwargs)
        dn = CatBoostRegressor(random_seed=seed + 1, **cb_kwargs)
    else:
        raise ValueError(name)
    return {"name": name, "clf": clf, "up": up, "dn": dn}


print("families", CONFIG["ensemble"], "+", ENSEMBLE_NAME)


families ['LR', 'RF', 'XGB', 'LGBM', 'CatBoost'] + ENSEMBLE


In [ ]:
# CELL: TRAIN
def confusion(y_true: np.ndarray, y_pred: np.ndarray, n_classes: int = 3) -> np.ndarray:
    mat = np.zeros((n_classes, n_classes), dtype=np.int64)
    for t, p in zip(y_true.tolist(), y_pred.tolist()):
        if 0 <= int(t) < n_classes and 0 <= int(p) < n_classes:
            mat[int(t), int(p)] += 1
    return mat


def class_precision(mat: np.ndarray, cls: int) -> float:
    col = mat[:, cls]
    denom = float(col.sum())
    if denom <= 0:
        return float("nan")
    return float(mat[cls, cls]) / denom


def balanced_accuracy(mat: np.ndarray) -> float:
    recalls = []
    for i in range(mat.shape[0]):
        denom = float(mat[i].sum())
        recalls.append(float(mat[i, i]) / denom if denom else float("nan"))
    finite = [r for r in recalls if np.isfinite(r)]
    return float(np.mean(finite)) if finite else float("nan")


def corr(a: np.ndarray, b: np.ndarray) -> float:
    if a.size < 3:
        return float("nan")
    if float(np.std(a)) < 1e-12 or float(np.std(b)) < 1e-12:
        return float("nan")
    return float(np.corrcoef(a, b)[0, 1])


def as_cls(arr: np.ndarray) -> np.ndarray:
    return np.asarray(arr).reshape(-1).astype(np.int64)


def as_1d(arr: np.ndarray) -> np.ndarray:
    return np.asarray(arr, dtype=np.float64).reshape(-1)


def proba_of(clf: Any, X: np.ndarray) -> np.ndarray:
    """(n, 3) class probabilities in LONG/SHORT/NO_TRADE order regardless of estimator."""
    p = np.asarray(clf.predict_proba(X), dtype=np.float64)
    if p.ndim == 1:
        p = p.reshape(len(X), -1)
    classes = getattr(clf, "classes_", None)
    if classes is None:
        return p
    full = np.zeros((p.shape[0], len(LABEL_NAMES)), dtype=np.float64)
    for j, c in enumerate(np.asarray(classes).reshape(-1).astype(int)):
        full[:, c] = p[:, j]
    return full


def classifier_report(y_true: np.ndarray, y_pred: np.ndarray) -> Dict[str, Any]:
    cm = confusion(y_true, y_pred)
    return {
        "precision_LONG": class_precision(cm, LABEL_LONG),
        "precision_SHORT": class_precision(cm, LABEL_SHORT),
        "n_LONG": float(np.sum(y_pred == LABEL_LONG)),
        "n_SHORT": float(np.sum(y_pred == LABEL_SHORT)),
        "balanced_accuracy": balanced_accuracy(cm),
        "confusion": cm.tolist(),
    }


_PNL_COL = {LABEL_LONG: "long_pnl_pct", LABEL_SHORT: "short_pnl_pct"}
_SIDE_NAME = {LABEL_LONG: "LONG", LABEL_SHORT: "SHORT"}


def net_pnl(cls: int, rows: np.ndarray, cost: Optional[float] = None) -> np.ndarray:
    """Net-of-cost realised P&L (fraction of price) for taking ``cls`` on ``rows``."""
    c = float(CONFIG["round_trip_cost_pct"]) if cost is None else float(cost)
    return labeled[_PNL_COL[int(cls)]].to_numpy(dtype=np.float64)[rows] - c


def signal_expectancy(
    pred_cls: np.ndarray, rows: np.ndarray, cost: Optional[float] = None
) -> Dict[str, float]:
    """Per-side expectancy, t-stat, profit factor and total for the predicted signals."""
    out: Dict[str, float] = {}
    for cls, name in _SIDE_NAME.items():
        take = pred_cls == cls
        pnl = net_pnl(cls, rows[take], cost) if take.any() else np.zeros(0)
        pnl = pnl[np.isfinite(pnl)]
        n = int(pnl.size)
        out[f"{name}_n"] = float(n)
        out[f"{name}_expectancy"] = float(pnl.mean()) if n else float("nan")
        std = float(pnl.std(ddof=1)) if n > 1 else float("nan")
        out[f"{name}_tstat"] = (
            float(pnl.mean() / (std / math.sqrt(n))) if n > 1 and std > 0 else float("nan")
        )
        wins = float(pnl[pnl > 0].sum())
        losses = float(-pnl[pnl < 0].sum())
        out[f"{name}_profit_factor"] = (
            wins / losses if losses > 0 else (float("inf") if wins > 0 else float("nan"))
        )
        out[f"{name}_total_pct"] = float(pnl.sum())
    return out


def expectancy_rank_score(exp_report: Dict[str, float]) -> float:
    """Mean expectancy over sides with >= ``min_signals`` trades; -inf when none qualify."""
    min_n = int(CONFIG["min_signals"])
    parts: List[float] = []
    for name in ("LONG", "SHORT"):
        n = float(exp_report.get(f"{name}_n", 0.0))
        e = float(exp_report.get(f"{name}_expectancy", float("nan")))
        if n >= min_n and np.isfinite(e):
            parts.append(e)
    return float(np.mean(parts)) if parts else float("-inf")


def choose_thresholds(proba: np.ndarray, rows: np.ndarray) -> Dict[str, Optional[float]]:
    """Per side, the probability threshold that maximises calibration expectancy.

    Candidates are quantiles of the side's probability on the calibration rows.
    A side with no candidate reaching ``min_signals`` trades is disabled (None).
    """
    min_n = int(CONFIG["min_signals"])
    eligible = signal_eligible(rows)
    out: Dict[str, Optional[float]] = {}
    for cls, name in _SIDE_NAME.items():
        other = LABEL_SHORT if cls == LABEL_LONG else LABEL_LONG
        p_side = proba[:, cls]
        p_other = proba[:, other]
        pnl_all = net_pnl(cls, rows)
        best_thr: Optional[float] = None
        best_e = float("-inf")
        if not eligible.any():
            out[name] = None
            continue
        for q in CONFIG["signal_quantiles"]:
            thr = float(np.quantile(p_side[eligible], float(q)))
            take = eligible & (p_side >= thr) & (p_side > p_other) & np.isfinite(pnl_all)
            n = int(take.sum())
            if n < min_n:
                continue
            e = float(pnl_all[take].mean())
            if e > best_e:
                best_thr, best_e = thr, e
        out[name] = best_thr
    return out


def apply_thresholds(
    proba: np.ndarray, thr: Dict[str, Optional[float]], eligible: Optional[np.ndarray] = None
) -> np.ndarray:
    """Live decision rule.

    Abstain unless the candle is eligible (TP distance pays for cost), P(side)
    clears the side's threshold and beats the other side.
    """
    pred = np.full(proba.shape[0], LABEL_NO_TRADE, dtype=np.int64)
    ok = np.ones(proba.shape[0], dtype=bool) if eligible is None else np.asarray(eligible, dtype=bool)
    p_long, p_short = proba[:, LABEL_LONG], proba[:, LABEL_SHORT]
    if thr.get("LONG") is not None:
        pred[ok & (p_long >= float(thr["LONG"])) & (p_long > p_short)] = LABEL_LONG
    if thr.get("SHORT") is not None:
        pred[ok & (p_short >= float(thr["SHORT"])) & (p_short > p_long)] = LABEL_SHORT
    return pred


def score_probas(
    p_calib: np.ndarray,
    rows_calib: np.ndarray,
    p_eval: np.ndarray,
    rows_eval: np.ndarray,
    y_eval: np.ndarray,
) -> Dict[str, Any]:
    thr = choose_thresholds(p_calib, rows_calib)
    pred = apply_thresholds(p_eval, thr, signal_eligible(rows_eval))
    exp_rep = signal_expectancy(pred, rows_eval)
    cls_rep = classifier_report(y_eval, pred)
    return {
        "thresholds": thr,
        "cls": {k: v for k, v in cls_rep.items() if k != "confusion"},
        "confusion": cls_rep["confusion"],
        "exp": exp_rep,
        "score": expectancy_rank_score(exp_rep),
        "pred": pred,
    }


def fit_fold(
    name: str, fold: str, fit_sel: np.ndarray, calib_sel: np.ndarray, eval_sel: np.ndarray, *, seed: int
) -> Dict[str, Any]:
    """Fit one family on ``fit_sel`` rows, return probabilities on calibration and eval rows."""
    mean, std = fit_scaler(all_x[fit_sel])
    pack = make_family(name, seed)
    pack["clf"].fit(scale_x(all_x[fit_sel], mean, std), all_y[fit_sel])
    result = {
        "name": name,
        "fold": fold,
        "clf": pack["clf"],
        "scaler": (mean, std),
        "p_calib": proba_of(pack["clf"], scale_x(all_x[calib_sel], mean, std)),
        "p_eval": proba_of(pack["clf"], scale_x(all_x[eval_sel], mean, std)),
    }
    if fold == "final":
        pack["up"].fit(X_train, R_train[:, 0])
        pack["dn"].fit(X_train, R_train[:, 1])
        result["up"] = pack["up"]
        result["dn"] = pack["dn"]
    return result


# Fold plan: expanding walk-forward folds inside development, then the final split.
# Every fold has disjoint fit / calibration / evaluation rows; thresholds are chosen
# on calibration rows only, so evaluation expectancy is out-of-sample for both the
# model and the decision rule.
fold_plan: List[Tuple[str, np.ndarray, np.ndarray, np.ndarray]] = []
for fold_i, (tr_idx, va_idx) in enumerate(wf_folds, start=1):
    fit_idx, calib_idx = inner_calibration_split(
        tr_idx, calib_fraction=float(CONFIG["calib_fraction"]), purge_bars=int(CONFIG["purge_bars"])
    )
    fold_plan.append((f"wf{fold_i}", rows_in(fit_idx), rows_in(calib_idx), rows_in(va_idx)))
fold_plan.append(("final", sel_train, sel_val, sel_test))
for fold, f_sel, c_sel, e_sel in fold_plan:
    print(f"{fold}: fit={int(f_sel.sum())} calib={int(c_sel.sum())} eval={int(e_sel.sum())}")

FAMILIES: List[str] = list(CONFIG["ensemble"])
runs: Dict[str, Dict[str, Dict[str, Any]]] = {fold: {} for fold, *_ in fold_plan}
print("Fitting", FAMILIES, "x", [f for f, *_ in fold_plan], "in parallel...")
t_start = time.time()
with ThreadPoolExecutor(max_workers=len(FAMILIES)) as pool:
    futs = {}
    for fold_i, (fold, f_sel, c_sel, e_sel) in enumerate(fold_plan):
        for name in FAMILIES:
            fut = pool.submit(
                fit_fold, name, fold, f_sel, c_sel, e_sel, seed=int(CONFIG["seed"]) + fold_i
            )
            futs[fut] = (fold, name)
    for fut in as_completed(futs):
        fold, name = futs[fut]
        try:
            runs[fold][name] = fut.result()
            print(f"  done {name:9s} {fold}  ({time.time() - t_start:.0f}s)")
        except Exception as exc:  # keep the bake-off alive if one family fails
            print(f"  FAILED {name} {fold}: {exc!r}")

# Soft-vote ensemble over the members that succeeded in a fold.
for fold, f_sel, c_sel, e_sel in fold_plan:
    members = [runs[fold][n] for n in FAMILIES if n in runs[fold]]
    if len(members) >= 2:
        runs[fold][ENSEMBLE_NAME] = {
            "name": ENSEMBLE_NAME,
            "fold": fold,
            "members": [m["name"] for m in members],
            "p_calib": np.mean([m["p_calib"] for m in members], axis=0),
            "p_eval": np.mean([m["p_eval"] for m in members], axis=0),
        }

CANDIDATES: List[str] = [n for n in FAMILIES + [ENSEMBLE_NAME] if n in runs["final"]]
scores: Dict[str, Dict[str, Dict[str, Any]]] = {name: {} for name in CANDIDATES}
for fold, f_sel, c_sel, e_sel in fold_plan:
    rows_c, rows_e, y_e = all_rows[c_sel], all_rows[e_sel], all_y[e_sel]
    for name in CANDIDATES:
        if name not in runs[fold]:
            continue
        r = runs[fold][name]
        scores[name][fold] = score_probas(r["p_calib"], rows_c, r["p_eval"], rows_e, y_e)

WF_FOLDS: List[str] = [fold for fold, *_ in fold_plan if fold != "final"]


def walk_forward_summary(name: str) -> Dict[str, float]:
    vals = [scores[name][f]["score"] for f in WF_FOLDS if f in scores[name]]
    finite = [v for v in vals if np.isfinite(v)]
    return {
        "wf_folds_scored": float(len(finite)),
        "wf_mean_score": float(np.mean(finite)) if finite else float("-inf"),
        "wf_positive_share": float(np.mean([v > 0 for v in finite])) if finite else 0.0,
        "wf_min_score": float(np.min(finite)) if finite else float("-inf"),
    }


leaderboard: Dict[str, Dict[str, Any]] = {}
for name in CANDIDATES:
    wf = walk_forward_summary(name)
    fin = scores[name].get("final")
    row: Dict[str, Any] = {
        **wf,
        "val_thresholds": fin["thresholds"] if fin else None,
        "wf_folds": {
            f: {"score": scores[name][f]["score"], "exp": scores[name][f]["exp"], "thresholds": scores[name][f]["thresholds"]}
            for f in WF_FOLDS
            if f in scores[name]
        },
    }
    if name != ENSEMBLE_NAME and "up" in runs["final"].get(name, {}):
        r = runs["final"][name]
        up_c = corr(as_1d(r["up"].predict(X_val)), R_val[:, 0])
        dn_c = corr(as_1d(r["dn"].predict(X_val)), R_val[:, 1])
        row["upside_corr"] = up_c
        row["downside_corr"] = dn_c
        row["reg_score"] = float(np.nanmean([up_c, dn_c]))
    leaderboard[name] = row
    print(
        f"{name:9s} WF mean={wf['wf_mean_score']:+.6f} min={wf['wf_min_score']:+.6f} "
        f"pos_share={wf['wf_positive_share']:.2f} folds={wf['wf_folds_scored']:.0f} "
        f"val_thr={row['val_thresholds']} reg={row.get('reg_score', float('nan')):.3f}"
    )


def _selection_key(name: str) -> Tuple[float, float]:
    wf = leaderboard[name]
    return (wf["wf_mean_score"], wf["wf_min_score"])


cls_winner_name = max(CANDIDATES, key=_selection_key)
reg_candidates = [n for n in CANDIDATES if "reg_score" in leaderboard[n]]
reg_winner_name = max(reg_candidates, key=lambda n: float(leaderboard[n]["reg_score"]))
cls_winner = {"name": cls_winner_name, **leaderboard[cls_winner_name]}
reg_winner = {"name": reg_winner_name, "reg_score": leaderboard[reg_winner_name]["reg_score"]}
print("best classifier (walk-forward):", cls_winner_name, "| best regressor family (val):", reg_winner_name)


wf1: fit=23167 calib=4130 eval=27527
wf2: fit=46587 calib=8262 eval=27527
wf3: fit=70006 calib=12395 eval=27527
final: fit=90489 calib=19414 eval=19367
Fitting ['LR', 'RF', 'XGB', 'LGBM', 'CatBoost'] x ['wf1', 'wf2', 'wf3', 'final'] in parallel...


  done LR        wf1  (17s)


  done LR        wf2  (50s)


  done CatBoost  wf1  (132s)


  done RF        wf1  (155s)


...

RF        WF mean=-0.001699 min=-0.002215 pos_share=0.00 folds=3 val_thr={'LONG': 0.24739546981498692, 'SHORT': 0.29627762169189475} reg=0.264


XGB       WF mean=-0.001460 min=-0.001786 pos_share=0.00 folds=3 val_thr={'LONG': 0.38516576647758494, 'SHORT': 0.4131194517016412} reg=0.292


/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


LGBM      WF mean=-0.001171 min=-0.001257 pos_share=0.00 folds=3 val_thr={'LONG': 0.29829407363837745, 'SHORT': 0.44344961094272717} reg=0.316
CatBoost  WF mean=-0.001621 min=-0.001815 pos_share=0.00 folds=3 val_thr={'LONG': 0.3373549774793285, 'SHORT': 0.33358963559986105} reg=0.216
ENSEMBLE  WF mean=-0.001488 min=-0.001590 pos_share=0.00 folds=3 val_thr={'LONG': 0.28468028444223015, 'SHORT': 0.35231765725614456} reg=nan
best classifier (walk-forward): LGBM | best regressor family (val): LGBM


In [ ]:
# CELL: EVAL
def _unscale_reg(pred: np.ndarray, col: int) -> np.ndarray:
    return as_1d(pred) * float(reg_std[0, col]) + float(reg_mean[0, col])


rows_test = all_rows[sel_test]
final_scores = {name: scores[name]["final"] for name in CANDIDATES}
win = final_scores[cls_winner_name]
pred_cls = win["pred"]
cm = np.asarray(win["confusion"])
print("TEST winner clf=", cls_winner_name, "thresholds=", win["thresholds"], "reg=", reg_winner_name)
print(
    f"test candles eligible for a signal (TP distance >= {MIN_SIGNAL_TP_PCT:.4%}): "
    f"{float(signal_eligible(rows_test).mean()):.1%}"
)
print("test confusion rows=true cols=pred LONG/SHORT/NO_TRADE")
print(cm)
print("test signals", {k: v for k, v in win["exp"].items()})

# Cost sensitivity of the winner's test signals (same decisions, different cost).
cost_sensitivity = {}
for c in CONFIG["cost_sensitivity_pct"]:
    rep = signal_expectancy(pred_cls, rows_test, cost=float(c))
    cost_sensitivity[f"{float(c):.4f}"] = {
        "score": expectancy_rank_score(rep),
        "LONG_expectancy": rep["LONG_expectancy"],
        "SHORT_expectancy": rep["SHORT_expectancy"],
    }
    print(f"cost {float(c):.4%}: score={cost_sensitivity[f'{float(c):.4f}']['score']:+.6f}")

member_test: Dict[str, Dict[str, Any]] = {}
for name in CANDIDATES:
    s = final_scores[name]
    member_test[name] = {**s["cls"], "exp": s["exp"], "score": s["score"], "thresholds": s["thresholds"]}
    if name != ENSEMBLE_NAME and "up" in runs["final"][name]:
        r = runs["final"][name]
        member_test[name]["upside_corr"] = corr(_unscale_reg(r["up"].predict(X_test), 0), r_test[:, 0])
        member_test[name]["downside_corr"] = corr(_unscale_reg(r["dn"].predict(X_test), 1), r_test[:, 1])
    print(
        name,
        "test",
        {
            "score": round(s["score"], 6) if np.isfinite(s["score"]) else s["score"],
            "LONG": (int(s["exp"]["LONG_n"]), round(s["exp"]["LONG_expectancy"], 6) if np.isfinite(s["exp"]["LONG_expectancy"]) else None),
            "SHORT": (int(s["exp"]["SHORT_n"]), round(s["exp"]["SHORT_expectancy"], 6) if np.isfinite(s["exp"]["SHORT_expectancy"]) else None),
            "P_LONG": round(s["cls"]["precision_LONG"], 3) if np.isfinite(s["cls"]["precision_LONG"]) else None,
            "P_SHORT": round(s["cls"]["precision_SHORT"], 3) if np.isfinite(s["cls"]["precision_SHORT"]) else None,
        },
    )

best_up = runs["final"][reg_winner_name]["up"]
best_dn = runs["final"][reg_winner_name]["dn"]
up_hat = _unscale_reg(best_up.predict(X_test), 0)
dn_hat = _unscale_reg(best_dn.predict(X_test), 1)
naive_up_mae = float(np.mean(np.abs(r_test[:, 0] - float(reg_mean[0, 0]))))
naive_dn_mae = float(np.mean(np.abs(r_test[:, 1] - float(reg_mean[0, 1]))))
reg_eval = {
    "upside_corr": corr(up_hat, r_test[:, 0]),
    "downside_corr": corr(dn_hat, r_test[:, 1]),
    "upside_mae_skill": 1.0 - float(np.mean(np.abs(r_test[:, 0] - up_hat))) / naive_up_mae,
    "downside_mae_skill": 1.0 - float(np.mean(np.abs(r_test[:, 1] - dn_hat))) / naive_dn_mae,
}
print("regression test", reg_eval)

eval_report = {
    "n_test": int(len(y_test)),
    "best_classifier": cls_winner_name,
    "best_regressor_family": reg_winner_name,
    "decision_rule": {
        "type": "probability_threshold_abstain",
        "thresholds": win["thresholds"],
        "min_signals": int(CONFIG["min_signals"]),
        "min_signal_tp_pct": MIN_SIGNAL_TP_PCT,
        "test_eligible_share": float(signal_eligible(rows_test).mean()),
    },
    "val_leaderboard": leaderboard,
    "test_members": member_test,
    "confusion": cm.tolist(),
    "precision_LONG": win["cls"]["precision_LONG"],
    "precision_SHORT": win["cls"]["precision_SHORT"],
    "n_LONG": win["cls"]["n_LONG"],
    "n_SHORT": win["cls"]["n_SHORT"],
    "balanced_accuracy": win["cls"]["balanced_accuracy"],
    "signal_expectancy": win["exp"],
    "score": win["score"],
    "cost_sensitivity": cost_sensitivity,
    "regression": reg_eval,
    "note": (
        "Classifier chosen by mean walk-forward thresholded expectancy (both sides, "
        f"min {int(CONFIG['min_signals'])} signals per side). Thresholds for the test come from "
        "validation only. Test is scored once. Balanced accuracy is diagnostic only."
    ),
}
print(json.dumps({k: v for k, v in eval_report.items() if k not in ("val_leaderboard", "test_members")}, indent=2, default=float))


TEST winner clf= LGBM thresholds= {'LONG': 0.29829407363837745, 'SHORT': 0.44344961094272717} reg= LGBM
test candles eligible for a signal (TP distance >= 0.6400%): 61.7%
test confusion rows=true cols=pred LONG/SHORT/NO_TRADE
[[  281    19  3536]
 [  239    47  3550]
 [  432    55 11208]]
test signals {'LONG_n': 952.0, 'LONG_expectancy': -0.0011642403087217044, 'LONG_tstat': -5.837730657170643, 'LONG_profit_factor': 0.6659922716795151, 'LONG_total_pct': -1.1083567739030626, 'SHORT_n': 121.0, 'SHORT_expectancy': 0.00019161665114819968, 'SHORT_tstat': 0.3475185086901917, 'SHORT_profit_factor': 1.0727383122228158, 'SHORT_total_pct': 0.023185614788932163}
cost 0.1000%: score=+0.000114
cost 0.1600%: score=-0.000486
cost 0.2000%: score=-0.000886
LR test {'score': -0.001193, 'LONG': (95, -0.000418), 'SHORT': (2387, -0.001193), 'P_LONG': 0.4, 'P_SHORT': 0.263}


RF test {'score': -0.001187, 'LONG': (686, -0.00216), 'SHORT': (184, -0.000214), 'P_LONG': 0.249, 'P_SHORT': 0.451}


XGB test {'score': 3.5e-05, 'LONG': (94, -0.001414), 'SHORT': (109, 3.5e-05), 'P_LONG': 0.277, 'P_SHORT': 0.394}


/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


LGBM test {'score': -0.000486, 'LONG': (952, -0.001164), 'SHORT': (121, 0.000192), 'P_LONG': 0.295, 'P_SHORT': 0.388}
CatBoost test {'score': -0.001061, 'LONG': (116, -0.00145), 'SHORT': (394, -0.000672), 'P_LONG': 0.31, 'P_SHORT': 0.388}
ENSEMBLE test {'score': -0.00041, 'LONG': (502, -0.001319), 'SHORT': (140, 0.000498), 'P_LONG': 0.309, 'P_SHORT': 0.486}


/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


regression test {'upside_corr': 0.1662886167530582, 'downside_corr': 0.13476691111686417, 'upside_mae_skill': -0.12374308103952258, 'downside_mae_skill': -0.07401898873027823}
{
  "n_test": 19367,
  "best_classifier": "LGBM",
  "best_regressor_family": "LGBM",
  "decision_rule": {
    "type": "probability_threshold_abstain",
    "thresholds": {
      "LONG": 0.29829407363837745,
      "SHORT": 0.44344961094272717
    },
    "min_signals": 100,
    "min_signal_tp_pct": 0.0064,
    "test_eligible_share": 0.616822429906542
  },
  "confusion": [
    [
      281,
      19,
      3536
    ],
    [
      239,
      47,
      3550
    ],
    [
      432,
      55,
      11208
    ]
  ],
  "precision_LONG": 0.29516806722689076,
  "precision_SHORT": 0.3884297520661157,
  "n_LONG": 952.0,
  "n_SHORT": 121.0,
  "balanced_accuracy": 0.3479546693023037,
  "signal_expectancy": {
    "LONG_n": 952.0,
    "LONG_expectancy": -0.0011642403087217044,
    "LONG_tstat": -5.837730657170643,
    "LONG_profit_

In [ ]:
# CELL: PROMOTE
def promotion_gate(eval_rep: Dict[str, Any], lb_row: Dict[str, Any]) -> Dict[str, Any]:
    """Decide ``ready_to_promote`` from held-out evidence only.

    Requirements: positive thresholded expectancy on the single held-out test,
    every active side has >= ``min_signals`` trades and t-stat >= ``promote_min_tstat``,
    expectancy stays positive at the highest cost in ``cost_sensitivity_pct``, and the
    walk-forward positive share is >= ``promote_min_wf_positive_share``.
    """
    reasons: List[str] = []
    min_n = int(CONFIG["min_signals"])
    min_t = float(CONFIG["promote_min_tstat"])
    exp = eval_rep["signal_expectancy"]
    active_sides = [s for s in ("LONG", "SHORT") if float(exp.get(f"{s}_n", 0.0)) > 0]
    if not active_sides:
        reasons.append("no_signals_on_test")
    if not (np.isfinite(eval_rep["score"]) and eval_rep["score"] > 0.0):
        reasons.append("test_expectancy_not_positive")
    for side in active_sides:
        if float(exp[f"{side}_n"]) < min_n:
            reasons.append(f"{side.lower()}_too_few_test_signals")
        t = float(exp.get(f"{side}_tstat", float("nan")))
        if not (np.isfinite(t) and t >= min_t):
            reasons.append(f"{side.lower()}_tstat_below_{min_t:g}")
        if not (np.isfinite(exp[f"{side}_expectancy"]) and exp[f"{side}_expectancy"] > 0.0):
            reasons.append(f"{side.lower()}_expectancy_not_positive")
    worst_cost = max(float(c) for c in CONFIG["cost_sensitivity_pct"])
    worst = eval_rep["cost_sensitivity"][f"{worst_cost:.4f}"]["score"]
    if not (np.isfinite(worst) and worst > 0.0):
        reasons.append("fragile_to_higher_cost")
    if float(lb_row.get("wf_positive_share", 0.0)) < float(CONFIG["promote_min_wf_positive_share"]):
        reasons.append("walk_forward_not_consistently_positive")
    if float(lb_row.get("wf_folds_scored", 0.0)) < 2:
        reasons.append("walk_forward_insufficient_folds")
    return {
        "ready_to_promote": len(reasons) == 0,
        "reasons": reasons,
        "test_score": eval_rep["score"],
        "test_signals": {s: exp.get(f"{s}_n") for s in ("LONG", "SHORT")},
        "test_tstats": {s: exp.get(f"{s}_tstat") for s in ("LONG", "SHORT")},
        "wf_positive_share": lb_row.get("wf_positive_share"),
        "wf_mean_score": lb_row.get("wf_mean_score"),
        "worst_cost_score": worst,
    }


promotion = promotion_gate(eval_report, leaderboard[cls_winner_name])
ready_to_promote = bool(promotion["ready_to_promote"])
print(json.dumps(promotion, indent=2, default=float))
if ready_to_promote:
    print("VERDICT: held-out evidence supports paper-trading this bundle (still research-only here).")
else:
    print("VERDICT: NOT ready. Do not trade this bundle live. Reasons:", promotion["reasons"])


{
  "ready_to_promote": false,
  "reasons": [
    "test_expectancy_not_positive",
    "long_tstat_below_2",
    "long_expectancy_not_positive",
    "short_tstat_below_2",
    "fragile_to_higher_cost",
    "walk_forward_not_consistently_positive"
  ],
  "test_score": -0.00048631182878675235,
  "test_signals": {
    "LONG": 952.0,
    "SHORT": 121.0
  },
  "test_tstats": {
    "LONG": -5.837730657170643,
    "SHORT": 0.3475185086901917
  },
  "wf_positive_share": 0.0,
  "wf_mean_score": -0.0011706111747708096,
  "worst_cost_score": -0.0008863118287867524
}
VERDICT: NOT ready. Do not trade this bundle live. Reasons: ['test_expectancy_not_positive', 'long_tstat_below_2', 'long_expectancy_not_positive', 'short_tstat_below_2', 'fragile_to_higher_cost', 'walk_forward_not_consistently_positive']


In [ ]:
# CELL: EXPORT
EXPORT_DIR.mkdir(parents=True, exist_ok=True)
final_run = runs["final"][cls_winner_name]
if cls_winner_name == ENSEMBLE_NAME:
    clf_payload: Any = {
        "members": {m: runs["final"][m]["clf"] for m in final_run["members"]},
        "member_scalers": {m: runs["final"][m]["scaler"] for m in final_run["members"]},
    }
else:
    clf_payload = {"members": {cls_winner_name: final_run["clf"]}, "member_scalers": {cls_winner_name: final_run["scaler"]}}
bundle = {
    "best_classifier": cls_winner_name,
    "best_regressor_family": reg_winner_name,
    "clf": clf_payload,
    "thresholds": eval_report["decision_rule"]["thresholds"],
    "up": best_up,
    "dn": best_dn,
    "feat_mean": feat_mean,
    "feat_std": feat_std,
    "reg_mean": reg_mean,
    "reg_std": reg_std,
    "feature_cols": list(FEATURE_COLS),
    "label_names": list(LABEL_NAMES),
}
with (EXPORT_DIR / "best_models.pkl").open("wb") as handle:
    pickle.dump(bundle, handle)
(EXPORT_DIR / "leaderboard.json").write_text(
    json.dumps(leaderboard, indent=2, default=float), encoding="utf-8"
)
metadata = {
    "model_family": f"jacksparrow_opportunity_{CONFIG['resolution']}_tabular_ensemble",
    "ensemble": list(CONFIG["ensemble"]),
    "candidates": CANDIDATES,
    "best_classifier": cls_winner_name,
    "best_regressor_family": reg_winner_name,
    "timeframe": str(CONFIG["resolution"]),
    "resolution": str(CONFIG["resolution"]),
    "horizon_bars": int(CONFIG["horizon_bars"]),
    "horizon_minutes": int(CONFIG["horizon_minutes"]),
    "tp_atr_mult": float(CONFIG["tp_atr_mult"]),
    "sl_atr_mult": float(CONFIG["sl_atr_mult"]),
    "round_trip_cost_pct": float(CONFIG["round_trip_cost_pct"]),
    "feature_cols": list(FEATURE_COLS),
    "label_names": list(LABEL_NAMES),
    "decision_rule": {
        **eval_report["decision_rule"],
        "description": (
            f"At a closed {CONFIG['resolution']} candle: if tp_atr_mult * ATR(14) / close < min_signal_tp_pct emit "
            "NO_TRADE (quiet-market guard). Otherwise scale features with the member scaler(s), "
            "average member predict_proba when best_classifier == ENSEMBLE, then emit LONG when "
            "P(LONG) >= thresholds.LONG and P(LONG) > P(SHORT); SHORT symmetrically; else NO_TRADE. "
            "A null threshold disables that side."
        ),
    },
    "ready_to_promote": ready_to_promote,
    "promote_reason": (
        "held_out_gate_passed_paper_trade_first" if ready_to_promote else ";".join(promotion["reasons"])
    ),
    "promotion": promotion,
    "eval": eval_report,
    "label_gate": label_report,
    "walk_forward": {name: leaderboard[name]["wf_folds"] for name in CANDIDATES},
    "config": CONFIG,
}
(EXPORT_DIR / "metadata.json").write_text(
    json.dumps(metadata, indent=2, default=float), encoding="utf-8"
)
(EXPORT_DIR / "eval_report.json").write_text(
    json.dumps(eval_report, indent=2, default=float), encoding="utf-8"
)
print("wrote", EXPORT_DIR / "best_models.pkl")
print("best classifier", cls_winner_name, "best regressor", reg_winner_name)
print("ready_to_promote=", ready_to_promote)
try:
    from google.colab import files
    import shutil

    shutil.make_archive(str(EXPORT_DIR), "zip", root_dir=str(EXPORT_DIR))
    files.download(str(EXPORT_DIR) + ".zip")
except ImportError:
    print("Not Colab; files remain under", EXPORT_DIR)


wrote export/opportunity_10m/best_models.pkl
best classifier LGBM best regressor LGBM
ready_to_promote= False


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

*File Operation*: `download` on `/content/export/opportunity_10m.zip`